In [ ]:
# %% [markdown]
# # CausalLOB — Step 2 Automatic Data Rebuild
#
# This notebook behaves like the original M1 notebook:
#
# - mounts Google Drive automatically;
# - searches local Colab storage and Drive for the required FI-2010 files;
# - searches inside ZIP files, including nested ZIPs;
# - if the files are not found, downloads the FI-2010 Kaggle mirror automatically;
# - reconstructs the canonical 394,337-row raw dataset;
# - infers 50 day×security boundary-separated sequences;
# - creates the six leakage-safe CausalLOB state variables;
# - creates the 20-step volatility regime variable;
# - builds the lag-valid temporal checkpoint for lags 1–5;
# - saves everything to Google Drive;
# - reuses valid checkpoints on later runs.
#
# No manual upload is needed unless KaggleHub itself fails.

# %%
# ==========================================================================================
# CELL 1 — INSTALL / IMPORT / DRIVE / CONFIG
# ==========================================================================================

import sys, subprocess, importlib.util, warnings, time, json, hashlib, gc, shutil, zipfile
from pathlib import Path

REQUIRED_PACKAGES = {
    "kagglehub": "kagglehub",
    "pyarrow": "pyarrow",
}
for module_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import numpy as np
import pandas as pd
import kagglehub

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    else:
        print("Google Drive already mounted.")
except Exception:
    pass

T0 = time.time()

# ---------------- USER SETTINGS ----------------
FORCE_REBUILD = False

ROOT = Path("/content/drive/MyDrive/CausalLOB/STEP2_REBUILD")
CHECKPOINT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
for p in [ROOT, CHECKPOINT_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CHECKPOINT_VERSION = "STEP2_REBUILD_v1_2026-09-17"

EXPECTED_ROWS = 394_337
EXPECTED_DAYS = 10
EXPECTED_SEGMENTS_PER_DAY = 5
EXPECTED_SEQUENCES = 50

DATASET_HANDLE = "ulfricirons/fi-2010"

RAW_CKPT = CHECKPOINT_DIR / "01_canonical_raw.npz"
BOUNDARY_CKPT = CHECKPOINT_DIR / "02_boundaries.npz"
FEATURE_PARQUET = CHECKPOINT_DIR / "03_features.parquet"
FEATURE_NPZ = CHECKPOINT_DIR / "03_features.npz"
TEMPORAL_NPZ = CHECKPOINT_DIR / "04_temporal_primary.npz"
QUALITY_JSON = CHECKPOINT_DIR / "05_quality_report.json"
SOURCE_MANIFEST = CHECKPOINT_DIR / "source_manifest.json"
RUN_MANIFEST = CHECKPOINT_DIR / "RUN_MANIFEST.json"

REQUIRED_NAMES = (
    ["Train_Dst_NoAuction_DecPre_CF_1.txt"]
    + [f"Test_Dst_NoAuction_DecPre_CF_{i}.txt" for i in range(1, 10)]
)
REQUIRED_SET = set(REQUIRED_NAMES)

RAW_FEATURE_NAMES = []
for level in range(1, 11):
    RAW_FEATURE_NAMES += [
        f"ask_price_l{level}",
        f"ask_volume_l{level}",
        f"bid_price_l{level}",
        f"bid_volume_l{level}",
    ]

LABEL_NAMES = [
    "label_h10",
    "label_h20",
    "label_h30",
    "label_h50",
    "label_h100",
]

PRIMARY_CAUSAL_VARS = [
    "log_relative_spread",
    "log_total_depth_l5",
    "imbalance_l1",
    "imbalance_l2_l5",
    "quote_ofi_depth_norm",
    "log_mid_return_1",
]

REGIME_VARS = ["event_volatility_20"]

print("="*100)
print("CAUSALLOB — STEP 2 AUTOMATIC DATA REBUILD")
print("="*100)
print("Checkpoint root :", CHECKPOINT_DIR)
print("Force rebuild   :", FORCE_REBUILD)
print("Dataset handle  :", DATASET_HANDLE)

# %%
# ==========================================================================================
# CELL 2 — HELPERS + AUTOMATIC DATASET DISCOVERY / DOWNLOAD
# ==========================================================================================

def sha256_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def atomic_savez_compressed(path, **kwargs):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp.npz")
    np.savez_compressed(tmp, **kwargs)
    tmp.replace(path)

def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2), encoding="utf-8")
    tmp.replace(path)

def validate_version(z):
    return (
        "checkpoint_version" in z.files
        and str(z["checkpoint_version"].item()) == CHECKPOINT_VERSION
    )

def find_required_txt_files(root):
    root = Path(root)
    found = {}
    if not root.exists():
        return found
    try:
        for path in root.rglob("*.txt"):
            if path.name in REQUIRED_SET:
                found[path.name] = path
    except Exception:
        pass
    return found

def extract_required_from_zip(zip_path, output_dir, seen=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    extracted = {}

    if seen is None:
        seen = set()

    zip_path = Path(zip_path)
    key = str(zip_path.resolve()) if zip_path.exists() else str(zip_path)
    if key in seen:
        return extracted
    seen.add(key)

    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            members = zf.namelist()
            basename_to_member = {Path(m).name: m for m in members}

            for required_name in REQUIRED_NAMES:
                if required_name in basename_to_member:
                    destination = output_dir / required_name
                    if not destination.exists():
                        with zf.open(basename_to_member[required_name]) as src, open(destination, "wb") as dst:
                            shutil.copyfileobj(src, dst)
                    extracted[required_name] = destination

            nested = [
                m for m in members
                if m.lower().endswith(".zip")
                and ("decpre" in m.lower() or "noauction" in m.lower() or "fi2010" in m.lower())
            ]

            for member in nested:
                nested_path = output_dir / Path(member).name
                if not nested_path.exists():
                    with zf.open(member) as src, open(nested_path, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                extracted.update(
                    extract_required_from_zip(nested_path, output_dir, seen=seen)
                )

    except (zipfile.BadZipFile, PermissionError, OSError):
        pass

    return extracted

def search_zip_roots(roots, extract_dir):
    found = {}
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        try:
            for zp in root.rglob("*.zip"):
                name = zp.name.lower()
                if (
                    "fi" in name
                    or "auction" in name
                    or "decpre" in name
                    or "orderbook" in name
                ):
                    found.update(extract_required_from_zip(zp, extract_dir))
                    if len(found) == 10:
                        return found
        except Exception:
            continue
    return found

def locate_or_download_sources():
    """
    1) Search already-extracted files.
    2) Search ZIPs already present in Colab / CausalLOB Drive folders.
    3) Download Kaggle FI-2010 mirror automatically.
    4) Search/extract downloaded files.
    """
    extract_dir = Path("/content/fi2010_decpre")
    extract_dir.mkdir(parents=True, exist_ok=True)

    local_roots = [
        Path("/content/fi2010_decpre"),
        Path("/content/fi2010_download"),
        Path("/content"),
        Path("/content/drive/MyDrive/CausalLOB"),
    ]

    found = {}

    # Search direct TXT files first.
    for root in local_roots:
        found.update(find_required_txt_files(root))
        if len(found) == 10:
            print("✓ Found all 10 FI-2010 Decimal-Precision files locally.")
            return found

    print(f"Found {len(found)}/10 required TXT files directly.")

    # Search ZIP archives already present.
    zip_roots = [
        Path("/content"),
        Path("/content/drive/MyDrive/CausalLOB"),
    ]
    found.update(search_zip_roots(zip_roots, extract_dir))
    found.update(find_required_txt_files(extract_dir))

    if len(found) == 10:
        print("✓ Extracted all 10 files from an existing local/Drive ZIP.")
        return found

    # Kaggle fallback.
    print(f"Still only {len(found)}/10 files available.")
    print("Downloading FI-2010 Kaggle mirror automatically...")

    dataset_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
    print("Kaggle dataset path:", dataset_path)

    found.update(find_required_txt_files(dataset_path))

    if len(found) < 10:
        found.update(search_zip_roots([dataset_path], extract_dir))

    if len(found) < 10:
        # Some Kaggle mirrors have arbitrary ZIP names; inspect all of them.
        try:
            for zp in dataset_path.rglob("*.zip"):
                found.update(extract_required_from_zip(zp, extract_dir))
                if len(found) == 10:
                    break
        except Exception:
            pass

    found.update(find_required_txt_files(extract_dir))
    found.update(find_required_txt_files(dataset_path))

    missing = REQUIRED_SET - set(found)
    if missing:
        raise FileNotFoundError(
            "Automatic acquisition failed. Missing files:\n"
            + "\n".join(sorted(missing))
            + "\n\nIf KaggleHub is unavailable, upload the FI-2010 ZIP anywhere under /content "
              "and rerun this cell; the notebook will find it automatically."
        )

    print("✓ Automatic acquisition complete.")
    return found

def load_fi2010_matrix(path):
    path = Path(path)

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline()
    first_line_count = len(first_line.split())

    flat = np.fromfile(path, dtype=np.float32, sep=" ")
    if flat.size % 149 != 0:
        raise ValueError(
            f"{path.name}: total values {flat.size:,} not divisible by 149."
        )

    if first_line_count == 149:
        matrix = flat.reshape(-1, 149).T
    else:
        matrix = flat.reshape(149, -1)

    if matrix.shape[0] != 149:
        raise RuntimeError(f"{path.name}: unexpected matrix shape {matrix.shape}")

    return matrix

print("Automatic acquisition helpers ready.")

# %%
# ==========================================================================================
# CELL 3 — CANONICAL RAW DATASET
# ==========================================================================================

def build_canonical_raw_checkpoint():
    found = locate_or_download_sources()

    day_file_map = {
        1: found["Train_Dst_NoAuction_DecPre_CF_1.txt"]
    }
    for cf in range(1, 10):
        day_file_map[cf + 1] = found[f"Test_Dst_NoAuction_DecPre_CF_{cf}.txt"]

    X_days, Y_days, days, sample_ids = [], [], [], []
    source_rows = []

    for d in range(1, 11):
        path = day_file_map[d]
        print(f"Loading day {d}: {path}")
        M = load_fi2010_matrix(path)

        Xd = M[0:40, :].T.astype(np.float32, copy=True)
        Yd = np.rint(M[144:149, :].T).astype(np.int8)

        if Xd.shape[1] != 40:
            raise RuntimeError("Expected 40 raw LOB columns.")
        if not np.isfinite(Xd).all():
            raise RuntimeError(f"Non-finite raw values on day {d}.")
        if not np.isin(Yd, [1,2,3]).all():
            raise RuntimeError(f"Unexpected labels on day {d}: {np.unique(Yd)}")

        n = len(Xd)
        X_days.append(Xd)
        Y_days.append(Yd)
        days.append(np.full(n, d, dtype=np.int8))
        sample_ids.append(np.arange(1, n+1, dtype=np.int32))

        source_rows.append({
            "day": d,
            "file": str(path),
            "rows": int(n),
            "sha256": sha256_file(path),
        })

    X_raw = np.concatenate(X_days, axis=0)
    Y = np.concatenate(Y_days, axis=0)
    day = np.concatenate(days, axis=0)
    sample_in_day = np.concatenate(sample_ids, axis=0)

    if len(X_raw) != EXPECTED_ROWS:
        raise RuntimeError(
            f"Expected {EXPECTED_ROWS:,} rows, found {len(X_raw):,}."
        )

    atomic_savez_compressed(
        RAW_CKPT,
        checkpoint_version=np.array(CHECKPOINT_VERSION),
        X_raw=X_raw,
        Y=Y,
        day=day,
        sample_in_day=sample_in_day,
        raw_feature_names=np.array(RAW_FEATURE_NAMES),
        label_names=np.array(LABEL_NAMES),
    )
    atomic_json(SOURCE_MANIFEST, {
        "checkpoint_version": CHECKPOINT_VERSION,
        "dataset_handle": DATASET_HANDLE,
        "sources": source_rows,
    })

    return X_raw, Y, day, sample_in_day

if RAW_CKPT.exists() and not FORCE_REBUILD:
    with np.load(RAW_CKPT, allow_pickle=False) as z:
        if not validate_version(z):
            raise RuntimeError(
                "Existing raw checkpoint version mismatch. "
                "Set FORCE_REBUILD=True once."
            )
        X_raw = z["X_raw"]
        Y = z["Y"]
        day = z["day"]
        sample_in_day = z["sample_in_day"]
    print("✓ Reused raw checkpoint:", RAW_CKPT)
else:
    X_raw, Y, day, sample_in_day = build_canonical_raw_checkpoint()
    print("✓ Built raw checkpoint:", RAW_CKPT)

print("Raw shape:", X_raw.shape)
print("Labels:", Y.shape)
print("Days:", np.unique(day))

# %%
# ==========================================================================================
# CELL 4 — INFER 50 BOUNDARY-SEPARATED SEQUENCES
#
# FI-2010 concatenates five anonymous instruments inside each day.
# Boundaries are inferred from discontinuities in the raw LOB representation.
# ==========================================================================================

def robust_row_distance(X):
    """
    Distance between adjacent rows after robust coordinate scaling.
    Used only to locate large discontinuities that separate anonymous stocks.
    """
    X64 = X.astype(np.float64, copy=False)
    scale = np.median(np.abs(np.diff(X64, axis=0)), axis=0)
    scale = np.where(scale < 1e-12, 1.0, scale)

    D = np.abs(np.diff(X64, axis=0)) / scale
    return np.median(D, axis=1)

def infer_five_segments_per_day(X_raw, day):
    instrument_segment = np.empty(len(X_raw), dtype=np.int8)
    sequence_id = np.empty(len(X_raw), dtype=np.int16)
    is_segment_start = np.zeros(len(X_raw), dtype=bool)

    diagnostics = []
    seq_counter = 0

    for d in range(1, EXPECTED_DAYS+1):
        idx = np.flatnonzero(day == d)
        Xd = X_raw[idx]

        dist = robust_row_distance(Xd)

        # Four largest internal discontinuities = boundaries between five anonymous stocks.
        top4 = np.argsort(dist)[-4:]
        boundaries_local = np.sort(top4 + 1)

        starts = np.r_[0, boundaries_local]
        ends = np.r_[boundaries_local, len(idx)]

        selected = np.sort(dist[top4])
        nonselected = np.delete(dist, top4)
        ratio_min = float(selected.min() / np.median(nonselected))
        ratio_median = float(np.median(selected) / np.median(nonselected))

        for seg, (s,e) in enumerate(zip(starts, ends), start=1):
            gidx = idx[s:e]
            instrument_segment[gidx] = seg
            sequence_id[gidx] = seq_counter
            is_segment_start[gidx[0]] = True
            seq_counter += 1

        diagnostics.append({
            "day": d,
            "boundaries_local": boundaries_local.tolist(),
            "segment_lengths": (ends-starts).tolist(),
            "min_separation_ratio": ratio_min,
            "median_separation_ratio": ratio_median,
        })

    if seq_counter != EXPECTED_SEQUENCES:
        raise RuntimeError(
            f"Expected {EXPECTED_SEQUENCES} sequences, found {seq_counter}."
        )

    return instrument_segment, sequence_id, is_segment_start, diagnostics

if BOUNDARY_CKPT.exists() and not FORCE_REBUILD:
    with np.load(BOUNDARY_CKPT, allow_pickle=False) as z:
        if not validate_version(z):
            raise RuntimeError("Boundary checkpoint version mismatch.")
        instrument_segment = z["instrument_segment"]
        sequence_id = z["sequence_id"]
        is_segment_start = z["is_segment_start"]
    diagnostics = json.loads((CHECKPOINT_DIR/"02_boundary_diagnostics.json").read_text())
    print("✓ Reused boundary checkpoint.")
else:
    instrument_segment, sequence_id, is_segment_start, diagnostics = \
        infer_five_segments_per_day(X_raw, day)

    atomic_savez_compressed(
        BOUNDARY_CKPT,
        checkpoint_version=np.array(CHECKPOINT_VERSION),
        instrument_segment=instrument_segment,
        sequence_id=sequence_id,
        is_segment_start=is_segment_start,
    )
    atomic_json(CHECKPOINT_DIR/"02_boundary_diagnostics.json", diagnostics)
    print("✓ Built boundary checkpoint.")

print("Sequences:", len(np.unique(sequence_id)))
print("Min separation ratio:",
      min(x["min_separation_ratio"] for x in diagnostics))
print("Median separation ratio:",
      np.median([x["median_separation_ratio"] for x in diagnostics]))

# %%
# ==========================================================================================
# CELL 5 — BUILD LEAKAGE-SAFE MICROSTRUCTURE FEATURES
# ==========================================================================================

def sequence_safe_shift(arr, seq, lag=1):
    arr = np.asarray(arr)
    out = np.full(arr.shape, np.nan, dtype=np.float64)

    if lag == 0:
        return arr.astype(np.float64, copy=True)

    for sid in np.unique(seq):
        idx = np.flatnonzero(seq == sid)
        if len(idx) <= lag:
            continue
        out[idx[lag:]] = arr[idx[:-lag]]
    return out

def sequence_safe_rolling_std(x, seq, window=20):
    out = np.full(len(x), np.nan, dtype=np.float64)
    for sid in np.unique(seq):
        idx = np.flatnonzero(seq == sid)
        s = pd.Series(x[idx])
        out[idx] = s.rolling(window=window, min_periods=window).std(ddof=0).to_numpy()
    return out

if FEATURE_NPZ.exists() and FEATURE_PARQUET.exists() and not FORCE_REBUILD:
    with np.load(FEATURE_NPZ, allow_pickle=False) as z:
        if not validate_version(z):
            raise RuntimeError("Feature checkpoint version mismatch.")
    df = pd.read_parquet(FEATURE_PARQUET)
    print("✓ Reused feature checkpoint.")
else:
    ask_price = X_raw[:,0::4].astype(np.float64)
    ask_vol   = X_raw[:,1::4].astype(np.float64)
    bid_price = X_raw[:,2::4].astype(np.float64)
    bid_vol   = X_raw[:,3::4].astype(np.float64)

    ask1 = ask_price[:,0]
    bid1 = bid_price[:,0]
    askv1 = ask_vol[:,0]
    bidv1 = bid_vol[:,0]

    mid = 0.5*(ask1+bid1)
    spread = ask1-bid1
    relative_spread = spread/mid

    total_depth_l5 = (
        ask_vol[:,:5].sum(axis=1)
        + bid_vol[:,:5].sum(axis=1)
    )

    imbalance_l1 = (
        (bidv1-askv1)
        /(bidv1+askv1)
    )

    bid_l2_l5 = bid_vol[:,1:5].sum(axis=1)
    ask_l2_l5 = ask_vol[:,1:5].sum(axis=1)
    imbalance_l2_l5 = (
        (bid_l2_l5-ask_l2_l5)
        /(bid_l2_l5+ask_l2_l5)
    )

    bid1_prev = sequence_safe_shift(bid1, sequence_id, 1)
    ask1_prev = sequence_safe_shift(ask1, sequence_id, 1)
    bidv1_prev = sequence_safe_shift(bidv1, sequence_id, 1)
    askv1_prev = sequence_safe_shift(askv1, sequence_id, 1)
    mid_prev = sequence_safe_shift(mid, sequence_id, 1)

    bid_component = (
        (bid1 >= bid1_prev).astype(np.float64)*bidv1
        - (bid1 <= bid1_prev).astype(np.float64)*bidv1_prev
    )
    ask_component = (
        (ask1 >= ask1_prev).astype(np.float64)*askv1_prev
        - (ask1 <= ask1_prev).astype(np.float64)*askv1
    )
    quote_ofi = bid_component + ask_component

    avg_l1_depth = 0.5*((bidv1+askv1)+(bidv1_prev+askv1_prev))
    quote_ofi_depth_norm = quote_ofi / avg_l1_depth

    log_mid_return_1 = np.log(mid/mid_prev)
    event_volatility_20 = sequence_safe_rolling_std(
        log_mid_return_1,
        sequence_id,
        window=20
    )

    df = pd.DataFrame({
        "day": day,
        "sample_in_day": sample_in_day,
        "instrument_segment": instrument_segment,
        "sequence_id": sequence_id,
        "is_segment_start": is_segment_start,
        "log_relative_spread": np.log(relative_spread),
        "log_total_depth_l5": np.log(total_depth_l5),
        "imbalance_l1": imbalance_l1,
        "imbalance_l2_l5": imbalance_l2_l5,
        "quote_ofi_depth_norm": quote_ofi_depth_norm,
        "log_mid_return_1": log_mid_return_1,
        "event_volatility_20": event_volatility_20,
    })

    for j,name in enumerate(LABEL_NAMES):
        df[name] = Y[:,j]

    df.to_parquet(FEATURE_PARQUET, index=False)

    atomic_savez_compressed(
        FEATURE_NPZ,
        checkpoint_version=np.array(CHECKPOINT_VERSION),
        features=df[PRIMARY_CAUSAL_VARS+REGIME_VARS].to_numpy(np.float64),
        Y=Y,
        day=day,
        instrument_segment=instrument_segment,
        sequence_id=sequence_id,
        sample_in_day=sample_in_day,
        feature_names=np.array(PRIMARY_CAUSAL_VARS+REGIME_VARS),
        label_names=np.array(LABEL_NAMES),
    )

    print("✓ Built feature checkpoint.")

print("Feature frame:", df.shape)
print(df[PRIMARY_CAUSAL_VARS+REGIME_VARS].isna().sum())

# %%
# ==========================================================================================
# CELL 6 — TEMPORAL CHECKPOINT FOR LAGS 1–5 + FINAL QA
# ==========================================================================================

TAU_MAX_PRIMARY = 5

def sequence_safe_lag_valid_mask(seq, current_valid, tau_max):
    valid = current_valid.copy()
    for lag in range(1, tau_max+1):
        same = np.zeros(len(seq), dtype=bool)
        same[lag:] = seq[lag:] == seq[:-lag]

        lag_valid = np.zeros(len(seq), dtype=bool)
        lag_valid[lag:] = current_valid[:-lag]

        valid &= same & lag_valid
    return valid

current_valid = np.isfinite(
    df[PRIMARY_CAUSAL_VARS+REGIME_VARS].to_numpy(np.float64)
).all(axis=1)

full_lag_valid = sequence_safe_lag_valid_mask(
    sequence_id,
    current_valid,
    TAU_MAX_PRIMARY,
)

if TEMPORAL_NPZ.exists() and not FORCE_REBUILD:
    with np.load(TEMPORAL_NPZ, allow_pickle=False) as z:
        if not validate_version(z):
            raise RuntimeError("Temporal checkpoint version mismatch.")
    print("✓ Reused temporal checkpoint.")
else:
    X_current = df.loc[current_valid, PRIMARY_CAUSAL_VARS].to_numpy(np.float64)
    X_state_with_regime = df.loc[
        current_valid, PRIMARY_CAUSAL_VARS+REGIME_VARS
    ].to_numpy(np.float64)

    Y_temporal = df.loc[current_valid, LABEL_NAMES].to_numpy(np.int8)
    day_temporal = df.loc[current_valid, "day"].to_numpy(np.int8)
    segment_temporal = df.loc[current_valid, "instrument_segment"].to_numpy(np.int8)
    seq_temporal = df.loc[current_valid, "sequence_id"].to_numpy(np.int16)
    sample_temporal = df.loc[current_valid, "sample_in_day"].to_numpy(np.int32)

    lag_mask_current = full_lag_valid[current_valid]

    atomic_savez_compressed(
        TEMPORAL_NPZ,
        checkpoint_version=np.array(CHECKPOINT_VERSION),
        X_current=X_current,
        X_state_with_regime=X_state_with_regime,
        Y=Y_temporal,
        day=day_temporal,
        instrument_segment=segment_temporal,
        sequence_id=seq_temporal,
        sample_in_day=sample_temporal,
        feature_names=np.array(PRIMARY_CAUSAL_VARS),
        regime_feature_names=np.array(REGIME_VARS),
        label_names=np.array(LABEL_NAMES),
        full_lag_valid=lag_mask_current,
    )
    print("✓ Built temporal checkpoint.")

# ---------------------- QA ----------------------
n_sequences = len(np.unique(sequence_id))
n_current = int(current_valid.sum())
n_full_lag = int(full_lag_valid.sum())
warmup_invalid = len(df) - n_current

qa = {
    "checkpoint_version": CHECKPOINT_VERSION,
    "raw_rows": int(len(X_raw)),
    "raw_shape": list(X_raw.shape),
    "label_shape": list(Y.shape),
    "days": int(len(np.unique(day))),
    "boundary_separated_sequences": int(n_sequences),
    "current_valid_rows": n_current,
    "warmup_invalid_rows": int(warmup_invalid),
    "fully_t_to_t_minus_5_valid_rows": n_full_lag,
    "publisher_engineered_features_used": False,
    "primary_features": PRIMARY_CAUSAL_VARS,
    "regime_features": REGIME_VARS,
    "label_names": LABEL_NAMES,
}

# Expected values from the canonical rebuild.
checks = {
    "raw_rows_394337": len(X_raw) == EXPECTED_ROWS,
    "days_10": len(np.unique(day)) == 10,
    "sequences_50": n_sequences == 50,
    "current_valid_393337": n_current == 393_337,
    "warmups_1000": warmup_invalid == 1000,
    "full_lag_valid_393087": n_full_lag == 393_087,
    "labels_valid": bool(np.isin(Y,[1,2,3]).all()),
}

qa["checks"] = checks
qa["all_checks_pass"] = bool(all(checks.values()))

atomic_json(QUALITY_JSON, qa)
atomic_json(RUN_MANIFEST, {
    "version": CHECKPOINT_VERSION,
    "root": str(ROOT),
    "checkpoints": {
        "raw": str(RAW_CKPT),
        "boundaries": str(BOUNDARY_CKPT),
        "features_parquet": str(FEATURE_PARQUET),
        "features_npz": str(FEATURE_NPZ),
        "temporal": str(TEMPORAL_NPZ),
        "quality": str(QUALITY_JSON),
    },
})

print("="*100)
print("STEP 2 FINAL QA")
print("="*100)
print(json.dumps(qa, indent=2))

if not qa["all_checks_pass"]:
    raise RuntimeError(
        "Step 2 QA failed. Do not continue to structure learning."
    )

print("\n✓ STEP 2 COMPLETE — ALL QA CHECKS PASS")
print("✓ No manual dataset upload required.")
print("✓ Checkpoints saved to:", CHECKPOINT_DIR)
print(f"Runtime this session: {time.time()-T0:.2f} sec")


Google Drive already mounted.
CAUSALLOB — STEP 2 AUTOMATIC DATA REBUILD
Checkpoint root : /content/drive/MyDrive/CausalLOB/STEP2_REBUILD/checkpoints
Force rebuild   : False
Dataset handle  : ulfricirons/fi-2010
Automatic acquisition helpers ready.
Found 0/10 required TXT files directly.
Still only 0/10 files available.


100%|██████████| 1.95G/1.95G [01:48<00:00, 19.3MB/s]

Extracting files...


Kaggle dataset path: /root/.cache/kagglehub/datasets/ulfricirons/fi-2010/versions/1
✓ Automatic acquisition complete.
Loading day 1: /root/.cache/kagglehub/datasets/ulfricirons/fi-2010/versions/1/BenchmarkDatasets/NoAuction/3.NoAuction_DecPre/NoAuction_DecPre_Training/Train_Dst_NoAuction_DecPre_CF_1.txt
Loading day 2: /root/.cache/kagglehub/datasets/ulfricirons/fi-2010/versions/1/BenchmarkDatasets/NoAuction/3.NoAuction_DecPre/NoAuction_DecPre_Testing/Test_Dst_NoAuction_DecPre_CF_1.txt
Loading day 3: /root/.cache/kagglehub/datasets/ulfricirons/fi-2010/versions/1/BenchmarkDatasets/NoAuction/3.NoAuction_DecPre/NoAuction_DecPre_Testing/Test_Dst_NoAuction_DecPre_CF_2.txt
Loading day 4: /root/.cache/kagglehub/datasets/ulfricirons/fi-2010/versions/1/BenchmarkDatasets/NoAuction/3.NoAuction_DecPre/NoAuction_DecPre_Testing/Test_Dst_NoAuction_DecPre_CF_3.txt
Loading day 5: /root/.cache/kagglehub/datasets/ulfricirons/fi-2010/versions/1/BenchmarkDatasets/NoAuction/3.NoAuction_DecPre/NoAuction_DecPr

In [ ]:
import numpy as np
import pandas as pd

boundary_audit = []
total_mismatches = 0

for d in range(1, 11):
    idx = np.flatnonzero(day == d)

    # Independent reconstruction using the original M1 method
    mid = (
        X_raw[idx, 0].astype(np.float64)
        + X_raw[idx, 2].astype(np.float64)
    ) / 2.0

    jumps = np.abs(np.diff(mid))
    ranked = np.sort(jumps)[::-1]

    ratio = ranked[3] / max(ranked[4], 1e-12)

    original_cuts = np.sort(
        np.argpartition(jumps, -4)[-4:] + 1
    )

    # Boundaries selected by the new Step 2
    new_cuts = np.flatnonzero(
        np.diff(sequence_id[idx]) != 0
    ) + 1

    matches = np.array_equal(original_cuts, new_cuts)

    total_mismatches += int(not matches)

    boundary_audit.append({
        "day": d,
        "original_cuts": original_cuts.tolist(),
        "new_cuts": new_cuts.tolist(),
        "separation_ratio": float(ratio),
        "exact_match": bool(matches)
    })

audit = pd.DataFrame(boundary_audit)

display(audit)

assert np.isfinite(audit["separation_ratio"]).all(), \
    "Non-finite boundary separation!"

assert (audit["separation_ratio"] > 10).all(), \
    "Weak security-boundary separation!"

assert total_mismatches == 0, \
    "New and original boundary methods disagree!"

assert int(is_segment_start.sum()) == 50, \
    "Expected exactly 50 segment starts."

print("BOUNDARY INDEPENDENT AUDIT: PASS")

,day,original_cuts,new_cuts,separation_ratio,exact_match
0,1,"[3454, 9772, 14694, 25413]","[3454, 9772, 14694, 25413]",50.562585,True
1,2,"[5079, 11201, 17166, 23878]","[5079, 11201, 17166, 23878]",31.888876,True
2,3,"[3903, 9314, 13341, 18527]","[3903, 9314, 13341, 18527]",64.141772,True
3,4,"[2806, 9798, 15140, 25959]","[2806, 9798, 15140, 25959]",21.095325,True
4,5,"[3030, 9758, 15704, 22082]","[3030, 9758, 15704, 22082]",80.250627,True
5,6,"[2263, 8506, 14113, 21120]","[2263, 8506, 14113, 21120]",44.285916,True
6,7,"[2801, 9861, 16601, 24455]","[2801, 9861, 16601, 24455]",52.222209,True
7,8,"[2647, 11309, 19900, 33129]","[2647, 11309, 19900, 33129]",31.111102,True
8,9,"[1873, 11144, 21180, 34060]","[1873, 11144, 21180, 34060]",63.998914,True
9,10,"[1888, 7016, 12738, 18559]","[1888, 7016, 12738, 18559]",67.307318,True


BOUNDARY INDEPENDENT AUDIT: PASS


In [ ]:
# %% [markdown]
# # CausalLOB — Step 3: Preflight and frozen structure discovery
#
# **GYP (adversarial research brief).** First attempt to falsify Step 2 using an independent feature reconstruction, exact comparison with the previous M1 checkpoint when present, non-circular boundary verification, and boundary-safe lag checks. **Fail closed** on any discrepancy. Only then run the **historically frozen** LPCMCI specification on Days 1–5. Save graph, edge marks, configuration/data fingerprints, and the preflight report. Do not access Days 6–10 market-state arrays for graph discovery or select edges from test performance. Do not invent additional edges to improve predictive metrics.
#
# Run cells in order on Colab. It finds your *existing* Step 2 checkpoints in Drive; **no raw ZIP download and no preprocessing rerun**. The LPCMCI call can be expensive. Completed discovery is reused only if its SHA-256 data and configuration keys match. Nothing modifies the older M1/M2 checkpoints.
#
# **Interpretation:** LPCMCI returns observational lagged candidate relationships under its assumptions, not intervention-identified causal effects. Circle endpoints are not proven hidden confounding. ParCorr values are conditional dependence statistics, not causal effect sizes. FI-2010 lags denote representation steps, not seconds or exchange ticks. Old exploration already viewed Days 8–10; they cannot be claimed researcher-blind external data in later evaluations.

# %%
# CELL 1 — Environment and read-only inputs
import sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path
import json, hashlib, time
import numpy as np
import pandas as pd

DESIRED_TIGRAMITE = '5.2.10.1'
try:
    installed = importlib.metadata.version('tigramite')
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != DESIRED_TIGRAMITE:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'tigramite=={DESIRED_TIGRAMITE}'])

from tigramite import data_processing as pp
from tigramite.lpcmci import LPCMCI
from tigramite.independence_tests.parcorr import ParCorr

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('Google Drive already mounted.')
except ImportError:
    pass

ROOT=Path('/content/drive/MyDrive/CausalLOB')
IN=ROOT/'STEP2_REBUILD'/'checkpoints'
OUT=ROOT/'STEP3_REBUILD_FROZEN'
CKPT=OUT/'checkpoints'
RESULTS=OUT/'results'
for p in [CKPT, RESULTS]: p.mkdir(parents=True, exist_ok=True)
OLD=ROOT/'M1_G9'/'checkpoints'
INPUT_VERSION='STEP2_REBUILD_v1_2026-09-17'
STAGE_VERSION='STEP3_DISCOVERY_v1_2026-09-17'
DISCOVERY_DAYS=(1,2,3,4,5)
VALIDATION_DAYS=(6,7)
RESERVED_DAYS=(8,9,10)
FEATURES=['log_relative_spread','log_total_depth_l5','imbalance_l1',
          'imbalance_l2_l5','quote_ofi_depth_norm','log_mid_return_1']
REGIME='event_volatility_20'
LAGS=5
FORCE_DISCOVERY=False  # only set True for intentional *identical-config* rerun

for filename in ['01_canonical_raw.npz','02_boundaries.npz','03_features.parquet',
                 '04_temporal_primary.npz','source_manifest.json','05_quality_report.json']:
    if not (IN/filename).exists():
        raise FileNotFoundError(f'Step 2 checkpoint missing: {IN/filename}')

T0=time.time()
print('Input (read-only):',IN)
print('New output root  :',OUT)
print('Discovery D1–5; internal validation D6–7; historically exposed D8–10.')

# %%
# CELL 2 — Independent preflight; stop on discrepancies; no model fitting

def sha_arrays(*arrays):
    h=hashlib.sha256()
    for arr in arrays:
        arr=np.ascontiguousarray(arr)
        h.update(str(arr.shape).encode()); h.update(str(arr.dtype).encode())
        h.update(arr.view(np.uint8))
    return h.hexdigest()

def atomic_json(path, obj):
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False), encoding='utf-8')
    tmp.replace(path)

def chk(z, name):
    if str(z['checkpoint_version'].item()) != INPUT_VERSION:
        raise RuntimeError(f'Wrong Step 2 version in {name}')

with np.load(IN/'01_canonical_raw.npz',allow_pickle=False) as z:
    chk(z,'raw')
    raw=z['X_raw']; labels=z['Y']; raw_day=z['day']; sample=z['sample_in_day']
with np.load(IN/'02_boundaries.npz',allow_pickle=False) as z:
    chk(z,'boundaries')
    seq=z['sequence_id']; segments=z['instrument_segment']; starts=z['is_segment_start']
with np.load(IN/'04_temporal_primary.npz',allow_pickle=False) as z:
    chk(z,'temporal')
    Xt=z['X_current']; Xreg=z['X_state_with_regime']; yt=z['Y']
    dt=z['day']; st=z['sequence_id']; segt=z['instrument_segment']; pos=z['sample_in_day']
    names=z['feature_names'].astype(str).tolist()
    old_lag_mask=z['full_lag_valid']
df=pd.read_parquet(IN/'03_features.parquet')
manifest=json.loads((IN/'source_manifest.json').read_text(encoding='utf-8'))
q=json.loads((IN/'05_quality_report.json').read_text(encoding='utf-8'))

assert raw.shape==(394337,40) and labels.shape==(394337,5)
assert df.shape[0]==len(raw) and names==FEATURES
assert q.get('all_checks_pass') is True
assert np.array_equal(np.unique(raw_day),np.arange(1,11))
assert set(np.unique(labels))=={1,2,3}
assert len(np.unique(seq))==50 and np.count_nonzero(starts)==50
assert np.all((raw[:,0]>raw[:,2]) & (raw[:,0]>0) & (raw[:,2]>0))
assert np.all(raw[:,1::4]>=0) and np.all(raw[:,3::4]>=0)
assert np.all(raw[:,1]>0) and np.all(raw[:,3]>0)
assert len(manifest.get('sources',[]))==10
assert len({s['day'] for s in manifest['sources']})==10
assert all(len(s['sha256'])==64 for s in manifest['sources'])

# Full exact comparison to PREVIOUS M1 when accessible; no assumption that path exists.
original_comparison={}
for name, keys in [('01_canonical_raw.npz',['X_raw','Y','day','sample_in_day']),
                   ('02_boundaries.npz',['sequence_id','instrument_segment','is_segment_start']),
                   ('04_temporal_primary.npz',['X_current','X_state_with_regime','Y','day',
                                               'sequence_id','sample_in_day','full_lag_valid'])]:
    prev=OLD/name
    if prev.exists():
        with np.load(IN/name,allow_pickle=False) as a, np.load(prev,allow_pickle=False) as b:
            for key in keys:
                if key not in b.files: raise RuntimeError(f'Old M1 missing {name}:{key}')
                exact=bool(np.array_equal(a[key],b[key],equal_nan=True)) if np.issubdtype(a[key].dtype,np.number) else bool(np.array_equal(a[key],b[key]))
                original_comparison[f'{name}:{key}']=exact
                if not exact: raise RuntimeError(f'NEW vs original M1 mismatch: {name}:{key}')
    else:
        original_comparison[name]='OLD_CHECKPOINT_NOT_AVAILABLE'

# Independently recover boundaries from raw best-level midpoint jumps.
boundary_rows=[]
for d in range(1,11):
    ix=np.flatnonzero(raw_day==d)
    assert np.array_equal(sample[ix],np.arange(1,len(ix)+1))
    midpoint=(raw[ix,0].astype(np.float64)+raw[ix,2].astype(np.float64))/2
    jumps=np.abs(np.diff(midpoint))
    ranked=np.sort(jumps)[::-1]
    ratio=float(ranked[3]/max(ranked[4],1e-12))
    independent=np.sort(np.argpartition(jumps,-4)[-4:]+1)
    found=np.flatnonzero(np.diff(seq[ix])!=0)+1
    assert np.array_equal(independent,found) and np.isfinite(ratio) and ratio>10
    boundary_rows.append({'day':d,'fourth_to_fifth_jump_ratio':ratio,
                          'exact_cut_match':True,'cuts':independent.tolist()})
assert np.all(starts == np.r_[True, seq[1:]!=seq[:-1]])
assert np.array_equal(segments,seq%5+1)
assert np.array_equal(raw_day,seq//5+1)

# Independent full-array reconstruction of *all* 7 model/regime features.
a=raw[:,0::4].astype(np.float64); av=raw[:,1::4].astype(np.float64)
b=raw[:,2::4].astype(np.float64); bv=raw[:,3::4].astype(np.float64)
mid=(a[:,0]+b[:,0])/2
prev=np.arange(len(raw))-1
prev[starts]=-1
valid_prev=prev>=0
features_ref=np.full((len(raw),7),np.nan,dtype=np.float64)
features_ref[:,0]=np.log((a[:,0]-b[:,0])/mid)
features_ref[:,1]=np.log(av[:,:5].sum(axis=1)+bv[:,:5].sum(axis=1))
features_ref[:,2]=(bv[:,0]-av[:,0])/(bv[:,0]+av[:,0])
features_ref[:,3]=(bv[:,1:5].sum(axis=1)-av[:,1:5].sum(axis=1))/(bv[:,1:5].sum(axis=1)+av[:,1:5].sum(axis=1))
i=np.flatnonzero(valid_prev); p=prev[i]
bid_comp=(b[i,0]>=b[p,0])*bv[i,0]-(b[i,0]<=b[p,0])*bv[p,0]
ask_comp=(a[i,0]>=a[p,0])*av[p,0]-(a[i,0]<=a[p,0])*av[i,0]
avg_depth=0.5*(bv[i,0]+av[i,0]+bv[p,0]+av[p,0])
features_ref[i,4]=(bid_comp+ask_comp)/avg_depth
features_ref[i,5]=np.log(mid[i]/mid[p])
for sid in np.unique(seq):
    ix=np.flatnonzero(seq==sid)
    features_ref[ix,6]=pd.Series(features_ref[ix,5]).rolling(20,min_periods=20).std(ddof=0).to_numpy()
max_feature_error={}
for k,name in enumerate(FEATURES+[REGIME]):
    observed=df[name].to_numpy(dtype=np.float64)
    expected=features_ref[:,k]
    if not np.array_equal(np.isnan(observed),np.isnan(expected)):
        raise RuntimeError(f'Feature missingness differs: {name}')
    delta=float(np.nanmax(np.abs(observed-expected)))
    max_feature_error[name]=delta
    if not np.allclose(observed, expected, rtol=1e-12,atol=1e-12,equal_nan=True):
        raise RuntimeError(f'Independent feature reconstruction mismatch: {name}; maximum error {delta}')

# Verify exact metadata, labels and end-of-sequence availability masks.
for name, expected in [('day',raw_day),('sample_in_day',sample),('sequence_id',seq),
                       ('instrument_segment',segments),('is_segment_start',starts)]:
    assert np.array_equal(df[name].to_numpy(),expected),name
for j,name in enumerate(['label_h10','label_h20','label_h30','label_h50','label_h100']):
    assert np.array_equal(df[name].to_numpy(),labels[:,j]),name
current_valid=np.isfinite(features_ref).all(axis=1)
assert current_valid.sum()==393337
assert np.allclose(Xt,features_ref[current_valid,:6],rtol=1e-12,atol=1e-12)
assert np.allclose(Xreg,features_ref[current_valid],rtol=1e-12,atol=1e-12)
assert np.array_equal(yt,labels[current_valid])
assert np.array_equal(dt,raw_day[current_valid]) and np.array_equal(st,seq[current_valid])
assert np.array_equal(segt,segments[current_valid]) and np.array_equal(pos,sample[current_valid])

independent_lag=np.zeros(len(raw),dtype=bool)
end_purge={h:0 for h in (10,20,30,50,100)}
for sid in np.unique(seq):
    ix=np.flatnonzero(seq==sid)
    assert len(ix)>120
    assert np.count_nonzero(~current_valid[ix])==20
    for p0 in range(25,len(ix)):
        independent_lag[ix[p0]]=current_valid[ix[p0-5:p0+1]].all()
    for h in end_purge:
        end_purge[h]+=min(h,len(ix))
assert independent_lag.sum()==393087
assert np.array_equal(old_lag_mask,independent_lag[current_valid])
assert end_purge=={10:500,20:1000,30:1500,50:2500,100:5000}

# Governance: only Days 1..5 become discovery data. D6–10 are not exposed to learner.
train_selector=np.isin(dt,DISCOVERY_DAYS)
assert np.count_nonzero(train_selector)>0 and np.all(dt[train_selector]<=5)
DATA_FINGERPRINT=sha_arrays(Xt[train_selector],dt[train_selector],st[train_selector])
preflight={'version':STAGE_VERSION, 'new_input_version':INPUT_VERSION,
           'boundary_audit':boundary_rows,'feature_max_abs_error':max_feature_error,
           'original_m1_exact_comparison':original_comparison,
           'full_lag_valid_rows':int(independent_lag.sum()),
           'train_days':list(DISCOVERY_DAYS),'validation_days':list(VALIDATION_DAYS),
           'historically_exposed_days':list(RESERVED_DAYS),
           'future_horizon_end_purge_counts':{str(k):v for k,v in end_purge.items()},
           'discovery_data_fingerprint':DATA_FINGERPRINT,
           'warning':'Publisher target labels not reconstructed yet; horizon-specific end purges required for downstream evaluation. D8-10 previously inspected, not external-blind.'}
atomic_json(RESULTS/'01_STEP2_INDEPENDENT_PREFLIGHT.json',preflight)
print('PREFLIGHT PASS: independent 7-feature reconstruction and all 10 boundary days.')
print('Old M1 exact checks:',sum(v is True for v in original_comparison.values()),
      'available; unverified:',sum(v=='OLD_CHECKPOINT_NOT_AVAILABLE' for v in original_comparison.values()))
print('Max absolute feature differences:',max_feature_error)
print('Lag-valid:',int(independent_lag.sum()),' | Discovery data hash:',DATA_FINGERPRINT)

# %%
# CELL 3 — Prepare ONLY D1–D5; sequence-wise scaling like the frozen historical M2
# Distinct sequences enter Tigramite as separate datasets; no pooled boundary crossings.
from collections import Counter

s0=np.flatnonzero(np.r_[True, st[1:]!=st[:-1]])
e0=np.r_[s0[1:],len(st)]
discovery_dict={}
norm_rows=[]
training_ids=[]
for j,(start,end) in enumerate(zip(s0,e0)):
    sid=int(st[start]); d=int(dt[start])
    assert np.all(st[start:end]==sid) and np.all(dt[start:end]==d)
    if d not in DISCOVERY_DAYS: continue
    arr=Xt[start:end]
    mu=arr.mean(axis=0)
    sd=arr.std(axis=0,ddof=0)
    if np.any(~np.isfinite(mu)) or np.any(sd<1e-12):
        raise RuntimeError(f'Non-finite/near-constant training feature in sequence {sid}')
    z=(arr-mu)/sd
    if not np.isfinite(z).all(): raise RuntimeError('Non-finite normalized training state')
    training_ids.append(sid)
    discovery_dict[len(discovery_dict)]=z
    for name,m,v in zip(FEATURES,mu,sd):
        norm_rows.append({'sequence_id':sid,'day':d,'feature':name,'mean':float(m),'std':float(v)})
assert len(discovery_dict)==25 and len(set(training_ids))==25
assert set(int(dt[np.flatnonzero(st==sid)[0]]) for sid in training_ids)==set(DISCOVERY_DAYS)
discovery_df=pp.DataFrame(data=discovery_dict,var_names=FEATURES,analysis_mode='multiple')
pd.DataFrame(norm_rows).to_csv(RESULTS/'02_D1_D5_discovery_scaling.csv',index=False)
print('Discovery sequences:',len(discovery_dict),' | Rows:',sum(len(a) for a in discovery_dict.values()))
print('Scaling: independently within each discovery sequence, using no D6–D10 state values.')

# %%
# CELL 4 — Frozen LPCMCI only; strict fingerprinted checkpoint, no stale M2 cache
LPCMCI_KWARGS=dict(
    tau_min=1,tau_max=5,pc_alpha=0.01,n_preliminary_iterations=1,
    max_p_global=5,max_p_non_ancestral=5,max_q_global=25,max_pds_set=15,
    prelim_only=False,no_non_ancestral_phase=False,break_once_separated=True,
    parents_of_lagged=True,fix_all_edges_before_final_orientation=True,
)
MODEL_SPEC={'algorithm':'LPCMCI','tigramite_version':DESIRED_TIGRAMITE,
            'test':'ParCorr','significance':'analytic',
            'kwargs':LPCMCI_KWARGS,'feature_names':FEATURES,
            'scale':'per-sequence z-score, ddof=0',
            'training_days':list(DISCOVERY_DAYS),'data_fingerprint':DATA_FINGERPRINT}
SPEC_HASH=hashlib.sha256(json.dumps(MODEL_SPEC,sort_keys=True).encode()).hexdigest()
PRIMARY=CKPT/'01_LPCMCI_fingerprinted.npz'
PRIMARY_META=CKPT/'01_LPCMCI_fingerprinted.json'

if PRIMARY.exists() and PRIMARY_META.exists() and not FORCE_DISCOVERY:
    previous=json.loads(PRIMARY_META.read_text(encoding='utf-8'))
    if previous.get('spec_hash')!=SPEC_HASH:
        raise RuntimeError('Existing discovery checkpoint is STALE: spec or data mismatch. '
                           'Preserve it and use a fresh output directory; do not silently load.')
    with np.load(PRIMARY,allow_pickle=False) as z:
        if str(z['spec_hash'].item())!=SPEC_HASH:
            raise RuntimeError('Checkpoint file/manifest fingerprint disagreement.')
        graph=z['graph']; p_matrix=z['p_matrix']; val_matrix=z['val_matrix']
    print('Reused EXACT-SPEC discovery checkpoint:',PRIMARY)
else:
    if PRIMARY.exists() and not FORCE_DISCOVERY and not PRIMARY_META.exists():
        raise RuntimeError('Checkpoint exists without manifest; manual review required.')
    print('Running frozen LPCMCI on D1–D5; this can take a while...')
    begin=time.time()
    learner=LPCMCI(dataframe=discovery_df,
                   cond_ind_test=ParCorr(significance='analytic'),verbosity=0)
    result=learner.run_lpcmci(**LPCMCI_KWARGS)
    graph=result['graph']; p_matrix=result['p_matrix']; val_matrix=result['val_matrix']
    runtime=time.time()-begin
    tmp=PRIMARY.with_name(PRIMARY.name+'.tmp.npz')
    np.savez_compressed(tmp,graph=graph,p_matrix=p_matrix,val_matrix=val_matrix,
                        feature_names=np.asarray(FEATURES),spec_hash=np.asarray(SPEC_HASH))
    tmp.replace(PRIMARY)
    atomic_json(PRIMARY_META,{'spec_hash':SPEC_HASH,'model_spec':MODEL_SPEC,
                              'runtime_sec':runtime,'checkpoint':str(PRIMARY)})
    print(f'Primary discovery saved in {runtime:.1f}s.')

assert graph.shape==(6,6,6),f'Unexpected graph shape {graph.shape}'
assert p_matrix.shape==graph.shape and val_matrix.shape==graph.shape
assert not np.any(graph[:,:,0]!=''),'Lag-zero edges violate frozen spec.'
assert np.isfinite(val_matrix[graph!='']).all()
print('Primary discovery fingerprint:',SPEC_HASH)

# %%
# CELL 5 — Parse all edge marks conservatively; export candidate families, no selection yet
records=[]
for lag in range(1,LAGS+1):
    for source in range(6):
        for target in range(6):
            mark=str(graph[source,target,lag])
            if not mark: continue
            orientation=('FULLY_DIRECTED_MARK' if mark=='-->'
                         else 'AMBIGUOUS_OR_OTHER_MARK')
            records.append({'source':FEATURES[source],'target':FEATURES[target],
                            'lag':lag,'edge_mark':mark,'orientation_class':orientation,
                            'self_edge':bool(source==target),
                            'parcorr_stat':float(val_matrix[source,target,lag]),
                            'algorithmic_p_value':float(p_matrix[source,target,lag])})

cols=['source','target','lag','edge_mark','orientation_class','self_edge',
      'parcorr_stat','algorithmic_p_value']
edges=pd.DataFrame(records,columns=cols)
edges.to_csv(RESULTS/'03_LPCMCI_ALL_MARKS.csv',index=False)

# Family table is descriptive only. Robustness and edge selection occur later on D6–D7.
cross=edges.loc[~edges.self_edge].copy()
family=[]
for (s,t),group in cross.groupby(['source','target'],sort=True):
    family.append({'source':s,'target':t,
                   'lags':','.join(map(str,sorted(group.lag.astype(int).unique()))),
                   'edge_marks':','.join(sorted(group.edge_mark.unique())),
                   'has_nonfully_directed_mark':bool((group.orientation_class!='FULLY_DIRECTED_MARK').any()),
                   'n_lag_slots':int(len(group))})
families=pd.DataFrame(family,columns=['source','target','lags','edge_marks',
                                    'has_nonfully_directed_mark','n_lag_slots'])
families.to_csv(RESULTS/'04_DESCRIPTIVE_CROSS_FAMILIES.csv',index=False)
mark_count={str(k):int(v) for k,v in Counter(edges.edge_mark).items()}
summary={'version':STAGE_VERSION,'status':'DISCOVERY_ONLY_NOT_VALIDATED',
         'training_days':[1,2,3,4,5],'validation_days_not_used':[6,7],
         'previously_exposed_days_not_used':[8,9,10],
         'fingerprint':SPEC_HASH,'retained_lag_slots':int(len(edges)),
         'self_slots':int(edges.self_edge.sum()),
         'cross_slots':int((~edges.self_edge).sum()),
         'candidate_cross_families':int(len(families)),
         'mark_counts':mark_count,
         'caveat':'Graph and family CSVs are candidates only. Do not call these identified causal effects or frozen M3 edges.'}
atomic_json(RESULTS/'05_DISCOVERY_ONLY_SUMMARY.json',summary)
print('STEP 3 DISCOVERY COMPLETE (not validated / not graph-frozen):')
print(json.dumps(summary,indent=2))
print('\nDiscovered candidate families:')
print(families.to_string(index=False))
print('\nNext step: D6–D7 prespecified family falsification and PCMCI+ sensitivity; do NOT access D8–D10 for graph selection.')

Google Drive already mounted.
Input (read-only): /content/drive/MyDrive/CausalLOB/STEP2_REBUILD/checkpoints
New output root  : /content/drive/MyDrive/CausalLOB/STEP3_REBUILD_FROZEN
Discovery D1–5; internal validation D6–7; historically exposed D8–10.
PREFLIGHT PASS: independent 7-feature reconstruction and all 10 boundary days.
Old M1 exact checks: 14 available; unverified: 0
Max absolute feature differences: {'log_relative_spread': 0.0, 'log_total_depth_l5': 0.0, 'imbalance_l1': 0.0, 'imbalance_l2_l5': 0.0, 'quote_ofi_depth_norm': 0.0, 'log_mid_return_1': 0.0, 'event_volatility_20': 0.0}
Lag-valid: 393087  | Discovery data hash: 674705f93839244b4f06917736a3b383c3a95d794d10f9ff836c9d16a559b0e9
Discovery sequences: 25  | Rows: 177752
Scaling: independently within each discovery sequence, using no D6–D10 state values.
Running frozen LPCMCI on D1–D5; this can take a while...
Primary discovery saved in 1590.2s.
Primary discovery fingerprint: fa32b947d8ece5ff253e8784eb0ae2cfa58e068a9951841e

In [ ]:
# %% [markdown]
# # CausalLOB — Step 4: D6–D7 validation and graph freeze
#
# **GYP / adversarial research instruction:** Reproduce the pre-existing M2 v2 validation protocol on
# the NEW, fingerprinted Step 3 discovery output. Do not rerun LPCMCI. Independently check its graph
# and D1–D5 fingerprint; never use D8–D10 state or target data to select edges. Test each directed
# source→target family against an equal-history AR(5) Ridge baseline on D6 and D7, an independently
# specified nonlinear model, an explicitly anti-causal lead placebo, and a 100-replicate
# within-sequence phase-alignment null. Apply BH correction across ALL directed cross-families.
# Run PCMCI+ sensitivity on D1–D5 ONLY. Audit all 150 cross-variable lag slots with matching rows
# and a target×lag-matched random-set analysis (descriptive, not causal significance).
# Freeze AR(5) self-history plus ONLY families that satisfy the historical ROBUST rule; a null
# result or empty graph is permitted. Checkpoint stages to prevent expensive reruns.
#
# The graph is observational, not an identified causal model. A phase null is not an exchangeable
# statistical randomization test; the lead placebo is deliberately non-deployable. The D6–D7
# evidence has been inspected in previous CausalLOB research, so this is a reproducibility audit,
# NOT a newly prospective independent validation. D8–D10 are historically exposed too.
#
# Run every cell in sequence in Google Colab (CPU). Source and discovery checkpoints are READ-ONLY.
# Output: /content/drive/MyDrive/CausalLOB/STEP4_VALIDATION_FROZEN

# %%
# CELL 1 — exact provenance, chronology isolation and immutable audit configuration
import sys, subprocess, importlib.metadata, importlib.util
from pathlib import Path
from collections import Counter
import json, hashlib, time
import numpy as np
import pandas as pd
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('Google Drive already mounted.')
except ImportError:
    pass

ROOT=Path('/content/drive/MyDrive/CausalLOB')
STEP2=ROOT/'STEP2_REBUILD'/'checkpoints'
STEP3=ROOT/'STEP3_REBUILD_FROZEN'
IN_CKPT=STEP3/'checkpoints'
IN_RESULTS=STEP3/'results'
OUT=ROOT/'STEP4_VALIDATION_FROZEN'
CKPT=OUT/'checkpoints'
RESULTS=OUT/'results'
FAMILY_CACHE=CKPT/'family_jobs'
PCMCI_CACHE=CKPT/'pcmci_jobs'
for folder in (CKPT,RESULTS,FAMILY_CACHE,PCMCI_CACHE):
    folder.mkdir(parents=True,exist_ok=True)

FEATURES=['log_relative_spread','log_total_depth_l5','imbalance_l1',
          'imbalance_l2_l5','quote_ofi_depth_norm','log_mid_return_1']
IDX={x:i for i,x in enumerate(FEATURES)}
TRAIN_DAYS=(1,2,3,4,5)
VAL_DAYS=(6,7)
EXPOSED_DAYS=(8,9,10)
MAX_LAG=5
RIDGE_ALPHA=1.0
HGB_KWARGS=dict(loss='squared_error',learning_rate=0.05,max_iter=120,
                max_leaf_nodes=15,min_samples_leaf=50,l2_regularization=1.0,
                early_stopping=False,random_state=5162026)
PHASE_REPS=100
PHASE_SEED=5162026
PCMCI_TAUS=(1,3,5)
PCMCI_VERSION='5.2.10.1'
STAGE_VERSION='STEP4_VALIDATION_v1_2026-09-18'
PERMUTATIONS=5000
T0=time.time()

# Never add a FORCE flag: a mismatched output directory is a HARD FAILURE.
def sha_arrays(*arrays):
    h=hashlib.sha256()
    for a in arrays:
        a=np.ascontiguousarray(a)
        h.update(str(a.shape).encode());h.update(str(a.dtype).encode())
        h.update(a.view(np.uint8))
    return h.hexdigest()

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

def atomic_json(path,obj):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False),encoding='utf-8')
    tmp.replace(path)

def atomic_csv(frame,path):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    frame.to_csv(tmp,index=False);tmp.replace(path)

def atomic_npz(path,**arrays):
    path=Path(path);tmp=path.with_name(path.name+'.tmp.npz')
    np.savez_compressed(tmp,**arrays);tmp.replace(path)

def require_file(path):
    if not path.is_file():
        raise FileNotFoundError('Required immutable input missing: '+str(path))
    return path

raw_npz=require_file(STEP2/'04_temporal_primary.npz')
preflight=require_file(IN_RESULTS/'01_STEP2_INDEPENDENT_PREFLIGHT.json')
discovery_meta=require_file(IN_CKPT/'01_LPCMCI_fingerprinted.json')
discovery_npz=require_file(IN_CKPT/'01_LPCMCI_fingerprinted.npz')
discovery_summary=require_file(IN_RESULTS/'05_DISCOVERY_ONLY_SUMMARY.json')
scaling_csv=require_file(IN_RESULTS/'02_D1_D5_discovery_scaling.csv')
old_pre=json.loads(preflight.read_text())
meta=json.loads(discovery_meta.read_text())
sum3=json.loads(discovery_summary.read_text())
assert old_pre['version']=='STEP3_DISCOVERY_v1_2026-09-17'
assert sum3['status']=='DISCOVERY_ONLY_NOT_VALIDATED'
assert sum3['training_days']==list(TRAIN_DAYS)
assert sum3['validation_days_not_used']==list(VAL_DAYS)
assert sum3['previously_exposed_days_not_used']==list(EXPOSED_DAYS)
assert old_pre['train_days']==list(TRAIN_DAYS)
assert old_pre['validation_days']==list(VAL_DAYS)
assert all(row['exact_cut_match'] and row['fourth_to_fifth_jump_ratio']>10
           for row in old_pre['boundary_audit'])
assert sum(v is False for v in old_pre['original_m1_exact_comparison'].values())==0
assert all(v==0.0 for v in old_pre['feature_max_abs_error'].values())
assert sum3['retained_lag_slots']==59 and sum3['mark_counts']=={'-->':54,'<->':5}

# The NPZ holds all ten days, but isolate D1–D7 immediately; labels are NEVER loaded.
with np.load(raw_npz,allow_pickle=False) as z:
    assert str(z['checkpoint_version'].item())=='STEP2_REBUILD_v1_2026-09-17'
    assert z['feature_names'].astype(str).tolist()==FEATURES
    all_day=z['day'];all_seq=z['sequence_id']
    assert np.array_equal(np.unique(all_day),np.arange(1,11))
    use=all_day<=7
    X=z['X_current'][use].copy()
    day=all_day[use].copy()
    seq=all_seq[use].copy()
    del all_day,all_seq,use
assert X.ndim==2 and X.shape[1]==6 and np.isfinite(X).all()
assert np.array_equal(np.unique(day),np.arange(1,8))
assert len(np.unique(seq))==35
assert np.all(np.diff(day)>=0)
DISCOVERY_HASH=sha_arrays(X[day<=5],day[day<=5],seq[day<=5])
assert DISCOVERY_HASH==old_pre['discovery_data_fingerprint'], 'D1–D5 state differs from Stage 3!'
assert DISCOVERY_HASH==meta['model_spec']['data_fingerprint']
assert meta['model_spec']['training_days']==list(TRAIN_DAYS)
assert meta['model_spec']['feature_names']==FEATURES
assert meta['model_spec']['scale']=='per-sequence z-score, ddof=0'
assert meta['model_spec']['tigramite_version']==PCMCI_VERSION
assert meta['model_spec']['test']=='ParCorr'
assert meta['model_spec']['kwargs']['tau_min']==1
assert meta['model_spec']['kwargs']['tau_max']==5
assert meta['model_spec']['kwargs']['pc_alpha']==0.01
assert hashlib.sha256(json.dumps(meta['model_spec'],sort_keys=True).encode()).hexdigest()==meta['spec_hash']
assert meta['spec_hash']==sum3['fingerprint']

with np.load(discovery_npz,allow_pickle=False) as z:
    assert str(z['spec_hash'].item())==meta['spec_hash']
    assert z['feature_names'].astype(str).tolist()==FEATURES
    graph=z['graph'].copy();pvals=z['p_matrix'].copy();statistics=z['val_matrix'].copy()
assert graph.shape==(6,6,6) and pvals.shape==graph.shape and statistics.shape==graph.shape
assert not np.any(graph[:,:,0]!='')
assert set(graph.flatten())<= {'','-->','<->'}
assert int(np.count_nonzero(graph!=''))==59
assert int(np.count_nonzero(graph=='-->'))==54
assert int(np.count_nonzero(graph=='<->'))==5

edges=[]
for lag in range(1,6):
    for source in range(6):
        for target in range(6):
            mark=str(graph[source,target,lag])
            if mark:
                edges.append({'source':FEATURES[source],'target':FEATURES[target],
                              'lag':lag,'edge_mark':mark,'self_edge':source==target,
                              'parcorr_stat':float(statistics[source,target,lag])})
all_edges=pd.DataFrame(edges)
assert len(all_edges)==59
assert sum((all_edges.edge_mark=='-->') & (~all_edges.self_edge))==27
# Independently verify that the saved Step 3 tabular graph matches the actual checkpoint.
step3_csv=require_file(IN_RESULTS/'03_LPCMCI_ALL_MARKS.csv')
stored=pd.read_csv(step3_csv)
key=lambda d:set(zip(d.source.astype(str),d.target.astype(str),d.lag.astype(int),d.edge_mark.astype(str)))
assert key(all_edges)==key(stored),'Step 3 CSV/NPZ disagreement'

# Historical v2 only validates FULLY DIRECTED cross families; do not silently orient ambiguous edges.
directed_cross=all_edges[(all_edges.edge_mark=='-->') & (~all_edges.self_edge)].copy()
ambiguous=all_edges[all_edges.edge_mark!='-->'].copy()
family_info=[]
for (source,target),grp in directed_cross.groupby(['source','target'],sort=True):
    reciprocal=((directed_cross.source==target)&(directed_cross.target==source)).any()
    ambiguous_pair=((ambiguous.source==source)&(ambiguous.target==target) |
                    (ambiguous.source==target)&(ambiguous.target==source)).any()
    family_info.append({'source':source,'target':target,
                        'discovery_lags':','.join(map(str,sorted(grp.lag.astype(int).unique()))),
                        'reciprocal_directed_family':bool(reciprocal),
                        'pair_has_ambiguous_endpoint':bool(ambiguous_pair)})
families=pd.DataFrame(family_info)
assert len(families)==16,f'Expected 16 directed families, got {len(families)}'

# Validate history slices and stored D1–D5 normalization. No scaling from D6–D7 is used for training.
starts=np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]])
ends=np.r_[starts[1:],len(seq)]
SLICES={}
for a,b in zip(starts,ends):
    sid=int(seq[a]);d=int(day[a]);assert b-a>20
    assert np.all(seq[a:b]==sid) and np.all(day[a:b]==d)
    SLICES[sid]=(slice(int(a),int(b)),d)
assert len(SLICES)==35
saved_scaling=pd.read_csv(scaling_csv)
assert len(saved_scaling)==150
for sid,(sl,d) in SLICES.items():
    if d>5:continue
    arr=X[sl];mu=arr.mean(axis=0);sd=arr.std(axis=0,ddof=0)
    row=saved_scaling[saved_scaling.sequence_id==sid].set_index('feature').loc[FEATURES]
    assert np.allclose(row['mean'].to_numpy(),mu,rtol=1e-12,atol=1e-12)
    assert np.allclose(row['std'].to_numpy(),sd,rtol=1e-12,atol=1e-12)

VALIDATION_HASH=sha_arrays(X[day>=6],day[day>=6],seq[day>=6])
RUN_SPEC={'stage':STAGE_VERSION,'discovery_spec_sha256':meta['spec_hash'],
          'discovery_checkpoint_sha256':sha_file(discovery_npz),
          'train_data_sha256':DISCOVERY_HASH,'validation_data_sha256':VALIDATION_HASH,
          'features':FEATURES,'days_train':list(TRAIN_DAYS),'days_validation':list(VAL_DAYS),
          'ridge_alpha':RIDGE_ALPHA,'hgb_kwargs':HGB_KWARGS,
          'phase_reps':PHASE_REPS,'phase_seed':PHASE_SEED,
          'pcmci_taus':list(PCMCI_TAUS),'pcmci_version':PCMCI_VERSION,
          'permutations':PERMUTATIONS,'sklearn_version':sklearn.__version__,
          'method':'historical M2 v2 robustness gates; all-slot matched-row diagnostic'}
RUN_HASH=hashlib.sha256(json.dumps(RUN_SPEC,sort_keys=True).encode()).hexdigest()
run_path=CKPT/'IMMUTABLE_RUN_SPEC.json'
if run_path.exists():
    prior=json.loads(run_path.read_text())
    if prior['run_hash']!=RUN_HASH:
        raise RuntimeError('Existing Step 4 outputs have a DIFFERENT data/spec/version fingerprint. '
                           'Preserve them and use a NEW output directory; NEVER mix caches.')
else:
    # If an incomplete/corrupt run left caches but no manifest, do not accept unversioned artifacts.
    if any(FAMILY_CACHE.iterdir()) or any(PCMCI_CACHE.iterdir()):
        raise RuntimeError('Output contains work but no provenance manifest. Manual inspection required.')
    atomic_json(run_path,{'run_hash':RUN_HASH,'spec':RUN_SPEC})

atomic_csv(all_edges,RESULTS/'01_DISCOVERY_GRAPH_VERIFIED.csv')
atomic_csv(families,RESULTS/'02_DIRECTED_CROSS_FAMILIES.csv')
print('STEP 4 PREFLIGHT PASS; no D8–D10 state or label used.')
print('Step 3 discovery fingerprint:',meta['spec_hash'])
print('Step 4 run fingerprint:',RUN_HASH)
print('59 candidate links / 27 directed cross slots / 16 directed families / 5 ambiguous marks.')
print('Train D1–5:',int((day<=5).sum()),'Validation D6–7:',int((day>=6).sum()))
print('Outputs:',OUT)

# %%
# CELL 2 — frozen family-testing helpers; equality-of-rows guards

def family_dataset(source,target,lags,days,mode='lag',fixed_end=None):
    """Paired AR5 and AR5+family matrices; same rows, same own-history, no crossings."""
    lags=tuple(sorted(set(map(int,lags))))
    assert lags and set(lags)<=set(range(1,6))
    assert source!=target and source in IDX and target in IDX
    assert mode in ('lag','lead')
    mx=max(lags) if fixed_end is None else int(fixed_end)
    assert mx>=max(lags) and mx<=5
    si=IDX[source];ti=IDX[target]
    bx=[];ax=[];yy=[];dd=[];ss=[]
    for sid,(sl,d) in SLICES.items():
        if d not in days:continue
        arr=X[sl]
        pos=np.arange(5,len(arr)-mx,dtype=np.int64)
        if not len(pos):continue
        baseline=np.column_stack([arr[pos-l,ti] for l in range(1,6)])
        suffix=np.column_stack([arr[pos-l if mode=='lag' else pos+l,si] for l in lags])
        bx.append(baseline);ax.append(np.column_stack([baseline,suffix]))
        yy.append(arr[pos,ti]);dd.append(np.full(len(pos),d,dtype=np.int8))
        ss.append(np.full(len(pos),sid,dtype=np.int16))
    if not bx:raise RuntimeError('No eligible rows')
    B=np.vstack(bx);A=np.vstack(ax);y=np.concatenate(yy)
    days_arr=np.concatenate(dd);seq_arr=np.concatenate(ss)
    assert A.shape[0]==B.shape[0]==len(y)==len(days_arr)==len(seq_arr)
    assert A.shape[1]==5+len(lags) and np.array_equal(A[:,:5],B)
    assert np.isfinite(A).all() and np.isfinite(B).all() and np.isfinite(y).all()
    return B,A,y,days_arr,seq_arr

def paired_ridge(train_b,train_a,train_y):
    mu=float(train_y.mean());sd=float(train_y.std(ddof=0))
    assert np.isfinite(mu) and np.isfinite(sd) and sd>1e-12
    z=(train_y-mu)/sd
    baseline=make_pipeline(StandardScaler(),Ridge(alpha=RIDGE_ALPHA))
    augmented=make_pipeline(StandardScaler(),Ridge(alpha=RIDGE_ALPHA))
    baseline.fit(train_b,z);augmented.fit(train_a,z)
    return baseline,augmented,mu,sd

def prediction_deltas(b,a,y,days,model_b,model_a,mu,sd):
    yz=(y-mu)/sd
    pb=model_b.predict(b);pa=model_a.predict(a)
    out={}
    for d in VAL_DAYS:
        ix=(days==d)
        assert ix.sum()>0
        out[d]={'n':int(ix.sum()),'delta_mse':float(np.mean((yz[ix]-pb[ix])**2)-
                                                      np.mean((yz[ix]-pa[ix])**2))}
    pooled=float(np.mean((yz-pb)**2)-np.mean((yz-pa)**2))
    assert abs(pooled-np.average([out[d]['delta_mse'] for d in VAL_DAYS],
                                 weights=[out[d]['n'] for d in VAL_DAYS]))<1e-10
    return out,pooled

def family_seed(source,target):
    raw=f'{PHASE_SEED}|{RUN_HASH}|{source}|{target}'.encode()
    return int.from_bytes(hashlib.sha256(raw).digest()[:8],'big')

def phase_null(source,target,lags,mb,ma,mu,sd,B,A,y,days,seq_ids):
    yz=(y-mu)/sd
    base_pred=mb.predict(B)
    actual_pred=ma.predict(A)
    base_mse=float(np.mean((yz-base_pred)**2))
    actual_gain=base_mse-float(np.mean((yz-actual_pred)**2))
    pos_by_seq=[np.flatnonzero(seq_ids==sid) for sid in np.unique(seq_ids)]
    rng=np.random.default_rng(family_seed(source,target))
    source_block=A[:,5:].copy()
    null_gains=np.empty(PHASE_REPS,dtype=np.float64)
    for j in range(PHASE_REPS):
        perturbed=A.copy()
        for indices in pos_by_seq:
            n=len(indices)
            low=max(10,2*max(lags)+1)
            high=n-low
            if high<=low:
                raise RuntimeError('Validation sequence too short for controlled phase shift')
            k=int(rng.integers(low,high))
            perturbed[indices,5:]=np.roll(source_block[indices],k,axis=0)
        null_pred=ma.predict(perturbed)
        null_gains[j]=base_mse-float(np.mean((yz-null_pred)**2))
    p=(1+int(np.count_nonzero(null_gains>=actual_gain)))/(PHASE_REPS+1)
    return dict(phase_empirical_p=float(p),phase_null_mean_gain=float(null_gains.mean()),
                phase_null_q95_gain=float(np.quantile(null_gains,0.95)),
                phase_actual_gain=float(actual_gain),phase_null_reps=PHASE_REPS)

def bh_qvalues(pvals):
    p=np.asarray(pvals,dtype=np.float64)
    assert p.ndim==1 and len(p)>0 and np.isfinite(p).all() and np.all((p>=0)&(p<=1))
    order=np.argsort(p)
    tmp=p[order]*len(p)/np.arange(1,len(p)+1)
    tmp=np.minimum.accumulate(tmp[::-1])[::-1]
    out=np.empty(len(p));out[order]=np.clip(tmp,0,1)
    return out

def evaluate_one_family(info):
    source=str(info.source);target=str(info.target)
    lags=tuple(map(int,str(info.discovery_lags).split(',')))
    Btr,Atr,ytr,dtr,str_=family_dataset(source,target,lags,TRAIN_DAYS)
    Bva,Ava,yva,dva,sva=family_dataset(source,target,lags,VAL_DAYS)
    assert set(np.unique(dtr))==set(TRAIN_DAYS)
    assert set(np.unique(dva))==set(VAL_DAYS)
    mb,ma,mu,sd=paired_ridge(Btr,Atr,ytr)
    ridge,delta=prediction_deltas(Bva,Ava,yva,dva,mb,ma,mu,sd)
    ztr=(ytr-mu)/sd
    hgb_b=HistGradientBoostingRegressor(**HGB_KWARGS)
    hgb_a=HistGradientBoostingRegressor(**HGB_KWARGS)
    hgb_b.fit(Btr,ztr);hgb_a.fit(Atr,ztr)
    hgb,hgb_delta=prediction_deltas(Bva,Ava,yva,dva,hgb_b,hgb_a,mu,sd)
    # Explicit anti-causal negative control. Do NOT use lead observations in any forecasting model.
    Btlead,Atlead,ytlead,dtlead,_=family_dataset(source,target,lags,TRAIN_DAYS,'lead')
    Bvlead,Avlead,yvlead,dvlead,_=family_dataset(source,target,lags,VAL_DAYS,'lead')
    assert np.array_equal(Btr,Btlead) and np.array_equal(Bva,Bvlead)
    assert np.array_equal(ytr,ytlead) and np.array_equal(yva,yvlead)
    lead=make_pipeline(StandardScaler(),Ridge(alpha=RIDGE_ALPHA))
    lead.fit(Atlead,ztr)
    lead_day,lead_delta=prediction_deltas(Bvlead,Avlead,yvlead,dvlead,mb,lead,mu,sd)
    null=phase_null(source,target,lags,mb,ma,mu,sd,Bva,Ava,yva,dva,sva)
    assert abs(null['phase_actual_gain']-delta)<1e-10
    return {'source':source,'target':target,'discovery_lags':info.discovery_lags,
            'n_train':int(len(ytr)),'n_D6':int(ridge[6]['n']),'n_D7':int(ridge[7]['n']),
            'ridge_D6_delta_mse':ridge[6]['delta_mse'],
            'ridge_D7_delta_mse':ridge[7]['delta_mse'],
            'ridge_positive_days':int(sum(ridge[d]['delta_mse']>0 for d in VAL_DAYS)),
            'ridge_weighted_delta_mse':delta,
            'hgb_D6_delta_mse':hgb[6]['delta_mse'],
            'hgb_D7_delta_mse':hgb[7]['delta_mse'],
            'hgb_weighted_delta_mse':hgb_delta,
            'hgb_positive_days':int(sum(hgb[d]['delta_mse']>0 for d in VAL_DAYS)),
            'lead_placebo_weighted_delta_mse':lead_delta,
            'lag_minus_lead_gain':delta-lead_delta,
            'reciprocal_directed_family':bool(info.reciprocal_directed_family),
            'pair_has_ambiguous_endpoint':bool(info.pair_has_ambiguous_endpoint),
            **null}

# %%
# CELL 3 — D6–D7 family falsification; durable checkpoint per family
family_rows=[]
for number,info in enumerate(families.itertuples(index=False),start=1):
    target_file=FAMILY_CACHE/f'family_{number:02d}.json'
    if target_file.exists():
        job=json.loads(target_file.read_text())
        if job.get('run_hash')!=RUN_HASH or job.get('source')!=info.source or job.get('target')!=info.target:
            raise RuntimeError('Family job checkpoint fingerprint/identity mismatch: '+str(target_file))
        row=job['result']
        print(f'Cached family {number:02d}/{len(families)}: {info.source} -> {info.target}')
    else:
        started=time.time()
        row=evaluate_one_family(info)
        atomic_json(target_file,{'run_hash':RUN_HASH,'source':info.source,
                                 'target':info.target,'result':row,
                                 'runtime_sec':time.time()-started})
        print(f'Finished family {number:02d}/{len(families)}: {info.source} -> {info.target}; '
              f'Ridge ΔMSE={row["ridge_weighted_delta_mse"]:+.6g}; '
              f'phase p={row["phase_empirical_p"]:.4f}')
    family_rows.append(row)

family_df=pd.DataFrame(family_rows)
assert len(family_df)==16 and not family_df[['ridge_weighted_delta_mse',
                                           'hgb_weighted_delta_mse','phase_empirical_p']].isna().any().any()
family_df['phase_bh_q']=bh_qvalues(family_df.phase_empirical_p.to_numpy())
atomic_csv(family_df,RESULTS/'03_ALL_FAMILY_FALSIFICATION.csv')
print('\nD6–D7 directed-family falsification complete; p/q are descriptive alignment diagnostics.')
print(family_df[['source','target','discovery_lags','ridge_D6_delta_mse',
                 'ridge_D7_delta_mse','hgb_weighted_delta_mse','lag_minus_lead_gain',
                 'phase_empirical_p','phase_bh_q']].to_string(index=False))

# %%
# CELL 4 — train-only PCMCI+ [tau=1,3,5] model-family sensitivity
try:
    installed=importlib.metadata.version('tigramite')
except importlib.metadata.PackageNotFoundError:
    installed=None
if installed!=PCMCI_VERSION:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',f'tigramite=={PCMCI_VERSION}'])
from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

normal_data={}
for sid,(sl,d) in SLICES.items():
    if d not in TRAIN_DAYS:continue
    arr=X[sl]
    norm_data=(arr-arr.mean(axis=0))/arr.std(axis=0,ddof=0)
    assert np.isfinite(norm_data).all()
    normal_data[len(normal_data)]=norm_data
assert len(normal_data)==25
train_frame=pp.DataFrame(data=normal_data,var_names=FEATURES,analysis_mode='multiple')
PCMCI_KWARGS=dict(tau_min=1,pc_alpha=0.01,contemp_collider_rule='majority',
                  conflict_resolution=True,reset_lagged_links=False,max_conds_dim=5,
                  max_combinations=1,max_conds_py=5,max_conds_px=5,
                  max_conds_px_lagged=5,fdr_method='fdr_bh')

pcmci_sets={}
pcmi_rows=[]
for tau in PCMCI_TAUS:
    spec={'step4_run_hash':RUN_HASH,'algorithm':'PCMCI+',
          'tigramite':PCMCI_VERSION,'test':'ParCorr analytic',
          'tau_max':tau,'kwargs':PCMCI_KWARGS,'days':[1,2,3,4,5]}
    hash_pcmci=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()
    path=PCMCI_CACHE/f'pcmci_tau{tau}.npz'
    manifest=PCMCI_CACHE/f'pcmci_tau{tau}.json'
    if path.exists() or manifest.exists():
        if not (path.exists() and manifest.exists()):
            raise RuntimeError(f'Incomplete PCMCI+ tau={tau} cache; manual inspection required')
        j=json.loads(manifest.read_text())
        assert j['spec_hash']==hash_pcmci
        with np.load(path,allow_pickle=False) as z:
            assert str(z['spec_hash'].item())==hash_pcmci
            pg=z['graph'].copy();pv=z['p_matrix'].copy();vv=z['val_matrix'].copy()
        print(f'Cached D1–D5 PCMCI+ tau={tau}.')
    else:
        print(f'Running D1–D5 PCMCI+ tau={tau}; no D6–D10 used...')
        start=time.time()
        learner=PCMCI(dataframe=train_frame,cond_ind_test=ParCorr(significance='analytic'),verbosity=0)
        result=learner.run_pcmciplus(tau_max=tau,**PCMCI_KWARGS)
        pg=result['graph'];pv=result['p_matrix'];vv=result['val_matrix']
        assert pg.shape==(6,6,tau+1)
        atomic_npz(path,graph=pg,p_matrix=pv,val_matrix=vv,spec_hash=np.array(hash_pcmci))
        atomic_json(manifest,{'spec_hash':hash_pcmci,'spec':spec,
                              'runtime_sec':time.time()-start})
    assert pg.shape==(6,6,tau+1) and pv.shape==pg.shape and vv.shape==pg.shape
    lagged={(FEATURES[s],FEATURES[t],int(k))
            for k in range(1,tau+1) for s in range(6) for t in range(6)
            if str(pg[s,t,k]) not in ('','<--')}
    pcmci_sets[tau]=lagged
    eligible={(str(r.source),str(r.target),int(r.lag))
              for r in directed_cross.itertuples() if int(r.lag)<=tau}
    common=eligible & lagged
    union=eligible | lagged
    pcmi_rows.append({'tau_max':tau,'pcmci_lagged_slots':len(lagged),
                      'eligible_lpcmci_directed_slots':len(eligible),
                      'common_slots':len(common),
                      'lpcmci_coverage':len(common)/len(eligible) if eligible else None,
                      'jaccard':len(common)/len(union) if union else None})
pcmi_df=pd.DataFrame(pcmi_rows)
atomic_csv(pcmi_df,RESULTS/'04_PCMCIplus_tau_sensitivity.csv')

supports=[]
for r in families.itertuples(index=False):
    lags=tuple(map(int,r.discovery_lags.split(',')))
    by_lag=[]
    for lag in lags:
        eligible_tau=[t for t in PCMCI_TAUS if t>=lag]
        if eligible_tau:
            by_lag.append(np.mean([(r.source,r.target,lag) in pcmci_sets[t]
                                   for t in eligible_tau]))
    supports.append({'source':r.source,'target':r.target,
                     'pcmci_family_mean_support':float(np.mean(by_lag)),
                     'pcmci_family_max_support':float(np.max(by_lag))})
support_df=pd.DataFrame(supports)
atomic_csv(support_df,RESULTS/'05_PCMCIplus_family_support.csv')
print('\nPCMCI+ sensitivity complete — model-family agreement is descriptive, not causal proof.')
print(pcmi_df.to_string(index=False))

# %%
# CELL 5 — all 150 cross-slots, equal-row comparisons, target×lag-matched set sensitivity
slots_file=CKPT/'150_slots_fingerprinted.csv'
slots_meta=CKPT/'150_slots_fingerprinted.json'
if slots_file.exists() or slots_meta.exists():
    if not (slots_file.exists() and slots_meta.exists()):
        raise RuntimeError('Incomplete all-slot checkpoint; inspect before continuing')
    audit=json.loads(slots_meta.read_text())
    if audit['run_hash']!=RUN_HASH or audit['sha256']!=sha_file(slots_file):
        raise RuntimeError('All-slot checkpoint hash mismatch')
    slots=pd.read_csv(slots_file)
    print('Reused fingerprinted 150-slot checkpoint.')
else:
    rows=[]
    for target in FEATURES:
        for source in FEATURES:
            if source==target:continue
            for lag in range(1,6):
                # fixed_end=5 for EVERY candidate, so lag comparisons use matched rows.
                Bt,At,yt,dt_,st_=family_dataset(source,target,[lag],TRAIN_DAYS,fixed_end=5)
                Bv,Av,yv,dv,sv=family_dataset(source,target,[lag],VAL_DAYS,fixed_end=5)
                mb,ma,mu,sd=paired_ridge(Bt,At,yt)
                by_day,gain=prediction_deltas(Bv,Av,yv,dv,mb,ma,mu,sd)
                rows.append({'source':source,'target':target,'lag':lag,
                             'n_D6':by_day[6]['n'],'n_D7':by_day[7]['n'],
                             'D6_delta_mse':by_day[6]['delta_mse'],
                             'D7_delta_mse':by_day[7]['delta_mse'],
                             'weighted_delta_mse':gain})
        print(f'All-slot evaluation finished target: {target}')
    slots=pd.DataFrame(rows)
    assert len(slots)==150
    directed_set={(r.source,r.target,int(r.lag)) for r in directed_cross.itertuples()}
    slots['discovered_directed']=[(r.source,r.target,int(r.lag)) in directed_set
                                  for r in slots.itertuples()]
    atomic_csv(slots,slots_file)
    atomic_json(slots_meta,{'run_hash':RUN_HASH,'sha256':sha_file(slots_file),
                            'rows':len(slots),'matched_rows_across_lags':True})

assert len(slots)==150 and int(slots.discovered_directed.sum())==27
assert len(set(zip(slots.source,slots.target,slots.lag)))==150
assert slots[['n_D6','n_D7']].nunique().max()==1
atomic_csv(slots,RESULTS/'06_ALL_150_CROSS_SLOTS.csv')

values=slots.weighted_delta_mse.to_numpy(np.float64)
selected=slots.discovered_directed.to_numpy(dtype=bool)
actual=float(values[selected].mean())
n=int(selected.sum())
rng=np.random.default_rng(5162026)
uniform=np.asarray([rng.choice(values,size=n,replace=False).mean()
                    for _ in range(PERMUTATIONS)])
# Control both target AND lag: sample the same count of source alternatives within each stratum.
strata=[]
for (t,lag),grp in slots.groupby(['target','lag'],sort=True):
    indices=grp.index.to_numpy(dtype=np.int64)
    count=int(selected[indices].sum())
    strata.append((indices,count))
matched=np.empty(PERMUTATIONS)
for rep in range(PERMUTATIONS):
    sample=np.concatenate([rng.choice(indices,size=count,replace=False)
                           for indices,count in strata if count])
    assert len(sample)==n
    matched[rep]=values[sample].mean()
assert np.isfinite(matched).all()
enrichment={'n_slots':int(len(slots)),'n_discovered_directed':n,
            'mean_gain_discovered':actual,
            'mean_gain_other':float(values[~selected].mean()),
            'uniform_random_set_p_descriptive':float((1+np.sum(uniform>=actual))/(PERMUTATIONS+1)),
            'target_lag_matched_set_p_descriptive':float((1+np.sum(matched>=actual))/(PERMUTATIONS+1)),
            'target_lag_matched_null_mean':float(matched.mean()),
            'target_lag_matched_null_q95':float(np.quantile(matched,.95)),
            'permutations':PERMUTATIONS,
            'caution':'These are post-discovery, D6–D7 descriptive permutation references. Slots share '
                      'autocorrelation, overlapping samples and selected conditioning. Not a causal p-value '
                      'or an independent external validation.'}
atomic_json(RESULTS/'07_SLOT_ENRICHMENT_DESCRIPTIVE.json',enrichment)
print('All-slot enrichment:',json.dumps(enrichment,indent=2))

# %%
# CELL 6 — apply historical M2 v2 tier rule; freeze sparse parent mask, no D8–D10
final=family_df.merge(support_df,on=['source','target'],how='left',validate='one_to_one')
assert len(final)==16 and final.pcmci_family_mean_support.notna().all()

def evidence_tier(r):
    robust=(r.ridge_positive_days==2 and r.ridge_weighted_delta_mse>0
            and r.hgb_weighted_delta_mse>0 and r.lag_minus_lead_gain>0
            and r.phase_bh_q<=.10)
    supported=(r.ridge_positive_days==2 and r.ridge_weighted_delta_mse>0
               and r.lag_minus_lead_gain>0
               and (r.hgb_weighted_delta_mse>0 or r.phase_bh_q<=.20))
    return ('ROBUST_PREDICTIVE_FAMILY' if robust else
            'SUPPORTED_PREDICTIVE_FAMILY' if supported else 'EXPLORATORY_FAMILY')

def orientation_flag(r):
    if r.reciprocal_directed_family:return 'CAVEATED_RECIPROCAL_FEEDBACK'
    if r.pair_has_ambiguous_endpoint:return 'CAVEATED_AMBIGUOUS_ENDPOINT'
    if r.lag_minus_lead_gain<=0:return 'CAVEATED_LEAD_PLACEBO'
    return 'CLEANER_TEMPORAL_ORIENTATION'

final['predictive_evidence_tier']=final.apply(evidence_tier,axis=1)
final['orientation_confidence']=final.apply(orientation_flag,axis=1)
primary=final[final.predictive_evidence_tier=='ROBUST_PREDICTIVE_FAMILY'].copy()
sensitivity=final[final.predictive_evidence_tier.isin([
    'ROBUST_PREDICTIVE_FAMILY','SUPPORTED_PREDICTIVE_FAMILY'])].copy()
atomic_csv(final,RESULTS/'08_FINAL_DIRECTED_FAMILY_EVIDENCE.csv')
atomic_csv(primary,RESULTS/'09_PRIMARY_ROBUST_FAMILIES.csv')
atomic_csv(sensitivity,RESULTS/'10_SENSITIVITY_FAMILIES.csv')

mask=np.zeros((6,6,6),dtype=bool)
fixed_self=[];cross=[]
for v,name in enumerate(FEATURES):
    for lag in range(1,6):
        mask[v,v,lag]=True
        fixed_self.append({'source':name,'target':name,'lag':lag,'role':'FIXED_AR5_SELF'})
for r in primary.itertuples(index=False):
    for lag in map(int,str(r.discovery_lags).split(',')):
        s=IDX[r.source];t=IDX[r.target]
        assert s!=t and str(graph[s,t,lag])=='-->'
        mask[s,t,lag]=True
        cross.append({'source':r.source,'target':r.target,'lag':lag,
                      'role':'ROBUST_DISCOVERY_CROSS',
                      'orientation_confidence':r.orientation_confidence})
assert not mask[:,:,0].any()
assert sum(mask[s,t,k] for s in range(6) for t in range(6) for k in range(1,6) if s==t)==30
assert int(mask.sum())==30+len(cross)
atomic_csv(pd.DataFrame(fixed_self),RESULTS/'11_FROZEN_SELF_AR5.csv')
atomic_csv(pd.DataFrame(cross,columns=['source','target','lag','role','orientation_confidence']),
           RESULTS/'12_FROZEN_CROSS_SLOTS.csv')
FROZEN=CKPT/'PRIMARY_FROZEN_PARENT_MASK.npz'
final_hash=sha_arrays(mask.astype(np.uint8))
if FROZEN.exists():
    with np.load(FROZEN,allow_pickle=False) as z:
        assert str(z['run_hash'].item())==RUN_HASH
        assert str(z['mask_hash'].item())==final_hash
        assert np.array_equal(z['parent_mask'],mask)
else:
    atomic_npz(FROZEN,parent_mask=mask,feature_names=np.array(FEATURES),
               run_hash=np.array(RUN_HASH),mask_hash=np.array(final_hash),
               discovery_spec_hash=np.array(meta['spec_hash']))

counts=Counter(final.predictive_evidence_tier)
summary={'stage':STAGE_VERSION,'status':'VALIDATED_AND_FROZEN_INTERNAL_ONLY',
         'run_hash':RUN_HASH,'discovery_spec_hash':meta['spec_hash'],
         'D1_D5_discovery_only':list(TRAIN_DAYS),
         'D6_D7_family_validation_only':list(VAL_DAYS),
         'D8_D10_used_for_selection':False,
         'D8_D10_historically_researcher_exposed':True,
         'directed_family_count':int(len(final)),
         'robust_families':int(counts['ROBUST_PREDICTIVE_FAMILY']),
         'supported_families':int(counts['SUPPORTED_PREDICTIVE_FAMILY']),
         'exploratory_families':int(counts['EXPLORATORY_FAMILY']),
         'self_AR5_slots':30,'cross_primary_slots':len(cross),
         'total_primary_slots':int(mask.sum()),
         'lpcmci_candidate_slots':59,'ambiguous_marks_not_oriented':5,
         'pcmci_sensitivity_only':pcmi_rows,
         'slot_enrichment_descriptive':enrichment,
         'primary_families':primary[['source','target','discovery_lags',
                                     'orientation_confidence']].to_dict(orient='records'),
         'warnings':[
             'LPCMCI edges are observational lagged candidates, NOT proven causal effects.',
             'Family tier criteria reproduce prior M2 v2 research rules; they are NOT newly preregistered.',
             'D6–D7 were inspected in prior CausalLOB work; this is reproducibility, not virgin validation.',
             'D8–D10 were also previously examined; not an external researcher-blind test.',
             'Phase-shift empirical p and slot-set permutation references are descriptive, not exact causal tests.',
             'A lead-placebo is deliberately non-deployable and cannot identify causality.',
             'No labels were accessed in structure selection; publisher labels still require separate audit.',
             'Self histories are fixed AR5, not limited to LPCMCI-discovered self slots.'
         ]}
atomic_json(RESULTS/'STEP4_FINAL_SUMMARY.json',summary)
print('\n'+'='*95)
print('STEP 4 INTERNAL VALIDATION COMPLETE — FROZEN GRAPH; NO D8–D10 SELECTION')
print('='*95)
print('Family counts:',dict(counts))
print('Primary slots: 30 self +',len(cross),'cross =',int(mask.sum()))
print('Frozen mask hash:',final_hash)
print('Primary families:\n',primary[['source','target','discovery_lags',
                                   'orientation_confidence']].to_string(index=False))
print('Output:',OUT)
print(f'Runtime: {time.time()-T0:.1f}s')
print('NEXT: independent target-label audit, then frozen parametric DBN; never retune graph on D8–D10.')

Mounted at /content/drive
STEP 4 PREFLIGHT PASS; no D8–D10 state or label used.
Step 3 discovery fingerprint: fa32b947d8ece5ff253e8784eb0ae2cfa58e068a9951841edffc0622f61e33b0
Step 4 run fingerprint: f146c37e25cbe4c084f40def8d6c8d766e3d0f1c91b9cd6c7eda218009721e33
59 candidate links / 27 directed cross slots / 16 directed families / 5 ambiguous marks.
Train D1–5: 177752 Validation D6–7: 76298
Outputs: /content/drive/MyDrive/CausalLOB/STEP4_VALIDATION_FROZEN
Finished family 01/16: imbalance_l1 -> imbalance_l2_l5; Ridge ΔMSE=+0.000237227; phase p=0.0099
Finished family 02/16: imbalance_l1 -> log_mid_return_1; Ridge ΔMSE=+0.0282089; phase p=0.0099
Finished family 03/16: imbalance_l1 -> log_relative_spread; Ridge ΔMSE=-9.15537e-05; phase p=0.8515
Finished family 04/16: imbalance_l1 -> log_total_depth_l5; Ridge ΔMSE=-1.31469e-06; phase p=0.9604
Finished family 05/16: imbalance_l1 -> quote_ofi_depth_norm; Ridge ΔMSE=+0.00469371; phase p=0.0099
Finished family 06/16: imbalance_l2_l5 -> imbalan

In [ ]:
# CausalLOB Step 5 — independent publisher-label and horizon audit.
# GYP: Try to falsify chronology and label interpretation. The graph and labels are immutable.
# Do not make D8–D10-driven modeling decisions or tune label thresholds from match rates.
# FI-2010 source: Ntakaris et al., arXiv:1705.03233, Sec. 3 (labels).
# Source says next k=1,2,3,5,10 representation events, a future-midprice mean and
# threshold 0.002 called a 'percentage change'. We audit BOTH documented-number
# interpretations, not fit a threshold or silently assign a unique label recipe.

import os, json, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('Google Drive already mounted.')
except ImportError:
    pass

T0 = time.time()
ROOT = Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
S2 = ROOT/'STEP2_REBUILD'/'checkpoints'
S4 = ROOT/'STEP4_VALIDATION_FROZEN'
OUT = ROOT/'STEP5_LABEL_AUDIT_READONLY'
RESULTS = OUT/'results'
RESULTS.mkdir(parents=True,exist_ok=True)
VERSION = 'STEP5_LABEL_AUDIT_v1_2026-09-18'
FEATURE_NAMES = ['log_relative_spread','log_total_depth_l5','imbalance_l1',
                 'imbalance_l2_l5','quote_ofi_depth_norm','log_mid_return_1']
HORIZONS = [(10,1),(20,2),(30,3),(50,5),(100,10)]
DAYS_AUDIT = np.arange(1,8)


def require(p):
    if not p.is_file():
        raise FileNotFoundError('Missing immutable prerequisite: '+str(p))
    return p


def digest_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):
            h.update(block)
    return h.hexdigest()


def hash_arrays(*arrays):
    h=hashlib.sha256()
    for arr in arrays:
        a=np.ascontiguousarray(arr)
        h.update(str(a.shape).encode())
        h.update(str(a.dtype).encode())
        h.update(a.view(np.uint8))
    return h.hexdigest()


def write_json_once(path,data):
    text=json.dumps(data,indent=2,sort_keys=True,allow_nan=False)+'\n'
    if path.exists():
        if path.read_text(encoding='utf8')!=text:
            raise RuntimeError('Refusing to overwrite changed audit output: '+str(path))
        return
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(text,encoding='utf8')
    tmp.replace(path)


def write_csv_once(path,df):
    text=df.to_csv(index=False,float_format='%.12g')
    if path.exists():
        if path.read_text(encoding='utf8')!=text:
            raise RuntimeError('Refusing to overwrite changed audit output: '+str(path))
        return
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(text,encoding='utf8')
    tmp.replace(path)

raw_path=require(S2/'01_canonical_raw.npz')
feat_path=require(S2/'03_features.npz')
bnd_path=require(S2/'02_boundaries.npz')
temp_path=require(S2/'04_temporal_primary.npz')
mask_path=require(S4/'checkpoints'/'PRIMARY_FROZEN_PARENT_MASK.npz')
s4_path=require(S4/'results'/'STEP4_FINAL_SUMMARY.json')

# These fingerprints are computed before label inspection and bind this audit to immutable inputs.
input_hashes={p.name:digest_file(p) for p in (raw_path,feat_path,bnd_path,temp_path,mask_path,s4_path)}
run_spec={'version':VERSION,'raw_sha256':input_hashes[raw_path.name],
          'features_sha256':input_hashes[feat_path.name],
          'boundaries_sha256':input_hashes[bnd_path.name],
          'temporal_sha256':input_hashes[temp_path.name],
          'frozen_mask_sha256':input_hashes[mask_path.name],
          'step4_summary_sha256':input_hashes[s4_path.name],
          'audited_days':DAYS_AUDIT.tolist(),
          'horizons_representation_steps':[1,2,3,5,10],
          'threshold_interpretations':['ratio_0.002','percent_0.002_equals_ratio_0.00002'],
          'selection':'none; descriptive only; D8-D10 not evaluated'}
run_hash=hashlib.sha256(json.dumps(run_spec,sort_keys=True).encode()).hexdigest()
previous=RESULTS/'00_AUDIT_SPEC.json'
if previous.is_file():
    old=json.loads(previous.read_text())
    if old.get('fingerprint')!=run_hash:
        raise RuntimeError('Audit already exists for DIFFERENT inputs; refusing overwrite. '+str(previous))
write_json_once(previous,{'fingerprint':run_hash,'spec':run_spec})

s4=json.loads(s4_path.read_text())
assert s4['status']=='VALIDATED_AND_FROZEN_INTERNAL_ONLY'
assert s4['D8_D10_used_for_selection'] is False
assert s4['total_primary_slots']==39
with np.load(mask_path,allow_pickle=False) as z:
    mask=z['parent_mask'].copy()
    assert mask.shape==(6,6,6) and mask.dtype==bool and int(mask.sum())==39
    assert z['feature_names'].astype(str).tolist()==FEATURE_NAMES
    mask_hash=str(z['mask_hash'].item())
assert hash_arrays(mask.astype(np.uint8))==mask_hash, 'Frozen mask content/hash mismatch.'
assert mask_hash=='eb38b17c1431c0a98d46ed1c792953c932c58572799b64aee79165dfc58439da', 'Unexpected frozen parent mask.'

# The compressed NPZ contains all dates; arrays are immediately restricted to D1-D7.
# Loading its container requires decompression; D8-D10 are NOT scored or used for decisions.
with np.load(raw_path,allow_pickle=False) as z:
    assert str(z['checkpoint_version'].item())=='STEP2_REBUILD_v1_2026-09-17'
    all_day=z['day']; audit_mask=all_day<=7
    assert len(all_day)==394337 and np.array_equal(np.unique(all_day),np.arange(1,11))
    X=z['X_raw'][audit_mask].copy()
    Y=z['Y'][audit_mask].copy()
    day=all_day[audit_mask].copy()
    sample=z['sample_in_day'][audit_mask].copy()
    del all_day
with np.load(bnd_path,allow_pickle=False) as z:
    assert str(z['checkpoint_version'].item())=='STEP2_REBUILD_v1_2026-09-17'
    seq=z['sequence_id'][audit_mask].copy()
    segment=z['instrument_segment'][audit_mask].copy()
    starts=z['is_segment_start'][audit_mask].copy()
with np.load(feat_path,allow_pickle=False) as z:
    assert str(z['checkpoint_version'].item())=='STEP2_REBUILD_v1_2026-09-17'
    names=z['feature_names'].astype(str).tolist()
    assert names[:6]==FEATURE_NAMES and 'event_volatility_20' in names
    ft=z['features'][audit_mask].copy()
    assert np.array_equal(z['Y'][audit_mask],Y)
    assert np.array_equal(z['day'][audit_mask],day)
    assert np.array_equal(z['sequence_id'][audit_mask],seq)
current=np.isfinite(ft).all(axis=1)
with np.load(temp_path,allow_pickle=False) as z:
    assert str(z['checkpoint_version'].item())=='STEP2_REBUILD_v1_2026-09-17'
    assert z['feature_names'].astype(str).tolist()==FEATURE_NAMES
    td=z['day']; use=td<=7
    tx=z['X_current'][use].copy()
    ty=z['Y'][use].copy()
    tday=td[use].copy()
    tseq=z['sequence_id'][use].copy()
    tlag=z['full_lag_valid'][use].copy()

assert X.shape==(len(day),40) and Y.shape==(len(day),5)
assert np.array_equal(np.unique(day),np.arange(1,8))
assert len(np.unique(seq))==35 and starts.sum()==35
assert np.array_equal(np.flatnonzero(starts),np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]]))
assert np.isin(Y,[1,2,3]).all() and np.isfinite(X).all()
assert np.array_equal(ty,Y[current])
assert np.array_equal(tseq,seq[current]) and np.array_equal(tday,day[current])
assert np.array_equal(tx,ft[current,:6])
assert int((~current).sum())==35*20
assert len(tx)==len(Y[current])

lagvalid=current.copy()
for lag in range(1,6):
    compatible=np.zeros(len(seq),dtype=bool)
    compatible[lag:]=(seq[lag:]==seq[:-lag]) & current[:-lag]
    lagvalid &= compatible
assert np.array_equal(tlag,lagvalid[current]), 'Lag-valid mask mismatch.'
assert not np.any(lagvalid[starts]), 'Boundary-crossing lag row.'

# Prior M1 exact parity is mandatory if old checkpoint available; otherwise disclose missing.
old_raw=ROOT/'M1_G9'/'checkpoints'/'01_canonical_raw.npz'
old_equality={'available':old_raw.exists(),'raw_exact':None,'labels_exact':None,
              'day_exact':None,'sequence_exact':None}
if old_raw.exists():
    with np.load(old_raw,allow_pickle=False) as z:
        old_equality['raw_exact']=bool(np.array_equal(z['X_raw'][:len(X)],X))
        old_equality['labels_exact']=bool(np.array_equal(z['Y'][:len(Y)],Y))
        old_equality['day_exact']=bool(np.array_equal(z['day'][:len(day)],day))
    old_boundary=ROOT/'M1_G9'/'checkpoints'/'02_boundaries.npz'
    if old_boundary.exists():
        with np.load(old_boundary,allow_pickle=False) as z:
            old_equality['sequence_exact']=bool(np.array_equal(z['sequence_id'][:len(seq)],seq))
    assert old_equality['raw_exact'] and old_equality['labels_exact'] and old_equality['day_exact']
    if old_equality['sequence_exact'] is not None:
        assert old_equality['sequence_exact']

# Independent chronology and label-horizon bookkeeping: all 35 audited sequences are isolated.
seq_ranges=[]
for sid in np.unique(seq):
    ids=np.flatnonzero(seq==sid)
    assert np.all(np.diff(ids)==1)
    assert np.unique(day[ids]).size==1 and np.unique(segment[ids]).size==1
    seq_ranges.append((int(sid),ids))

# Original paper labels: Up=1, Stationary=2, Down=3.
# h10/h20/... is named in original raw-message horizon units; the paper's label
# horizon over the provided 10-message representations is k=1/2/3/5/10.
# Earlier project benchmarks used a more conservative h=10/20/... representation-row purge.
label_counts=[]
purge=[]
for col,(h,k) in enumerate(HORIZONS):
    for cohort,cohort_days in [('D1_D5',range(1,6)),('D6_D7',range(6,8))]:
        sel=np.isin(day,np.array(list(cohort_days)))
        vals,cnt=np.unique(Y[sel,col],return_counts=True)
        countmap=dict(zip(vals.tolist(),cnt.tolist()))
        label_counts.append({'cohort':cohort,'horizon_raw_messages':h,
                             'horizon_representation_steps':k,'n':int(sel.sum()),
                             'up_label_1':countmap.get(1,0),'neutral_label_2':countmap.get(2,0),
                             'down_label_3':countmap.get(3,0),
                             'majority_share':max(countmap.values())/int(sel.sum())})
    for policy,tail in [('publisher_minimum_k',k),('frozen_conservative_h',h)]:
        # For clean target audit, exclude the LAST tail rows of EACH anonymous sequence.
        eligible=np.zeros(len(day),dtype=bool)
        for sid,ids in seq_ranges:
            assert len(ids)>tail+20
            eligible[ids[:-tail]]=True
        assert eligible.sum()==len(day)-35*tail
        purge.append({'horizon_raw_messages':h,
                      'horizon_representation_steps':k,
                      'policy':policy,'tail_rows_per_sequence':tail,
                      'removed_D1_D7':int((~eligible).sum()),
                      'eligible_raw_D1_D7':int(eligible.sum()),
                      'eligible_current_state_D1_D7':int((eligible&current).sum()),
                      'eligible_lag_valid_D1_D7':int((eligible&lagvalid).sum())})

# Literal paper formula tested at two FIXED unit conventions. No threshold/offset search.
# Approximation caveat: these are only last raw-LOB snapshots available per 10-event block;
# original underlying ten message-by-message midprices are NOT present in DecPre rows 1-40.
mid=(X[:,0].astype(np.float64)+X[:,2].astype(np.float64))/2
assert np.all(mid>0)
thresholds=[('ratio_literal_0.002',.002),('percentage_units_0.002pct',.002/100)]
formula_rows=[]

for col,(h,k) in enumerate(HORIZONS):
    for cohort,days_here in [('D1_D5',range(1,6)),('D6_D7',range(6,8))]:
        for name,cutoff in thresholds:
            total=0;correct=0;prevalence=np.zeros(3,dtype=np.int64)
            for sid,ids in seq_ranges:
                if int(day[ids[0]]) not in days_here:continue
                n=len(ids)
                anchor=mid[ids[0]]
                dev=mid[ids]-anchor
                cum=np.r_[0.0,np.cumsum(dev,dtype=np.float64)]
                i=np.arange(n-k,dtype=np.int64)
                fut=anchor+(cum[i+k+1]-cum[i+1])/k
                delta=(fut-mid[ids[i]])/mid[ids[i]]
                yp=np.where(delta>=cutoff,1,np.where(delta<=-cutoff,3,2))
                yt=Y[ids[i],col]
                total+=len(i)
                correct+=int((yp==yt).sum())
                prevalence+=np.bincount(yp,minlength=4)[1:4]
            assert total>0
            reference=next(r for r in label_counts
                           if r['cohort']==cohort and r['horizon_raw_messages']==h)
            formula_rows.append({'cohort':cohort,'horizon_raw_messages':h,
                                 'horizon_representation_steps':k,
                                 'formula':'future_represented_mid_mean_vs_current_mid',
                                 'threshold_convention':name,
                                 'threshold_as_ratio':cutoff,
                                 'evaluated_pairs':int(total),
                                 'agreement':correct/total,
                                 'majority_class_reference':reference['majority_share'],
                                 'agreement_minus_majority_reference':correct/total-reference['majority_share'],
                                 'predicted_up_share':prevalence[0]/total,
                                 'predicted_neutral_share':prevalence[1]/total,
                                 'predicted_down_share':prevalence[2]/total})

# Descriptive association with current versus PREVIOUS return: lagged historical signal may
# genuinely be weaker; association by itself does NOT establish target leakage.
return_now=ft[:,5]
sign_now=np.where(return_now>1e-12,1,np.where(return_now< -1e-12,3,2)).astype(np.int8)
sign_prev=np.zeros(len(day),dtype=np.int8)
for sid,ids in seq_ranges:
    sign_prev[ids[1:]]=sign_now[ids[:-1]]

def balanced_acc(a,b):
    return float(np.mean([(b[a==j]==j).mean() for j in [1,2,3] if np.any(a==j)]))

sign_rows=[]
for col,(h,k) in enumerate(HORIZONS):
    for cohort,days_here in [('D1_D5',range(1,6)),('D6_D7',range(6,8))]:
        sub=np.isin(day,list(days_here))&current&lagvalid
        for descriptor,pred in [('current_return_sign',sign_now),('prior_return_sign',sign_prev)]:
            assert pred[sub].min()>=1
            sign_rows.append({'cohort':cohort,'horizon_raw_messages':h,
                              'observation':descriptor,'n':int(sub.sum()),
                              'balanced_accuracy_descriptive':balanced_acc(Y[sub,col],pred[sub]),
                              'ordinary_accuracy_descriptive':float((Y[sub,col]==pred[sub]).mean())})

label_df=pd.DataFrame(label_counts)
purge_df=pd.DataFrame(purge)
formula_df=pd.DataFrame(formula_rows)
sign_df=pd.DataFrame(sign_rows)
write_csv_once(RESULTS/'01_label_distributions_D1_D7.csv',label_df)
write_csv_once(RESULTS/'02_horizon_tail_purge_D1_D7.csv',purge_df)
write_csv_once(RESULTS/'03_fixed_formula_hypotheses_D1_D7.csv',formula_df)
write_csv_once(RESULTS/'04_return_sign_diagnostic_D1_D7.csv',sign_df)

# Do not convert the best observed reconstruction into a new label recipe.
# Without uncompressed message-level mids and publisher generation code, exact semantic
# reproduction is NOT proven, even if a formula achieves high approximate agreement.
summary={'version':VERSION,'status':'AUDIT_COMPLETED_WITH_LABEL_PROVENANCE_LIMITATION',
         'fingerprint':run_hash,'input_hashes':input_hashes,
         'days_audited':[1,2,3,4,5,6,7],
         'days_8_to_10_scored':False,
         'frozen_graph_unchanged':True,'frozen_mask_hash':mask_hash,
         'n_audited_raw_rows':int(len(X)),
         'n_audited_sequences':len(seq_ranges),
         'n_audited_current_valid':int(current.sum()),
         'n_audited_five_lag_valid':int(lagvalid.sum()),
         'publisher_label_map':{'1':'up','2':'stationary','3':'down'},
         'horizon_representation_steps':{str(h):k for h,k in HORIZONS},
         'frozen_conservative_tail_purge_representation_rows_per_sequence':{str(h):h for h,k in HORIZONS},
         'old_M1_comparison':old_equality,
         'matched_publisher_label_array_to_temporal':True,
         'all_sequence_and_lag_checks_passed':True,
         'best_approximate_reconstruction_agreement_is_not_ground_truth':True,
         'limitations':[
             'FI-2010 labels are supplied by publisher; original message-level within-block midprices and generation code are not in this checkpoint.',
             'Paper gives a threshold 0.002 for a percentage change; literal ratio and percentage-unit interpretations are both descriptive hypotheses, never selected for relabeling.',
             'A mismatch with reconstructed labels does not establish leakage; exact source-label recreation cannot be proven from ten-event representation snapshots alone.',
             'Days 6-7 and 8-10 were historically inspected in earlier work. D8-D10 not scored in this audit.',
             'Reported prediction horizons h=10..100 are underlying raw-message/event labels; equivalent k=1,2,3,5,10 representation steps. The previous h-row purge remains intentionally conservative.',
             'Current state spans t-1 to t for OFI/return; only subsequent representations can enter official future labels. Any current-return association is not by itself leakage.',
             'This stage does not create new labels or modify the graph, train models, choose thresholds, or declare trading PnL.'
         ]}
write_json_once(RESULTS/'05_LABEL_AUDIT_SUMMARY.json',summary)

print('\n'+'='*100)
print('STEP 5 LABEL/HORIZON AUDIT COMPLETE — frozen graph and source labels UNCHANGED')
print('='*100)
print('Audited raw rows:',len(X),'| sequences:',len(seq_ranges),'| current valid:',int(current.sum()),
      '| complete-lag:',int(lagvalid.sum()))
print('Original M1 exact parity:',old_equality)
print('\nPublished-horizon interpretation and conservative purge:')
print(purge_df[purge_df.policy=='frozen_conservative_h'].to_string(index=False))
print('\nFixed, NON-TUNED formula reconstruction hypotheses (a low match is diagnostic, not leakage proof):')
print(formula_df[['cohort','horizon_raw_messages','threshold_convention','evaluated_pairs',
                  'agreement','majority_class_reference']].to_string(index=False))
print('\nReturn-sign association diagnostic (not a predictive or leakage test):')
print(sign_df[sign_df.horizon_raw_messages.isin([10,20])].to_string(index=False))
print('\nSummary:',RESULTS/'05_LABEL_AUDIT_SUMMARY.json')
print('NEXT: frozen parametric DBN using original publisher labels. No relabeling; no graph reopening.')
print(f'Runtime: {time.time()-T0:.1f}s')

Google Drive already mounted.

STEP 5 LABEL/HORIZON AUDIT COMPLETE — frozen graph and source labels UNCHANGED
Audited raw rows: 254750 | sequences: 35 | current valid: 254050 | complete-lag: 253875
Original M1 exact parity: {'available': True, 'raw_exact': True, 'labels_exact': True, 'day_exact': True, 'sequence_exact': True}

Published-horizon interpretation and conservative purge:
 horizon_raw_messages  horizon_representation_steps                policy  tail_rows_per_sequence  removed_D1_D7  eligible_raw_D1_D7  eligible_current_state_D1_D7  eligible_lag_valid_D1_D7
                   10                             1 frozen_conservative_h                      10            350              254400                        253700                    253525
                   20                             2 frozen_conservative_h                      20            700              254050                        253350                    253175
                   30                          

In [ ]:
# %% [markdown]
# CausalLOB — Step 6: frozen parametric dynamic Bayesian network
#
# GYP / adversarial self-brief:
# - Treat Step 4's 39-slot graph and Step 5's publisher labels as IMMUTABLE.
# - Fit transitions and five ordinal outcome heads only on D1–D7.
# - Apply horizon-specific conservative tail purges to BOTH training and evaluation.
# - Evaluate the frozen procedure on D8–D10 once, as a historically exposed retrospective assessment.
# - Report negative results and calibration; neither accuracy nor a paper trading signal establishes PnL.
# - Preserve all prior folders; this step writes only to STEP6_FROZEN_DBN.
#
# Run cells in order on Google Colab (CPU); reruns load fingerprinted checkpoints.

# %%
# CELL 1 — setup, exact source specification and durable output directory
import os, sys, json, time, hashlib, importlib.util, subprocess
from pathlib import Path

for module, package in [('scipy','scipy'),('sklearn','scikit-learn')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])

import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import minimize
from sklearn.linear_model import Ridge
from sklearn.metrics import balanced_accuracy_score, f1_score, log_loss

try:
    from IPython.display import display
except ImportError:
    display = print
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('Google Drive already mounted.')
except ImportError:
    pass

T0=time.time()
ROOT=Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
S2=ROOT/'STEP2_REBUILD'/'checkpoints'
S4=ROOT/'STEP4_VALIDATION_FROZEN'
S5=ROOT/'STEP5_LABEL_AUDIT_READONLY'/'results'
OUT=ROOT/'STEP6_FROZEN_DBN'
CKPT=OUT/'checkpoints'
RESULTS=OUT/'results'
HEAD_DIR=CKPT/'outcome_heads'
for directory in (CKPT,RESULTS,HEAD_DIR):
    directory.mkdir(parents=True,exist_ok=True)

VERSION='STEP6_FROZEN_DBN_v1_2026-09-18'
FEATURES=['log_relative_spread','log_total_depth_l5','imbalance_l1',
          'imbalance_l2_l5','quote_ofi_depth_norm','log_mid_return_1']
HORIZONS=[10,20,30,50,100]
REPS=[1,2,3,5,10]
TRAIN_DAYS=[1,2,3,4,5,6,7]
EVAL_DAYS=[8,9,10]
L=5; N=6; ALPHA=1.0; ECE_BINS=15
MASK_HASH_EXPECTED='eb38b17c1431c0a98d46ed1c792953c932c58572799b64aee79165dfc58439da'

SOURCE={
 'temporal':S2/'04_temporal_primary.npz',
 'raw':S2/'01_canonical_raw.npz',
 'mask':S4/'checkpoints'/'PRIMARY_FROZEN_PARENT_MASK.npz',
 'step4_summary':S4/'results'/'STEP4_FINAL_SUMMARY.json',
 'step5_summary':S5/'05_LABEL_AUDIT_SUMMARY.json',
}
for name,path in SOURCE.items():
    if not path.is_file():
        raise FileNotFoundError(f'Missing READ-ONLY Step 6 prerequisite [{name}]: {path}')

def hash_file(path):
    digest=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()

def hash_arrays(*arrays):
    digest=hashlib.sha256()
    for arr in arrays:
        x=np.ascontiguousarray(arr)
        digest.update(str(x.shape).encode())
        digest.update(str(x.dtype).encode())
        digest.update(x.view(np.uint8))
    return digest.hexdigest()

def atomic_json(path,data):
    text=json.dumps(data,sort_keys=True,indent=2,allow_nan=False)+'\n'
    path=Path(path); tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(text,encoding='utf8');tmp.replace(path)

def atomic_npz(path,**arrays):
    path=Path(path);tmp=path.with_name(path.name+'.tmp.npz')
    np.savez_compressed(tmp,**arrays);tmp.replace(path)

def safe_csv(path,frame):
    path=Path(path); tmp=path.with_name(path.name+'.tmp')
    frame.to_csv(tmp,index=False,float_format='%.12g');tmp.replace(path)

print('CausalLOB Step 6 — read-only source root:',ROOT)
print('Outputs (new directory only):',OUT)

# %%
# CELL 2 — rigorous provenance, complete sequence isolation and horizon eligibility
s4=json.loads(SOURCE['step4_summary'].read_text())
s5=json.loads(SOURCE['step5_summary'].read_text())
assert s4['status']=='VALIDATED_AND_FROZEN_INTERNAL_ONLY'
assert s4['D8_D10_used_for_selection'] is False
assert s4['total_primary_slots']==39 and s4.get('cross_primary_slots',9)==9
assert s5['status']=='AUDIT_COMPLETED_WITH_LABEL_PROVENANCE_LIMITATION'
assert s5['frozen_graph_unchanged'] and s5['days_8_to_10_scored'] is False
assert s5['matched_publisher_label_array_to_temporal']
assert s5['frozen_mask_hash']==MASK_HASH_EXPECTED
assert s5['frozen_conservative_tail_purge_representation_rows_per_sequence']=={str(h):h for h in HORIZONS}
assert all(s5['old_M1_comparison'][k] for k in ['raw_exact','labels_exact','day_exact','sequence_exact'])

hashes={k:hash_file(p) for k,p in SOURCE.items()}
assert hashes['mask']==s5['input_hashes']['PRIMARY_FROZEN_PARENT_MASK.npz']
assert hashes['raw']==s5['input_hashes']['01_canonical_raw.npz']
assert hashes['temporal']==s5['input_hashes']['04_temporal_primary.npz']
assert hashes['step4_summary']==s5['input_hashes']['STEP4_FINAL_SUMMARY.json']
with np.load(SOURCE['mask'],allow_pickle=False) as z:
    mask=z['parent_mask'].copy()
    assert z['feature_names'].astype(str).tolist()==FEATURES
    assert str(z['mask_hash'].item())==MASK_HASH_EXPECTED
assert mask.shape==(6,6,6) and mask.dtype==bool and int(mask.sum())==39
assert hash_arrays(mask.astype(np.uint8))==MASK_HASH_EXPECTED
assert not mask[:,:,0].any()
assert all(mask[i,i,1:].all() for i in range(6))
assert sum(mask[s,t,l] for s in range(6) for t in range(6)
           for l in range(1,6) if s!=t)==9

with np.load(SOURCE['temporal'],allow_pickle=False) as z:
    assert str(z['checkpoint_version'].item())=='STEP2_REBUILD_v1_2026-09-17'
    assert z['feature_names'].astype(str).tolist()==FEATURES
    X=z['X_current'].astype(np.float64)
    Y=z['Y'].astype(np.int8)
    day=z['day'].astype(np.int8)
    seq=z['sequence_id'].astype(np.int16)
    lagvalid=z['full_lag_valid'].copy()
assert X.shape==(393337,6) and Y.shape==(393337,5)
assert len(day)==len(seq)==len(lagvalid)==len(X)
assert np.isfinite(X).all() and np.isin(Y,[1,2,3]).all()
assert np.array_equal(np.unique(day),np.arange(1,11))
assert len(np.unique(seq))==50 and int(lagvalid.sum())==393087
assert np.all(np.diff(day)>=0)

# Independent label array cross-check against the raw checkpoint, accounting for
# exactly 20 discarded warm-up representations per sequence.
with np.load(SOURCE['raw'],allow_pickle=False) as z:
    raw_y=z['Y'].copy()
    raw_day=z['day'].copy()
assert raw_y.shape==(394337,5) and np.array_equal(np.unique(raw_day),np.arange(1,11))
raw_seq_start=np.flatnonzero(np.r_[True,raw_day[1:]!=raw_day[:-1]])
# Raw identity is established by Step 5's exact raw/sequence audit; reconstruct
# its current-valid alignment from the original Step 2 first-20-rows rule.
with np.load(S2/'02_boundaries.npz',allow_pickle=False) as z:
    raw_seq=z['sequence_id'].copy()
raw_starts=np.flatnonzero(np.r_[True,raw_seq[1:]!=raw_seq[:-1]])
assert len(raw_starts)==50
raw_ends=np.r_[raw_starts[1:],len(raw_seq)]
raw_current=np.concatenate([np.arange(a+20,b) for a,b in zip(raw_starts,raw_ends)])
assert len(raw_current)==len(X)
assert np.array_equal(raw_y[raw_current],Y)
assert np.array_equal(raw_day[raw_current],day)
assert np.array_equal(raw_seq[raw_current],seq)
del raw_y,raw_day,raw_seq

starts=np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]])
ends=np.r_[starts[1:],len(seq)]
assert len(starts)==50
remaining=np.empty(len(seq),dtype=np.int32)
train_rows=[];eval_rows=[]
for a,b in zip(starts,ends):
    assert b-a>110 and np.all(seq[a:b]==seq[a]) and np.all(day[a:b]==day[a])
    expected_lags=np.r_[np.zeros(L,dtype=bool),np.ones(b-a-L,dtype=bool)]
    assert np.array_equal(lagvalid[a:b],expected_lags), 'Inconsistent Step2 lag validity.'
    remaining[a:b]=b-1-np.arange(a,b)
    index=np.arange(a+L,b,dtype=np.int64)
    if int(day[a]) in TRAIN_DAYS: train_rows.append(index)
    elif int(day[a]) in EVAL_DAYS: eval_rows.append(index)
train_idx=np.concatenate(train_rows);eval_idx=np.concatenate(eval_rows)
assert len(train_idx)==253875 and len(eval_idx)==139212
assert np.array_equal(np.unique(day[train_idx]),TRAIN_DAYS)
assert np.array_equal(np.unique(day[eval_idx]),EVAL_DAYS)
assert set(seq[train_idx]).isdisjoint(set(seq[eval_idx]))
assert lagvalid[train_idx].all() and lagvalid[eval_idx].all()

run_spec={'version':VERSION,'source_sha256':hashes,'frozen_mask_hash':MASK_HASH_EXPECTED,
          'train_days':TRAIN_DAYS,'assessment_days_historically_exposed':EVAL_DAYS,
          'ridge_alpha':ALPHA,'max_lag':L,'horizons':HORIZONS,
          'conservative_tail_purge_representation_rows':HORIZONS,
          'head':'six-current-state ordered probit, probit link, no intercept, analytic-gradient L-BFGS-B maxiter=160',
          'innovation_covariance':'diagonal; correlated residuals descriptive only',
          'metric_policy':'same transition rows, label tail purge per sequence',
          'selection':'none after Step4, no hyperparameter optimization on D8-D10'}
RUN_HASH=hashlib.sha256(json.dumps(run_spec,sort_keys=True).encode()).hexdigest()
manifest_path=CKPT/'IMMUTABLE_STEP6_SPEC.json'
if manifest_path.exists():
    old=json.loads(manifest_path.read_text())
    if old.get('run_hash')!=RUN_HASH:
        raise RuntimeError('Existing Step6 checkpoint has DIFFERENT inputs/specification. STOP.')
else:
    atomic_json(manifest_path,{'run_hash':RUN_HASH,'spec':run_spec})
print('PREFLIGHT PASS: Step 2/4/5 inputs exact, publisher labels unmodified, mask fixed.')
print('Train transition rows:',len(train_idx),' | retrospective assessment rows:',len(eval_idx))
print('Graph: 30 self + 9 cross =',int(mask.sum()),' | run hash:',RUN_HASH)

# %%
# CELL 3 — train-only standardization, vectorized Ridge CPDs, state-space freeze
normalization_mask=np.isin(day,TRAIN_DAYS)
mu=X[normalization_mask].mean(axis=0)
sigma=X[normalization_mask].std(axis=0,ddof=0)
assert np.isfinite(mu).all() and (sigma>1e-12).all()
XZ=(X-mu)/sigma
Ztr=np.concatenate([XZ[train_idx-lag] for lag in range(1,L+1)],axis=1)
Zev=np.concatenate([XZ[eval_idx-lag] for lag in range(1,L+1)],axis=1)
Ytr=XZ[train_idx]; Yev=XZ[eval_idx]
assert Ztr.shape==(253875,30) and Zev.shape==(139212,30)
assert np.isfinite(Ztr).all() and np.isfinite(Zev).all()

def fit_transition(parent_mask):
    B=np.zeros((6,30),dtype=float)
    intercept=np.zeros(6,dtype=float)
    variance=np.zeros(6,dtype=float)
    for target in range(6):
        parents=[(lag-1)*6+source for lag in range(1,6)
                 for source in range(6) if parent_mask[source,target,lag]]
        assert parents and len(parents)==len(set(parents))
        reg=Ridge(alpha=ALPHA,fit_intercept=True)
        reg.fit(Ztr[:,parents],Ytr[:,target])
        B[target,parents]=reg.coef_
        intercept[target]=reg.intercept_
        error=Ytr[:,target]-(Ztr[:,parents]@reg.coef_+reg.intercept_)
        variance[target]=np.mean(error*error)
    assert (variance>1e-12).all()
    assert np.all(B[~np.array([[parent_mask[j%6,t,j//6+1]
                   for j in range(30)] for t in range(6)])]==0)
    return B,intercept,variance

def state_space(B,c,v):
    F=np.zeros((30,30),dtype=float)
    F[:6,:]=B
    for lag in range(1,5):
        F[lag*6:(lag+1)*6,(lag-1)*6:lag*6]=np.eye(6)
    offset=np.zeros(30,dtype=float);offset[:6]=c
    Q=np.zeros((30,30),dtype=float);Q[:6,:6]=np.diag(v)
    return F,offset,Q

model_path=CKPT/'01_FROZEN_STATE_SPACE.npz'
ar_mask=np.zeros_like(mask)
for variable in range(6): ar_mask[variable,variable,1:]=True
if model_path.exists():
    with np.load(model_path,allow_pickle=False) as z:
        assert str(z['run_hash'].item())==RUN_HASH,'Transition checkpoint fingerprint mismatch.'
        B_ar=z['B_ar'];c_ar=z['c_ar'];v_ar=z['v_ar']
        B_sp=z['B_sparse'];c_sp=z['c_sparse'];v_sp=z['v_sparse']
        F_ar=z['F_ar'];off_ar=z['offset_ar'];Q_ar=z['Q_ar']
        F_sp=z['F_sparse'];off_sp=z['offset_sparse'];Q_sp=z['Q_sparse']
        assert np.array_equal(z['train_mean'],mu)
        assert np.array_equal(z['train_std'],sigma)
        assert np.array_equal(z['frozen_parent_mask'],mask)
    print('Reused fingerprinted Step 6 transition checkpoint.')
else:
    B_ar,c_ar,v_ar=fit_transition(ar_mask)
    B_sp,c_sp,v_sp=fit_transition(mask)
    F_ar,off_ar,Q_ar=state_space(B_ar,c_ar,v_ar)
    F_sp,off_sp,Q_sp=state_space(B_sp,c_sp,v_sp)
    atomic_npz(model_path,run_hash=np.array(RUN_HASH),
               frozen_parent_mask=mask,feature_names=np.array(FEATURES),
               train_mean=mu,train_std=sigma,
               B_ar=B_ar,c_ar=c_ar,v_ar=v_ar,
               B_sparse=B_sp,c_sparse=c_sp,v_sparse=v_sp,
               F_ar=F_ar,offset_ar=off_ar,Q_ar=Q_ar,
               F_sparse=F_sp,offset_sparse=off_sp,Q_sparse=Q_sp)
    print('Saved frozen AR(5) and sparse 39-slot Gaussian transitions.')

assert F_sp.shape==(30,30) and np.array_equal(np.diag(Q_sp)[:6],v_sp)
assert np.allclose(F_sp[6:,:-6],np.eye(24)) # exact deterministic 4-block shift
train_error=Ytr-(Ztr@B_sp.T+c_sp)
resid_cov=np.cov(train_error,rowvar=False,ddof=0)
assert np.allclose(np.diag(resid_cov),v_sp,rtol=1e-8)
parameter_rows=[]
for model_name,B in [('AR5',B_ar),('SPARSE_39',B_sp)]:
    for t in range(6):
        for lag in range(1,6):
            for s in range(6):
                if (ar_mask if model_name=='AR5' else mask)[s,t,lag]:
                    parameter_rows.append({'model':model_name,'source':FEATURES[s],
                        'target':FEATURES[t],'lag':lag,
                        'coefficient_standardized':float(B[t,(lag-1)*6+s])})
safe_csv(RESULTS/'01_FROZEN_PARAMETER_TABLE.csv',pd.DataFrame(parameter_rows))
safe_csv(RESULTS/'01_TRAIN_RESIDUAL_CORRELATION.csv',
         pd.DataFrame(np.corrcoef(train_error,rowvar=False),columns=FEATURES,index=FEATURES)
           .rename_axis('target').reset_index())
print('Transition parameters frozen; residual innovation correlations are DESCRIPTIVE, not used as Q.')

# %%
# CELL 4 — same-row, no-tuning D8–D10 transition assessment

def transition_scores(true,pred,v):
    e=true-pred
    mse=np.mean(e*e,axis=0)
    nll=0.5*(np.log(2*np.pi*v)+mse/v)
    cover=np.mean(np.abs(e)<=norm.ppf(0.95)*np.sqrt(v),axis=0)
    return mse,nll,cover

p_ar=Zev@B_ar.T+c_ar
p_sp=Zev@B_sp.T+c_sp
rows=[]
for name,pred,variance in [('AR5',p_ar,v_ar),('SPARSE_39',p_sp,v_sp)]:
    for label,where in [('POOLED',np.ones(len(eval_idx),dtype=bool))]+[
        (str(d),day[eval_idx]==d) for d in EVAL_DAYS]:
        m,nll,coverage=transition_scores(Yev[where],pred[where],variance)
        for t in range(6):
            rows.append({'model':name,'day':label,'target':FEATURES[t],
                         'n':int(where.sum()),'rmse':float(np.sqrt(m[t])),
                         'nll':float(nll[t]),'coverage90':float(coverage[t]),
                         'abs_coverage_error90':float(abs(coverage[t]-.90))})
transitions=pd.DataFrame(rows)
safe_csv(RESULTS/'02_TRANSITION_D8_D10.csv',transitions)
pooled=transitions[transitions.day=='POOLED'].pivot(index='target',columns='model',values='rmse')
pooled['rmse_gain_AR5_minus_sparse']=pooled['AR5']-pooled['SPARSE_39']
print('\nFrozen transition assessment (standardized state; positive gain means sparse improves):')
display(pooled.round(6))

# %%
# CELL 5 — fit immutable publisher-label ordered probit heads, one per horizon
# Horizon h is in original messages; 1/2/3/5/10 representation steps.
# The conservative h-REPRESENTATION-row tail purge is frozen by Step 5, not tuned.

def eligible_indices(candidate_idx,h):
    idx=candidate_idx[remaining[candidate_idx]>=h]
    assert np.isin(day[idx], TRAIN_DAYS+EVAL_DAYS).all()
    assert lagvalid[idx].all()
    return idx

def encode_publisher(y):
    assert np.isin(y,[1,2,3]).all()
    return (3-y).astype(np.int8)  # publisher 3=Down ->0; 2=Neutral ->1; 1=Up ->2

def ordered_probs(Xstate,beta,cut):
    eta=Xstate@np.asarray(beta,dtype=float)
    c1,c2=map(float,cut)
    down=norm.cdf(c1-eta)
    upto_neutral=norm.cdf(c2-eta)
    P=np.column_stack([down,upto_neutral-down,1-upto_neutral])
    assert np.isfinite(P).all() and np.min(P)>=-1e-12
    P=np.maximum(P,1e-15)
    P/=P.sum(axis=1,keepdims=True)
    return P

def ordinal_nll_gradient(theta,Z,y):
    """Exact 3-class probit log-likelihood and analytic gradient, per observation.

    Parameterization: c1 = theta[6], c2 = c1 + exp(theta[7]).
    The intercept is identified by thresholds; no separate intercept is fitted.
    """
    beta=np.asarray(theta[:6],dtype=float)
    c1=float(theta[6]);gap=float(np.exp(theta[7]));c2=c1+gap
    eta=Z@beta
    z1=c1-eta; z2=c2-eta
    cdf1=norm.cdf(z1); cdf2=norm.cdf(z2)
    sf1=norm.sf(z1); sf2=norm.sf(z2)
    p0=cdf1
    p1=np.where(z1>0,sf1-sf2,cdf2-cdf1)
    p2=sf2
    p=np.where(y==0,p0,np.where(y==1,p1,p2))
    # Clipping is solely numerical, applied equally to likelihood and gradient.
    p=np.maximum(p,1e-14)
    phi1=norm.pdf(z1);phi2=norm.pdf(z2)
    d_eta=np.where(y==0,-phi1,np.where(y==1,phi1-phi2,phi2))
    d_c1=np.where(y==0,phi1,np.where(y==1,-phi1,0.0))
    d_c2=np.where(y==2,-phi2,np.where(y==1,phi2,0.0))
    n=len(y)
    loss=float(-np.mean(np.log(p)))
    score_eta=d_eta/p
    grad_beta=-(Z.T@score_eta)/n
    grad_c1=-np.mean((d_c1+d_c2)/p)
    grad_gap=-gap*np.mean(d_c2/p)
    gradient=np.r_[grad_beta,grad_c1,grad_gap]
    if not np.isfinite(loss) or not np.isfinite(gradient).all():
        raise FloatingPointError('Nonfinite ordinal-probit loss/gradient; STOP.')
    return loss,gradient

def fit_head(h,training_indices):
    column=HORIZONS.index(h)
    y=encode_publisher(Y[training_indices,column])
    Z=XZ[training_indices]
    freq=np.bincount(y,minlength=3).astype(float)/len(y)
    assert (freq>0).all(),'Missing class in training set.'
    c1=float(norm.ppf(freq[0]))
    c2=float(norm.ppf(freq[0]+freq[1]))
    start=np.r_[np.zeros(6),c1,np.log(c2-c1)]
    fit=minimize(ordinal_nll_gradient,start,args=(Z,y),method='L-BFGS-B',
                 jac=True,options={'maxiter':160,'ftol':1e-9,'gtol':1e-6})
    if not fit.success:
        raise RuntimeError(f'Ordered probit h{h} did NOT converge: {fit.message}. '
                           'STOP; never tune using D8–D10.')
    beta=fit.x[:6]; thresholds=[float(fit.x[6]),float(fit.x[6]+np.exp(fit.x[7]))]
    if not thresholds[0]<thresholds[1]:
        raise RuntimeError(f'Invalid ordered thresholds at h{h}.')
    return {'run_hash':RUN_HASH,'horizon_raw_messages':h,
            'horizon_representation_steps':REPS[column],
            'n_train':int(len(training_indices)),
            'train_index_sha256':hash_arrays(training_indices),
            'feature_names':FEATURES,'beta':beta.tolist(),
            'thresholds':thresholds,'converged':bool(fit.success),
            'llf':float(-len(y)*fit.fun),
            'optimizer':'analytic-gradient L-BFGS-B','iterations':int(fit.nit)}

heads={}
for h in HORIZONS:
    tr=eligible_indices(train_idx,h)
    hfile=HEAD_DIR/f'ordered_probit_h{h}.json'
    if hfile.exists():
        head=json.loads(hfile.read_text())
        assert head['run_hash']==RUN_HASH
        assert head['train_index_sha256']==hash_arrays(tr)
        assert head['n_train']==len(tr) and head['feature_names']==FEATURES
        assert head['converged'] and len(head['beta'])==6
        print(f'Reused immutable ordered-probit h{h} checkpoint.')
    else:
        print(f'Fitting frozen ordered-probit h{h}; eligible train rows={len(tr):,} ...')
        hstart=time.time()
        head=fit_head(h,tr)
        atomic_json(hfile,head)
        print(f'Saved h{h} in {time.time()-hstart:.1f}s.')
    heads[h]=head

# %%
# CELL 6 — frozen retrospective price-state assessment and inspectable probabilities

def ece(y,P,bins=ECE_BINS):
    conf=P.max(axis=1); pred=P.argmax(axis=1); hit=(pred==y)
    limits=np.linspace(0,1,bins+1)
    acc=0.0
    for i in range(bins):
        m=(conf>=limits[i]) & (conf < limits[i+1] if i<bins-1 else conf<=limits[i+1])
        if m.any():acc+=m.mean()*abs(hit[m].mean()-conf[m].mean())
    return float(acc)

def classification_metrics(y,P):
    pred=np.argmax(P,axis=1)
    brier=np.mean(np.sum((P-np.eye(3)[y])**2,axis=1))
    return {'n':int(len(y)),'accuracy':float(np.mean(pred==y)),
            'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
            'macro_f1':float(f1_score(y,pred,labels=[0,1,2],average='macro',zero_division=0)),
            'log_loss':float(log_loss(y,P,labels=[0,1,2])),
            'brier_multiclass':float(brier),'ece_confidence':ece(y,P)}

classification=[]
for h in HORIZONS:
    tr=eligible_indices(train_idx,h)
    te=eligible_indices(eval_idx,h)
    column=HORIZONS.index(h)
    assert set(seq[tr]).isdisjoint(set(seq[te]))
    yt=encode_publisher(Y[tr,column]);ye=encode_publisher(Y[te,column])
    prior=np.bincount(yt,minlength=3).astype(float)
    prior/=prior.sum()
    P=ordered_probs(XZ[te],heads[h]['beta'],heads[h]['thresholds'])
    Pprior=np.broadcast_to(prior,P.shape)
    outfile=RESULTS/f'03_PREDICTED_CLASS_PROBABILITIES_h{h}.npz'
    # Full array stores labels and row identity. It is assessment-only; never use for tuning.
    if outfile.exists():
        with np.load(outfile,allow_pickle=False) as z:
            assert str(z['run_hash'].item())==RUN_HASH
            assert np.array_equal(z['row_index'],te)
            assert np.array_equal(z['p_down_neutral_up'],P), 'Prediction drift on rerun.'
    else:
        atomic_npz(outfile,run_hash=np.array(RUN_HASH),
                   row_index=te,day=day[te],sequence_id=seq[te],
                   original_label=Y[te,column],label_ordered=ye,
                   p_down_neutral_up=P,prior_down_neutral_up=prior,
                   class_names=np.array(['Down','Neutral','Up']),
                   horizon_raw_messages=np.array(h),
                   horizon_representation_steps=np.array(REPS[column]))
    for label,m in [('POOLED',np.ones(len(te),dtype=bool))]+[(str(d),day[te]==d) for d in EVAL_DAYS]:
        for name,pred in [('PRIOR_ONLY',Pprior),('ORDERED_PROBIT_SPARSE_DBN_HEAD',P)]:
            classification.append({'horizon_raw_messages':h,
                'horizon_representation_steps':REPS[column],
                'day':label,'model':name,**classification_metrics(ye[m],pred[m])})
classification=pd.DataFrame(classification)
safe_csv(RESULTS/'04_OUTCOME_METRICS_D8_D10.csv',classification)
print('\nD8–D10 retrospective assessment; Day 8–10 historically inspected in older research:')
display(classification[classification.day=='POOLED'][[
    'horizon_raw_messages','model','n','accuracy','balanced_accuracy',
    'macro_f1','log_loss','brier_multiclass','ece_confidence']].round(6))
print('\nExample h10 output:')
with np.load(RESULTS/'03_PREDICTED_CLASS_PROBABILITIES_h10.npz',allow_pickle=False) as z:
    preview=pd.DataFrame(z['p_down_neutral_up'][:5],columns=['P_Down','P_Neutral','P_Up'])
    preview.insert(0,'day',z['day'][:5])
    preview.insert(1,'publisher_label',z['original_label'][:5])
display(preview.round(6))

# %%
# CELL 7 — frozen final manifest; DO NOT claim trading profitability
summary={'version':VERSION,'status':'PARAMETRIC_DBN_FITTED_AND_ASSESSED_RETROSPECTIVELY',
         'run_hash':RUN_HASH,'frozen_mask_hash':MASK_HASH_EXPECTED,
         'structural_slots':39,'self_slots':30,'cross_slots':9,
         'train_days':TRAIN_DAYS,'retrospective_assessment_days':EVAL_DAYS,
         'train_transition_rows':int(len(train_idx)),
         'assessment_transition_rows':int(len(eval_idx)),
         'transition_models':['AR5','SPARSE_39'],
         'innovation_Q':'diagonal Gaussian primary; no contemporaneous DAG asserted',
         'outcome_heads':HORIZONS,
         'label_alignment':'Publisher original labels; 20 warmups and h representation-row tail purges',
         'model_selection_after_step4':False,
         'historical_researcher_blind_assessment':False,
         'label_generation_exactly_reconstructed':False,
         'execution_costs_measured':False,'realized_trading_profit_estimated':False,
         'limitations':[
            'D6–D7 graph selection and D8–D10 earlier researcher exposure limit inferential independence.',
            'FI-2010 supplied classification labels do not encode executable future profits.',
            'No fees, spreads on completed round trips, fill/queue position, slippage or latency simulated.',
            'A fixed directed cross-lag graph is not identified intervention-level causality.',
            'Ordered probit predicts conditional label probabilities, not conditional return magnitudes.',
            'Correlated contemporaneous residuals invalidate a strict independent-CPD interpretation if full Q is used.'
         ],'next_step':'Exact missing-state Gaussian inference with the frozen sparse DBN; covariance sensitivity separate.'}
atomic_json(RESULTS/'STEP6_FINAL_SUMMARY.json',summary)
print('\n'+'='*95)
print('STEP 6 COMPLETE — FROZEN GAUSSIAN DBN + FIVE ORDERED-PROBIT HEADS')
print('='*95)
print('All output artifacts:',RESULTS)
print('No PnL claim: accuracy is not net returns.')
print('Runtime:',round(time.time()-T0,1),'seconds')

Google Drive already mounted.
CausalLOB Step 6 — read-only source root: /content/drive/MyDrive/CausalLOB
Outputs (new directory only): /content/drive/MyDrive/CausalLOB/STEP6_FROZEN_DBN
PREFLIGHT PASS: Step 2/4/5 inputs exact, publisher labels unmodified, mask fixed.
Train transition rows: 253875  | retrospective assessment rows: 139212
Graph: 30 self + 9 cross = 39  | run hash: 9831f8e1db702848b8588a535088f9392c32e9a09b66eb5cafe40fc6d4bd1d8c
Saved frozen AR(5) and sparse 39-slot Gaussian transitions.
Transition parameters frozen; residual innovation correlations are DESCRIPTIVE, not used as Q.

Frozen transition assessment (standardized state; positive gain means sparse improves):


model,AR5,SPARSE_39,rmse_gain_AR5_minus_sparse
target,,,
imbalance_l1,0.561755,0.561755,0.000000
imbalance_l2_l5,0.406754,0.406754,0.000000
log_mid_return_1,0.833930,0.824720,0.009211
log_relative_spread,0.434254,0.434556,-0.000302
log_total_depth_l5,0.109082,0.109082,0.000000
quote_ofi_depth_norm,0.836648,0.836316,0.000332


Fitting frozen ordered-probit h10; eligible train rows=253,525 ...
Saved h10 in 6.2s.
Fitting frozen ordered-probit h20; eligible train rows=253,175 ...
Saved h20 in 2.6s.
Fitting frozen ordered-probit h30; eligible train rows=252,825 ...
Saved h30 in 2.4s.
Fitting frozen ordered-probit h50; eligible train rows=252,125 ...
Saved h50 in 1.9s.
Fitting frozen ordered-probit h100; eligible train rows=250,375 ...
Saved h100 in 2.1s.

D8–D10 retrospective assessment; Day 8–10 historically inspected in older research:


,horizon_raw_messages,model,n,accuracy,balanced_accuracy,macro_f1,log_loss,brier_multiclass,ece_confidence
0,10,PRIOR_ONLY,139062,0.707008,0.333333,0.276120,0.830344,0.472694,0.101577
1,10,ORDERED_PROBIT_SPARSE_DBN_HEAD,139062,0.877493,0.727693,0.797387,0.499358,0.237289,0.088780
8,20,PRIOR_ONLY,138912,0.620976,0.333333,0.255392,0.958605,0.566575,0.126579
9,20,ORDERED_PROBIT_SPARSE_DBN_HEAD,138912,0.781228,0.623119,0.682000,0.715190,0.386840,0.135478
16,30,PRIOR_ONLY,138762,0.561595,0.333333,0.239753,1.030661,0.619175,0.143100
17,30,ORDERED_PROBIT_SPARSE_DBN_HEAD,138762,0.685137,0.538449,0.565151,0.937313,0.540732,0.195836
24,50,PRIOR_ONLY,138462,0.274624,0.333333,0.143637,1.109434,0.673787,0.068967
25,50,ORDERED_PROBIT_SPARSE_DBN_HEAD,138462,0.322327,0.407045,0.281089,1.070878,0.643074,0.103171
32,100,PRIOR_ONLY,137712,0.342664,0.333333,0.170141,1.160453,0.700311,0.063303
33,100,ORDERED_PROBIT_SPARSE_DBN_HEAD,137712,0.369982,0.374541,0.296589,1.146834,0.690095,0.086855



Example h10 output:


,day,publisher_label,P_Down,P_Neutral,P_Up
0,8,3,0.791804,0.207566,0.000631
1,8,3,0.106839,0.771845,0.121316
2,8,1,0.000661,0.211368,0.787972
3,8,1,0.000000,0.000000,1.000000
4,8,2,0.222088,0.728111,0.049801



STEP 6 COMPLETE — FROZEN GAUSSIAN DBN + FIVE ORDERED-PROBIT HEADS
All output artifacts: /content/drive/MyDrive/CausalLOB/STEP6_FROZEN_DBN/results
No PnL claim: accuracy is not net returns.
Runtime: 28.6 seconds


In [ ]:
# CausalLOB Step 7 — exact partial-evidence inference; one-cell Colab-compatible script.
# Research brief (GYP): Challenge uncertainty claims with exactly matched masks, rows,
# frozen graph/head parameters and train-only covariance sensitivity. Report failures;
# do not tune on D8–D10, infer causality, or claim executable profits.

import os, json, time, hashlib, importlib.util, subprocess, sys
from pathlib import Path

for module, package in [('scipy','scipy'),('sklearn','scikit-learn')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.covariance import LedoitWolf
from sklearn.metrics import balanced_accuracy_score, f1_score

try:
    from IPython.display import display
except ImportError:
    display = print
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('Google Drive already mounted.')
except ImportError:
    pass

T0=time.time()
ROOT=Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
S2=ROOT/'STEP2_REBUILD'/'checkpoints'
S4=ROOT/'STEP4_VALIDATION_FROZEN'/'checkpoints'
S5=ROOT/'STEP5_LABEL_AUDIT_READONLY'/'results'
S6=ROOT/'STEP6_FROZEN_DBN'
OUT=ROOT/'STEP7_MISSING_STATE_READONLY'
CKPT=OUT/'checkpoints'; RESULTS=OUT/'results'; PROBS=RESULTS/'focal_probabilities'
VERSION='STEP7_EXACT_MISSING_STATE_v1_2026-09-18'
EXPECTED_MASK_HASH='eb38b17c1431c0a98d46ed1c792953c932c58572799b64aee79165dfc58439da'
FEATURES=['log_relative_spread','log_total_depth_l5','imbalance_l1',
          'imbalance_l2_l5','quote_ofi_depth_norm','log_mid_return_1']
HORIZONS=[10,20,30,50,100]
TRAIN_DAYS=[1,2,3,4,5,6,7]
EVAL_DAYS=[8,9,10]
MASK_SCENARIOS={
    'FULL_OBSERVED':[],
    'MISS_L1_IMBALANCE':['imbalance_l1'],
    'MISS_OFI_PROXY':['quote_ofi_depth_norm'],
    'MISS_BOTH_IMBALANCES':['imbalance_l1','imbalance_l2_l5'],
    'MISS_LIQUIDITY':['log_relative_spread','log_total_depth_l5'],
    'MISS_RETURN_AND_OFI':['log_mid_return_1','quote_ofi_depth_norm'],
    'HISTORY_ONLY':FEATURES.copy(),
}
FOCAL_SCENARIOS=['MISS_RETURN_AND_OFI','HISTORY_ONLY']
FOCAL_HORIZONS=[10,20]
MASK_IDX={name:np.array(sorted(FEATURES.index(x) for x in vars_),dtype=np.int64)
          for name,vars_ in MASK_SCENARIOS.items()}

SOURCE={
 'temporal':S2/'04_temporal_primary.npz',
 'step4_mask':S4/'PRIMARY_FROZEN_PARENT_MASK.npz',
 'step5_summary':S5/'05_LABEL_AUDIT_SUMMARY.json',
 'step6_state':S6/'checkpoints'/'01_FROZEN_STATE_SPACE.npz',
 'step6_summary':S6/'results'/'STEP6_FINAL_SUMMARY.json',
 'step6_outcome_metrics':S6/'results'/'04_OUTCOME_METRICS_D8_D10.csv',
}
for h in HORIZONS:
    SOURCE[f'head_h{h}']=S6/'checkpoints'/'outcome_heads'/f'ordered_probit_h{h}.json'
    SOURCE[f'pred_h{h}']=S6/'results'/f'03_PREDICTED_CLASS_PROBABILITIES_h{h}.npz'


def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()


def atomic_json(path,obj):
    path=Path(path); tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj,sort_keys=True,indent=2,allow_nan=False)+'\n',encoding='utf8')
    tmp.replace(path)


def atomic_npz(path,**kwargs):
    path=Path(path);tmp=path.with_name(path.name+'.tmp.npz')
    np.savez_compressed(tmp,**kwargs);tmp.replace(path)


def atomic_csv(path,df):
    path=Path(path);tmp=path.with_name(path.name+'.tmp')
    df.to_csv(tmp,index=False,float_format='%.12g');tmp.replace(path)


def assert_spd(Q,name):
    assert np.allclose(Q,Q.T,rtol=1e-12,atol=1e-12),f'{name} not symmetric'
    ev=np.linalg.eigvalsh(Q)
    assert np.isfinite(ev).all() and ev.min()>1e-12,f'{name} covariance not SPD'
    return ev


def reference_ordered_probs(Z,beta,cut):
    """Identical to frozen Step 6 head evaluation, for strict parity."""
    eta=Z@np.asarray(beta,dtype=float)
    c1,c2=map(float,cut)
    down=norm.cdf(c1-eta)
    upto_neutral=norm.cdf(c2-eta)
    P=np.column_stack([down,upto_neutral-down,1-upto_neutral])
    assert np.isfinite(P).all() and np.min(P)>=-1e-12
    P=np.maximum(P,1e-15)
    P/=P.sum(axis=1,keepdims=True)
    return P


def condition_exact(prior,actual,Q,missing):
    """One-step p(X_t,miss | exact X_t,obs, exact X_{t-1:t-5}).

    Observed components are *copied* from observed input. Their posterior variance is 0.
    No evaluation-period parameter fitting or future-state smoothing is involved.
    """
    missing=np.asarray(missing,dtype=int)
    observed=np.array([j for j in range(6) if j not in missing],dtype=int)
    n=len(prior)
    assert prior.shape==actual.shape==(n,6)
    post=actual.copy()
    Sfull=np.zeros((6,6),dtype=np.float64)
    if not len(missing):
        return post,Sfull,np.zeros((0,0))
    if not len(observed):
        Sfull=Q.copy()
        return prior.copy(),Sfull,Q.copy()
    Soo=Q[np.ix_(observed,observed)]
    Smo=Q[np.ix_(missing,observed)]
    Smm=Q[np.ix_(missing,missing)]
    # Positive definiteness asserted; solve, not pseudoinverse, avoids hiding defects.
    K=np.linalg.solve(Soo,Smo.T).T
    post[:,missing]=prior[:,missing]+(actual[:,observed]-prior[:,observed])@K.T
    Scond=Smm-K@Smo.T
    Scond=(Scond+Scond.T)*0.5
    assert_spd(Scond,'posterior missing state')
    Sfull[np.ix_(missing,missing)]=Scond
    assert np.array_equal(post[:,observed],actual[:,observed])
    return post,Sfull,Scond


def state_metrics(actual,post,Scond,missing):
    missing=np.asarray(missing,dtype=int)
    if not len(missing):
        return {'state_rmse':0.0,'state_mae':0.0,'state_joint_nll':np.nan,
                'coverage90':np.nan,'abs_coverage_error90':np.nan,
                'mean_interval_width90':np.nan}
    err=actual[:,missing]-post[:,missing]
    rmse=float(np.sqrt(np.mean(err**2)))
    mae=float(np.mean(np.abs(err)))
    eig=assert_spd(Scond,'state score covariance')
    sign,logdet=np.linalg.slogdet(Scond)
    assert sign>0
    whitened=np.linalg.solve(Scond,err.T).T
    quad=np.einsum('ij,ij->i',err,whitened)
    assert np.isfinite(quad).all() and quad.min()>-1e-8
    d=len(missing)
    nll=float(np.mean(0.5*(d*np.log(2*np.pi)+logdet+quad)))
    halfwidth=norm.ppf(.95)*np.sqrt(np.diag(Scond))
    cov=float(np.mean(np.abs(err)<=halfwidth))
    return {'state_rmse':rmse,'state_mae':mae,'state_joint_nll':nll,
            'coverage90':cov,'abs_coverage_error90':abs(cov-.90),
            'mean_interval_width90':float(np.mean(2*halfwidth))}


def probs_integrated(post,Sfull,head):
    beta=np.asarray(head['beta'],dtype=np.float64)
    c1,c2=map(float,head['thresholds'])
    assert beta.shape==(6,) and c1<c2
    extra=float(beta@Sfull@beta)
    assert extra>=-1e-12
    sd=np.sqrt(1.0+max(0.,extra))
    eta=post@beta
    below=norm.cdf((c1-eta)/sd)
    upto=norm.cdf((c2-eta)/sd)
    P=np.column_stack([below,upto-below,1.0-upto])
    assert np.isfinite(P).all() and P.min()>=-1e-12
    P=np.maximum(P,1e-15)
    P/=P.sum(axis=1,keepdims=True)
    return P,extra


def metrics(y,P):
    y=np.asarray(y,dtype=int)
    assert np.isin(y,[0,1,2]).all() and P.shape==(len(y),3)
    assert np.allclose(P.sum(axis=1),1.,atol=1e-12)
    pred=P.argmax(axis=1)
    n=len(y)
    chosen=P[np.arange(n),y]
    ll=float(-np.mean(np.log(np.maximum(chosen,1e-15))))
    brier=float(np.mean(np.sum((P-np.eye(3)[y])**2,axis=1)))
    conf=P.max(axis=1); correct=pred==y; ece=0.
    edges=np.linspace(0,1,16)
    for j in range(15):
        inside=(conf>=edges[j]) & ((conf<=edges[j+1]) if j==14 else (conf<edges[j+1]))
        if inside.any():
            ece+=inside.mean()*abs(correct[inside].mean()-conf[inside].mean())
    return {'n':n,'accuracy':float(correct.mean()),
            'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
            'macro_f1':float(f1_score(y,pred,labels=[0,1,2],average='macro',zero_division=0)),
            'log_loss':ll,'brier_multiclass':brier,'ece_confidence':float(ece)}


def eligible(indices,remaining,h):
    return indices[remaining[indices]>=h]


# Local function-level analytic checks; these run before using real checkpoints.
def self_test():
    rng=np.random.default_rng(5162026)
    prior=rng.normal(size=(17,6));actual=rng.normal(size=(17,6))
    A=rng.normal(size=(6,6));Q=A@A.T+np.eye(6)*.2
    for missing in [[],[2],[4,5],[0,1,2,3,4,5]]:
        post,Sfull,S=condition_exact(prior,actual,Q,missing)
        observed=[j for j in range(6) if j not in missing]
        if observed:
            assert np.array_equal(post[:,observed],actual[:,observed])
            assert np.all(Sfull[np.ix_(observed,observed)]==0)
        if len(missing)==6:
            assert np.array_equal(post,prior) and np.array_equal(S,Q)
        if not missing:
            assert np.array_equal(post,actual) and np.count_nonzero(Sfull)==0
        else:
            z=state_metrics(actual,post,S,missing)
            assert np.isfinite(z['state_joint_nll'])
    diag=np.diag(np.arange(1.,7.))
    p,Sfull,S=condition_exact(prior,actual,diag,[4,5])
    assert np.array_equal(p[:,[4,5]],prior[:,[4,5]])
    dummy={'beta':np.array([.2,-.1,.4,0,.3,.5]),'thresholds':[-.3,.7]}
    P,e=probs_integrated(actual,np.zeros((6,6)),dummy)
    assert np.array_equal(P,reference_ordered_probs(actual,dummy['beta'],dummy['thresholds']))
    assert np.array_equal(P.sum(axis=1).round(12),np.ones(17))
    assert e==0.
    P2,e2=probs_integrated(p,Sfull,dummy)
    assert e2>0 and np.isfinite(P2).all()
    print('SELF-TEST PASS: zero/full/partial masks, SPD, diagonal no cross-conditioning, ordinal integration')

self_test()

for name,path in SOURCE.items():
    if not path.is_file():
        raise FileNotFoundError(f'Missing READ-ONLY prerequisite [{name}]: {path}')

source_hashes={name:sha256(path) for name,path in SOURCE.items()}
s5=json.loads(SOURCE['step5_summary'].read_text())
s6=json.loads(SOURCE['step6_summary'].read_text())
assert s5['frozen_mask_hash']==EXPECTED_MASK_HASH
assert s6['status']=='PARAMETRIC_DBN_FITTED_AND_ASSESSED_RETROSPECTIVELY'
assert s6['frozen_mask_hash']==EXPECTED_MASK_HASH
assert s6['structural_slots']==39 and s6['model_selection_after_step4'] is False
assert s6['historical_researcher_blind_assessment'] is False
with np.load(SOURCE['step4_mask'],allow_pickle=False) as z:
    frozen_mask=z['parent_mask'].copy()
    assert str(z['mask_hash'].item())==EXPECTED_MASK_HASH
    assert z['feature_names'].astype(str).tolist()==FEATURES
assert frozen_mask.shape==(6,6,6) and frozen_mask.sum()==39
with np.load(SOURCE['temporal'],allow_pickle=False) as z:
    assert str(z['checkpoint_version'].item())=='STEP2_REBUILD_v1_2026-09-17'
    assert z['feature_names'].astype(str).tolist()==FEATURES
    X=z['X_current'].astype(np.float64)
    Y=z['Y'].astype(np.int8)
    day=z['day'].astype(np.int8)
    seq=z['sequence_id'].astype(np.int16)
    lagvalid=z['full_lag_valid'].astype(bool)
assert X.shape==(393337,6) and Y.shape==(393337,5)
assert len(np.unique(seq))==50 and np.isfinite(X).all()
assert lagvalid.sum()==393087 and np.isin(Y,[1,2,3]).all()
assert np.all(np.diff(day)>=0)
with np.load(SOURCE['step6_state'],allow_pickle=False) as z:
    assert str(z['run_hash'].item())==s6['run_hash']
    assert z['feature_names'].astype(str).tolist()==FEATURES
    assert np.array_equal(z['frozen_parent_mask'],frozen_mask)
    mu=z['train_mean'].copy();sigma=z['train_std'].copy()
    B_ar=z['B_ar'].copy();c_ar=z['c_ar'].copy();v_ar=z['v_ar'].copy()
    B_sp=z['B_sparse'].copy();c_sp=z['c_sparse'].copy();v_sp=z['v_sparse'].copy()
    F_sp=z['F_sparse'].copy();Q_sp=z['Q_sparse'].copy()
assert B_ar.shape==B_sp.shape==(6,30) and c_ar.shape==c_sp.shape==(6,)
assert np.allclose(F_sp[:6],B_sp) and np.allclose(Q_sp[:6,:6],np.diag(v_sp))
assert np.array_equal(mu,X[np.isin(day,TRAIN_DAYS)].mean(axis=0))
assert np.array_equal(sigma,X[np.isin(day,TRAIN_DAYS)].std(axis=0,ddof=0))
assert (sigma>0).all()
heads={}
for h in HORIZONS:
    head=json.loads(SOURCE[f'head_h{h}'].read_text())
    assert head['run_hash']==s6['run_hash'] and head['converged']
    assert head['feature_names']==FEATURES and len(head['beta'])==6
    heads[h]=head

starts=np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]])
ends=np.r_[starts[1:],len(seq)]
remaining=np.empty(len(seq),dtype=np.int32)
train_rows=[];eval_rows=[]
for a,b in zip(starts,ends):
    assert b-a>110 and np.all(seq[a:b]==seq[a]) and np.all(day[a:b]==day[a])
    expected_lags=np.r_[np.zeros(5,dtype=bool),np.ones(b-a-5,dtype=bool)]
    assert np.array_equal(lagvalid[a:b],expected_lags)
    remaining[a:b]=b-1-np.arange(a,b)
    indices=np.arange(a+5,b,dtype=np.int64)
    if int(day[a]) in TRAIN_DAYS:train_rows.append(indices)
    elif int(day[a]) in EVAL_DAYS:eval_rows.append(indices)
train_idx=np.concatenate(train_rows);eval_idx=np.concatenate(eval_rows)
assert len(train_idx)==253875 and len(eval_idx)==139212
assert set(seq[train_idx]).isdisjoint(set(seq[eval_idx]))
assert np.isin(day[train_idx],TRAIN_DAYS).all()
assert np.isin(day[eval_idx],EVAL_DAYS).all()

# Verify all original Step 6 probability arrays and exactly matched row identities.
XZ=(X-mu)/sigma
reference={}
for h in HORIZONS:
    idx=eligible(eval_idx,remaining,h)
    with np.load(SOURCE[f'pred_h{h}'],allow_pickle=False) as z:
        assert str(z['run_hash'].item())==s6['run_hash']
        assert np.array_equal(z['row_index'],idx)
        assert np.array_equal(z['sequence_id'],seq[idx])
        assert np.array_equal(z['day'],day[idx])
        assert np.array_equal(z['original_label'],Y[idx,HORIZONS.index(h)])
        pref=z['p_down_neutral_up'].copy()
    pcheck=reference_ordered_probs(XZ[idx],heads[h]['beta'],heads[h]['thresholds'])
    assert np.array_equal(pref,pcheck),f'Head h{h}: not exact Step 6 parity!'
    reference[h]=pref

spec={
 'version':VERSION,'source_sha256':source_hashes,
 'step6_run_hash':s6['run_hash'],'frozen_mask_hash':EXPECTED_MASK_HASH,
 'train_days':TRAIN_DAYS,'retrospective_days_historically_exposed':EVAL_DAYS,
 'missing_evidence_masks':MASK_SCENARIOS,'horizons':HORIZONS,
 'transition':'read frozen AR5 and sparse39, no new transition fitting',
 'covariances':'Step6 frozen diagonal primary; LedoitWolf training-residual shrinkage sensitivity',
 'inference':'exact one-step Gaussian conditioning, full exact history and noiseless observed state',
 'outcome':'analytically integrated immutable Step6 ordered probit heads',
 'tail_purge':'h representation rows per sequence, inherited Step5/6 conservative policy',
 'model_selection':'none; no D8–D10 parameter fitting or hyperparameter/graph/head selection',
}
run_hash=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()
for p in [CKPT,RESULTS,PROBS]:p.mkdir(parents=True,exist_ok=True)
immutable=CKPT/'IMMUTABLE_STEP7_SPEC.json'
if immutable.exists():
    prior_spec=json.loads(immutable.read_text())
    assert prior_spec['run_hash']==run_hash,'Existing Step7 outputs have different inputs/spec! STOP.'
else:
    atomic_json(immutable,{'run_hash':run_hash,'spec':spec})
print('PREFLIGHT PASS: frozen 39-slot graph and Step6 h10...h100 outputs reproduced bitwise.')
print('Train-only residual rows:',len(train_idx),' | D8–D10 retrospective rows:',len(eval_idx))
print('Run hash:',run_hash)

# Training-only covariance estimation, sensitive to contemporaneous residual correlations.
Ztr=np.concatenate([XZ[train_idx-lag] for lag in range(1,6)],axis=1)
Zev=np.concatenate([XZ[eval_idx-lag] for lag in range(1,6)],axis=1)
Ytr=XZ[train_idx];Yev=XZ[eval_idx]
assert Ztr.shape==(253875,30) and Zev.shape==(139212,30)
variants={}
for name,B,c,v in [('AR5',B_ar,c_ar,v_ar),('SPARSE_39',B_sp,c_sp,v_sp)]:
    residual=Ytr-(Ztr@B.T+c)
    assert np.allclose(np.mean(residual*residual,axis=0),v,rtol=5e-10,atol=5e-10)
    lw=LedoitWolf().fit(residual)
    for covname,Q,shrinkage in [
        ('DIAG_FROZEN',np.diag(v),None),
        ('SHRINK_TRAIN_SENSITIVITY',lw.covariance_,float(lw.shrinkage_)),
    ]:
        tag=name+'__'+covname
        ev=assert_spd(Q,tag)
        variants[tag]={'B':B,'c':c,'Q':Q,'structure':name,'covariance':covname,
                       'shrinkage':shrinkage,'eigenvalues':ev}

cov_rows=[]
for name,d in variants.items():
    Q=d['Q'];sd=np.sqrt(np.diag(Q));corr=Q/np.outer(sd,sd)
    cov_rows.append({'model':name,'structure':d['structure'],'covariance':d['covariance'],
                     'shrinkage_train_only':d['shrinkage'],
                     'min_eigenvalue':float(d['eigenvalues'].min()),
                     'condition_number':float(d['eigenvalues'].max()/d['eigenvalues'].min()),
                     'max_abs_offdiag_correlation':float(np.max(np.abs(corr-np.eye(6))))})
atomic_csv(RESULTS/'01_TRAIN_ONLY_COVARIANCE_DIAGNOSTICS.csv',pd.DataFrame(cov_rows))
if not (CKPT/'01_TRAIN_ONLY_COVARIANCES.npz').exists():
    atomic_npz(CKPT/'01_TRAIN_ONLY_COVARIANCES.npz',run_hash=np.array(run_hash),
        Q_ar_diag=variants['AR5__DIAG_FROZEN']['Q'],
        Q_sparse_diag=variants['SPARSE_39__DIAG_FROZEN']['Q'],
        Q_ar_shrink=variants['AR5__SHRINK_TRAIN_SENSITIVITY']['Q'],
        Q_sparse_shrink=variants['SPARSE_39__SHRINK_TRAIN_SENSITIVITY']['Q'])
else:
    with np.load(CKPT/'01_TRAIN_ONLY_COVARIANCES.npz',allow_pickle=False) as z:
        assert str(z['run_hash'].item())==run_hash
        assert np.array_equal(z['Q_sparse_diag'],variants['SPARSE_39__DIAG_FROZEN']['Q'])
        assert np.array_equal(z['Q_sparse_shrink'],variants['SPARSE_39__SHRINK_TRAIN_SENSITIVITY']['Q'])
        assert np.array_equal(z['Q_ar_diag'],variants['AR5__DIAG_FROZEN']['Q'])
        assert np.array_equal(z['Q_ar_shrink'],variants['AR5__SHRINK_TRAIN_SENSITIVITY']['Q'])
print('Train-only innovation covariances ready. Shrinkage is sensitivity, NOT a replacement for frozen diagonal DBN.')

# Exact inference and identical D8–D10 subsets for each variant and missingness scenario.
state_rows=[];outcome_rows=[]
for name,d in variants.items():
    prior=Zev@d['B'].T+d['c']
    for scenario,missing in MASK_IDX.items():
        posterior,Sfull,Scond=condition_exact(prior,Yev,d['Q'],missing)
        for group,m in [('POOLED',np.ones(len(eval_idx),dtype=bool))]+[
            (str(daynum),day[eval_idx]==daynum) for daynum in EVAL_DAYS]:
            state_rows.append({'model':name,'structure':d['structure'],
                'covariance':d['covariance'],'scenario':scenario,'day':group,
                'n':int(m.sum()),'n_missing':len(missing),
                **state_metrics(Yev[m],posterior[m],Scond,missing)})
        for h in HORIZONS:
            idx=eligible(eval_idx,remaining,h)
            loc=np.searchsorted(eval_idx,idx)
            assert np.array_equal(eval_idx[loc],idx)
            y=(3-Y[idx,HORIZONS.index(h)]).astype(int)
            if not len(missing):
                P=reference_ordered_probs(posterior[loc],heads[h]['beta'],heads[h]['thresholds'])
                assert np.array_equal(P,reference[h]),f'Full-observed Step6 drift h{h}'
                latent_extra=0.0
            else:
                P,latent_extra=probs_integrated(posterior[loc],Sfull,heads[h])
            for group,m in [('POOLED',np.ones(len(idx),dtype=bool))]+[
                (str(daynum),day[idx]==daynum) for daynum in EVAL_DAYS]:
                outcome_rows.append({'model':name,'structure':d['structure'],
                    'covariance':d['covariance'],'scenario':scenario,'horizon':h,
                    'day':group,'latent_extra_variance':latent_extra,**metrics(y[m],P[m])})
            if (scenario in FOCAL_SCENARIOS and h in FOCAL_HORIZONS
                    and name in ['SPARSE_39__DIAG_FROZEN','SPARSE_39__SHRINK_TRAIN_SENSITIVITY']):
                out_file=PROBS/f'{name}__{scenario}__h{h}.npz'
                if out_file.is_file():
                    with np.load(out_file,allow_pickle=False) as z:
                        assert str(z['run_hash'].item())==run_hash
                        assert np.array_equal(z['row_index'],idx)
                        assert np.array_equal(z['p_down_neutral_up'],P)
                else:
                    atomic_npz(out_file,run_hash=np.array(run_hash),row_index=idx,
                        day=day[idx],sequence_id=seq[idx],label_ordered=y,
                        p_down_neutral_up=P,
                        class_names=np.array(['Down','Neutral','Up']),
                        scenario=np.array(scenario),horizon=np.array(h),
                        posterior_latent_extra_variance=np.array(latent_extra))
        print('Finished:',name,scenario)

state=pd.DataFrame(state_rows)
outcome=pd.DataFrame(outcome_rows)
assert len(state)==4*7*4 and len(outcome)==4*7*5*4
atomic_csv(RESULTS/'02_EXACT_STATE_METRICS.csv',state)
atomic_csv(RESULTS/'03_EXACT_OUTCOME_METRICS.csv',outcome)

# Parity of Step 7 full-evidence metrics against Step 6's frozen classifier table.
old=pd.read_csv(SOURCE['step6_outcome_metrics'])
for h in HORIZONS:
    for group in ['POOLED']+[str(d) for d in EVAL_DAYS]:
        a=old[(old['horizon_raw_messages']==h)&(old['day'].astype(str)==group)&
              (old['model']=='ORDERED_PROBIT_SPARSE_DBN_HEAD')]
        b=outcome[(outcome.horizon==h)&(outcome.day==group)&
              (outcome.model=='SPARSE_39__DIAG_FROZEN')&
              (outcome.scenario=='FULL_OBSERVED')]
        assert len(a)==len(b)==1
        for field in ['accuracy','balanced_accuracy','macro_f1','log_loss','brier_multiclass','ece_confidence']:
            assert abs(float(a.iloc[0][field])-float(b.iloc[0][field]))<5e-10,(h,group,field)
print('STEP6↔STEP7 METRIC PARITY PASS: all 5 horizons, pooled and each of 3 days.')

# Contrasts: gains positive if challenger improves; target/mask/horizon matched.
state_p=state[state.day=='POOLED'].pivot(index='scenario',columns='model',values=['state_rmse','state_joint_nll','abs_coverage_error90'])
outcome_p=outcome[outcome.day=='POOLED'].pivot(index=['scenario','horizon'],columns='model',values=['log_loss','brier_multiclass','balanced_accuracy'])
comparisons=[
    ('CROSS_STRUCTURE_DIAG_PRIMARY','AR5__DIAG_FROZEN','SPARSE_39__DIAG_FROZEN'),
    ('COVARIANCE_SENSITIVITY_SPARSE','SPARSE_39__DIAG_FROZEN','SPARSE_39__SHRINK_TRAIN_SENSITIVITY'),
    ('CROSS_STRUCTURE_SHRINK_SENSITIVITY','AR5__SHRINK_TRAIN_SENSITIVITY','SPARSE_39__SHRINK_TRAIN_SENSITIVITY'),
]
contrast_state=[];contrast_outcome=[]
for label,baseline,challenger in comparisons:
    for scenario in MASK_SCENARIOS:
        if scenario!='FULL_OBSERVED':
            contrast_state.append({'contrast':label,'scenario':scenario,
                **{field+'_gain':float(state_p.loc[scenario,(field,baseline)]-
                                      state_p.loc[scenario,(field,challenger)])
                   for field in ['state_rmse','state_joint_nll','abs_coverage_error90']}})
        for h in HORIZONS:
            values={}
            for field in ['log_loss','brier_multiclass']:
                values[field+'_gain']=float(outcome_p.loc[(scenario,h),(field,baseline)]-
                                           outcome_p.loc[(scenario,h),(field,challenger)])
            values['balanced_accuracy_gain']=float(outcome_p.loc[(scenario,h),('balanced_accuracy',challenger)]-
                                                  outcome_p.loc[(scenario,h),('balanced_accuracy',baseline)])
            contrast_outcome.append({'contrast':label,'scenario':scenario,'horizon':h,**values})
state_abl=pd.DataFrame(contrast_state);outcome_abl=pd.DataFrame(contrast_outcome)
atomic_csv(RESULTS/'04_STATE_ABLATIONS.csv',state_abl)
atomic_csv(RESULTS/'05_OUTCOME_ABLATIONS.csv',outcome_abl)

# Day consistency on explicitly small 3-day blocks (not independent tests).
stability=[]
for label,baseline,challenger in comparisons:
    for scenario in MASK_SCENARIOS:
        if scenario=='FULL_OBSERVED':continue
        delta=[]
        for d in EVAL_DAYS:
            a=state[(state.model==baseline)&(state.scenario==scenario)&(state.day==str(d))].iloc[0]
            b=state[(state.model==challenger)&(state.scenario==scenario)&(state.day==str(d))].iloc[0]
            delta.append(float(a.state_rmse-b.state_rmse))
        stability.append({'contrast':label,'scenario':scenario,'days_positive':int(np.sum(np.array(delta)>0)),
                          'days_total':3,'day8_gain':delta[0],'day9_gain':delta[1],
                          'day10_gain':delta[2]})
atomic_csv(RESULTS/'06_DAY_LEVEL_STATE_CONSISTENCY.csv',pd.DataFrame(stability))

summary={'version':VERSION,'status':'EXACT_PARTIAL_EVIDENCE_ASSESSED_RETROSPECTIVELY',
         'run_hash':run_hash,'step6_run_hash':s6['run_hash'],
         'frozen_mask_hash':EXPECTED_MASK_HASH,
         'training_rows':int(len(train_idx)),'retrospective_rows':int(len(eval_idx)),
         'n_state_metrics':len(state),'n_outcome_metrics':len(outcome),
         'probability_columns':['Down','Neutral','Up'],
         'complete_evidence_exact_parity_with_step6':True,
         'graph_and_heads_retrained':False,
         'primary_model':'SPARSE_39__DIAG_FROZEN',
         'sensitivity':'LedoitWolf covariance from D1-D7 residuals, NOT original DBN CPDs',
         'assumption':'exact five-step historical state and noiseless observed contemporaneous state',
         'D8_D10_historically_researcher_blind':False,
         'label_formula_reconstructed_exactly':False,
         'limitations':['Three evaluation days were inspected in earlier project research.',
             'The shrinkage covariance creates correlated residual innovations; it is a sensitivity model, not the original independent-innovation DBN.',
             'Only one-step filtering with exact historical states, not multistep gap smoothing.',
             'Outcomes are publisher labels; accuracy does not establish after-cost profitability.',
             'Do not tune structure, thresholds, Q selection or masks against D8-D10.'],
         'output_files':[p.name for p in sorted(RESULTS.glob('*.csv'))]}
atomic_json(RESULTS/'STEP7_FINAL_SUMMARY.json',summary)

print('\n'+'='*104)
print('STEP 7 COMPLETE — EXACT MISSING-STATE INFERENCE (retrospective, graph unchanged)')
print('='*104)
print('Frozen sparse vs AR5 under diagonal Q: state RMSE improvements (+ = sparse improves):')
display(state_abl[state_abl.contrast=='CROSS_STRUCTURE_DIAG_PRIMARY'][['scenario','state_rmse_gain','state_joint_nll_gain','abs_coverage_error90_gain']].round(6))
print('\nCovariance sensitivity vs frozen sparse diagonal (positive = shrinkage improves):')
display(state_abl[state_abl.contrast=='COVARIANCE_SENSITIVITY_SPARSE'][['scenario','state_rmse_gain','state_joint_nll_gain','abs_coverage_error90_gain']].round(6))
print('\nFocal outcomes h10/h20:')
display(outcome_abl[(outcome_abl.scenario.isin(FOCAL_SCENARIOS))&
                    (outcome_abl.horizon.isin(FOCAL_HORIZONS))][
                   ['contrast','scenario','horizon','log_loss_gain','brier_multiclass_gain','balanced_accuracy_gain']].round(6))
print('\nOutputs:',RESULTS)
print('No trading-profit or identified-causal-effect claim. Runtime:',round(time.time()-T0,1),'s')

Google Drive already mounted.
SELF-TEST PASS: zero/full/partial masks, SPD, diagonal no cross-conditioning, ordinal integration
PREFLIGHT PASS: frozen 39-slot graph and Step6 h10...h100 outputs reproduced bitwise.
Train-only residual rows: 253875  | D8–D10 retrospective rows: 139212
Run hash: 12ac66dc723a267c9082067776d32078dd476241c69d8b69df3e8534b6d8683c
Train-only innovation covariances ready. Shrinkage is sensitivity, NOT a replacement for frozen diagonal DBN.
Finished: AR5__DIAG_FROZEN FULL_OBSERVED
Finished: AR5__DIAG_FROZEN MISS_L1_IMBALANCE
Finished: AR5__DIAG_FROZEN MISS_OFI_PROXY
Finished: AR5__DIAG_FROZEN MISS_BOTH_IMBALANCES
Finished: AR5__DIAG_FROZEN MISS_LIQUIDITY
Finished: AR5__DIAG_FROZEN MISS_RETURN_AND_OFI
Finished: AR5__DIAG_FROZEN HISTORY_ONLY
Finished: AR5__SHRINK_TRAIN_SENSITIVITY FULL_OBSERVED
Finished: AR5__SHRINK_TRAIN_SENSITIVITY MISS_L1_IMBALANCE
Finished: AR5__SHRINK_TRAIN_SENSITIVITY MISS_OFI_PROXY
Finished: AR5__SHRINK_TRAIN_SENSITIVITY MISS_BOTH_IMBALANCE

,scenario,state_rmse_gain,state_joint_nll_gain,abs_coverage_error90_gain
0,MISS_L1_IMBALANCE,0.000000,0.000000,0.000000
1,MISS_OFI_PROXY,0.000332,0.000661,-0.000115
2,MISS_BOTH_IMBALANCES,0.000000,0.000000,0.000000
3,MISS_LIQUIDITY,-0.000207,-0.000438,0.000672
4,MISS_RETURN_AND_OFI,0.004752,0.013698,-0.005984
5,HISTORY_ONLY,0.002209,0.013261,-0.001771



Covariance sensitivity vs frozen sparse diagonal (positive = shrinkage improves):


,scenario,state_rmse_gain,state_joint_nll_gain,abs_coverage_error90_gain
6,MISS_L1_IMBALANCE,0.158365,0.332344,-0.002356
7,MISS_OFI_PROXY,0.248267,0.361488,0.009065
8,MISS_BOTH_IMBALANCES,0.087599,0.347351,-0.001609
9,MISS_LIQUIDITY,-0.000349,0.043611,0.000230
10,MISS_RETURN_AND_OFI,0.063880,0.476854,0.000248
11,HISTORY_ONLY,0.000000,0.522124,-0.000048



Focal outcomes h10/h20:


,contrast,scenario,horizon,log_loss_gain,brier_multiclass_gain,balanced_accuracy_gain
25,CROSS_STRUCTURE_DIAG_PRIMARY,MISS_RETURN_AND_OFI,10,0.008034,0.001527,0.028693
26,CROSS_STRUCTURE_DIAG_PRIMARY,MISS_RETURN_AND_OFI,20,0.005350,-0.000719,0.007611
30,CROSS_STRUCTURE_DIAG_PRIMARY,HISTORY_ONLY,10,0.008104,0.001626,0.031000
31,CROSS_STRUCTURE_DIAG_PRIMARY,HISTORY_ONLY,20,0.006005,0.000196,0.024094
60,COVARIANCE_SENSITIVITY_SPARSE,MISS_RETURN_AND_OFI,10,0.047918,0.033997,0.149928
61,COVARIANCE_SENSITIVITY_SPARSE,MISS_RETURN_AND_OFI,20,0.029641,0.023273,0.097961
65,COVARIANCE_SENSITIVITY_SPARSE,HISTORY_ONLY,10,-0.014945,-0.010293,-0.002633
66,COVARIANCE_SENSITIVITY_SPARSE,HISTORY_ONLY,20,-0.010476,-0.007126,0.000457
95,CROSS_STRUCTURE_SHRINK_SENSITIVITY,MISS_RETURN_AND_OFI,10,0.015055,0.010884,0.026359
96,CROSS_STRUCTURE_SHRINK_SENSITIVITY,MISS_RETURN_AND_OFI,20,0.010098,0.005201,-0.007395



Outputs: /content/drive/MyDrive/CausalLOB/STEP7_MISSING_STATE_READONLY/results
No trading-profit or identified-causal-effect claim. Runtime: 55.9 s


In [ ]:
# CausalLOB STEP 8 — frozen graph-specific contribution: static, AR5,
# sparse CausalLOB, dense VAR5, 100 matched random graphs.
# GYP self-brief: falsify graph-specific claims. Reuse the Step6 graph/head without
# any selection on previously inspected D8–D10; all new models D1–D7 only.
# Challenge sparse graph against equally complex source-randomized graphs,
# isolate history via static baseline, preserve negative outcomes, never claim
# post-cost profitability or identified causal effects.
import os, json, time, hashlib, importlib.util, subprocess
from pathlib import Path
from collections import defaultdict
for module,package in [('scipy','scipy'),('sklearn','scikit-learn')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([__import__('sys').executable,'-m','pip','install','-q',package])
import numpy as np
import pandas as pd
from scipy.special import ndtr
try:
    from IPython.display import display
except ImportError:
    display=print
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
    else:print('Google Drive already mounted.')
except ImportError:
    pass
T0=time.monotonic()
ROOT=Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
S2=ROOT/'STEP2_REBUILD/checkpoints'
S4=ROOT/'STEP4_VALIDATION_FROZEN/checkpoints'
S6=ROOT/'STEP6_FROZEN_DBN'
S7=ROOT/'STEP7_MISSING_STATE_READONLY/results'
OUT=ROOT/'STEP8_GRAPH_SPECIFIC_READONLY'
CKPT,RESULTS=OUT/'checkpoints',OUT/'results'
for p in (CKPT,RESULTS):p.mkdir(parents=True,exist_ok=True)
TRAIN_DAYS=[1,2,3,4,5,6,7]
EVAL_DAYS=[8,9,10]
HORIZONS=(10,20)
SEED=5162026
N_RANDOM=100
RIDGE_ALPHA=1.0
N=6
L=5
EXPECTED_MASK_HASH='eb38b17c1431c0a98d46ed1c792953c932c58572799b64aee79165dfc58439da'
EXPECTED=['log_relative_spread','log_total_depth_l5','imbalance_l1',
          'imbalance_l2_l5','quote_ofi_depth_norm','log_mid_return_1']
MASKS={
    'MISS_L1_IMBALANCE':(2,),
    'MISS_OFI_PROXY':(4,),
    'MISS_BOTH_IMBALANCES':(2,3),
    'MISS_LIQUIDITY':(0,1),
    'MISS_RETURN_AND_OFI':(4,5),
    'HISTORY_ONLY':tuple(range(6)),
}
M1=S2/'04_temporal_primary.npz'
M2_MASK=S4/'PRIMARY_FROZEN_PARENT_MASK.npz'
M3=S6/'checkpoints/01_FROZEN_STATE_SPACE.npz'
M3_FREEZE=S6/'results/STEP6_FINAL_SUMMARY.json'
S7_FREEZE=S7/'STEP7_FINAL_SUMMARY.json'
S7_STATES=S7/'02_EXACT_STATE_METRICS.csv'
S7_OUTCOMES=S7/'03_EXACT_OUTCOME_METRICS.csv'
HEADS=S6/'checkpoints/outcome_heads'
SOURCES=[M1,M2_MASK,M3,M3_FREEZE,S7_FREEZE,S7_STATES,S7_OUTCOMES]+[
    HEADS/f'ordered_probit_h{h}.json' for h in HORIZONS]
missing_paths=[str(p) for p in SOURCES if not p.is_file()]
if missing_paths:
    raise FileNotFoundError('READ-ONLY Step8 prerequisites absent; do not train or rebuild:\n'+'\n'.join(missing_paths))
with np.load(M1,allow_pickle=False) as z:
    assert str(z['checkpoint_version'].item())=='STEP2_REBUILD_v1_2026-09-17'
    X=z['X_current'].astype(np.float64)
    Y=z['Y'].astype(np.int8)
    day=z['day'].astype(np.int8)
    sid=z['sequence_id'].astype(np.int16)
    lagvalid=z['full_lag_valid'].astype(bool)
    names=z['feature_names'].astype(str).tolist()
with np.load(M2_MASK,allow_pickle=False) as z:
    frozen_mask=z['parent_mask'].copy()
    assert z['feature_names'].astype(str).tolist()==EXPECTED
    assert str(z['mask_hash'].item())==EXPECTED_MASK_HASH
with np.load(M3,allow_pickle=False) as z:
    frozen_run_hash=str(z['run_hash'].item())
    B_ar_ref=z['B_ar'].astype(np.float64)
    B_sparse_ref=z['B_sparse'].astype(np.float64)
    c_ar_ref=z['c_ar'].astype(np.float64)
    c_sparse_ref=z['c_sparse'].astype(np.float64)
    v_ar_ref=z['v_ar'].astype(np.float64)
    v_sparse_ref=z['v_sparse'].astype(np.float64)
    train_mean=z['train_mean'].astype(np.float64)
    train_std=z['train_std'].astype(np.float64)
    assert np.array_equal(z['frozen_parent_mask'],frozen_mask)
    assert np.array_equal(z['Q_sparse'][:N,:N],np.diag(v_sparse_ref))
    assert np.array_equal(z['Q_ar'][:N,:N],np.diag(v_ar_ref))
    assert z['feature_names'].astype(str).tolist()==EXPECTED
freeze=json.loads(M3_FREEZE.read_text())
s7freeze=json.loads(S7_FREEZE.read_text())
assert freeze['run_hash']==frozen_run_hash
assert freeze['frozen_mask_hash']==EXPECTED_MASK_HASH
assert freeze['structural_slots']==39
assert freeze['model_selection_after_step4'] is False
assert freeze['historical_researcher_blind_assessment'] is False
assert s7freeze['step6_run_hash']==frozen_run_hash
assert s7freeze['frozen_mask_hash']==EXPECTED_MASK_HASH
assert s7freeze['complete_evidence_exact_parity_with_step6'] is True
assert names==EXPECTED and frozen_mask.shape==(6,6,6)
assert int(frozen_mask.sum())==39 and not frozen_mask[:,:,0].any()
assert all(frozen_mask[j,j,1:].all() for j in range(N))
heads={h:json.loads((HEADS/f'ordered_probit_h{h}.json').read_text()) for h in HORIZONS}
for h,head in heads.items():
    assert head['run_hash']==frozen_run_hash and head['converged']
    assert head['feature_names']==EXPECTED and len(head['beta'])==N
assert len(X)==393337 and Y.shape==(393337,5)
assert np.isfinite(X).all() and np.isin(Y,[1,2,3]).all()
assert np.array_equal(np.unique(day),np.arange(1,11))
# Boundary-safe construction: matching EXACT Step6/7 transition rows.
starts=np.flatnonzero(np.r_[True,sid[1:]!=sid[:-1]])
ends=np.r_[starts[1:],len(X)]
assert len(starts)==50 and len(np.unique(sid))==50
remaining=np.empty(len(X),dtype=np.int32)
tr_list,te_list=[],[]
for a,b in zip(starts,ends):
    assert b-a>110 and np.all(sid[a:b]==sid[a]) and np.all(day[a:b]==day[a])
    assert np.array_equal(lagvalid[a:b],np.r_[np.zeros(5,bool),np.ones(b-a-5,bool)])
    remaining[a:b]=b-1-np.arange(a,b)
    idx=np.arange(a+L,b,dtype=np.int64)
    if int(day[a]) in TRAIN_DAYS:tr_list.append(idx)
    elif int(day[a]) in EVAL_DAYS:te_list.append(idx)
tr_idx=np.concatenate(tr_list);te_idx=np.concatenate(te_list)
assert len(tr_idx)==253875 and len(te_idx)==139212
assert set(sid[tr_idx]).isdisjoint(set(sid[te_idx]))
assert np.max(np.abs(X[np.isin(day,TRAIN_DAYS)].mean(0)-train_mean))<1e-10
assert np.max(np.abs(X[np.isin(day,TRAIN_DAYS)].std(0)-train_std))<1e-10
assert np.all(train_std>0)
XZ=(X-train_mean)/train_std
Ztr=np.concatenate([XZ[tr_idx-k] for k in range(1,L+1)],axis=1)
Zte=np.concatenate([XZ[te_idx-k] for k in range(1,L+1)],axis=1)
Ytr=XZ[tr_idx];Yte=XZ[te_idx];te_days=day[te_idx]
assert Ztr.shape==(253875,30) and Zte.shape==(139212,30)
assert np.isfinite(Ztr).all() and np.isfinite(Zte).all()
# Decode frozen mask directly: no hand-entered / reselected graph.
actual_slots=tuple(sorted(
    [(s,t,lag) for s in range(N) for t in range(N) if s!=t
     for lag in range(1,L+1) if frozen_mask[s,t,lag]],
    key=lambda r:(r[1],r[2],r[0])))
assert len(actual_slots)==9 and len(set(actual_slots))==9
assert sum(1 for s,t,l in actual_slots if s==t)==0
print('STEP8 PREFLIGHT PASS: frozen graph/head, D1-D7 fitting, D8-D10 previously inspected; nothing modified upstream.')
print('Source Step6 hash:',frozen_run_hash,' | rows:',len(tr_idx),len(te_idx))

# Target+lag-matched graph-null construction: preserve the number of selected
# parents for each (target, lag), randomize only source identity without replacement.
# This is much stricter than selecting nine arbitrary edges from 150 candidates.
groups = defaultdict(list)
for source,target,lag in actual_slots: groups[(target,lag)].append(source)
rng = np.random.default_rng(SEED)
random_graphs, seen = [], set()
for rep in range(N_RANDOM):
    for attempt in range(10000):
        candidate = []
        for (target,lag),existing_sources in sorted(groups.items()):
            pool = [src for src in range(N) if src != target]
            sampled = rng.choice(pool,size=len(existing_sources),replace=False)
            candidate.extend((int(src),int(target),int(lag)) for src in sampled)
        g = tuple(sorted(candidate,key=lambda r:(r[1],r[2],r[0])))
        if g not in seen and g != actual_slots:
            seen.add(g)
            random_graphs.append(g)
            break
    else:
        raise RuntimeError('Could not obtain 100 distinct target+lag-matched graphs; STOP.')
assert all(len(g)==9 for g in random_graphs)
assert all({(t,l):sum((tt,ll)==(t,l) for _,tt,ll in g) for (t,l) in groups} ==
           {(t,l):len(srcs) for (t,l),srcs in groups.items()} for g in random_graphs)

# Precompute sufficient statistics for exact M3-equivalent Ridge(alpha=1, fit_intercept=True).
# This reduces the 100 full-data regressions to solving tiny normal equations.
z_mean, y_mean = Ztr.mean(axis=0), Ytr.mean(axis=0)
Zc = Ztr - z_mean
Yc = Ytr - y_mean
G = Zc.T @ Zc
H = Zc.T @ Yc
Yss = np.einsum('ij,ij->j',Yc,Yc)
del Zc, Yc

def fit_graph(graph):
    B = np.zeros((N,N*L),dtype=np.float64)
    c = np.zeros(N,dtype=np.float64)
    v = np.zeros(N,dtype=np.float64)
    for target in range(N):
        cols = [target+N*(lag-1) for lag in range(1,L+1)]
        cols += [source+N*(lag-1) for source,t,lag in graph if t==target]
        assert len(set(cols)) == len(cols), 'Repeated parents; STOP.'
        # Sorting does not affect fitted coefficients; map back to correct design columns.
        cols = np.asarray(sorted(cols),dtype=np.int64)
        gram = G[np.ix_(cols,cols)]
        cross = H[cols,target]
        b = np.linalg.solve(gram+RIDGE_ALPHA*np.eye(len(cols)),cross)
        B[target,cols] = b
        c[target] = y_mean[target]-z_mean[cols]@b
        rss = Yss[target]-2*b@cross+b@gram@b
        v[target] = max(float(rss/len(Ytr)),1e-10)
    return B,c,v

B_ar_fit,c_ar_fit,v_ar_fit = fit_graph(())
B_sparse_fit,c_sparse_fit,v_sparse_fit = fit_graph(actual_slots)
checks = {
    'AR5_B':float(np.max(np.abs(B_ar_fit-B_ar_ref))),
    'AR5_c':float(np.max(np.abs(c_ar_fit-c_ar_ref))),
    'AR5_var':float(np.max(np.abs(v_ar_fit-v_ar_ref))),
    'SPARSE_B':float(np.max(np.abs(B_sparse_fit-B_sparse_ref))),
    'SPARSE_c':float(np.max(np.abs(c_sparse_fit-c_sparse_ref))),
    'SPARSE_var':float(np.max(np.abs(v_sparse_fit-v_sparse_ref))),
}
print('Exact frozen-Step6 reproduction diagnostics:',checks)
if max(checks.values())>2e-6:
    raise RuntimeError('Ridge refit DOES NOT reproduce frozen Step6. STOP; do not evaluate another graph.')

# Original baselines are evaluated with their *frozen parameters*, not refitted surrogates.
static_mu=Ytr.mean(axis=0)
static_var=np.mean((Ytr-static_mu)**2,axis=0)
assert np.isfinite(static_var).all() and np.all(static_var>1e-12)
core = {
    'STATIC_DIAG_TRAIN_ONLY':(np.zeros((N,N*L)),static_mu,static_var),
    'FROZEN_AR5':(B_ar_ref,c_ar_ref,v_ar_ref),
    'FROZEN_CAUSALLOB':(B_sparse_ref,c_sparse_ref,v_sparse_ref),
    'DENSE_VAR5':fit_graph(tuple((s,t,l) for t in range(N) for l in range(1,L+1) for s in range(N) if s!=t)),
}

# Two-horizon outcomes: reuse the SAME frozen D1–D7 probit heads in all graphs.
future_masks = {h:remaining[te_idx]>=h for h in HORIZONS}
labels = {}
for hi,h in enumerate((10,20,30,50,100)):
    if h not in HORIZONS: continue
    yy = Y[te_idx[future_masks[h]],hi]
    lab = np.full(len(yy),-1,dtype=np.int8)
    for source,dest in ((3,0),(2,1),(1,2)): lab[yy==source] = dest
    assert (lab>=0).all(), 'Unrecognized FI labels; STOP.'
    labels[h] = lab

def score_state(x, pred, var, miss):
    # Scores are on standardized M3 scale; compare only within the same mask.
    e = x[:,miss]-pred[:,miss]
    vv = var[miss]
    rmse = float(np.sqrt(np.mean(e*e)))
    nll = float(np.mean(0.5*np.sum(np.log(2*np.pi*vv)+(e*e)/vv,axis=1)))
    cover = float(np.mean(np.abs(e) <= 1.6448536269514722*np.sqrt(vv)))
    return rmse,nll,cover

def score_direction(y,prob):
    prob = np.maximum(prob,1e-15)
    prob /= prob.sum(axis=1,keepdims=True)
    pred = prob.argmax(axis=1)
    mat = np.bincount(y*3+pred,minlength=9).reshape(3,3)
    recall = np.diag(mat)/np.maximum(mat.sum(axis=1),1)
    precision = np.diag(mat)/np.maximum(mat.sum(axis=0),1)
    macro_f1 = float(np.mean(np.divide(2*precision*recall,precision+recall,
                                      out=np.zeros(3),where=(precision+recall)>0)))
    return dict(log_loss=float(-np.mean(np.log(prob[np.arange(len(y)),y]))),
                brier=float(np.mean(np.sum((prob-np.eye(3)[y])**2,axis=1))),
                balanced_accuracy=float(np.mean(recall)),macro_f1=macro_f1)

def score_model(model_name, fit, graph, scenarios=MASKS):
    B,c,var = fit
    mu = Zte @ B.T + c
    residual = Yte-mu
    trans = []
    states = []
    outcomes = []
    unique_days = ['POOLED']+EVAL_DAYS
    for d in unique_days:
        ix = np.ones(len(te_idx),dtype=bool) if d=='POOLED' else (te_days==d)
        err = residual[ix]
        # Joint transition log likelihood under original diagonal innovation assumption.
        terms = 0.5*(np.log(2*np.pi*var)+(err*err)/var)
        trans.append(dict(model=model_name,day=d,n=int(ix.sum()),
                          transition_rmse=float(np.sqrt(np.mean(err**2))),
                          transition_joint_nll=float(np.mean(np.sum(terms,axis=1))),
                          return_rmse=float(np.sqrt(np.mean(err[:,5]**2))),
                          return_nll=float(np.mean(terms[:,5]))))
    # Conditioning under diagonal innovations leaves observed X exactly observed, and
    # replaces missing dimensions with the model prediction. No false Q correlations.
    for scenario,miss_tuple in scenarios.items():
        miss=np.asarray(miss_tuple,dtype=int)
        for d in unique_days:
            ix=np.ones(len(te_idx),dtype=bool) if d=='POOLED' else te_days==d
            rmse,nll,cover = score_state(Yte[ix],mu[ix],var,miss)
            states.append(dict(model=model_name,scenario=scenario,day=d,n=int(ix.sum()),
                               state_rmse=rmse,state_nll=nll,coverage90=cover))
        for h in HORIZONS:
            keep=future_masks[h]
            head=heads[h]
            beta=np.asarray(head['beta'],dtype=float)
            t1,t2=map(float,head['thresholds'])
            assert t1<t2, 'Invalid probit thresholds; STOP.'
            # Start from actual *observed* state, impute only missing coordinates.
            latent = Yte[keep] @ beta + (mu[keep][:,miss]-Yte[keep][:,miss]) @ beta[miss]
            scale = np.sqrt(1.0+np.sum(var[miss]*beta[miss]**2))
            p0=ndtr((t1-latent)/scale)
            upto=ndtr((t2-latent)/scale)
            p1=upto-p0
            p2=1.0-upto
            P=np.column_stack((p0,p1,p2))
            for d in unique_days:
                which=np.ones(len(labels[h]),dtype=bool) if d=='POOLED' else te_days[keep]==d
                met=score_direction(labels[h][which],P[which].copy())
                outcomes.append(dict(model=model_name,scenario=scenario,horizon=h,day=d,
                                     n=int(which.sum()),**met))
    return trans,states,outcomes

# Incremental checkpoint: ensure interrupted runs can resume without retraining finished random models.
progress_file = CKPT/'random_progress.npz'
progress_meta = CKPT/'random_progress_manifest.json'
def sha256_stream(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''):h.update(block)
    return h.hexdigest()
inputs_sha = {str(p):sha256_stream(p) for p in SOURCES}
base_id = hashlib.sha256(json.dumps({'input_hashes':inputs_sha,'seed':SEED,'n_random':N_RANDOM,
                                    'alpha':RIDGE_ALPHA,'masks':MASKS,'horizons':HORIZONS},
                                   sort_keys=True).encode('utf-8')).hexdigest()
all_B=np.zeros((N_RANDOM,N,N*L),dtype=np.float64)
all_c=np.zeros((N_RANDOM,N),dtype=np.float64)
all_v=np.zeros((N_RANDOM,N),dtype=np.float64)
completed=0
if progress_file.exists() and progress_meta.exists():
    meta=json.loads(progress_meta.read_text(encoding='utf-8'))
    if meta.get('run_id')!=base_id:
        raise RuntimeError('Existing random checkpoint belongs to a DIFFERENT input/configuration; STOP.')
    with np.load(progress_file,allow_pickle=False) as z:
        completed=int(z['completed'].item())
        all_B[:completed]=z['B'][:completed]
        all_c[:completed]=z['c'][:completed]
        all_v[:completed]=z['var'][:completed]
    assert 0<=completed<=N_RANDOM
    print(f'Loaded random-model checkpoint: {completed}/{N_RANDOM} complete')

def save_progress(n):
    tmp=CKPT/'random_progress.tmp.npz'
    np.savez_compressed(tmp,completed=np.array(n),B=all_B[:n],c=all_c[:n],var=all_v[:n])
    tmp.replace(progress_file)
    progress_meta.write_text(json.dumps({'run_id':base_id,'completed':n},indent=2),encoding='utf-8')

# Train ONLY; no D8–D10 metric is accessed during graph fitting or graph sampling.
for r in range(completed,N_RANDOM):
    all_B[r],all_c[r],all_v[r] = fit_graph(random_graphs[r])
    if (r+1)%10==0 or r==N_RANDOM-1:
        save_progress(r+1)
        print(f'Fitted target+lag-matched random graphs: {r+1}/{N_RANDOM}')

# Compute evaluation outputs. These graphs cannot be selected/retrained based on results.
transition_rows,state_rows,outcome_rows = [],[],[]
for name,fit in core.items():
    t,s,o=score_model(name,fit,actual_slots if name=='FROZEN_CAUSALLOB' else ())
    transition_rows.extend(t);state_rows.extend(s);outcome_rows.extend(o)
FOCAL_MASKS={key:MASKS[key] for key in ('MISS_RETURN_AND_OFI','HISTORY_ONLY')}
# Resume-safe evaluation checkpoint: keeps already scored random graphs on restart.
EV_CKPT=CKPT/'evaluation_progress'
EV_CKPT.mkdir(exist_ok=True)
EV_META=EV_CKPT/'manifest.json'
EV_FILES={k:EV_CKPT/f'{k}.csv' for k in ('transition','state','outcome')}
finished_eval=0
if EV_META.exists():
    em=json.loads(EV_META.read_text())
    assert em['run_id']==base_id and em['focal_masks']==list(FOCAL_MASKS)
    finished_eval=int(em['completed'])
    assert 0<=finished_eval<=N_RANDOM and all(f.is_file() for f in EV_FILES.values())
    prev={key:pd.read_csv(path) for key,path in EV_FILES.items()}
    for k,frame in prev.items():
        expected={f'RANDOM_MATCHED_{r:03d}' for r in range(finished_eval)}
        assert set(frame.model.unique())==expected,(k,'incomplete/corrupt evaluation checkpoint')
    transition_rows.extend(prev['transition'].to_dict('records'))
    state_rows.extend(prev['state'].to_dict('records'))
    outcome_rows.extend(prev['outcome'].to_dict('records'))
    print(f'Resumed random evaluation: {finished_eval}/{N_RANDOM}')

def save_eval(n):
    expected={f'RANDOM_MATCHED_{r:03d}' for r in range(n)}
    for key,rows in [('transition',transition_rows),('state',state_rows),('outcome',outcome_rows)]:
        frame=pd.DataFrame(rows)
        frame=frame[frame.model.isin(expected)]
        tmp=EV_FILES[key].with_name(EV_FILES[key].name+'.tmp')
        frame.to_csv(tmp,index=False,float_format='%.14g')
        tmp.replace(EV_FILES[key])
    manifest_tmp=EV_META.with_name(EV_META.name+'.tmp')
    manifest_tmp.write_text(json.dumps({'run_id':base_id,'completed':n,
        'focal_masks':list(FOCAL_MASKS)},sort_keys=True))
    manifest_tmp.replace(EV_META)

for r in range(finished_eval,N_RANDOM):
    g=random_graphs[r]
    t,s,o=score_model(f'RANDOM_MATCHED_{r:03d}',(all_B[r],all_c[r],all_v[r]),g,scenarios=FOCAL_MASKS)
    transition_rows.extend(t);state_rows.extend(s);outcome_rows.extend(o)
    if (r+1)%10==0 or r==N_RANDOM-1:
        save_eval(r+1)
        print(f'Evaluated and checkpointed random graph {r+1}/{N_RANDOM}')

transition_df=pd.DataFrame(transition_rows)
state_df=pd.DataFrame(state_rows)
outcome_df=pd.DataFrame(outcome_rows)
# CSV resume reads numeric day keys as strings, while fresh results carry ints.
# Canonicalize once before grouping: never mix integer 8 with text '8'.
for frame in (transition_df,state_df,outcome_df):
    frame['day']=frame['day'].astype(str)
# Cross-experiment parity is mandatory. The score definitions and row identities
# must reproduce Step7's original frozen-AR5 and CausalLOB diagonal evidence.
s7states=pd.read_csv(S7_STATES)
s7outcomes=pd.read_csv(S7_OUTCOMES)
for graph_name,s7_name in [('FROZEN_AR5','AR5__DIAG_FROZEN'),
                           ('FROZEN_CAUSALLOB','SPARSE_39__DIAG_FROZEN')]:
    for scenario in MASKS:
        for d in ['POOLED']+[str(z) for z in EVAL_DAYS]:
            q=state_df[(state_df.model==graph_name)&(state_df.scenario==scenario)&
                       (state_df.day.astype(str)==d)]
            r=s7states[(s7states.model==s7_name)&(s7states.scenario==scenario)&
                       (s7states.day.astype(str)==d)]
            assert len(q)==len(r)==1
            for a,b in [('state_rmse','state_rmse'),('state_nll','state_joint_nll'),
                        ('coverage90','coverage90')]:
                assert abs(float(q.iloc[0][a])-float(r.iloc[0][b]))<3e-8,(graph_name,scenario,d,a)
            for h in HORIZONS:
                q2=outcome_df[(outcome_df.model==graph_name)&(outcome_df.scenario==scenario)&
                              (outcome_df.horizon==h)&(outcome_df.day.astype(str)==d)]
                r2=s7outcomes[(s7outcomes.model==s7_name)&(s7outcomes.scenario==scenario)&
                              (s7outcomes.horizon==h)&(s7outcomes.day.astype(str)==d)]
                assert len(q2)==len(r2)==1
                for a,b in [('log_loss','log_loss'),('brier','brier_multiclass'),
                            ('balanced_accuracy','balanced_accuracy'),('macro_f1','macro_f1')]:
                    assert abs(float(q2.iloc[0][a])-float(r2.iloc[0][b]))<3e-8,(graph_name,scenario,h,d,a)
print('STEP7↔STEP8 PARITY PASS: state and h10/h20 outcome metrics for both frozen models.')

state_df['coverage90_abs_error'] = (state_df['coverage90'] - 0.90).abs()
outcome_df['balanced_accuracy_error'] = 1.0-outcome_df['balanced_accuracy']
outcome_df['macro_f1_error'] = 1.0-outcome_df['macro_f1']
transition_df.to_csv(RESULTS/'01_transition_all_models.csv',index=False)
state_df.to_csv(RESULTS/'02_missing_state_all_models.csv',index=False)
outcome_df.to_csv(RESULTS/'03_outcome_all_models.csv',index=False)

# Tail rank is an empirical random-reference diagnostic, NOT a valid causal-discovery p-value:
# the observed graph was previously selected on D1–D7 and D8–D10 were historically inspected.
def reference_summary(df,fields,keys,filename):
    rows=[]
    for key,frame in df.groupby(keys,sort=False):
        k=key if isinstance(key,tuple) else (key,)
        sel=dict(zip(keys,k))
        null=frame[frame.model.str.startswith('RANDOM_MATCHED_')]
        ref=frame[frame.model=='FROZEN_CAUSALLOB']
        ar=frame[frame.model=='FROZEN_AR5']
        dense=frame[frame.model=='DENSE_VAR5']
        assert len(ref)==len(ar)==len(dense)==1 and len(null)==N_RANDOM, 'Missing comparison row; STOP.'
        for field in fields:
            observed=float(ref.iloc[0][field])
            vals=null[field].to_numpy(dtype=float)
            rows.append(dict(**sel,metric=field,causallob=observed,ar5=float(ar.iloc[0][field]),
                             dense_var5=float(dense.iloc[0][field]),
                             null_median=float(np.median(vals)),null_q05=float(np.quantile(vals,.05)),
                             null_q95=float(np.quantile(vals,.95)),
                             fraction_random_worse=float(np.mean(vals>observed)),
                             sparse_minus_ar5_gain=float(ar.iloc[0][field]-observed),
                             sparse_minus_dense_gain=float(dense.iloc[0][field]-observed),
                             n_random=N_RANDOM))
    out=pd.DataFrame(rows)
    out.to_csv(RESULTS/filename,index=False)
    return out

trans_summary=reference_summary(transition_df,['transition_rmse','transition_joint_nll','return_rmse','return_nll'],
                                ['day'],'04_graph_null_transition_summary.csv')
state_summary=reference_summary(state_df[state_df.scenario.isin(FOCAL_MASKS)],['state_rmse','state_nll','coverage90_abs_error'],
                                ['scenario','day'],'05_graph_null_missing_state_summary.csv')
out_summary=reference_summary(outcome_df[outcome_df.scenario.isin(FOCAL_MASKS)],['log_loss','brier','balanced_accuracy_error','macro_f1_error'],
                              ['scenario','horizon','day'],'06_graph_null_outcome_summary.csv')

# Static-vs-temporal decomposition using error metrics: positive gain improves.
decomposition=[]
for sc in MASKS:
    for d in ['POOLED']+[str(z) for z in EVAL_DAYS]:
        s=state_df[(state_df.scenario==sc)&(state_df.day.astype(str)==d)].set_index('model')
        for base,challenge,label in [
            ('STATIC_DIAG_TRAIN_ONLY','FROZEN_AR5','HISTORY_VS_STATIC'),
            ('FROZEN_AR5','FROZEN_CAUSALLOB','SPARSE_EDGES_VS_AR5'),
            ('STATIC_DIAG_TRAIN_ONLY','FROZEN_CAUSALLOB','FULL_TEMPORAL_VS_STATIC'),
            ('DENSE_VAR5','FROZEN_CAUSALLOB','SPARSE_VS_DENSE'),
        ]:
            decomposition.append({'scenario':sc,'horizon':0,'day':d,'comparison':label,
                'state_rmse_gain':float(s.loc[base,'state_rmse']-s.loc[challenge,'state_rmse']),
                'state_nll_gain':float(s.loc[base,'state_nll']-s.loc[challenge,'state_nll']),
                'outcome_logloss_gain':np.nan,'outcome_brier_gain':np.nan})
            for h in HORIZONS:
                o=outcome_df[(outcome_df.scenario==sc)&(outcome_df.horizon==h)&
                             (outcome_df.day.astype(str)==d)].set_index('model')
                decomposition.append({'scenario':sc,'horizon':h,'day':d,'comparison':label,
                    'state_rmse_gain':np.nan,'state_nll_gain':np.nan,
                    'outcome_logloss_gain':float(o.loc[base,'log_loss']-o.loc[challenge,'log_loss']),
                    'outcome_brier_gain':float(o.loc[base,'brier']-o.loc[challenge,'brier'])})
decomp=pd.DataFrame(decomposition)
decomp.to_csv(RESULTS/'09_static_dynamic_decomposition.csv',index=False,float_format='%.12g')

# Save exact sampled graphs in a stable, audit-friendly format.
graph_rows=[]
for source,target,lag in actual_slots:
    graph_rows.append(dict(graph='FROZEN_CAUSALLOB',source=names[source],target=names[target],lag=lag))
for r,g in enumerate(random_graphs):
    for source,target,lag in g:
        graph_rows.append(dict(graph=f'RANDOM_MATCHED_{r:03d}',source=names[source],target=names[target],lag=lag))
pd.DataFrame(graph_rows).to_csv(RESULTS/'07_all_graph_slots.csv',index=False)

manifest={
    'version':'STEP8_GRAPH_SPECIFIC_READONLY_v1_2026-09-18',
    'run_id':base_id,'step6_hash':frozen_run_hash,'original_source_hashes':inputs_sha,'train_days':TRAIN_DAYS,
    'evaluation_days':EVAL_DAYS,'n_train_rows':len(tr_idx),'n_eval_rows':len(te_idx),
    'lag_order':L,'variables':names,'ridge_alpha':RIDGE_ALPHA,
    'original_primary':'FROZEN_CAUSALLOB; diagonal Q; static Gaussian is additional train-only control',
    'comparison_models':['STATIC_DIAG_TRAIN_ONLY','FROZEN_AR5','FROZEN_CAUSALLOB','DENSE_VAR5'],
    'n_random':N_RANDOM,'random_seed':SEED,'random_graph_reference_masks':list(FOCAL_MASKS),'frozen_model_masks':list(MASKS),
    'random_null':'Distinct nine-slot graphs exactly matched on cross-parent count per (target,lag); randomize sources excluding self.',
    'null_caveat':'Diagnostic conditional graph reference only: graph selection on D1–D7 is not replicated for random graphs, so no valid structure-selection p-value.',
    'reference_orientation':'All summary metrics are errors: smaller better. balanced_accuracy_error=1-BA; macro_f1_error=1-F1; coverage90_abs_error=abs(coverage-0.90).',
    'historical_caveat':'D8–D10 historically inspected: NOT researcher-blind external validation; no model selection or hyperparameter tuning here.',
    'conditional_inference':'Same frozen ordered-probit D1–D7 heads; diagonal innovation covariances fitted on train for new graphs. Missing-value masks, not interventions.',
    'label_end_purge':'h10 and h20 require >=h later representations in same inferred sequence.',
    'n_structural_slots':{'Static':0,'AR5':30,'CausalLOB':39,'DenseVAR5':180},
    'frozen_m3_reproduction_max_error':checks,
    'note':'Only new-model coefficients estimated on D1–D7; no M2/M3 original artifacts altered. Three test days provide descriptive sign checks only.',
}
(RESULTS/'08_graph_value_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')

print('\n'+'='*112)
print('GRAPH-SPECIFIC VALUE AUDIT COMPLETE (post-hoc, not external validation)')
print('Train:',len(tr_idx),' | Eval:',len(te_idx),' | Random graphs:',N_RANDOM,
      ' | Runtime:',round(time.monotonic()-T0,1),'sec')
print('\nPOOLED TRANSITION: lower errors better; fraction_random_worse closer to 1 means sparse beats more random graphs')
display(trans_summary[trans_summary.day.astype(str)=='POOLED'].round(6))
print('\nFOCAL MISSING-STATE RMSE:')
display(state_summary[(state_summary.day.astype(str)=='POOLED') &
                      state_summary.scenario.isin(['MISS_RETURN_AND_OFI','HISTORY_ONLY']) &
                      (state_summary.metric=='state_rmse')].round(6))
print('\nFOCAL h10/h20 OUTCOME LOG LOSS:')
display(out_summary[(out_summary.day.astype(str)=='POOLED') &
                     out_summary.scenario.isin(['MISS_RETURN_AND_OFI','HISTORY_ONLY']) &
                     (out_summary.metric=='log_loss')].round(6))
print('\nSaved nine results to:',RESULTS)
print('\nStatic/history/edge decomposition — pooled focal state gains:')
print(decomp[(decomp.day=='POOLED')&(decomp.horizon==0)&
             (decomp.scenario.isin(['MISS_RETURN_AND_OFI','HISTORY_ONLY']))][
             ['scenario','comparison','state_rmse_gain','state_nll_gain']].round(6).to_string(index=False))

Google Drive already mounted.
STEP8 PREFLIGHT PASS: frozen graph/head, D1-D7 fitting, D8-D10 previously inspected; nothing modified upstream.
Source Step6 hash: 9831f8e1db702848b8588a535088f9392c32e9a09b66eb5cafe40fc6d4bd1d8c  | rows: 253875 139212
Exact frozen-Step6 reproduction diagnostics: {'AR5_B': 1.0880185641326534e-14, 'AR5_c': 5.879452198535862e-15, 'AR5_var': 3.2899238888717264e-12, 'SPARSE_B': 1.0880185641326534e-14, 'SPARSE_c': 6.582625070028492e-15, 'SPARSE_var': 3.2901459334766514e-12}
Fitted target+lag-matched random graphs: 10/100
Fitted target+lag-matched random graphs: 20/100
Fitted target+lag-matched random graphs: 30/100
Fitted target+lag-matched random graphs: 40/100
Fitted target+lag-matched random graphs: 50/100
Fitted target+lag-matched random graphs: 60/100
Fitted target+lag-matched random graphs: 70/100
Fitted target+lag-matched random graphs: 80/100
Fitted target+lag-matched random graphs: 90/100
Fitted target+lag-matched random graphs: 100/100
Evaluated and c

,day,metric,causallob,ar5,dense_var5,null_median,null_q05,null_q95,fraction_random_worse,sparse_minus_ar5_gain,sparse_minus_dense_gain,n_random
0,POOLED,transition_rmse,0.586137,0.588347,0.585171,0.586529,0.586058,0.587859,0.85,0.002209,-0.000966,100
1,POOLED,transition_joint_nll,3.712316,3.725577,3.705322,3.714656,3.712059,3.722787,0.90,0.013261,-0.006994,100
2,POOLED,return_rmse,0.824720,0.833930,0.822980,0.826415,0.824574,0.831965,0.93,0.009211,-0.001739,100
3,POOLED,return_nll,1.251983,1.265020,1.249820,1.254092,1.251983,1.262273,0.95,0.013038,-0.002163,100



FOCAL MISSING-STATE RMSE:


,scenario,day,metric,causallob,ar5,dense_var5,null_median,null_q05,null_q95,fraction_random_worse,sparse_minus_ar5_gain,sparse_minus_dense_gain,n_random
0,MISS_RETURN_AND_OFI,POOLED,state_rmse,0.830538,0.835290,0.828739,0.831430,0.830383,0.834259,0.88,0.004752,-0.001799,100
12,HISTORY_ONLY,POOLED,state_rmse,0.586137,0.588347,0.585171,0.586529,0.586058,0.587859,0.85,0.002209,-0.000966,100



FOCAL h10/h20 OUTCOME LOG LOSS:


,scenario,horizon,day,metric,causallob,ar5,dense_var5,null_median,null_q05,null_q95,fraction_random_worse,sparse_minus_ar5_gain,sparse_minus_dense_gain,n_random
0,MISS_RETURN_AND_OFI,10,POOLED,log_loss,1.045064,1.053098,1.042070,1.046045,1.043646,1.051617,0.76,0.008034,-0.002993,100
16,MISS_RETURN_AND_OFI,20,POOLED,log_loss,1.060077,1.065426,1.057136,1.060856,1.058791,1.065006,0.71,0.005350,-0.002941,100
32,HISTORY_ONLY,10,POOLED,log_loss,1.044335,1.052438,1.041350,1.045323,1.042931,1.050936,0.76,0.008104,-0.002984,100
48,HISTORY_ONLY,20,POOLED,log_loss,1.053335,1.059340,1.050446,1.054184,1.052095,1.058741,0.72,0.006005,-0.002889,100



Saved nine results to: /content/drive/MyDrive/CausalLOB/STEP8_GRAPH_SPECIFIC_READONLY/results

Static/history/edge decomposition — pooled focal state gains:
           scenario              comparison  state_rmse_gain  state_nll_gain
MISS_RETURN_AND_OFI       HISTORY_VS_STATIC         0.003510        0.008282
MISS_RETURN_AND_OFI     SPARSE_EDGES_VS_AR5         0.004752        0.013698
MISS_RETURN_AND_OFI FULL_TEMPORAL_VS_STATIC         0.008262        0.021981
MISS_RETURN_AND_OFI         SPARSE_VS_DENSE        -0.001799       -0.004722
       HISTORY_ONLY       HISTORY_VS_STATIC         0.360443        4.488561
       HISTORY_ONLY     SPARSE_EDGES_VS_AR5         0.002209        0.013261
       HISTORY_ONLY FULL_TEMPORAL_VS_STATIC         0.362652        4.501822
       HISTORY_ONLY         SPARSE_VS_DENSE        -0.000966       -0.006994


In [ ]:
# CausalLOB Step 9 — multi-step missing-observation stress test, read-only upstream.
# GYP self-brief: Try to falsify the practical value of the frozen 39-slot sparse
# temporal graph. Unlike Step 7's one-step mask with exact history, propagate the
# full 30-dimensional Gaussian filtering distribution through consecutive missing
# observations. Keep the same endpoints for gap lengths 1/5/20; all graph, heads,
# covariance variants, missing masks, event stride and labels fixed in this file.
# Check against frozen one-step Step 7 and dense Step 8; report prior-only losses,
# adverse calibration, and any dense-model advantage. Never tune on D8–D10,
# claim causal effects, infer execution costs, or call D8–D10 new external validation.

import os, json, time, hashlib, sys, importlib.util, subprocess
from pathlib import Path

for module,package in [('scipy','scipy'),('sklearn','scikit-learn')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import numpy as np
import pandas as pd
from scipy.special import ndtr
from scipy.stats import norm
from sklearn.linear_model import Ridge
from sklearn.metrics import balanced_accuracy_score, f1_score
try:
    from IPython.display import display
except ImportError:
    display=print
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('Google Drive already mounted.')
except ImportError:
    pass

T0=time.monotonic()
ROOT=Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
S2=ROOT/'STEP2_REBUILD/checkpoints'
S4=ROOT/'STEP4_VALIDATION_FROZEN/checkpoints'
S6=ROOT/'STEP6_FROZEN_DBN'
S7=ROOT/'STEP7_MISSING_STATE_READONLY'
S8=ROOT/'STEP8_GRAPH_SPECIFIC_READONLY'
OUT=ROOT/'STEP9_GAP_STRESS_READONLY'
CKPT=OUT/'checkpoints'
RESULTS=OUT/'results'
VERSION='STEP9_GAP_STRESS_v1_2026-09-18'
MASK_HASH='eb38b17c1431c0a98d46ed1c792953c932c58572799b64aee79165dfc58439da'
FEATURES=['log_relative_spread','log_total_depth_l5','imbalance_l1',
          'imbalance_l2_l5','quote_ofi_depth_norm','log_mid_return_1']
N,L,D=6,5,30
TRAIN_DAYS=(1,2,3,4,5,6,7)
EVAL_DAYS=(8,9,10)
GAPS=(1,5,20)
HORIZONS=(10,20)
STRIDE=200
OFFSET=25
MAX_GAP=max(GAPS)
HMAX=max(HORIZONS)
MASKS={'MISS_RETURN_AND_OFI':(4,5),'HISTORY_ONLY':tuple(range(N))}
SOURCE={
 'temporal':S2/'04_temporal_primary.npz',
 'mask':S4/'PRIMARY_FROZEN_PARENT_MASK.npz',
 'state':S6/'checkpoints/01_FROZEN_STATE_SPACE.npz',
 'step6_summary':S6/'results/STEP6_FINAL_SUMMARY.json',
 'step7_summary':S7/'results/STEP7_FINAL_SUMMARY.json',
 'step7_covariances':S7/'checkpoints/01_TRAIN_ONLY_COVARIANCES.npz',
 'step8_manifest':S8/'results/08_graph_value_manifest.json',
 'step8_transitions':S8/'results/01_transition_all_models.csv',
}
for h in HORIZONS:
    SOURCE[f'head_h{h}']=S6/f'checkpoints/outcome_heads/ordered_probit_h{h}.json'
    SOURCE[f'pred_h{h}']=S6/f'results/03_PREDICTED_CLASS_PROBABILITIES_h{h}.npz'
    for sc in MASKS:
        SOURCE[f'step7_one_step_{sc}_h{h}']=(S7/'results/focal_probabilities'/
            f'SPARSE_39__DIAG_FROZEN__{sc}__h{h}.npz')
missing=[f'{name}: {path}' for name,path in SOURCE.items() if not path.is_file()]
if missing:
    raise FileNotFoundError('Step 9 requires completed Steps 2, 4, 6, 7 AND 8. Missing:\n'+'\n'.join(missing))

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''):
            h.update(block)
    return h.hexdigest()

def atomic_csv(path,df):
    tmp=path.with_name(path.name+'.tmp')
    df.to_csv(tmp,index=False,float_format='%.14g')
    tmp.replace(path)

def atomic_json(path,obj):
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,sort_keys=True,allow_nan=False)+'\n',encoding='utf8')
    tmp.replace(path)

def assert_spd(S,label):
    assert np.allclose(S,S.T,atol=2e-10),f'{label}: asymmetric covariance'
    eig=np.linalg.eigvalsh(S)
    assert np.isfinite(eig).all() and eig.min()>1e-11,(label,eig.min())

# A 30-dimensional lagged-state filter: S_t=(X_t,X_{t-1},...,X_{t-4}).
# The filter starts from five *actually observed* previous states before the gap.
# No later observations enter the filtering pass.
def build_state_space(B,c,Q6):
    assert B.shape==(N,N*L) and c.shape==(N,) and Q6.shape==(N,N)
    F=np.zeros((D,D),dtype=np.float64)
    F[:N,:]=B
    for k in range(1,L):
        F[k*N:(k+1)*N,(k-1)*N:k*N]=np.eye(N)
    off=np.zeros(D);off[:N]=c
    Q=np.zeros((D,D));Q[:N,:N]=Q6
    return F,off,Q

def kalman_update(m,P,observed,observations):
    """Batch means; shared covariance. Exact noiseless observation update."""
    obs=np.asarray(observed,dtype=int)
    if not len(obs):
        return m,P
    Poo=P[np.ix_(obs,obs)]
    assert_spd(Poo,'observed predictive covariance')
    K=np.linalg.solve(Poo,P[obs,:]).T
    m=m+(observations-m[:,obs])@K.T
    m[:,obs]=observations  # noiseless evidence, preserve exact observation
    P=P-K@P[obs,:]
    P=(P+P.T)/2
    P[obs,:]=0
    P[:,obs]=0
    assert np.isfinite(m).all() and np.isfinite(P).all()
    return m,P

def filter_gap(B,c,Q6,standardized,endpoints,gap,missing):
    """Online filtering, with a missing mask at *every* step of an outage."""
    F,off,Q=build_state_space(B,c,Q6)
    begins=endpoints-gap+1
    initial=np.concatenate([standardized[begins-j] for j in range(1,L+1)],axis=1)
    assert initial.shape==(len(endpoints),D)
    m=initial
    P=np.zeros((D,D),dtype=np.float64)
    observed=np.asarray([j for j in range(N) if j not in missing],dtype=int)
    for k in range(gap):
        m=m@F.T+off
        P=F@P@F.T+Q
        P=(P+P.T)/2
        evidence=standardized[begins+k][:,observed]
        m,P=kalman_update(m,P,observed,evidence)
        if len(observed):
            assert np.array_equal(m[:,observed],evidence)
    miss=np.asarray(missing,dtype=int)
    assert_spd(P[np.ix_(miss,miss)],'endpoint missing-state covariance')
    return m[:,:N],P[:N,:N]

def outcome_probs(m,S,head):
    beta=np.asarray(head['beta'],dtype=float)
    low,high=map(float,head['thresholds'])
    assert low<high
    extra=float(beta@S@beta)
    assert extra>=-1e-9
    scale=np.sqrt(1+max(0,extra))
    eta=m@beta
    p0=ndtr((low-eta)/scale)
    upto=ndtr((high-eta)/scale)
    p=np.column_stack((p0,upto-p0,1-upto))
    assert np.isfinite(p).all() and p.min()>-1e-10
    p=np.maximum(p,1e-15)
    p/=p.sum(axis=1,keepdims=True)
    return p,extra

def score_state(x,m,S,missing):
    miss=np.asarray(missing,dtype=int)
    V=S[np.ix_(miss,miss)]
    assert_spd(V,'scored missing covariance')
    err=x[:,miss]-m[:,miss]
    sign,ld=np.linalg.slogdet(V)
    assert sign>0
    quad=np.einsum('ij,ij->i',err,np.linalg.solve(V,err.T).T)
    nll=float(np.mean(.5*(len(miss)*np.log(2*np.pi)+ld+quad)))
    z=norm.ppf(.95)
    cov=np.abs(err)<=z*np.sqrt(np.diag(V))
    return dict(state_rmse=float(np.sqrt(np.mean(err**2))),
                state_nll=nll,coverage90=float(cov.mean()),
                abs_coverage_error90=float(abs(cov.mean()-.9)),
                mean_interval_width90=float(np.mean(2*z*np.sqrt(np.diag(V)))))

def score_outcome(y,p):
    pred=p.argmax(axis=1)
    conf=p.max(axis=1)
    right=pred==y
    edges=np.linspace(0,1,16)
    ece=0.
    for i in range(15):
        mask=(conf>=edges[i]) & ((conf<=edges[i+1]) if i==14 else (conf<edges[i+1]))
        if mask.any():
            ece+=mask.mean()*abs(right[mask].mean()-conf[mask].mean())
    return dict(n=len(y),accuracy=float(right.mean()),
                balanced_accuracy=float(balanced_accuracy_score(y,pred)),
                macro_f1=float(f1_score(y,pred,average='macro',labels=[0,1,2],zero_division=0)),
                log_loss=float(-np.mean(np.log(np.maximum(p[np.arange(len(y)),y],1e-15)))),
                brier=float(np.mean(np.sum((p-np.eye(3)[y])**2,axis=1))),
                ece_confidence=float(ece))

# Mathematical self-tests, including Kalman conditioning and propagated covariance.
def self_test():
    rng=np.random.default_rng(5162026)
    A=rng.normal(size=(N,N))
    Q=A@A.T+np.eye(N)*.5
    mp=rng.normal(size=(13,D))
    P=np.zeros((D,D));P[:N,:N]=Q
    known=[0,1,2,3]
    values=rng.normal(size=(13,len(known)))
    m,p=kalman_update(mp.copy(),P.copy(),known,values)
    K=Q[np.ix_([4,5],known)]@np.linalg.inv(Q[np.ix_(known,known)])
    expected=mp[:,[4,5]]+(values-mp[:,known])@K.T
    expected_var=Q[np.ix_([4,5],[4,5])]-K@Q[np.ix_(known,[4,5])]
    assert np.allclose(m[:,[4,5]],expected,rtol=0,atol=1e-11)
    assert np.allclose(p[np.ix_([4,5],[4,5])],expected_var,atol=1e-11)
    assert np.array_equal(m[:,known],values)
    assert np.array_equal(p[known,:],np.zeros((len(known),D)))
    B=np.zeros((N,D));B[:,0:N]=np.eye(N)*.5
    c=np.zeros(N)
    x=rng.normal(size=(55,N))
    ends=np.array([21,27,35,47],dtype=int)
    actual1,P1=filter_gap(B,c,Q,x,ends,1,(4,5))
    prior=np.concatenate([x[ends-k] for k in range(1,L+1)],axis=1)@B.T+c
    exp1=prior[:,[4,5]]+(x[ends][:,known]-prior[:,known])@K.T
    assert np.allclose(actual1[:,[4,5]],exp1,atol=1e-11)
    assert np.allclose(P1[np.ix_([4,5],[4,5])],expected_var,atol=1e-11)
    all1,V=filter_gap(B,c,Q,x,ends,1,tuple(range(N)))
    assert np.array_equal(all1,prior) and np.array_equal(V,Q)
    all2,V2=filter_gap(B,c,Q,x,ends,2,tuple(range(N)))
    F,off,QQ=build_state_space(B,c,Q)
    P2=F@QQ@F.T+QQ
    assert np.allclose(V2,P2[:N,:N],atol=1e-11)
    assert np.isfinite(all2).all()
    head={'beta':[.2,.1,.3,-.1,.5,1.2],'thresholds':[-.6,.7]}
    pp,ex=outcome_probs(all2,V2,head)
    assert np.allclose(pp.sum(1),1,atol=1e-12) and ex>=0
    print('SELF-TEST PASS: exact conditioning, noiseless measurements, one-step parity, two-step uncertainty propagation, probabilities.')

self_test()

# Verify fingerprints BEFORE scoring. Only STEP9_GAP_STRESS_READONLY may be written.
sums={key:sha256(path) for key,path in SOURCE.items()}
s6=json.loads(SOURCE['step6_summary'].read_text())
s7=json.loads(SOURCE['step7_summary'].read_text())
s8=json.loads(SOURCE['step8_manifest'].read_text())
assert s6['frozen_mask_hash']==MASK_HASH and s6['structural_slots']==39
assert s6['model_selection_after_step4'] is False
assert s6['historical_researcher_blind_assessment'] is False
assert s7['step6_run_hash']==s6['run_hash'] and s7['frozen_mask_hash']==MASK_HASH
assert s7['complete_evidence_exact_parity_with_step6']
assert s8['step6_hash']==s6['run_hash'] and s8['n_random']==100
assert s8['n_structural_slots']['CausalLOB']==39 and s8['n_structural_slots']['DenseVAR5']==180

with np.load(SOURCE['temporal'],allow_pickle=False) as z:
    assert str(z['checkpoint_version'].item())=='STEP2_REBUILD_v1_2026-09-17'
    assert z['feature_names'].astype(str).tolist()==FEATURES
    X=z['X_current'].astype(np.float64)
    Y=z['Y'].astype(np.int8)
    day=z['day'].astype(np.int8)
    seq=z['sequence_id'].astype(np.int16)
    lagvalid=z['full_lag_valid'].astype(bool)
with np.load(SOURCE['mask'],allow_pickle=False) as z:
    frozen_mask=z['parent_mask'].copy()
    assert str(z['mask_hash'].item())==MASK_HASH
with np.load(SOURCE['state'],allow_pickle=False) as z:
    assert str(z['run_hash'].item())==s6['run_hash']
    assert np.array_equal(z['frozen_parent_mask'],frozen_mask)
    mean=z['train_mean'].astype(float)
    std=z['train_std'].astype(float)
    B_ar=z['B_ar'].astype(float);c_ar=z['c_ar'].astype(float);v_ar=z['v_ar'].astype(float)
    B_sp=z['B_sparse'].astype(float);c_sp=z['c_sparse'].astype(float);v_sp=z['v_sparse'].astype(float)
    assert np.array_equal(z['Q_sparse'][:6,:6],np.diag(v_sp))
    assert np.array_equal(z['Q_ar'][:6,:6],np.diag(v_ar))
with np.load(SOURCE['step7_covariances'],allow_pickle=False) as z:
    assert str(z['run_hash'].item())==s7['run_hash']
    Q_shrink=z['Q_sparse_shrink'].astype(float)
    assert np.array_equal(z['Q_sparse_diag'],np.diag(v_sp))
assert_spd(Q_shrink,'train-fitted sensitivity covariance')
heads={}
priors={}
step7_reference={}
for h in HORIZONS:
    head=json.loads(SOURCE[f'head_h{h}'].read_text())
    assert head['run_hash']==s6['run_hash'] and head['converged']
    assert head['feature_names']==FEATURES
    heads[h]=head
    with np.load(SOURCE[f'pred_h{h}'],allow_pickle=False) as z:
        assert str(z['run_hash'].item())==s6['run_hash']
        priors[h]=z['prior_down_neutral_up'].astype(float)
    assert priors[h].shape==(3,) and np.isclose(priors[h].sum(),1) and (priors[h]>0).all()
    for sc in MASKS:
        with np.load(SOURCE[f'step7_one_step_{sc}_h{h}'],allow_pickle=False) as z:
            assert str(z['run_hash'].item())==s7['run_hash']
            step7_reference[(sc,h)]=(z['row_index'].copy(),z['p_down_neutral_up'].copy())
assert X.shape==(393337,N) and Y.shape==(393337,5)
assert np.isfinite(X).all() and np.isin(Y,[1,2,3]).all()
assert frozen_mask.shape==(6,6,6) and int(frozen_mask.sum())==39
assert sum(frozen_mask[j,j,1:].sum() for j in range(N))==30
assert (std>0).all()
XZ=(X-mean)/std
starts=np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]])
ends=np.r_[starts[1:],len(X)]
assert len(starts)==50
remaining=np.empty(len(X),dtype=np.int32)
train=[];eval_=[];event_ends=[]
for a,b in zip(starts,ends):
    assert b-a>110 and (seq[a:b]==seq[a]).all() and (day[a:b]==day[a]).all()
    assert np.array_equal(lagvalid[a:b],np.r_[np.zeros(5,bool),np.ones(b-a-5,bool)])
    remaining[a:b]=b-1-np.arange(a,b)
    rows=np.arange(a+L,b)
    if int(day[a]) in TRAIN_DAYS: train.append(rows)
    if int(day[a]) in EVAL_DAYS:
        eval_.append(rows)
        endpoint_start=a+L+OFFSET+MAX_GAP-1
        event_ends.extend(np.arange(endpoint_start,b-HMAX,STRIDE).tolist())
train=np.concatenate(train)
eval_=np.concatenate(eval_)
endpoint=np.asarray(event_ends,dtype=np.int64)
assert len(train)==253875 and len(eval_)==139212 and len(endpoint)>50
assert np.isin(day[endpoint],EVAL_DAYS).all()
assert (remaining[endpoint]>=HMAX).all()
assert (lagvalid[endpoint-MAX_GAP+1]).all()
assert (seq[endpoint-MAX_GAP+1]==seq[endpoint]).all()
assert (seq[endpoint+HMAX]==seq[endpoint]).all()
assert np.max(np.abs(X[np.isin(day,TRAIN_DAYS)].mean(0)-mean))<1e-10
assert np.max(np.abs(X[np.isin(day,TRAIN_DAYS)].std(0)-std))<1e-10

# Dense baseline is fitted with the SAME Step 8 ridge specification, train only.
Ztr=np.concatenate([XZ[train-j] for j in range(1,L+1)],axis=1)
Ytr=XZ[train]
B_dense=np.zeros((N,D));c_dense=np.zeros(N);v_dense=np.zeros(N)
for j in range(N):
    reg=Ridge(alpha=1.,fit_intercept=True).fit(Ztr,Ytr[:,j])
    B_dense[j]=reg.coef_;c_dense[j]=reg.intercept_
    err=Ytr[:,j]-reg.predict(Ztr)
    v_dense[j]=np.mean(err*err)
assert (v_dense>1e-11).all()
step8_transition=pd.read_csv(SOURCE['step8_transitions'])
ref=step8_transition[(step8_transition.model=='DENSE_VAR5') &
                     (step8_transition.day.astype(str)=='POOLED')]
assert len(ref)==1,'Step 8 dense reference missing; STOP.'
Zev=np.concatenate([XZ[eval_-j] for j in range(1,L+1)],axis=1)
e=(XZ[eval_]-(Zev@B_dense.T+c_dense))
rmse_dense=float(np.sqrt(np.mean(e*e)))
nll_dense=float(np.mean(np.sum(.5*(np.log(2*np.pi*v_dense)+e*e/v_dense),axis=1)))
assert abs(rmse_dense-float(ref.iloc[0].transition_rmse))<3e-7,(rmse_dense,ref.iloc[0].transition_rmse)
assert abs(nll_dense-float(ref.iloc[0].transition_joint_nll))<3e-7,(nll_dense,ref.iloc[0].transition_joint_nll)
print('STEP8 DENSE PARITY PASS; RMSE',round(rmse_dense,7),'joint NLL',round(nll_dense,7))

# Reverify train-only label priors, with Step 6's exact conservative tail purge.
for h in HORIZONS:
    eligible_train=train[remaining[train]>=h]
    y=(3-Y[eligible_train,(10,20,30,50,100).index(h)]).astype(int)
    calc=np.bincount(y,minlength=3)/len(y)
    assert np.array_equal(calc,priors[h]),f'h{h} train prior mismatch!'

models={
 'STATIC_DIAG_TRAIN':(np.zeros((N,D)),Ytr.mean(0),np.diag(np.mean((Ytr-Ytr.mean(0))**2,axis=0))),
 'AR5_DIAG_FROZEN':(B_ar,c_ar,np.diag(v_ar)),
 'CAUSALLOB_DIAG_FROZEN':(B_sp,c_sp,np.diag(v_sp)),
 'DENSE_VAR5_DIAG':(B_dense,c_dense,np.diag(v_dense)),
 'CAUSALLOB_SHRINK_SENSITIVITY':(B_sp,c_sp,Q_shrink),
}
for name,(B,c,Q) in models.items():
    assert B.shape==(6,30) and c.shape==(6,) and Q.shape==(6,6)
    assert_spd(Q,name)

spec={'version':VERSION,'input_sha256':sums,'step6_run_hash':s6['run_hash'],
      'step7_run_hash':s7['run_hash'],'step8_run_id':s8['run_id'],
      'gap_lengths':GAPS,'masks':MASKS,'endpoint_stride':STRIDE,
      'endpoint_initial_offset':OFFSET,'horizon_purge':HMAX,
      'horizons':HORIZONS,'endpoint_design':'same endpoints for 1/5/20; all initial five history states exact',
      'models':list(models),'dense_alpha':1.,'output':'endpoint state posterior + h10/h20 frozen heads',
      'evaluation_caveat':'D8–D10 historically inspected; fixed retrospective diagnostic; no p values or tuning'}
run_id=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()
CKPT.mkdir(parents=True,exist_ok=True);RESULTS.mkdir(parents=True,exist_ok=True)
immutable=CKPT/'IMMUTABLE_STEP9_SPEC.json'
if immutable.exists():
    old=json.loads(immutable.read_text())
    assert old['run_id']==run_id,'Existing Step9 results have different inputs/settings! STOP.'
else:
    atomic_json(immutable,{'run_id':run_id,'spec':spec})
print('STEP 9 PREFLIGHT PASS: fixed graph/heads, same endpoints, D1–D7-only dense refit, D8–D10 assessment.')
print('Endpoint events:',len(endpoint),'by day:',{d:int(np.sum(day[endpoint]==d)) for d in EVAL_DAYS})
print('Run ID:',run_id)

output_names=['01_gap_state_metrics.csv','02_gap_outcome_metrics.csv',
              '03_gap_contrasts.csv','04_day_consistency.csv',
              '05_gap_event_manifest.csv','06_endpoint_probability_examples.csv',
              'STEP9_FINAL_SUMMARY.json']
summary_path=RESULTS/output_names[-1]
if summary_path.exists():
    saved=json.loads(summary_path.read_text())
    assert saved['run_id']==run_id
    assert all((RESULTS/name).is_file() for name in output_names)
    print('STEP 9 ALREADY COMPLETE — VERIFIED MATCHING FINGERPRINT; NO REPROCESSING.')
    contrast=pd.read_csv(RESULTS/'03_gap_contrasts.csv')
    display(contrast[(contrast.day=='POOLED') &
                     (contrast.comparison.isin(['SPARSE_VS_AR5','SPARSE_VS_DENSE','SPARSE_VS_PRIOR']))][
                     ['scenario','gap','horizon','comparison','state_rmse_gain','outcome_logloss_gain']].round(6))
    print('Results:',RESULTS)
else:
    events=pd.DataFrame({'endpoint_row':endpoint,'day':day[endpoint],
                         'sequence_id':seq[endpoint],'remaining_rows':remaining[endpoint]})
    for gap in GAPS:
        events[f'begin_gap_{gap}']=endpoint-gap+1
    atomic_csv(RESULTS/'05_gap_event_manifest.csv',events)
    states=[];outcomes=[];examples=[]
    truth=XZ[endpoint]
    groups={'POOLED':np.ones(len(endpoint),dtype=bool)}
    groups.update({str(d):day[endpoint]==d for d in EVAL_DAYS})
    for scenario,missing in MASKS.items():
        for gap in GAPS:
            for name,(B,c,Q6) in models.items():
                if name=='STATIC_DIAG_TRAIN':
                    # Identical current unconditional Gaussian at every step, no memory.
                    # This equals filtering an F with a zero top transition row.
                    pass
                post,V=filter_gap(B,c,Q6,XZ,endpoint,gap,missing)
                if gap==1 and name=='CAUSALLOB_DIAG_FROZEN':
                    exact=np.concatenate([XZ[endpoint-j] for j in range(1,L+1)],axis=1)@B_sp.T+c_sp
                    if len(missing)==N:
                        # Both compute the same one-step Gaussian mean, but BLAS/augmented
                        # state matrix multiplications can differ by a few ulps. Bitwise
                        # equality is unjustified: check a tight NUMERICAL tolerance.
                        mean_err=float(np.max(np.abs(post-exact)))
                        cov_err=float(np.max(np.abs(V-Q6)))
                        assert np.allclose(post,exact,rtol=2e-12,atol=2e-12), (
                            'One-step mean parity failed',mean_err)
                        assert np.allclose(V,Q6,rtol=2e-12,atol=2e-12), (
                            'One-step covariance parity failed',cov_err)
                        print('ONE-STEP HISTORY-ONLY NUMERICAL PARITY PASS;'
                              'mean max abs error:',mean_err,
                              'covariance max abs error:',cov_err)
                    else:
                        observed=[j for j in range(N) if j not in missing]
                        assert np.array_equal(post[:,observed],XZ[endpoint][:,observed])
                        assert np.allclose(post[:,missing],exact[:,missing],atol=1e-12,rtol=0)
                        assert np.allclose(V[np.ix_(missing,missing)],Q6[np.ix_(missing,missing)],atol=1e-12)
                for label,keep in groups.items():
                    vals=score_state(truth[keep],post[keep],V,missing)
                    states.append({'model':name,'scenario':scenario,'gap':gap,'day':label,
                                   'n_events':int(keep.sum()),'n_missing':len(missing),**vals})
                for h in HORIZONS:
                    P,extra=outcome_probs(post,V,heads[h])
                    y=(3-Y[endpoint,(10,20,30,50,100).index(h)]).astype(int)
                    if gap==1 and name=='CAUSALLOB_DIAG_FROZEN':
                        # Numerical parity with original one-step inference on these rows.
                        diag=Q6[np.ix_(missing,missing)]
                        s=np.zeros((6,6));s[np.ix_(missing,missing)]=diag
                        oldp,_=outcome_probs(post,s,heads[h])
                        assert np.array_equal(P,oldp)
                        rix,refprob=step7_reference[(scenario,h)]
                        loc=np.searchsorted(rix,endpoint)
                        assert (loc<len(rix)).all() and np.array_equal(rix[loc],endpoint)
                        parity_error=float(np.max(np.abs(P-refprob[loc])))
                        assert parity_error<2e-11,('Step7 one-step parity FAILED',scenario,h,parity_error)
                    for label,keep in groups.items():
                        met=score_outcome(y[keep],P[keep])
                        outcomes.append({'model':name,'scenario':scenario,'gap':gap,
                            'horizon':h,'day':label,'latent_variance_extra':extra,**met})
                    if name=='CAUSALLOB_DIAG_FROZEN' and gap in (1,MAX_GAP):
                        for k in range(min(5,len(endpoint))):
                            examples.append({'scenario':scenario,'gap':gap,'horizon':h,
                                'endpoint_row':int(endpoint[k]),'day':int(day[endpoint[k]]),
                                'true_label':int(y[k]),'P_Down':float(P[k,0]),
                                'P_Neutral':float(P[k,1]),'P_Up':float(P[k,2]),
                                'latent_extra_variance':extra})
                print(f'Finished {name:32s} {scenario:21s} gap={gap:2d}')
    # A prior-only baseline on EXACTLY the same events; identical under either outage mask.
    for scenario in MASKS:
        for gap in GAPS:
            for h in HORIZONS:
                y=(3-Y[endpoint,(10,20,30,50,100).index(h)]).astype(int)
                P=np.broadcast_to(priors[h],(len(endpoint),3))
                for label,keep in groups.items():
                    met=score_outcome(y[keep],P[keep])
                    outcomes.append({'model':'TRAIN_ONLY_CLASS_PRIOR','scenario':scenario,'gap':gap,
                                     'horizon':h,'day':label,'latent_variance_extra':np.nan,**met})
    state_df=pd.DataFrame(states)
    outcome_df=pd.DataFrame(outcomes)
    assert len(state_df)==len(models)*len(MASKS)*len(GAPS)*len(groups)
    assert len(outcome_df)==(len(models)+1)*len(MASKS)*len(GAPS)*len(HORIZONS)*len(groups)
    atomic_csv(RESULTS/'01_gap_state_metrics.csv',state_df)
    atomic_csv(RESULTS/'02_gap_outcome_metrics.csv',outcome_df)
    atomic_csv(RESULTS/'06_endpoint_probability_examples.csv',pd.DataFrame(examples))
    comparisons=[('HISTORY_VS_STATIC','STATIC_DIAG_TRAIN','AR5_DIAG_FROZEN'),
                 ('SPARSE_VS_AR5','AR5_DIAG_FROZEN','CAUSALLOB_DIAG_FROZEN'),
                 ('SPARSE_VS_DENSE','DENSE_VAR5_DIAG','CAUSALLOB_DIAG_FROZEN'),
                 ('SHRINK_SENSITIVITY','CAUSALLOB_DIAG_FROZEN','CAUSALLOB_SHRINK_SENSITIVITY'),
                 ('SPARSE_VS_PRIOR', 'TRAIN_ONLY_CLASS_PRIOR','CAUSALLOB_DIAG_FROZEN')]
    comp=[]
    for scenario in MASKS:
        for gap in GAPS:
            for label in groups:
                for title,base,test in comparisons:
                    aa=state_df[(state_df.model==base)&(state_df.scenario==scenario)&
                                (state_df.gap==gap)&(state_df.day==label)]
                    bb=state_df[(state_df.model==test)&(state_df.scenario==scenario)&
                                (state_df.gap==gap)&(state_df.day==label)]
                    if base=='TRAIN_ONLY_CLASS_PRIOR':
                        state_gain=np.nan;state_nll_gain=np.nan
                    else:
                        assert len(aa)==len(bb)==1
                        state_gain=float(aa.iloc[0].state_rmse-bb.iloc[0].state_rmse)
                        state_nll_gain=float(aa.iloc[0].state_nll-bb.iloc[0].state_nll)
                    for h in HORIZONS:
                        oa=outcome_df[(outcome_df.model==base)&(outcome_df.scenario==scenario)&
                                      (outcome_df.gap==gap)&(outcome_df.horizon==h)&(outcome_df.day==label)]
                        ob=outcome_df[(outcome_df.model==test)&(outcome_df.scenario==scenario)&
                                      (outcome_df.gap==gap)&(outcome_df.horizon==h)&(outcome_df.day==label)]
                        assert len(oa)==len(ob)==1
                        comp.append({'scenario':scenario,'gap':gap,'horizon':h,'day':label,'comparison':title,
                                     'state_rmse_gain':state_gain,'state_nll_gain':state_nll_gain,
                                     'outcome_logloss_gain':float(oa.iloc[0].log_loss-ob.iloc[0].log_loss),
                                     'outcome_brier_gain':float(oa.iloc[0].brier-ob.iloc[0].brier),
                                     'outcome_ba_gain':float(ob.iloc[0].balanced_accuracy-oa.iloc[0].balanced_accuracy)})
    contrast=pd.DataFrame(comp)
    atomic_csv(RESULTS/'03_gap_contrasts.csv',contrast)
    consistency=[]
    for (sc,gap,h,comparison),f in contrast[contrast.day!='POOLED'].groupby(
        ['scenario','gap','horizon','comparison'],sort=False):
        assert set(f.day)=={'8','9','10'}
        gains=f['outcome_logloss_gain'].to_numpy()
        consistency.append({'scenario':sc,'gap':gap,'horizon':h,'comparison':comparison,
                            'days_positive_logloss_gain':int((gains>0).sum()),
                            'day8_gain':float(f[f.day=='8'].outcome_logloss_gain.iloc[0]),
                            'day9_gain':float(f[f.day=='9'].outcome_logloss_gain.iloc[0]),
                            'day10_gain':float(f[f.day=='10'].outcome_logloss_gain.iloc[0]),
                            'n_days':3})
    atomic_csv(RESULTS/'04_day_consistency.csv',pd.DataFrame(consistency))
    summary={'version':VERSION,'run_id':run_id,'status':'MULTISTEP_GAP_FILTERING_COMPLETE',
      'source_step6_run_hash':s6['run_hash'],'n_endpoint_events':len(endpoint),
      'events_by_day':{str(d):int(np.sum(day[endpoint]==d)) for d in EVAL_DAYS},
      'gap_lengths_representation_steps':list(GAPS),'scenarios':list(MASKS),
      'horizon_raw_messages':list(HORIZONS),
      'model_primary':'CAUSALLOB_DIAG_FROZEN',
      'sensitivity':'CAUSALLOB_SHRINK_SENSITIVITY is NOT original independent-CPD DBN',
      'dense_baseline_verified_step8_parity':True,
      'one_step_exact_saved_step7_probability_parity':True,
      'train_only_prior_baseline':True,
      'all_models_same_endpoints_each_gap':True,
      'no_future_observations_in_filtering':True,
      'historically_blind_external_test':False,
      'limitations':['Only three historically exposed evaluation days.',
         'Synthetic missingness patterns, not measured production outages.',
         'No online latency, order fills, fees or net-profit evidence.',
         'The frozen outcome heads were fitted using exact current states; missingness was not part of head fitting.',
         'One-step benchmark differs from full-sequence Step 7 only by fixed subsetting of endpoints.',
         'Compare forecasts to train-only prior; a negative gain means no useful probabilistic forecast.'],
      'outputs':output_names[:-1]}
    atomic_json(summary_path,summary)
    print('\n'+'='*115)
    print('STEP 9 COMPLETE — EXACT ONLINE FILTERING THROUGH CONSECUTIVE MISSING REPRESENTATION STEPS')
    print('='*115)
    print('Fixed matched endpoints:',len(endpoint),' by day:',summary['events_by_day'])
    print('POOLED state comparisons:')
    state_table=contrast[(contrast.day=='POOLED')&(contrast.horizon==10)&
                         (contrast.comparison.isin(['SPARSE_VS_AR5','SPARSE_VS_DENSE','SHRINK_SENSITIVITY']))]
    display(state_table[['scenario','gap','comparison','state_rmse_gain','state_nll_gain']].round(6))
    print('\nPOOLED outcome comparisons (negative prior gain means below train-only class prior):')
    display(contrast[(contrast.day=='POOLED')&
        (contrast.comparison.isin(['SPARSE_VS_AR5','SPARSE_VS_DENSE','SPARSE_VS_PRIOR']))][
        ['scenario','gap','horizon','comparison','outcome_logloss_gain','outcome_brier_gain']].round(6))
    print('Outputs:',RESULTS,'Runtime:',round(time.monotonic()-T0,1),'seconds')


# ==================================================================================================
# STEPS 10–11 — Integrity-checked synthesis and transparent final research package.
# This section runs after Step 9, in the SAME single notebook cell.
# GYP self-brief: test every comparison on matched rows; report all adverse results; reproduce
# originals without tuning; distinguish completed retrospective course analysis from an
# externally validated, financially profitable or NeurIPS-ready study.
# ==================================================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import zipfile as _zipfile
from datetime import datetime, timezone

FINAL=ROOT/'STEP10_11_FINAL_RESEARCH_PACKAGE'
FRES=FINAL/'results'
FFIG=FINAL/'figures'
for pp in (FRES, FFIG): pp.mkdir(parents=True,exist_ok=True)

# Source files must exist; never rebuild or change any prior artifact here.
INFILES={
 'step2_raw':S2/'01_canonical_raw.npz',
 'step2_temporal':S2/'04_temporal_primary.npz',
 'step4_mask':S4/'PRIMARY_FROZEN_PARENT_MASK.npz',
 'step4_summary':ROOT/'STEP4_VALIDATION_FROZEN/results/STEP4_FINAL_SUMMARY.json',
 'step5_summary':ROOT/'STEP5_LABEL_AUDIT_READONLY/results/05_LABEL_AUDIT_SUMMARY.json',
 'step6_state':SOURCE['state'],
 'step6_summary':SOURCE['step6_summary'],
 'step6_metrics':S6/'results/04_OUTCOME_METRICS_D8_D10.csv',
 'step7_summary':SOURCE['step7_summary'],
 'step7_state':S7/'results/02_EXACT_STATE_METRICS.csv',
 'step7_outcomes':S7/'results/03_EXACT_OUTCOME_METRICS.csv',
 'step8_manifest':SOURCE['step8_manifest'],
 'step8_transition':SOURCE['step8_transitions'],
 'step8_null_transition':S8/'results/04_graph_null_transition_summary.csv',
 'step8_null_state':S8/'results/05_graph_null_missing_state_summary.csv',
 'step8_null_outcome':S8/'results/06_graph_null_outcome_summary.csv',
 'step8_decomposition':S8/'results/09_static_dynamic_decomposition.csv',
 'step9_summary':RESULTS/'STEP9_FINAL_SUMMARY.json',
 'step9_states':RESULTS/'01_gap_state_metrics.csv',
 'step9_outcomes':RESULTS/'02_gap_outcome_metrics.csv',
 'step9_comparisons':RESULTS/'03_gap_contrasts.csv',
 'step9_days':RESULTS/'04_day_consistency.csv',
 'step9_events':RESULTS/'05_gap_event_manifest.csv',
}
missing_files=[f'{key}: {file}' for key,file in INFILES.items() if not file.is_file()]
if missing_files:
    raise FileNotFoundError('Final package cannot claim completeness: missing required existing outputs:\n'+'\n'.join(missing_files))
source_hash_before={name:sha256(path) for name,path in INFILES.items()}
S4SUMMARY=json.loads(INFILES['step4_summary'].read_text())
S5SUMMARY=json.loads(INFILES['step5_summary'].read_text())
S9SUMMARY=json.loads(INFILES['step9_summary'].read_text())
assert S9SUMMARY['status']=='MULTISTEP_GAP_FILTERING_COMPLETE'
assert S9SUMMARY['run_id']==run_id
assert S9SUMMARY['source_step6_run_hash']==s6['run_hash']
assert S9SUMMARY['events_by_day']=={'8':277,'9':260,'10':160}
assert S9SUMMARY['all_models_same_endpoints_each_gap']
assert S9SUMMARY['one_step_exact_saved_step7_probability_parity']
assert S9SUMMARY['no_future_observations_in_filtering']
assert s8['n_random']==100, 'The final graph audit must contain 100 random graphs.'
assert s8['step6_hash']==s6['run_hash'] and S5SUMMARY['frozen_graph_unchanged']
assert S5SUMMARY['old_M1_comparison']['raw_exact']
assert S5SUMMARY['old_M1_comparison']['labels_exact']
assert S5SUMMARY['old_M1_comparison']['sequence_exact']
assert S5SUMMARY['status']=='AUDIT_COMPLETED_WITH_LABEL_PROVENANCE_LIMITATION'
assert s6['historical_researcher_blind_assessment'] is False
assert s7['D8_D10_historically_researcher_blind'] is False
assert s6['model_selection_after_step4'] is False
assert s8['n_structural_slots']=={'Static':0,'AR5':30,'CausalLOB':39,'DenseVAR5':180}
with np.load(INFILES['step2_raw'],allow_pickle=False) as rawz:
    assert rawz['X_raw'].shape==(394337,40) and rawz['Y'].shape==(394337,5)
with np.load(INFILES['step2_temporal'],allow_pickle=False) as tz:
    d=tz['day'].copy(); seq_check=tz['sequence_id'].copy()
    assert len(d)==393337 and len(np.unique(seq_check))==50
    assert np.array_equal(np.unique(d),np.arange(1,11))
    days_count={str(v):int(np.sum(d==v)) for v in np.arange(1,11)}

m6=pd.read_csv(INFILES['step6_metrics'])
m9=pd.read_csv(INFILES['step9_outcomes'])
s9=pd.read_csv(INFILES['step9_states'])
c9=pd.read_csv(INFILES['step9_comparisons'])
d9=pd.read_csv(INFILES['step9_days'])
g8=pd.read_csv(INFILES['step8_null_transition'])
g8state=pd.read_csv(INFILES['step8_null_state'])
g8out=pd.read_csv(INFILES['step8_null_outcome'])
events=pd.read_csv(INFILES['step9_events'])
assert len(events)==697 and set(events.day)=={8,9,10}
assert events.endpoint_row.is_unique
assert all(events['begin_gap_'+str(g)].ge(0).all() for g in GAPS)
assert np.array_equal(events.day.value_counts().sort_index().values,[277,260,160])
assert set(m9['model'])==set(models)|{'TRAIN_ONLY_CLASS_PRIOR'}
assert set(m9['scenario'])==set(MASKS) and set(m9['gap'])==set(GAPS)
assert set(m9['horizon'])==set(HORIZONS)
assert set(s9['model'])==set(models)
assert set(s9['scenario'])==set(MASKS) and set(s9['gap'])==set(GAPS)
assert len(s9)==len(models)*len(MASKS)*len(GAPS)*4
assert len(m9)==(len(models)+1)*len(MASKS)*len(GAPS)*len(HORIZONS)*4
assert not m9[['accuracy','balanced_accuracy','log_loss','brier','ece_confidence']].isna().any().any()
assert not s9[['state_rmse','state_nll','coverage90','mean_interval_width90']].isna().any().any()
assert m9['n'].gt(0).all()
assert np.all((s9.coverage90>=0)&(s9.coverage90<=1))
assert s9['mean_interval_width90'].gt(0).all()
assert np.allclose(m9.groupby(['scenario','gap','horizon','model']).n.sum().values / 2,697), 'Pooled and per-day row counts disagree'
assert (m9[m9.day=='POOLED'].n==697).all()
assert (s9[s9.day=='POOLED'].n_events==697).all()
assert set(c9.comparison)=={'HISTORY_VS_STATIC','SPARSE_VS_AR5','SPARSE_VS_DENSE','SHRINK_SENSITIVITY','SPARSE_VS_PRIOR'}
assert set(g8.n_random)=={100}

# Fair full-observation comparisons must be on the same horizon-specific cohort.
pooled6=m6[m6.day.astype(str)=='POOLED'].copy()
full_records=[]
for h in (10,20,30,50,100):
    block=pooled6[pooled6.horizon_raw_messages==h]
    prior=block[block.model=='PRIOR_ONLY']
    head=block[block.model=='ORDERED_PROBIT_SPARSE_DBN_HEAD']
    assert len(prior)==len(head)==1
    r=head.iloc[0]; p=prior.iloc[0]
    assert int(r.n)==int(p.n)
    full_records.append({'horizon_underlying_events':h,
         'horizon_representation_steps':{10:1,20:2,30:3,50:5,100:10}[h],
         'n_matched':int(r.n),
         'probit_accuracy':float(r.accuracy),'probit_balanced_accuracy':float(r.balanced_accuracy),
         'probit_macro_f1':float(r.macro_f1),'probit_logloss':float(r.log_loss),
         'probit_brier':float(r.brier_multiclass),'probit_ece':float(r.ece_confidence),
         'prior_accuracy':float(p.accuracy),'prior_logloss':float(p.log_loss),
         'prior_brier':float(p.brier_multiclass),
         'prior_minus_probit_logloss':float(p.log_loss-r.log_loss),
         'prior_minus_probit_brier':float(p.brier_multiclass-r.brier_multiclass)})
full_df=pd.DataFrame(full_records)
assert np.array_equal(full_df.horizon_underlying_events.to_numpy(),np.array([10,20,30,50,100]))

# Recompute all comparisons from original state/outcome tables, not trusting only a printed gain.
for comparison,base,candidate in [
 ('HISTORY_VS_STATIC','STATIC_DIAG_TRAIN','AR5_DIAG_FROZEN'),
 ('SPARSE_VS_AR5','AR5_DIAG_FROZEN','CAUSALLOB_DIAG_FROZEN'),
 ('SPARSE_VS_DENSE','DENSE_VAR5_DIAG','CAUSALLOB_DIAG_FROZEN'),
 ('SHRINK_SENSITIVITY','CAUSALLOB_DIAG_FROZEN','CAUSALLOB_SHRINK_SENSITIVITY'),
 ('SPARSE_VS_PRIOR','TRAIN_ONLY_CLASS_PRIOR','CAUSALLOB_DIAG_FROZEN')]:
    check=c9[c9.comparison==comparison]
    for _,r in check.iterrows():
        w=(m9.scenario==r.scenario)&(m9.gap==r.gap)&(m9.horizon==r.horizon)&(m9.day==r.day)
        x=m9[w & (m9.model==base)]; y=m9[w & (m9.model==candidate)]
        assert len(x)==len(y)==1 and int(x.iloc[0].n)==int(y.iloc[0].n)
        delta=float(x.iloc[0].log_loss-y.iloc[0].log_loss)
        assert abs(delta-r.outcome_logloss_gain)<2e-10,(comparison,r.scenario,r.gap,r.horizon,delta,r.outcome_logloss_gain)

# Step 9 uses deliberately spaced endpoints; never call it a full 139,212-row gap experiment.
assert len(endpoint)==697 and len(eval_)==139212
horizons_looked_at=sorted(set(m9.horizon.tolist()))
primary_pooled=m9[(m9.day=='POOLED')&m9.model.isin(['CAUSALLOB_DIAG_FROZEN','TRAIN_ONLY_CLASS_PRIOR','AR5_DIAG_FROZEN','DENSE_VAR5_DIAG'])].copy()
pivot=primary_pooled.pivot_table(index=['scenario','gap','horizon'],columns='model',values=['log_loss','brier','balanced_accuracy','ece_confidence'])
master=[]
for sc in MASKS:
    for g in GAPS:
        for h in HORIZONS:
            piece=primary_pooled[(primary_pooled.scenario==sc)&(primary_pooled.gap==g)&(primary_pooled.horizon==h)]
            assert len(piece)==4
            vals=piece.set_index('model')
            a=vals.loc['CAUSALLOB_DIAG_FROZEN']
            p=vals.loc['TRAIN_ONLY_CLASS_PRIOR']
            ar=vals.loc['AR5_DIAG_FROZEN']
            dense=vals.loc['DENSE_VAR5_DIAG']
            s=s9[(s9.day=='POOLED')&(s9.scenario==sc)&(s9.gap==g)&(s9.model=='CAUSALLOB_DIAG_FROZEN')].iloc[0]
            master.append({'scenario':sc,'gap_representation_steps':g,
               'horizon_underlying_events':h,'n_matched_endpoints':int(a.n),
               'causallob_state_rmse':float(s.state_rmse),
               'causallob_state_nll':float(s.state_nll),
               'causallob_coverage90':float(s.coverage90),
               'causallob_interval_width90':float(s.mean_interval_width90),
               'causallob_logloss':float(a.log_loss),
               'prior_logloss':float(p.log_loss),'ar5_logloss':float(ar.log_loss),
               'dense_logloss':float(dense.log_loss),
               'prior_minus_causallob_logloss':float(p.log_loss-a.log_loss),
               'ar5_minus_causallob_logloss':float(ar.log_loss-a.log_loss),
               'dense_minus_causallob_logloss':float(dense.log_loss-a.log_loss),
               'causallob_brier':float(a.brier), 'prior_brier':float(p.brier),
               'causallob_balanced_accuracy':float(a.balanced_accuracy),
               'causallob_ece':float(a.ece_confidence),
               'beats_training_prior_on_logloss':bool(a.log_loss < p.log_loss)})
master_df=pd.DataFrame(master)
assert len(master_df)==2*3*2
assert master_df.n_matched_endpoints.eq(697).all()

# Structural results from 100 genuinely evaluated target/lag-matched random graphs.
trow=g8[(g8.day.astype(str)=='POOLED')&(g8.metric=='transition_rmse')]
assert len(trow)==1
tr=trow.iloc[0]
trans8_raw=pd.read_csv(INFILES['step8_transition'])
for model,field in [('FROZEN_CAUSALLOB','causallob'),('FROZEN_AR5','ar5'),('DENSE_VAR5','dense_var5')]:
    rr=trans8_raw[(trans8_raw.model==model)&(trans8_raw.day.astype(str)=='POOLED')]
    assert len(rr)==1 and abs(float(rr.iloc[0].transition_rmse)-float(tr[field]))<2e-10, (
        'Step8 graph summary disagrees with raw model scores',model)
assert 0<=tr.fraction_random_worse<=1
assert float(tr.ar5)>float(tr.causallob) and float(tr.dense_var5)<float(tr.causallob)
struct={'n_candidate_lag_slots':180,'frozen_self_slots':30,'frozen_cross_slots':9,
  'frozen_total_slots':39,'dense_total_slots':180,'sparse_fraction_of_dense':39/180,
  'transition_rmse_sparse':float(tr.causallob),'transition_rmse_ar5':float(tr.ar5),
  'transition_rmse_dense':float(tr.dense_var5),
  'fraction_100_matched_random_graphs_worse':float(tr.fraction_random_worse),
  'matched_random_median_rmse':float(tr.null_median),
  'interpretation':'Random graph ranking is descriptive, not a test of causal identification or selection-adjusted significance.'}

# Explicit evidence ledger and completion gates: do not infer missing experiments occurred.
ledger=[
 ('Dataset integrity','PASS','All 10 FI-2010 days reconstructed, 394337 raw rows; original M1 parity and no publisher engineered inputs.'),
 ('Boundary-safe chronology','PASS','50 inferred day/security sequences, independent boundary audit, 5 lag validity.'),
 ('Graph discovery','PASS_WITH_CAUSAL_SCOPE','D1–D5 only; 59 candidate lag slots, retained 39 for primary.'),
 ('Graph validation','INTERNAL_ONLY','D6–D7 family falsification; historically observed in earlier work.'),
 ('Label provenance','LIMITED','Publisher labels unchanged; exact underlying-event formula not reconstructed.'),
 ('Gaussian DBN and five heads','PASS_RETROSPECTIVE','Trained D1–D7, retrospectively scored D8–D10, not blind external test.'),
 ('Exact one-step missing evidence','PASS_RETROSPECTIVE','Analytic posterior verified against complete-evidence outputs.'),
 ('Graph-specific value','MIXED','100 target/lag-matched random graphs; dense VAR slightly lower transition error.'),
 ('Consecutive-gap stress','PASS_RETROSPECTIVE','697 fixed spaced endpoints; 2 masks, gaps 1/5/20, h10/h20; conditional posterior.'),
 ('DeepLOB CPU benchmark','UNVERIFIED','Completion and test metrics not demonstrated in attached data; integrate only if independent artifact supplied.'),
 ('Independent external validation','NOT_DONE','D8–D10 were inspected before; a new market/day dataset needed.'),
 ('Trading profitability after costs','NOT_DONE','No true execution, fees, latency, fill, adverse selection or PnL simulation.'),
 ('NeurIPS-level new method','PROPOSED_ONLY','Graph uncertainty / nonstationary regime method requires implementation and external replication.'),
]
ledger_df=pd.DataFrame(ledger,columns=['item','status','evidence_or_gap'])

# Figures: distinct plots without misleading confidence bands from 697 correlated endpoints.
plt.figure(figsize=(8.8,4.8))
for sc,grp in master_df[master_df.horizon_underlying_events==10].groupby('scenario',sort=False):
    plt.plot(grp.gap_representation_steps,grp.prior_minus_causallob_logloss,marker='o',label=sc.replace('_',' ').title())
plt.axhline(0,linewidth=.8,linestyle='--')
plt.xlabel('Consecutive missing representation steps')
plt.ylabel('Training prior minus CausalLOB log loss (positive = improvement)')
plt.title('Fixed-endpoint missingness stress: h10')
plt.xticks(list(GAPS));plt.legend(fontsize=8);plt.tight_layout()
fig1=FFIG/'01_prior_comparison_h10.png';plt.savefig(fig1,dpi=160);plt.close()

plt.figure(figsize=(8.8,4.8))
for sc,grp in s9[(s9.day=='POOLED') & (s9.model.isin(['AR5_DIAG_FROZEN','CAUSALLOB_DIAG_FROZEN','DENSE_VAR5_DIAG']))].groupby('scenario',sort=False):
    for model,sub in grp.groupby('model',sort=False):
        plt.plot(sub.gap,sub.state_rmse,marker='o',label=sc.replace('_',' ')+': '+model.replace('_DIAG_FROZEN','').replace('_DIAG',''))
plt.xlabel('Consecutive missing representation steps');plt.ylabel('Standardized missing-state RMSE')
plt.title('Matched-endpoint missing-state reconstruction')
plt.xticks(list(GAPS));plt.legend(fontsize=7,loc='best');plt.tight_layout()
fig2=FFIG/'02_gap_state_rmse.png';plt.savefig(fig2,dpi=160);plt.close()

plt.figure(figsize=(8.8,4.8))
plt.plot(full_df.horizon_underlying_events,full_df.probit_logloss,marker='o',label='Frozen ordered probit')
plt.plot(full_df.horizon_underlying_events,full_df.prior_logloss,marker='s',label='Training-only class prior')
plt.xlabel('Publisher horizon (underlying messages)');plt.ylabel('Multiclass log loss')
plt.title('Full-observation retrospective evaluation: matched rows')
plt.xticks([10,20,30,50,100]);plt.legend();plt.tight_layout()
fig3=FFIG/'03_full_observation_horizon_logloss.png';plt.savefig(fig3,dpi=160);plt.close()

# Optional DeepLOB artifact: do NOT scan broad Drive or substitute training progress for final results.
# This is a manifest check only. A separate properly verified benchmark is required in the paper.
deep_root=ROOT/'DEEPLOB_CPU_BASELINE'
deep_status='NOT_VERIFIED'
deep_notes='Previous h10 epoch progress is not a final validation-selected D8–D10 test result.'
for candidate in (deep_root/'results/FINAL_SUMMARY.json', deep_root/'results/DEEPlOB_FINAL_SUMMARY.json',
                  deep_root/'checkpoints/DONE.json'):
    if candidate.is_file():
        deep_status='ARTIFACT_FOUND_REQUIRES_METRIC_AND_SPLIT_REVIEW'
        deep_notes='Candidate file found: '+str(candidate)+'; not automatically equated with a fair benchmark.'
        break
ledger_df.loc[ledger_df.item=='DeepLOB CPU benchmark','status']=deep_status
ledger_df.loc[ledger_df.item=='DeepLOB CPU benchmark','evidence_or_gap']=deep_notes

atomic_csv(FRES/'01_evidence_ledger.csv',ledger_df)
atomic_csv(FRES/'02_full_observation_all_horizons.csv',full_df)
atomic_csv(FRES/'03_matched_gap_results.csv',master_df)
atomic_csv(FRES/'04_gap_day_consistency.csv',d9)
atomic_json(FRES/'05_graph_evidence.json',struct)

# Reproducible self-critique: evidence gates are computed, not manually cherry-picked.
prior_beats=int((master_df.prior_minus_causallob_logloss<0).sum())
prior_ties=int(np.isclose(master_df.prior_minus_causallob_logloss,0,atol=1e-10).sum())
shr=s9[(s9.day=='POOLED')&(s9.model=='CAUSALLOB_SHRINK_SENSITIVITY')].sort_values(['scenario','gap'])
prm=s9[(s9.day=='POOLED')&(s9.model=='CAUSALLOB_DIAG_FROZEN')].sort_values(['scenario','gap'])
assert len(shr)==len(prm)==6
sensitivity_better=int((prm.state_rmse.to_numpy()>shr.state_rmse.to_numpy()).sum())
claims={
 'all_ten_days_loaded':True,'all_ten_days_used_to_fit_graph':False,
 'discovery_days':[1,2,3,4,5],'internal_validation_days':[6,7],
 'parametric_training_days':[1,2,3,4,5,6,7],
 'retrospective_evaluation_days':[8,9,10],
 'evaluation_researcher_blind':False,
 'n_raw_rows':394337,'n_state_rows':393337,'n_full_lag_rows':393087,
 'n_transition_train':253875,'n_transition_assessment':139212,
 'n_gap_endpoints':697,'gap_endpoint_sample_fraction':697/139212,
 'n_gap_prior_beats_causallob_logloss':prior_beats,
 'n_gap_prior_ties_causallob_logloss':prior_ties,
 'n_gap_model_comparisons':len(master_df),
 'gap_covariance_sensitivity_lower_rmse_scenarios':sensitivity_better,
 'missingness_naturally_observed':False,
 'net_profit_evaluated':False,
 'observational_graph_identified_causal':False,
 'matched_random_graph_selection_adjusted_test':False,
 'independent_external_data_validation':False,
 'deep_lob_status':deep_status,
 'research_software_core_complete':True,
 'journal_or_neurips_publication_claim_supported':False,
}
atomic_json(FRES/'06_claims_and_completion_gates.json',claims)

# Write a report using the computed results, with explicit distinction between measured and proposed work.
floatstr=lambda v: f'{float(v):.6f}'
full_rows='\n'.join(
 f"| {int(r.horizon_underlying_events)} | {int(r.n_matched):,} | {floatstr(r.probit_balanced_accuracy)} | {floatstr(r.probit_logloss)} | {floatstr(r.prior_logloss)} | {floatstr(r.prior_minus_probit_logloss)} |"
 for _,r in full_df.iterrows())
gap_rows='\n'.join(
 f"| {r.scenario.replace('_',' ')} | {int(r.gap_representation_steps)} | {int(r.horizon_underlying_events)} | {floatstr(r.causallob_state_rmse)} | {floatstr(r.causallob_logloss)} | {floatstr(r.prior_logloss)} | {floatstr(r.prior_minus_causallob_logloss)} |"
 for _,r in master_df.iterrows())
proof_samples=int(len(m9[m9.day=='POOLED'])/len(MASKS)/len(GAPS)/len(HORIZONS))
report=f'''# CausalLOB: A Sparse Temporal Graphical Model for Partial-Observation Inference in Limit Order Books

**Research-status manuscript draft — dated {datetime.now(timezone.utc).strftime('%Y-%m-%d UTC')}.**
**Status:** retrospective computational study completed through Steps 2–11; new external validation, completed DeepLOB benchmarking and trading-cost evidence remain outstanding. This document is not a peer-reviewed publication.

## Abstract

We study probabilistic inference in a six-dimensional representation of high-frequency limit-order-book states using a sparsely parameterized, fifth-order Gaussian temporal model. LPCMCI is applied to Days 1–5 of FI-2010 to obtain observational lagged dependency candidates. Internal validation uses Days 6–7; model parameters and ordinal price-state heads are estimated on Days 1–7. Days 8–10 provide a historically exposed retrospective assessment, not an independent researcher-blind test. The frozen model has 39 lagged coefficients versus 180 in a dense fifth-order vector autoregression. Its transition RMSE is {floatstr(tr.causallob)} versus {floatstr(tr.ar5)} for AR(5) and {floatstr(tr.dense_var5)} for dense VAR(5). It has smaller error than {int(round(100*float(tr.fraction_random_worse)))}/100 target-and-lag-matched random sparse graphs, a descriptive reference rather than selection-adjusted causal evidence. Consecutive-gap filtering is evaluated at {len(events)} fixed, spaced endpoints. The training-prior baseline is explicitly retained, and negative probabilistic forecasting results are not discarded. There is no demonstrated post-cost trading profitability or identified intervention-level causal effect.

## Research question and hypotheses

**Question:** Can an observationally selected sparse temporal dependency graph support uncertainty-aware reconstruction and price-state forecasting when contemporaneous LOB measurements disappear, relative to static, autoregressive, matched-random and dense temporal comparators?

- **H1:** Historical information reduces missing-state reconstruction error relative to an unconditional static Gaussian.
- **H2:** The identities of selected cross-variable lagged edges add predictive information beyond the same-size autoregressive model and a matched-random graph reference.
- **H3:** Coherent probabilistic inference under missingness yields better log loss than the training-only three-class prior as outages lengthen.

These are evaluated descriptively because the evaluation days were inspected in earlier iterations and only three day-level blocks are available.

## Data, units and chronology

- FI-2010 NoAuction Decimal Precision: {394337:,} raw observations, 40 raw LOB variables and five original publisher labels; all ten days and 50 boundary-separated sequences are reconstructed.
- Each observation is a representation of ten underlying events. Horizons 10/20/30/50/100 refer to underlying-event labels, or approximately 1/2/3/5/10 representation steps. A deliberately conservative purge of h representation rows is applied before labeled evaluation.
- State nodes: log relative spread, log depth (levels 1–5), L1 imbalance, levels 2–5 imbalance, depth-normalized **quote-change OFI proxy**, and one-step log midpoint return. The seventh volatility measure is regime diagnostics only.
- Days 1–5: graph discovery; Days 6–7: internal family falsification; Days 1–7: frozen-graph parameter and ordinal-head fitting; Days 8–10: retrospective assessment. Sequences are never treated as 50 statistically independent replicates; day is the outer evaluation block.
- Lags 1–5 are in representation steps, not seconds or individual exchange messages. The exact source-label formula is not reconstructible from the representation snapshots alone; publisher labels remain unchanged.

## Probabilistic method

We fit a fifth-order Gaussian transition model with only the 39 frozen structural slots: \\(X_t=c+\\sum_{{\\tau=1}}^5 B_\\tau X_{{t-\\tau}}+\\varepsilon_t\\). The **primary** innovation covariance is diagonal, preserving an independent contemporaneous-innovation assumption. A Ledoit–Wolf covariance is reported separately as a sensitivity model with correlated innovations, not described as the original same conditional-independence factorization. The 30-dimensional augmented state \\(S_t=(X_t,\\ldots,X_{{t-4}})\\) is propagated with a Gaussian filter; noiseless observed components are conditioned on exactly at each missingness step. Five ordered-probit heads output \\(P(\\mathrm{{Down}}),P(\\mathrm{{Neutral}}),P(\\mathrm{{Up}})\\). All head coefficients are kept frozen when observations go missing.

We compare static Gaussian, autoregressive AR(5), frozen CausalLOB, dense VAR(5), and 100 target/lag-matched random sparse graphs. The consecutive-gap experiment uses 697 spaced endpoints, two prespecified masks (missing OFI and return; history-only), and gaps of 1, 5 and 20 representation steps. It evaluates h10 and h20 on the same endpoints for each gap. Prior-only scores use precisely the same endpoint labels.

## Results: complete current-state evidence

Matched retrospective scores, **lower log loss better**:

| Horizon (underlying events) | Evaluation rows | Balanced accuracy | Probit log loss | Training-prior log loss | Prior − probit log loss |
|---:|---:|---:|---:|---:|---:|
{full_rows}

These are label-prediction diagnostics, not estimates of executable market returns. In particular, severe deterioration at longer horizons limits a general long-horizon predictive claim.

## Results: structural contribution

- Sparse temporal transition RMSE {floatstr(tr.causallob)}; AR(5) {floatstr(tr.ar5)}; dense VAR(5) {floatstr(tr.dense_var5)}.
- The sparse structure has 39/180 = {100*39/180:.1f}% as many lagged slots as dense VAR(5). It does **not** outperform dense VAR(5) on transition RMSE.
- {int(round(100*float(tr.fraction_random_worse)))}/100 target/lag-matched random graphs have higher transition error, leaving {100-int(round(100*float(tr.fraction_random_worse)))}/100 with equal or better results. This random graph comparison is conditional and does not reproduce graph-selection uncertainty.
- Much of the static-vs-dynamic reconstruction gain arises from ordinary temporal persistence; graph-specific cross-edge gains are usually small. The original selected graph must not be retuned to improve this outcome.

## Results: consecutive missingness (exact same 697 endpoints)

Positive final column indicates improvement over a **training-only class prior** for precisely the same cohort:

| Missing-evidence scenario | Gap (representation steps) | Horizon | State RMSE | CausalLOB log loss | Prior log loss | Prior − CausalLOB |
|---|---:|---:|---:|---:|---:|---:|
{gap_rows}

Across these {len(master_df)} comparisons the prior has lower log loss in {prior_beats} cases and ties (within 1e-10) in {prior_ties} cases. Report these negative results alongside reconstruction advantages. Scores are based on simulated masks and only {len(events)} widely spaced events, not naturally recorded order-book-feed outages. Day-level results appear in `04_gap_day_consistency.csv`; do not calculate a confidence interval pretending 697 endpoints are independent.

## Interpretation and threats to validity

- **Causal scope:** LPCMCI selects observational lagged candidates under model assumptions, not identified intervention effects; reciprocal or ambiguous marks are not proof of latent confounding.
- **Graph contribution:** The method provides a compact structural parameterization, but dense VAR has a modest predictive advantage and temporal persistence explains most static-to-dynamic improvement.
- **Innovation distribution:** Diagonal Gaussian innovations omit important contemporaneous residual covariance. Shrinkage covariance may improve missing-state inference, but is a different model and can worsen history-only outcomes.
- **Outcome signal:** The current return and quote-OFI proxy share the current representation interval. Strong current-return/label association does not by itself show leakage, but stale features rapidly lose predictive association; the publisher's exact original event-level label formula is not available to reconstruct fully here.
- **Data exposure:** Earlier research consulted Days 6–10; Days 8–10 cannot be called pristine new independent validation. There are only three evaluation days.
- **Missingness:** Injected outages may not resemble true outages and do not identify missing-not-at-random mechanisms.
- **Baselines:** A complete validation-selected DeepLOB CPU test score is not established by the present artifacts ({deep_status}). Additional nonlinear XGBoost benchmark from other project runs is not assumed reproduced here.
- **No profitability:** No measured queue position, fills, execution prices, fees, market impact, slippage, latency or net PnL.

## Conclusion

The frozen sparse model can reconstruct missing LOB variables and produces coherent three-class probabilities. Its 39-slot temporal structure improves on AR(5) modestly in some missingness settings but does not establish general predictive superiority over dense VAR(5), matched random graphs, or a training-prior outcome baseline. The computational protocol is reproducible and preserves negative results. Stronger research claims require a genuinely unseen dataset and a method that propagates uncertainty about graph identity rather than treating one observational graph as known.

## Remaining actions before claiming a complete research study or NeurIPS-ready paper

1. Complete and verify the CPU DeepLOB benchmark: validation-selected checkpoint, matched population, per-horizon score and full protocol (if it was not actually finished).
2. Run a frozen protocol on new external LOB data or previously unexamined time periods without further tuning on the known FI-2010 evaluation days.
3. Implement and ablate a **new, principled graph-uncertainty or regime-aware inference method**, with proper training-only graph weighting, uncertainty scoring, matched compute and multiple independent seeds.
4. Verify genuine event-level label definitions and exact horizon units if the original event-level data become available; do not relabel the existing benchmark retrospectively.
5. Obtain execution-level data and a predeclared transaction-cost model before making any financial-profit claim.
6. Add full reference metadata, author contributions, academic approval and a submitted walkthrough/video if required by the course.

**Reference context (bibliographic metadata to verify before submission):** FI-2010 benchmark and DeepLOB; CausalStock (NeurIPS 2024) for probabilistic financial temporal graphs; FANTOM (NeurIPS 2025) for regime-dependent causal modeling. CausalStock and FANTOM are conceptual comparators, not same-dataset numerical baselines.
'''
report_path=FRES/'07_research_report_draft.md'
report_path.write_text(report,encoding='utf8')

# Simple concise summary for professor/viva; still evidence-based, never polished away negatives.
summary_text=f'''# CausalLOB — Final course-project evidence summary

- Dataset used: all 10 FI-2010 days loaded; Days 1–5 graph discovery, 6–7 internal validation, 1–7 transition/head fitting, 8–10 retrospective assessment. **Not trained on all 10 days.**
- Graph: 39 lagged slots (30 self-memory + 9 cross) versus dense 180.
- Retrospective transition RMSE: CausalLOB {tr.causallob:.6f}, AR(5) {tr.ar5:.6f}, dense VAR(5) {tr.dense_var5:.6f}; sparse beat {int(round(100*float(tr.fraction_random_worse)))}/100 matched random graphs.
- Gap stress: {len(events)} spaced assessment endpoints from Days 8–10, gaps 1/5/20, two masks, h10/h20, same labels/endpoints and training-only prior.
- Training-only prior scores better than frozen CausalLOB in {prior_beats}/{len(master_df)} pooled gap/horizon conditions; **do not conceal this.**
- Strong same-step price-label association does not imply an executable trading edge; no after-cost PnL claims.
- The course computational research pipeline is complete through this synthesis. Independent external validation, a verified DeepLOB result and a novel publication-level method are **not** yet complete.

See `07_research_report_draft.md` and the CSVs for full methods, metrics, limitations and future work.
'''
(FRES/'08_professor_one_page.md').write_text(summary_text,encoding='utf8')
(FRES/'10_REPRODUCE_README.md').write_text("""# Reproduce CausalLOB research package

Run `CausalLOB_STEP9_TO_FINAL_ONE_CELL_COLAB.ipynb` as one code cell in Google Colab, using the original frozen Step 2, 4, 5, 6, 7 and 8 artifacts in `/content/drive/MyDrive/CausalLOB`. It will automatically mount Drive and reject missing prerequisites. Existing Step 9 completed data are reused only after matching the run fingerprint. Do not use D8–D10 for graph or model retuning. Generated outputs appear in `STEP9_GAP_STRESS_READONLY` and `STEP10_11_FINAL_RESEARCH_PACKAGE`. Install Python packages as requested by the notebook; no external dataset download occurs. ZIP excludes raw licensed data and checkpoints; hashes in the manifest allow source integrity verification. Researcher-blind external validation, full DeepLOB completion, and net trading PnL are not provided by this executable.
""",encoding='utf8')

# Export checksum/lineage manifest and bundle (exclude large raw data and frozen checkpoints).
artifact_paths=[FRES/n for n in ['01_evidence_ledger.csv','02_full_observation_all_horizons.csv',
 '03_matched_gap_results.csv','04_gap_day_consistency.csv','05_graph_evidence.json',
 '06_claims_and_completion_gates.json','07_research_report_draft.md','08_professor_one_page.md','10_REPRODUCE_README.md']]
artifact_paths += [fig1,fig2,fig3]
for p in artifact_paths: assert p.is_file() and p.stat().st_size>0
source_hash_after={name:sha256(path) for name,path in INFILES.items()}
assert source_hash_after==source_hash_before,'Upstream file unexpectedly changed; DO NOT TRUST final package.'
manifest={
 'status':'RETROSPECTIVE_COMPUTATIONAL_PACKAGE_COMPLETE_WITH_OPEN_EXTERNAL_REQUIREMENTS',
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'step9_run_id':run_id,'source_step6_run_hash':s6['run_hash'],
 'source_step7_run_hash':s7['run_hash'],'source_step8_run_id':s8['run_id'],
 'all_ten_days_loaded':True,'all_ten_days_trained':False,
 'researcher_blind_external_validation':False,
 'upstream_readonly_sha256':source_hash_after,
 'generated_sha256':{p.name:sha256(p) for p in artifact_paths},
 'limitations':S9SUMMARY['limitations'],
 'open_requirements':['independent_external_validation','verified_deeplob_test_metrics',
                     'novel_publication_level_method','no_net_pnl_evidence'],
}
manifest_path=FRES/'09_reproducibility_manifest.json'
atomic_json(manifest_path,manifest)
assert source_hash_before=={name:sha256(path) for name,path in INFILES.items()}
zip_path=FINAL/'CausalLOB_FINAL_RETROSPECTIVE_PACKAGE.zip'
zip_tmp=FINAL/'CausalLOB_FINAL_RETROSPECTIVE_PACKAGE.zip.tmp'
with _zipfile.ZipFile(zip_tmp,'w',compression=_zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
    for p in artifact_paths+[manifest_path]:
        zf.write(p,arcname=('figures/' if p.parent==FFIG else 'results/')+p.name)
    for p in [INFILES['step4_summary'],INFILES['step5_summary'],INFILES['step6_summary'],
              INFILES['step7_summary'],INFILES['step8_manifest'],INFILES['step9_summary']]:
        zf.write(p,arcname='source_summaries/'+p.parent.parent.name+'_'+p.name)
zip_tmp.replace(zip_path)
with _zipfile.ZipFile(zip_path) as zf:
    assert zf.testzip() is None
    assert len(zf.namelist())==len(artifact_paths)+1+6

print('\n'+'='*112)
print('STEPS 9–11 COMPLETE: numerical Step9 fixed, evidence package produced; no claim of external research completion')
print('='*112)
print('ALL TEN days loaded; graph D1–5, validation D6–7, parameters D1–7, assessment D8–10 (historically exposed).')
print('Full-horizon retrospective head vs matched training-only prior:')
display(full_df[['horizon_underlying_events','n_matched','probit_balanced_accuracy',
                 'probit_logloss','prior_logloss','prior_minus_probit_logloss']].round(6))
print('Consecutive-gap PRIMARY vs training-only prior: negative gain means CausalLOB loses.')
display(master_df[['scenario','gap_representation_steps','horizon_underlying_events',
                   'causallob_state_rmse','causallob_logloss','prior_logloss',
                   'prior_minus_causallob_logloss']].round(6))
print('Prior wins:',prior_beats,'/',len(master_df),'matched pooled gap comparisons.')
print('Frozen sparse transition RMSE:',round(float(tr.causallob),6),'| dense:',round(float(tr.dense_var5),6))
print('Step9 outputs:',RESULTS)
print('Final report:',report_path)
print('One-page brief:',FRES/'08_professor_one_page.md')
print('Reproducibility ZIP:',zip_path)
print('UPSTREAM SOURCE HASHES UNCHANGED: PASS')
print('Final runtime:',round(time.monotonic()-T0,1),'seconds')

Google Drive already mounted.
SELF-TEST PASS: exact conditioning, noiseless measurements, one-step parity, two-step uncertainty propagation, probabilities.
STEP8 DENSE PARITY PASS; RMSE 0.5851714 joint NLL 3.7053221
STEP 9 PREFLIGHT PASS: fixed graph/heads, same endpoints, D1–D7-only dense refit, D8–D10 assessment.
Endpoint events: 697 by day: {8: 277, 9: 260, 10: 160}
Run ID: 1988063fc8dff88497d49170731e5b3c929b83e95f13112386b6cde990e96190
Finished STATIC_DIAG_TRAIN                MISS_RETURN_AND_OFI   gap= 1
Finished AR5_DIAG_FROZEN                  MISS_RETURN_AND_OFI   gap= 1
Finished CAUSALLOB_DIAG_FROZEN            MISS_RETURN_AND_OFI   gap= 1
Finished DENSE_VAR5_DIAG                  MISS_RETURN_AND_OFI   gap= 1
Finished CAUSALLOB_SHRINK_SENSITIVITY     MISS_RETURN_AND_OFI   gap= 1
Finished STATIC_DIAG_TRAIN                MISS_RETURN_AND_OFI   gap= 5
Finished AR5_DIAG_FROZEN                  MISS_RETURN_AND_OFI   gap= 5
Finished CAUSALLOB_DIAG_FROZEN            MISS_RETURN_AND_

,scenario,gap,comparison,state_rmse_gain,state_nll_gain
2,MISS_RETURN_AND_OFI,1,SPARSE_VS_AR5,0.002868,0.008321
4,MISS_RETURN_AND_OFI,1,SPARSE_VS_DENSE,-0.002317,-0.005236
6,MISS_RETURN_AND_OFI,1,SHRINK_SENSITIVITY,0.055947,0.405097
42,MISS_RETURN_AND_OFI,5,SPARSE_VS_AR5,0.003323,0.010372
44,MISS_RETURN_AND_OFI,5,SPARSE_VS_DENSE,0.000553,0.000866
46,MISS_RETURN_AND_OFI,5,SHRINK_SENSITIVITY,0.054801,0.399766
82,MISS_RETURN_AND_OFI,20,SPARSE_VS_AR5,0.003490,0.011057
84,MISS_RETURN_AND_OFI,20,SPARSE_VS_DENSE,0.001224,0.002118
86,MISS_RETURN_AND_OFI,20,SHRINK_SENSITIVITY,0.054600,0.399098
122,HISTORY_ONLY,1,SPARSE_VS_AR5,0.001434,0.009475



POOLED outcome comparisons (negative prior gain means below train-only class prior):


,scenario,gap,horizon,comparison,outcome_logloss_gain,outcome_brier_gain
2,MISS_RETURN_AND_OFI,1,10,SPARSE_VS_AR5,0.006303,0.001289
3,MISS_RETURN_AND_OFI,1,20,SPARSE_VS_AR5,0.003995,-0.000581
4,MISS_RETURN_AND_OFI,1,10,SPARSE_VS_DENSE,-0.004092,-0.003116
5,MISS_RETURN_AND_OFI,1,20,SPARSE_VS_DENSE,-0.004754,-0.003577
8,MISS_RETURN_AND_OFI,1,10,SPARSE_VS_PRIOR,-0.182523,-0.136152
9,MISS_RETURN_AND_OFI,1,20,SPARSE_VS_PRIOR,-0.086164,-0.065835
42,MISS_RETURN_AND_OFI,5,10,SPARSE_VS_AR5,0.005320,0.001392
43,MISS_RETURN_AND_OFI,5,20,SPARSE_VS_AR5,0.005495,0.000977
44,MISS_RETURN_AND_OFI,5,10,SPARSE_VS_DENSE,0.002895,0.001241
45,MISS_RETURN_AND_OFI,5,20,SPARSE_VS_DENSE,0.002728,0.001020


Outputs: /content/drive/MyDrive/CausalLOB/STEP9_GAP_STRESS_READONLY/results Runtime: 12.0 seconds

STEPS 9–11 COMPLETE: numerical Step9 fixed, evidence package produced; no claim of external research completion
ALL TEN days loaded; graph D1–5, validation D6–7, parameters D1–7, assessment D8–10 (historically exposed).
Full-horizon retrospective head vs matched training-only prior:


,horizon_underlying_events,n_matched,probit_balanced_accuracy,probit_logloss,prior_logloss,prior_minus_probit_logloss
0,10,139062,0.727693,0.499358,0.830344,0.330986
1,20,138912,0.623119,0.715190,0.958605,0.243416
2,30,138762,0.538449,0.937313,1.030661,0.093348
3,50,138462,0.407045,1.070878,1.109434,0.038556
4,100,137712,0.374541,1.146834,1.160453,0.013619


Consecutive-gap PRIMARY vs training-only prior: negative gain means CausalLOB loses.


,scenario,gap_representation_steps,horizon_underlying_events,causallob_state_rmse,causallob_logloss,prior_logloss,prior_minus_causallob_logloss
0,MISS_RETURN_AND_OFI,1,10,0.902347,1.049522,0.866999,-0.182523
1,MISS_RETURN_AND_OFI,1,20,0.902347,1.061962,0.975799,-0.086164
2,MISS_RETURN_AND_OFI,5,10,0.909648,1.047088,0.866999,-0.180089
3,MISS_RETURN_AND_OFI,5,20,0.909648,1.060570,0.975799,-0.084771
4,MISS_RETURN_AND_OFI,20,10,0.909519,1.047073,0.866999,-0.180074
5,MISS_RETURN_AND_OFI,20,20,0.909519,1.060496,0.975799,-0.084697
6,HISTORY_ONLY,1,10,0.632487,1.048717,0.866999,-0.181718
7,HISTORY_ONLY,1,20,0.632487,1.054068,0.975799,-0.078269
8,HISTORY_ONLY,5,10,0.742237,1.050559,0.866999,-0.183560
9,HISTORY_ONLY,5,20,0.742237,1.062099,0.975799,-0.086300


Prior wins: 12 / 12 matched pooled gap comparisons.
Frozen sparse transition RMSE: 0.586137 | dense: 0.585171
Step9 outputs: /content/drive/MyDrive/CausalLOB/STEP9_GAP_STRESS_READONLY/results
Final report: /content/drive/MyDrive/CausalLOB/STEP10_11_FINAL_RESEARCH_PACKAGE/results/07_research_report_draft.md
One-page brief: /content/drive/MyDrive/CausalLOB/STEP10_11_FINAL_RESEARCH_PACKAGE/results/08_professor_one_page.md
Reproducibility ZIP: /content/drive/MyDrive/CausalLOB/STEP10_11_FINAL_RESEARCH_PACKAGE/CausalLOB_FINAL_RETROSPECTIVE_PACKAGE.zip
UPSTREAM SOURCE HASHES UNCHANGED: PASS
Final runtime: 18.2 seconds


In [ ]:
# CAUSALLOB STEP 12 — FINAL EVIDENCE AUDIT + RESUMABLE DEEPLOB
# GYP SELF-BRIEF: Falsify unsupported claims, preserve frozen checkpoints and
# previously inspected D8–D10 status. Complete exact-day/matched-population audits.
# Resume original DeepLOB architecture/protocol without skipping data or epochs.
# Generate an external-validation preregistration, NOT invented results.

import os, sys, json, time, hashlib, random, warnings, gc, importlib.util, subprocess
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from datetime import datetime, timezone

# USER SETTINGS: override using environment variables for automation if desired.
RUN_DEEPLOB = os.environ.get('CAUSALLOB_RUN_DEEPLOB', '1') == '1'
ACCELERATOR = os.environ.get('CAUSALLOB_ACCELERATOR', 'auto').lower() # auto, cpu, cuda
if ACCELERATOR not in {'auto','cpu','cuda'}: raise ValueError('Invalid ACCELERATOR')
FORCE_RESTART = False # Never change on a scientific run without separately archiving checkpoints.
HORIZONS = (10,20,30,50,100)
SEED = 5162026
WINDOW = 100
BATCH_SIZE = 256        # ORIGINAL batch size; do not increase silently.
MAX_EPOCHS = 15         # ORIGINAL training budget.
PATIENCE = 3            # ORIGINAL early-stopping patience.
MIN_DELTA = 1e-4
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 5.0
VERSION = 'DEEPLOB_CPU_v1_2026-09-14' # Same checkpoint format/model; accelerator logged separately.
TRAIN_DAYS = (1,2,3,4,5)
VAL_DAYS = (6,7)
TEST_DAYS = (8,9,10)
LABEL_TO_ORD = {3:0,2:1,1:2}
ROOT = Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))

for module,package in [('torch','torch'),('sklearn','scikit-learn')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import balanced_accuracy_score,f1_score,log_loss
warnings.filterwarnings('ignore',category=FutureWarning)
try:
    from IPython.display import display
except ImportError:
    display=print
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists() and 'CAUSALLOB_ROOT' not in os.environ:
        drive.mount('/content/drive')
except ImportError:
    pass

if ACCELERATOR == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA requested but no GPU is available. Select Colab GPU runtime or ACCELERATOR=cpu.')
DEVICE = torch.device('cuda' if (ACCELERATOR!='cpu' and torch.cuda.is_available()) else 'cpu')
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True,warn_only=True)
    torch.set_float32_matmul_precision('highest')
# No AMP, FP16, TF32, graph/model/head changes, data subsampling, or altered early stopping.
CPU_THREADS = max(1,min(4,os.cpu_count() or 2))
torch.set_num_threads(CPU_THREADS)
try: torch.set_num_interop_threads(1)
except RuntimeError: pass
torch.backends.mkldnn.enabled=True
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if DEVICE.type=='cuda': torch.cuda.manual_seed_all(SEED)
START=time.monotonic()
S2=ROOT/'STEP2_REBUILD/checkpoints'
S4=ROOT/'STEP4_VALIDATION_FROZEN'
S5=ROOT/'STEP5_LABEL_AUDIT_READONLY'
S6=ROOT/'STEP6_FROZEN_DBN'
S7=ROOT/'STEP7_MISSING_STATE_READONLY'
S8=ROOT/'STEP8_GRAPH_SPECIFIC_READONLY'
S9=ROOT/'STEP9_GAP_STRESS_READONLY'
PREV=ROOT/'STEP10_11_FINAL_RESEARCH_PACKAGE'
DEEP=ROOT/'DEEPLOB_CPU_BASELINE'
DCK=DEEP/'checkpoints'
DR=DEEP/'results'
OUT=ROOT/'STEP12_EVIDENCE_AND_DEEPLOB'
RES=OUT/'results'; CKPT=OUT/'checkpoints'
for p in (OUT,RES,CKPT,DCK,DR):p.mkdir(parents=True,exist_ok=True)

FILES={
 'raw':S2/'01_canonical_raw.npz',
 'boundary':S2/'02_boundaries.npz',
 'temporal':S2/'04_temporal_primary.npz',
 'mask':S4/'checkpoints/PRIMARY_FROZEN_PARENT_MASK.npz',
 'step4':S4/'results/STEP4_FINAL_SUMMARY.json',
 'step5':S5/'results/05_LABEL_AUDIT_SUMMARY.json',
 'state':S6/'checkpoints/01_FROZEN_STATE_SPACE.npz',
 'step6_metrics':S6/'results/04_OUTCOME_METRICS_D8_D10.csv',
 'step7_state':S7/'results/02_EXACT_STATE_METRICS.csv',
 'step7_outcome':S7/'results/03_EXACT_OUTCOME_METRICS.csv',
 'step8_graph':S8/'results/04_graph_null_transition_summary.csv',
 'step9_state':S9/'results/01_gap_state_metrics.csv',
 'step9_outcome':S9/'results/02_gap_outcome_metrics.csv',
 'step9_days':S9/'results/04_day_consistency.csv',
 'step9_events':S9/'results/05_gap_event_manifest.csv',
 'step9_summary':S9/'results/STEP9_FINAL_SUMMARY.json',
}
for h in HORIZONS: FILES[f'probs_h{h}']=S6/f'results/03_PREDICTED_CLASS_PROBABILITIES_h{h}.npz'
missing={name:str(path) for name,path in FILES.items() if not path.is_file()}
if missing:raise FileNotFoundError('Required frozen evidence missing; stop. '+json.dumps(missing,indent=2))

def hashfile(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for c in iter(lambda:f.read(1<<20),b''):h.update(c)
 return h.hexdigest()

def write_json(p,obj):
 temp=Path(str(p)+'.tmp');temp.write_text(json.dumps(obj,indent=2,sort_keys=True,allow_nan=False,default=str)+'\n');temp.replace(p)

def write_csv(p,df):
 temp=Path(str(p)+'.tmp');df.to_csv(temp,index=False,float_format='%.14g');temp.replace(p)

def report_table(df):
 if df is None or df.empty:return '(not available)'
 try:return df.to_markdown(index=False,floatfmt='.6f')
 except ImportError:return '```\n'+df.to_string(index=False)+'\n```'

sha_before={k:hashfile(v) for k,v in FILES.items()}
with np.load(FILES['raw'],allow_pickle=False) as z:
 RAW=z['X_raw'].astype(np.float32,copy=False)
 LABELS=z['Y'].astype(np.int8,copy=False)
 DAY=z['day'].astype(np.int8,copy=False)
with np.load(FILES['boundary'],allow_pickle=False) as z:
 SEQ=z['sequence_id'].astype(np.int16,copy=False)
with np.load(FILES['temporal'],allow_pickle=False) as z:
 XSTATE=z['X_current']
 XDAY=z['day']
 XSEQ=z['sequence_id']
 XTARGET=z['Y']
 LAGVALID=z['full_lag_valid']
with np.load(FILES['mask'],allow_pickle=False) as z:
 maskdata={k:z[k] for k in z.files}
assert RAW.shape==(394337,40) and LABELS.shape==(394337,5)
assert len(np.unique(DAY))==10 and len(np.unique(SEQ))==50
assert sum(np.isin(DAY,TRAIN_DAYS+VAL_DAYS))==254750
starts=np.flatnonzero(np.r_[True,SEQ[1:]!=SEQ[:-1]])
ends=np.r_[starts[1:],len(SEQ)]
assert len(starts)==50
RAW_CURRENT=np.concatenate([np.arange(int(s)+20,int(e)) for s,e in zip(starts,ends)])
assert len(RAW_CURRENT)==len(XSTATE)==393337
assert np.array_equal(DAY[RAW_CURRENT],XDAY)
assert np.array_equal(SEQ[RAW_CURRENT],XSEQ)
assert np.array_equal(LABELS[RAW_CURRENT],XTARGET)
assert int(LAGVALID.sum())==393087
with np.load(FILES['state'],allow_pickle=False) as z:
 model_mask=z['frozen_parent_mask']; STATE_MU=z['train_mean'];STATE_SIGMA=z['train_std']
assert int(np.asarray(model_mask).sum())==39
assert np.isfinite(STATE_MU).all() and (STATE_SIGMA>0).all()
step5=json.loads(FILES['step5'].read_text())
step9=json.loads(FILES['step9_summary'].read_text())
assert step5.get('status')=='AUDIT_COMPLETED_WITH_LABEL_PROVENANCE_LIMITATION'
assert step9.get('n_endpoint_events')==697
assert step9.get('historically_blind_external_test') is False

# Every saved Step 6 probability file must agree with publisher labels, cohort and summary.
score6=pd.read_csv(FILES['step6_metrics'])
prob_store={}; parity=[]
for j,h in enumerate(HORIZONS):
 with np.load(FILES[f'probs_h{h}'],allow_pickle=False) as z:
  idx=z['row_index'].astype(np.int64)
  rrow=RAW_CURRENT[idx]
  p=z['p_down_neutral_up'].astype(np.float64)
  y=z['label_ordered'].astype(np.int64)
  yy=z['original_label'].astype(np.int64)
  prior=z['prior_down_neutral_up'].astype(np.float64)
  dd=z['day'].astype(np.int8)
 assert len(idx)>100000 and np.all(np.diff(idx)>0)
 assert np.array_equal(dd,DAY[rrow]) and np.isin(dd,TEST_DAYS).all()
 assert np.array_equal(yy,LABELS[rrow,j])
 assert np.array_equal(y,np.select([yy==3,yy==2,yy==1],[0,1,2],default=-1))
 assert np.isfinite(p).all() and (p>=0).all() and np.max(np.abs(p.sum(1)-1))<1e-10
 assert np.isclose(prior.sum(),1,atol=1e-10)
 for d in TEST_DAYS:
  ii=np.flatnonzero(dd==d)
  if len(ii):assert np.all(SEQ[rrow[ii]]==XSEQ[idx[ii]])
 m=score6[(score6.horizon_raw_messages==h)&(score6.day.astype(str)=='POOLED')]
 assert len(m)==2
 for model_name,P in [('PRIOR_ONLY',np.broadcast_to(prior,p.shape)),('ORDERED_PROBIT_SPARSE_DBN_HEAD',p)]:
  row=m[m.model==model_name].iloc[0]
  nll=float(-np.mean(np.log(np.maximum(P[np.arange(len(y)),y],1e-15))))
  assert int(row.n)==len(idx) and abs(float(row.log_loss)-nll)<5e-9,(h,model_name,nll,row.log_loss)
  parity.append({'horizon':h,'model':model_name,'n':len(y),'reproduced_logloss':nll,'saved_logloss':float(row.log_loss),'absolute_diff':abs(nll-row.log_loss)})
 prob_store[h]={'raw_rows':rrow,'day':dd,'label':y,'P':p,'prior':prior}
write_csv(RES/'01_exact_probability_parity.csv',pd.DataFrame(parity))

old_raw=ROOT/'M1_G9/checkpoints/01_canonical_raw.npz'
old_boundary=ROOT/'M1_G9/checkpoints/02_boundaries.npz'
old_parity={'old_M1_available':bool(old_raw.exists() and old_boundary.exists())}
if old_parity['old_M1_available']:
 with np.load(old_raw,allow_pickle=False) as z:
  old_parity['raw_identical']=bool(np.array_equal(z['X_raw'],RAW))
  old_parity['labels_identical']=bool(np.array_equal(z['Y'],LABELS))
  old_parity['days_identical']=bool(np.array_equal(z['day'],DAY))
 with np.load(old_boundary,allow_pickle=False) as z:
  old_parity['boundaries_identical']=bool(np.array_equal(z['sequence_id'],SEQ))
 assert all(v for k,v in old_parity.items() if k!='old_M1_available')
write_json(RES/'02_dataset_checkpoint_parity.json',old_parity)
print('PREFLIGHT PASS:',len(parity),'frozen probability parity comparisons; Step 2 labels raw-exact.')
print('Device:',DEVICE,'| DeepLOB:',RUN_DEEPLOB,'| old M1 parity:',old_parity)

# STEP 12A — completed retrospective evidence checks: same populations, no refitting.
state7=pd.read_csv(FILES['step7_state'])
outcome7=pd.read_csv(FILES['step7_outcome'])
null8=pd.read_csv(FILES['step8_graph'])
state9=pd.read_csv(FILES['step9_state'])
outcome9=pd.read_csv(FILES['step9_outcome'])
days9=pd.read_csv(FILES['step9_days'])
events9=pd.read_csv(FILES['step9_events'])

full6=score6[score6.day.astype(str)=='POOLED'].copy()
full6=full6.rename(columns={'horizon_raw_messages':'horizon'})
base6=full6[full6.model=='PRIOR_ONLY'].set_index('horizon')
head6=full6[full6.model=='ORDERED_PROBIT_SPARSE_DBN_HEAD'].set_index('horizon')
assert set(base6.index)==set(HORIZONS) and set(head6.index)==set(HORIZONS)
all_horizon=pd.DataFrame([{
 'horizon':h,'n':int(head6.loc[h,'n']),
 'probit_logloss':float(head6.loc[h,'log_loss']),
 'prior_logloss':float(base6.loc[h,'log_loss']),
 'prior_minus_probit_logloss':float(base6.loc[h,'log_loss']-head6.loc[h,'log_loss']),
 'probit_balanced_accuracy':float(head6.loc[h,'balanced_accuracy']),
 'probit_brier':float(head6.loc[h,'brier_multiclass']),
 'probit_ece':float(head6.loc[h,'ece_confidence'])
} for h in HORIZONS])
write_csv(RES/'03_all_horizon_matched_complete.csv',all_horizon)

# Step7 rows use the same per-horizon purge; assert exact N match before prior comparison.
assert set(outcome7.day.astype(str).unique())=={'POOLED','8','9','10'}
step7_vs_prior=[]
for _,r in outcome7[(outcome7.model=='SPARSE_39__DIAG_FROZEN')&(outcome7.scenario!='FULL_OBSERVED')].iterrows():
    sd=str(r.day)
    b=score6[(score6.horizon_raw_messages==int(r.horizon))&
             (score6.day.astype(str)==sd)&(score6.model=='PRIOR_ONLY')]
    assert len(b)==1 and int(b.iloc[0].n)==int(r.n),('Prior cohort mismatch',sd,r.scenario,r.horizon)
    step7_vs_prior.append({'scenario':r.scenario,'horizon':int(r.horizon),'day':sd,
                           'n':int(r.n),'causallob_logloss':float(r.log_loss),
                           'prior_logloss':float(b.iloc[0].log_loss),
                           'prior_minus_causallob':float(b.iloc[0].log_loss-r.log_loss),
                           'causallob_brier':float(r.brier_multiclass),
                           'prior_brier':float(b.iloc[0].brier_multiclass),
                           'prior_minus_causallob_brier':float(b.iloc[0].brier_multiclass-r.brier_multiclass)})
step7_prior=pd.DataFrame(step7_vs_prior)
write_csv(RES/'04_step7_matched_prior_all_days.csv',step7_prior)

# Step 7: report actual 90% interval coverage per day and covariance version.
cov7=state7[(state7.scenario!='FULL_OBSERVED')&state7.coverage90.notna()].copy()
cov7['coverage_minus_nominal']=cov7.coverage90-.90
cov7['abs_coverage_error_recomputed']=np.abs(cov7.coverage_minus_nominal)
assert np.allclose(cov7.abs_coverage_error_recomputed,cov7.abs_coverage_error90,rtol=0,atol=2e-10)
write_csv(RES/'05_step7_coverage_all_days.csv',cov7)

# Step 9: all comparisons same 697 spaced endpoint set, each day n=277/260/160.
assert set(events9.day.astype(int).unique())=={8,9,10}
counts=events9.groupby('day').size().to_dict()
assert {int(k):int(v) for k,v in counts.items()}=={8:277,9:260,10:160},counts
check9=[]
for (sc,gap,h,d),grp in outcome9.groupby(['scenario','gap','horizon','day']):
    assert len(grp)==6,(sc,gap,h,d,len(grp))
    assert grp.n.nunique()==1
    prior=grp[grp.model=='TRAIN_ONLY_CLASS_PRIOR'].iloc[0]
    sparse=grp[grp.model=='CAUSALLOB_DIAG_FROZEN'].iloc[0]
    check9.append({'scenario':sc,'gap':int(gap),'horizon':int(h),'day':str(d),'n':int(prior.n),
                   'causallob_logloss':float(sparse.log_loss),'prior_logloss':float(prior.log_loss),
                   'prior_minus_causallob_logloss':float(prior.log_loss-sparse.log_loss),
                   'causallob_brier':float(sparse.brier),'prior_brier':float(prior.brier),
                   'prior_minus_causallob_brier':float(prior.brier-sparse.brier)})
matched9=pd.DataFrame(check9)
assert len(matched9)==48 and (matched9[matched9.day=='POOLED'].n==697).all()
write_csv(RES/'06_step9_matched_prior_all_days.csv',matched9)

cov9=state9[(state9.model.isin(['CAUSALLOB_DIAG_FROZEN','CAUSALLOB_SHRINK_SENSITIVITY']))].copy()
cov9['coverage_minus_nominal']=cov9.coverage90-.90
assert np.allclose(cov9.abs_coverage_error90,abs(cov9.coverage_minus_nominal),atol=2e-10)
write_csv(RES/'07_step9_coverage_all_days.csv',cov9)

# Exactly three real assessment-day blocks. These are descriptive, NOT 697 independent observations.
daystat=[]
for (sc,gap,h),grp in matched9[matched9.day!='POOLED'].groupby(['scenario','gap','horizon']):
    assert set(grp.day)=={'8','9','10'}
    daystat.append({'scenario':sc,'gap':int(gap),'horizon':int(h),
                    'days_prior_lower_loss':int((grp.prior_minus_causallob_logloss<0).sum()),
                    'days_causallob_lower_loss':int((grp.prior_minus_causallob_logloss>0).sum()),
                    'min_prior_minus_causallob':float(grp.prior_minus_causallob_logloss.min()),
                    'max_prior_minus_causallob':float(grp.prior_minus_causallob_logloss.max())})
daystats=pd.DataFrame(daystat)
assert len(daystats)==12
write_csv(RES/'08_gap_three_day_descriptive_consistency.csv',daystats)

# Avoid stating that matching graphs imply a causal finding. Read graph null, do not select edges.
graph= null8[(null8.day=='POOLED')&(null8.metric=='transition_rmse')]
assert len(graph)==1 and int(graph.iloc[0].n_random)==100
G=graph.iloc[0]
assert abs(float(G.causallob)-.5861374731258596)<5e-5

# Existing labels are intact; exact original source event-level formula was not recovered.
label_audit={'publisher_labels_preserved':True,
 'exact_raw_M1_parity':old_parity,
 'source_label_exact_generation_reconstructed':False,
 'source_label_status':'UNRESOLVED: original event/message-level generation code and within-block midprices unavailable in the supplied snapshots',
 'threshold_hypotheses':'Diagnostic only; neither threshold was selected or used for relabeling',
 'horizons_underlying_events':[10,20,30,50,100],
 'horizon_steps_in_ten_event_representation':[1,2,3,5,10],
 'purge_policy':'Conservative: drop the last h REPRESENTATION rows of every security-day sequence',
 'researcher_blind_external_evaluation':False,
 'validation_and_test_historical_exposure':'D6-D10 consulted during earlier iterations; D8-D10 retrospective only'}
write_json(RES/'09_label_provenance_limitations.json',label_audit)

prereg='''# External replication protocol — preregistration DRAFT (not executed)

Status: **PROTOCOL ONLY. No external data were downloaded or evaluated in this step.**

## Fixed question
Can a 39-slot six-variable temporal structure support missing-state reconstruction
and calibrated downstream movement probabilities on genuinely unexamined LOB data,
relative to static Gaussian, AR(5), dense VAR(5), target/lag-matched random sparse
models, and a training-only three-class prior?

## Dataset and governance
- Pick a legally accessible external LOB dataset / untouched period BEFORE inspecting outcomes.
- Record instrument identifiers, feed source, timestamp conventions, event definitions,
  trading sessions, auction exclusions and missing-message conditions.
- Match the six economic feature definitions and distinguish quote-change OFI proxy
  from executed-trade order flow. Report when features are not semantically comparable.
- FI-2010 representation rows each aggregate ten events; re-establish horizon equivalence
  on the external feed; never simply reuse 10/20/30/50/100 as if units were identical.
- Reserve a chronological discovery/training/validation/test split and hash source bytes.
- No tuning, graph edits, thresholds or sample selection using external test outcomes.

## Locked comparisons and metrics
- Keep this FI-2010 model frozen for zero-shot transfer; if refitting is necessary,
  predeclare a separate external-train-only refit and keep results separate.
- Evaluate transition RMSE/NLL, missing-state RMSE/NLL, marginal 90% coverage,
  Up/Neutral/Down log loss, multiclass Brier and calibration against matched prior.
- Use full-observation and simulated gaps 1, 5, 20 with both prior masks; validate
  outage mechanism if true feed gaps are observed.
- Report individual test days/instruments, no pseudo-replication by overlapping events.
- Report costs and runtime of every comparator; train and validate on identical inputs.

## Hypotheses and failure criteria
- H1 history vs static: report both gains and losses for each missingness scenario.
- H2 edges vs AR5/random: report matched graph-null fraction; no post-hoc selection.
- H3 usable uncertainty: benchmark log loss and Brier against a test-cohort-matched
  training-only prior; a loss to the prior FAILS this hypothesis for that scenario.
- If label provenance or variable semantics cannot be reconciled, halt direct numerical
  comparisons and report separately without an invented cross-dataset leaderboard.

## Scope
No intervention-level causal claims, no claim of profitable execution from labels,
no NeurIPS acceptance or publication-readiness claim from the present FI-2010 results.
'''
(RES/'10_external_replication_prereg_DRAFT.md').write_text(prereg,encoding='utf8')
print('AUDIT PASS: exact labels/probabilities, matched priors, coverage, 3-day consistency and source provenance.')

# STEP 12B — original DeepLOB architecture, optimized batch gathering and optional FP32 GPU.
# ARCHITECTURE inserted verbatim from original CausalLOB_DEEPLOB_CPU_BASELINE_COLAB.
class ConvBNAct(nn.Module):
    def __init__(self,in_ch,out_ch,kernel_size,stride=1,padding=0):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(in_ch,out_ch,kernel_size=kernel_size,stride=stride,padding=padding,bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.01,inplace=True),
        )
    def forward(self,x): return self.net(x)

class DeepLOB(nn.Module):
    def __init__(self,n_classes=3):
        super().__init__()
        self.block1=nn.Sequential(
            ConvBNAct(1,32,(1,2),stride=(1,2)),
            ConvBNAct(32,32,(4,1),padding=(2,0)),
            ConvBNAct(32,32,(4,1),padding=(1,0)),
        )
        self.block2=nn.Sequential(
            ConvBNAct(32,32,(1,2),stride=(1,2)),
            ConvBNAct(32,32,(4,1),padding=(2,0)),
            ConvBNAct(32,32,(4,1),padding=(1,0)),
        )
        self.block3=nn.Sequential(
            ConvBNAct(32,32,(1,10)),
            ConvBNAct(32,32,(4,1),padding=(2,0)),
            ConvBNAct(32,32,(4,1),padding=(1,0)),
        )
        self.i1=nn.Sequential(ConvBNAct(32,64,(1,1)),ConvBNAct(64,64,(3,1),padding=(1,0)))
        self.i2=nn.Sequential(ConvBNAct(32,64,(1,1)),ConvBNAct(64,64,(5,1),padding=(2,0)))
        self.i3=nn.Sequential(nn.MaxPool2d((3,1),stride=(1,1),padding=(1,0)),ConvBNAct(32,64,(1,1)))
        self.lstm=nn.LSTM(input_size=192,hidden_size=64,num_layers=1,batch_first=True)
        self.fc=nn.Linear(64,n_classes)
    def forward(self,x):
        x=self.block1(x); x=self.block2(x); x=self.block3(x)
        a,b,c=self.i1(x),self.i2(x),self.i3(x)
        tmin=min(a.shape[2],b.shape[2],c.shape[2])
        x=torch.cat([a[:,:,:tmin,:],b[:,:,:tmin,:],c[:,:,:tmin,:]],dim=1)
        if x.shape[-1]!=1: raise RuntimeError(f'Expected width=1, got {x.shape}')
        x=x.squeeze(-1).permute(0,2,1).contiguous()
        x,_=self.lstm(x)
        return self.fc(x[:,-1,:])


# Original window, split, normalization, batch, Adam, loss and patience unchanged.
trainmask=np.isin(DAY,TRAIN_DAYS)
rawmean=RAW[trainmask].mean(axis=0,dtype=np.float64)
rawstd=RAW[trainmask].std(axis=0,ddof=0,dtype=np.float64)
assert np.isfinite(rawmean).all() and np.all(rawstd>1e-12)
scaler_path=DCK/'train_only_scaler.npz'
if scaler_path.exists():
 with np.load(scaler_path,allow_pickle=False) as z:
  assert np.allclose(z['mean'],rawmean,rtol=0,atol=1e-10),'Original DeepLOB mean mismatch'
  assert np.allclose(z['std'],rawstd,rtol=0,atol=1e-10),'Original DeepLOB std mismatch'
else:
 np.savez_compressed(scaler_path,mean=rawmean,std=rawstd,feature_names=np.array([f'raw_{i}' for i in range(40)]),train_days=np.array(TRAIN_DAYS))
XZ=((RAW-rawmean.astype(np.float32))/rawstd.astype(np.float32)).astype(np.float32,copy=False)
assert XZ.dtype==np.float32
REMAIN=np.empty(len(SEQ),dtype=np.int32)
for s,e in zip(starts,ends):
 REMAIN[s:e]=np.arange(e-s-1,-1,-1,dtype=np.int32)

def eligible(h,days):
 idx=[]
 for s,e in zip(starts,ends):
  if int(DAY[s]) not in days:continue
  lo=int(s)+WINDOW-1;hi=int(e)-1-int(h)
  if hi<lo:continue
  ix=np.arange(lo,hi+1,dtype=np.int64)
  assert np.all(SEQ[ix-WINDOW+1]==SEQ[ix]) and np.all(REMAIN[ix]>=h)
  idx.append(ix)
 return np.concatenate(idx) if idx else np.empty(0,dtype=np.int64)

window_offsets=np.arange(WINDOW-1,-1,-1,dtype=np.int64)
Y_ORD=np.zeros_like(LABELS,dtype=np.int8)
for v,i in LABEL_TO_ORD.items():Y_ORD[LABELS==v]=i
assert np.isin(LABELS,[1,2,3]).all()
class FastWindowDataset(Dataset):
 def __init__(self,indices,hidx):
  self.indices=np.asarray(indices,dtype=np.int64)
  self.hidx=int(hidx)
 def __len__(self):return len(self.indices)
 def __getitem__(self,k):
  t=int(self.indices[k]);s=t-WINDOW+1
  if SEQ[s]!=SEQ[t]:raise RuntimeError('Window crosses boundary')
  return (torch.from_numpy(XZ[s:t+1]).unsqueeze(0),
          torch.tensor(int(Y_ORD[t,self.hidx]),dtype=torch.long),
          torch.tensor(t,dtype=torch.long))
 def __getitems__(self,ks):
  # Equivalent data and exact index order to __getitem__, but ONE vectorized NumPy
  # gather per batch rather than ~256 Python window slices. No approximation.
  t=self.indices[np.asarray(ks,dtype=np.int64)]
  positions=t[:,None]-window_offsets[None,:]
  assert np.array_equal(SEQ[positions[:,0]],SEQ[t])
  x=np.ascontiguousarray(XZ[positions]).reshape(-1,1,WINDOW,40)
  y=np.ascontiguousarray(Y_ORD[t,self.hidx].astype(np.int64))
  return torch.from_numpy(x),torch.from_numpy(y),torch.from_numpy(t.copy())

def make_loader(rows,hidx,shuffle):
 return DataLoader(FastWindowDataset(rows,hidx),batch_size=BATCH_SIZE,
                   shuffle=shuffle,num_workers=0,pin_memory=False,
                   drop_last=False,collate_fn=lambda batch:batch)

# Gather parity: exact equality of fast and original data paths (not just close).
idx_check=eligible(10,TRAIN_DAYS)[:3]
ds_check=FastWindowDataset(idx_check,0)
fastx,fasty,fastt=next(iter(make_loader(idx_check,0,False)))
for i in range(len(idx_check)):
 slowx,slowy,slowt=ds_check[i]
 assert torch.equal(slowx,fastx[i]) and torch.equal(slowy,fasty[i]) and torch.equal(slowt,fastt[i])
print('DeepLOB fast dataloader EXACT INPUT PARITY PASS; FP32, batch=256, full data, same architecture.')
# Model sanity: identical architecture/output shape; no state mutated in frozen CausalLOB.
with torch.inference_mode():
 probe=DeepLOB(); assert probe(torch.zeros(2,1,WINDOW,40)).shape==(2,3)
del probe;gc.collect()
# Independent initial seed for this new sequence. Older resumed states retain model/optimizer.
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if DEVICE.type=='cuda':torch.cuda.manual_seed_all(SEED)


def torchload(path,map_location='cpu'):
 try:return torch.load(path,map_location=map_location,weights_only=False)
 except TypeError:return torch.load(path,map_location=map_location)

def atomic_torch(path,obj):
 tmp=Path(str(path)+'.tmp');torch.save(obj,tmp);tmp.replace(path)

def rng_capture():
 return {'torch_cpu_rng':torch.get_rng_state(),
         'torch_cuda_rng':torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else None,
         'numpy_rng':np.random.get_state(),
         'python_rng':random.getstate()}

def rng_restore(state):
 if 'torch_cpu_rng' not in state:return False
 torch.set_rng_state(state['torch_cpu_rng'])
 if DEVICE.type=='cuda' and state.get('torch_cuda_rng') is not None:
  torch.cuda.set_rng_state_all(state['torch_cuda_rng'])
 np.random.set_state(state['numpy_rng']);random.setstate(state['python_rng'])
 return True

def metrics(y,P):
 P=np.asarray(P,dtype=np.float64).copy()
 assert np.isfinite(P).all() and (P>=0).all()
 P/=P.sum(axis=1,keepdims=True)  # numerical simplex normalization, not calibration/retraining
 pred=P.argmax(axis=1)
 return {'n':int(len(y)),
         'accuracy':float(np.mean(pred==y)),
         'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
         'macro_f1':float(f1_score(y,pred,labels=[0,1,2],average='macro',zero_division=0)),
         'log_loss':float(log_loss(y,np.clip(P,1e-15,1),labels=[0,1,2])),
         'brier_multiclass':float(np.mean(np.sum((P-np.eye(3)[y])**2,axis=1)))}

@torch.inference_mode()
def evaluate(model,loader):
 model.eval();probs=[];ys=[];rr=[]
 for xb,yb,tb in loader:
  logits=model(xb.to(DEVICE,dtype=torch.float32))
  probs.append(torch.softmax(logits,dim=1).cpu().numpy().astype(np.float64))
  ys.append(yb.numpy());rr.append(tb.numpy())
 return np.concatenate(ys),np.concatenate(probs),np.concatenate(rr)

def checkpoint_paths(h):
 d=DCK/f'h{h}';d.mkdir(parents=True,exist_ok=True)
 return {'dir':d,'best':d/'best.pt','last':d/'last.pt','history':d/'history.csv',
         'done':d/'DONE.json','probs':d/f'test_probabilities_h{h}.parquet',
         'probs_npz':d/f'test_probabilities_h{h}.npz'}

# Legacy baseline version may not include restored RNG. This is a reproducibility
# limitation of the original code. A resumed run keeps EXACT saved optimizer/weights,
# but is not guaranteed bitwise identical to an uninterrupted historical CPU run.
resume_warnings=[]


def save_deeplob_probs(paths,df):
 if importlib.util.find_spec('pyarrow') or importlib.util.find_spec('fastparquet'):
  tmp=Path(str(paths['probs'])+'.tmp.parquet')
  df.to_parquet(tmp,index=False);tmp.replace(paths['probs'])
 else:
  tmp=Path(str(paths['probs_npz'])+'.tmp.npz')
  np.savez_compressed(tmp,**{col:df[col].to_numpy() for col in
      ('raw_row','day','y_ord','p_down','p_stationary','p_up')})
  tmp.replace(paths['probs_npz'])

def ensure_predictions(h,paths,result):
 # Prefer a stored untouched test-probability file; otherwise infer from frozen best.
 if paths['probs'].exists() or paths['probs_npz'].exists():
  if paths['probs'].exists():
   tdf=pd.read_parquet(paths['probs'])
  else:
   with np.load(paths['probs_npz'],allow_pickle=False) as zz:
    tdf=pd.DataFrame({key:zz[key] for key in ('raw_row','day','y_ord','p_down','p_stationary','p_up')})
  required={'raw_row','day','y_ord','p_down','p_stationary','p_up'}
  assert required.issubset(tdf.columns)
  rows=tdf.raw_row.to_numpy(dtype=np.int64)
  assert len(rows)==len(eligible(h,TEST_DAYS))
  assert np.array_equal(rows,eligible(h,TEST_DAYS))
  return tdf,False
 if not paths['best'].exists():raise FileNotFoundError('DeepLOB DONE exists, but best.pt missing: '+str(paths['best']))
 m=DeepLOB().to(DEVICE)
 best=torchload(paths['best'],map_location='cpu')
 assert best['version']==VERSION and best['horizon']==h
 m.load_state_dict(best['model_state'])
 y,P,rr=evaluate(m,make_loader(eligible(h,TEST_DAYS),HORIZONS.index(h),False))
 assert np.array_equal(rr,eligible(h,TEST_DAYS))
 assert np.array_equal(y,Y_ORD[rr,HORIZONS.index(h)])
 testmetric=metrics(y,P)
 if result and abs(float(result['test_pooled']['log_loss'])-testmetric['log_loss'])>0.003:
  raise RuntimeError('Stored DeepLOB DONE and newly inferred best checkpoint disagree materially; investigate before continuing.')
 df=pd.DataFrame({'raw_row':rr,'day':DAY[rr],'y_ord':y,
                  'p_down':P[:,0],'p_stationary':P[:,1],'p_up':P[:,2]})
 save_deeplob_probs(paths,df)
 del m;gc.collect()
 return df,True


def run_horizon(h):
 paths=checkpoint_paths(h)
 if paths['done'].exists():
  result=json.loads(paths['done'].read_text())
  assert result['version']==VERSION and int(result['horizon'])==h
  assert int(result['n_train_windows'])==len(eligible(h,TRAIN_DAYS))
  assert int(result['n_val_windows'])==len(eligible(h,VAL_DAYS))
  assert int(result['n_test_windows'])==len(eligible(h,TEST_DAYS))
  assert paths['best'].exists(), 'Completed horizon must retain its validation-selected best.pt'
  print('DeepLOB h',h,'already complete; no training or model selection rerun.')
  tdf,new=ensure_predictions(h,paths,result)
  return result,tdf,('recovered_probabilities' if new else 'reused_done_and_probabilities')
 if not RUN_DEEPLOB:return None,None,'not_run'
 if FORCE_RESTART:raise RuntimeError('FORCE_RESTART forbidden in frozen evidence run; archive original checkpoints before any deliberate restart.')
 train=eligible(h,TRAIN_DAYS);valid=eligible(h,VAL_DAYS);test=eligible(h,TEST_DAYS)
 trainloader=make_loader(train,HORIZONS.index(h),True)
 valloader=make_loader(valid,HORIZONS.index(h),False)
 net=DeepLOB().to(DEVICE)
 opt=torch.optim.Adam(net.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
 lossfn=nn.CrossEntropyLoss()
 start_epoch=1;bestval=float('inf');bad=0;history=[]
 if paths['last'].exists():
  ck=torchload(paths['last'],map_location='cpu')
  assert ck['version']==VERSION and ck['horizon']==h
  if paths['best'].exists():
   best_on_disk=torchload(paths['best'],map_location='cpu')
   assert best_on_disk['version']==VERSION and best_on_disk['horizon']==h
   if int(best_on_disk['epoch'])>int(ck['epoch']):
    raise RuntimeError('Checkpoint interrupted between best.pt and last.pt. Stop; archive and reconcile rather than silently overwrite best.')
  net.load_state_dict(ck['model_state']);opt.load_state_dict(ck['optimizer_state'])
  start_epoch=int(ck['epoch'])+1
  bestval=float(ck['best_val_nll']);bad=int(ck['bad_epochs']);history=list(ck.get('history',[]))
  original_accelerator=str(ck.get('accelerator','cpu (legacy checkpoint)'))
  if original_accelerator!=str(DEVICE):
   msg=f'h{h}: execution switched {original_accelerator} -> {DEVICE}; FP32 numerics and continuation are not bit-identical.'
   resume_warnings.append(msg);print('CAUTION:',msg)
  restored=rng_restore(ck)
  if not restored:
   msg=f'h{h}: legacy epoch checkpoint has no RNG state; exact uninterrupted-CPU trajectory cannot be guaranteed.'
   resume_warnings.append(msg);print('CAUTION:',msg)
  print('Resume h',h,'at epoch',start_epoch,'prior best val logloss',bestval)
  assert len(history)==start_epoch-1
 assert start_epoch<=MAX_EPOCHS+1
 if start_epoch==MAX_EPOCHS+1 and not paths['best'].exists():raise RuntimeError('Training budget exhausted, best checkpoint missing')
 if bad>=PATIENCE:print('Prior early-stopping criterion already satisfied; proceeding to best.pt evaluation.')
 for epoch in range(start_epoch,MAX_EPOCHS+1):
  if bad>=PATIENCE:break
  st=time.monotonic();net.train();total=0.0;n=0
  for xb,yb,_ in trainloader:
   xb=xb.to(DEVICE,dtype=torch.float32);yb=yb.to(DEVICE)
   opt.zero_grad(set_to_none=True)
   logits=net(xb);loss=lossfn(logits,yb)
   loss.backward();torch.nn.utils.clip_grad_norm_(net.parameters(),GRAD_CLIP);opt.step()
   total+=float(loss.item())*len(yb);n+=len(yb)
  vy,vp,_=evaluate(net,valloader)
  vm=metrics(vy,vp)
  val_nll=vm['log_loss']
  better=(bestval-val_nll)>MIN_DELTA
  if better:
   bestval=val_nll;bad=0
   atomic_torch(paths['best'],{'version':VERSION,'horizon':h,'epoch':epoch,
                                'model_state':net.state_dict(),'val_metrics':vm})
  else:bad+=1
  sec=time.monotonic()-st
  history.append({'epoch':epoch,'train_cross_entropy':total/max(1,n),'val_log_loss':val_nll,
                  'val_balanced_accuracy':vm['balanced_accuracy'],'val_macro_f1':vm['macro_f1'],
                  'seconds':sec,'improved':better})
  write_csv(paths['history'],pd.DataFrame(history))
  payload={'version':VERSION,'horizon':h,'epoch':epoch,'model_state':net.state_dict(),
           'optimizer_state':opt.state_dict(),'best_val_nll':bestval,
           'bad_epochs':bad,'history':history,'accelerator':str(DEVICE)}
  payload.update(rng_capture())
  atomic_torch(paths['last'],payload)
  print(f'h{h} epoch={epoch:02d}: train CE={total/max(1,n):.6f} val NLL={val_nll:.6f} '+
        f'val BA={vm["balanced_accuracy"]:.4f}, {sec/60:.1f} min, device={DEVICE}; '+
        ('BEST' if better else f'patience={bad}/{PATIENCE}'))
 if not paths['best'].exists():raise RuntimeError('No validation-selected best checkpoint: '+str(paths['best']))
 best=torchload(paths['best'],map_location='cpu')
 assert best['version']==VERSION and int(best['horizon'])==h
 net.load_state_dict(best['model_state'])
 yy,pp,rr=evaluate(net,make_loader(test,HORIZONS.index(h),False))
 assert np.array_equal(rr,test) and np.array_equal(yy,Y_ORD[rr,HORIZONS.index(h)])
 tdf=pd.DataFrame({'raw_row':rr,'day':DAY[rr],'y_ord':yy,
                   'p_down':pp[:,0],'p_stationary':pp[:,1],'p_up':pp[:,2]})
 save_deeplob_probs(paths,tdf)
 results={'version':VERSION,'horizon':h,'selected_epoch':int(best['epoch']),
  'selection_metric':'D6-D7 validation log loss',
  'best_validation':best['val_metrics'],'test_pooled':metrics(yy,pp),
  'test_by_day':[{'day':d,**metrics(yy[DAY[rr]==d],pp[DAY[rr]==d])} for d in TEST_DAYS],
  'n_train_windows':int(len(train)),'n_val_windows':int(len(valid)),
  'n_test_windows':int(len(test)),'cpu_threads':CPU_THREADS,
  'accelerator':str(DEVICE),'precision':'full float32, no AMP or TF32',
  'batch_size':BATCH_SIZE,'window':WINDOW,'history_checkpoint_fidelity':
  ('LEGACY_RNG_NOT_SAVED' if any(w.startswith(f'h{h}:') for w in resume_warnings) else 'RNG_CHECKPOINTED')}
 write_json(paths['done'],results)  # LAST: no DONE until predictions and full evaluation exist.
 del net,opt,trainloader,valloader;gc.collect()
 if DEVICE.type=='cuda':torch.cuda.empty_cache()
 print('DeepLOB h',h,'DONE; selected epoch',results['selected_epoch'],
       'test NLL',results['test_pooled']['log_loss'])
 return results,tdf,'completed_this_run'

# STEP 12C — train missing DeepLOB horizons (one cell; checkpoints after EACH epoch).
# If user interrupts, audit CSVs already exist and next execution resumes best/last.
deep_results=[];deep_rows=[];device_status=[]
print('\nDEEPlOB TRAINING POLICY:',
      'full FP32 GPU acceleration' if DEVICE.type=='cuda' else 'original CPU FP32; 4-thread max')
print('No shortened epochs, no subsampling, no changed batch size/architecture/schedule.')
if DEVICE.type=='cuda':
 print('CAUTION: GPU and CPU FP32 are NOT bitwise interchangeable. Scientific model/protocol is unchanged; numerical path may differ.')
for h in HORIZONS:
 res,tdf,status=run_horizon(h)
 device_status.append({'horizon':h,'status':status,'device':(res.get('accelerator','cpu (legacy DONE)') if res is not None else str(DEVICE)),
                       'done':res is not None})
 if res is None:
  print('DeepLOB h',h,'NOT COMPLETE; audit will mark missing.');continue
 deep_results.append(res)
 if tdf is None:raise RuntimeError('Completed DeepLOB must have row-indexed probabilities')
 # Exact same test observations for both models; no adjustment on D8-D10.
 original=prob_store[h]
 indices,ia,ib=np.intersect1d(tdf.raw_row.to_numpy(dtype=np.int64),
                               original['raw_rows'],assume_unique=True,return_indices=True)
 assert len(indices)==len(tdf),f'h{h}: DeepLOB cohort not a subset of Step6 cohort'
 yp=tdf.y_ord.to_numpy(dtype=np.int64)[ia]
 assert np.array_equal(yp,original['label'][ib]),'DeepLOB/DBN label identity mismatch'
 deepp=tdf[['p_down','p_stationary','p_up']].to_numpy(dtype=float)[ia]
 dbnp=original['P'][ib]
 prior=np.broadcast_to(original['prior'],dbnp.shape)
 dd=tdf.day.to_numpy(dtype=np.int8)[ia]
 assert np.isin(dd,TEST_DAYS).all() and len(yp)>100000
 for dayname,sel in [('POOLED',np.ones(len(yp),bool))]+[(str(d),dd==d) for d in TEST_DAYS]:
  assert sel.any()
  for name,P in [('DEEPLOB_SAME_COHORT',deepp),('FROZEN_DBN_HEAD_SAME_COHORT',dbnp),('TRAIN_PRIOR_SAME_COHORT',prior)]:
   deep_rows.append({'horizon':h,'day':dayname,'model':name,**metrics(yp[sel],P[sel])})
 write_csv(RES/'11_deeplob_matched_comparison.csv',pd.DataFrame(deep_rows))
 write_json(RES/'11_deeplob_progress.json',{'completed':device_status,'resume_warnings':resume_warnings,
       'accelerator':str(DEVICE),'full_precision':True,'architecture_changed':False,
       'training_subset_used':False,'validation_selected':True})
 print('Matched DeepLOB vs DBN/PRIOR complete h',h,'n',len(indices))
write_json(RES/'11_deeplob_progress.json',{'completed':device_status,'resume_warnings':resume_warnings,
     'accelerator':str(DEVICE),'full_precision':True,'architecture_changed':False,
     'training_subset_used':False,'validation_selected':True})
if deep_rows:write_csv(RES/'11_deeplob_matched_comparison.csv',pd.DataFrame(deep_rows))

# Source-fingerprint guard: NEVER modify the frozen scientific evidence.
sha_after={k:hashfile(v) for k,v in FILES.items()}
assert sha_before==sha_after,'An upstream frozen checkpoint changed; STOP and investigate.'

primary=matched9[matched9.day=='POOLED'].copy()
negative=int((primary.prior_minus_causallob_logloss<0).sum())
assert len(primary)==12 and negative==12,'Expected previously documented 12/12 negative gap results; do not silently replace'
coverage_summary=cov7[(cov7.day.astype(str)=='POOLED') &
                      (cov7.model=='SPARSE_39__DIAG_FROZEN')][
                      ['scenario','coverage90','abs_coverage_error90','state_rmse','state_joint_nll']].copy()
claim_gate={
 'all_ten_days_in_source':True,
 'graph_discovery_only_days_1_to_5':True,
 'internal_validation_days_6_to_7_historically_exposed':True,
 'model_fit_days_1_to_7':True,
 'retrospective_assessment_days_8_to_10_historically_exposed':True,
 'pristine_external_validation':False,
 'publisher_exact_label_formula_verified':False,
 'upstream_sha256_unchanged':True,
 'deep_lob_completed_horizons':[int(r['horizon']) for r in deep_results],
 'deep_lob_all_horizons_complete':len(deep_results)==5,
 'deep_lob_matched_test_comparison':len(deep_rows)>0,
 'deep_lob_precision':'float32; no AMP/TF32; optional GPU (not bitwise CPU)',
 'deep_lob_resume_rng_limitations':resume_warnings,
 'missingness_natural_feed_outages':False,
 'matched_gap_prior_better_cases':negative,
 'matched_gap_scenarios':12,
 'intervention_causality_identified':False,
 'execution_costs_and_realized_profit_evaluated':False,
 'neurips_ready_claim':False,
 'external_protocol_drafted_not_executed':True,
 'only_three_evaluation_day_blocks':True}
write_json(RES/'12_final_claims_and_completion_gates.json',claim_gate)

# Full reproducibility manifest: actual source paths and byte-level fingerprints.
manifest={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'script_version':'STEP12_FINAL_AUDIT_DEEPLOB_v1',
 'input_sha256':sha_after,
 'input_paths':{k:str(v) for k,v in FILES.items()},
 'deep_lob_checkpoint_root':str(DCK),
 'output_directory':str(RES),
 'raw_shape':list(RAW.shape),
 'time_split':{'discovery':[1,2,3,4,5],'validation':[6,7],
               'fit':[1,2,3,4,5,6,7],'retrospective_test':[8,9,10]},
 'sample_units':'one FI-2010 representation row comprises ten underlying events',
 'inference_unit':'DAY; 50 security-day sequences are NOT independent inference units',
 'run_status':claim_gate,
 'data_is_researched_blind':False}
write_json(RES/'13_reproducibility_manifest.json',manifest)

# A complete research report in concise academic prose (not a performance victory narrative).
dp=pd.DataFrame(deep_rows)
dp_pooled=dp[dp.day=='POOLED'] if len(dp) else pd.DataFrame()
if dp_pooled.empty:
 deeptext='**DeepLOB status: NOT VERIFIED AS COMPLETE.** No scores invented. Run this cell with RUN_DEEPLOB=True and a GPU runtime if possible.'
else:
 deeptext=('Completed horizons: '+', '.join(str(int(x)) for x in sorted(dp_pooled.horizon.unique()))+'. '
 'The table uses an exactly matched test cohort for each horizon, not the different original DeepLOB and DBN test row counts.\n\n'+report_table(dp_pooled[
 ['horizon','model','n','balanced_accuracy','log_loss','brier_multiclass']]))

report='''# CausalLOB — Consolidated evidence and final research status

Research audit generated: '''+datetime.now(timezone.utc).isoformat()+'''

## Research question
Can a sparse, observationally selected lagged graphical model reconstruct missing
limit-order-book states and deliver useful probabilistic price-state predictions
compared against autoregression, dense temporal models, matched random graphs,
and a training-only class prior?

## Protocol and units
FI-2010 NoAuction Decimal Precision, 394,337 raw representation observations,
40 raw LOB columns, six modeled variables, five publisher horizons and 50 boundary-
separated sequences. Representation steps summarize ten underlying exchange events.
Graph discovery D1–D5; internal falsification D6–D7; frozen transitions and five
ordered-probit heads fitted on D1–D7; retrospective scoring on D8–D10.
**D8–D10 were inspected in older experiments: not researcher-blind validation.**
The actual outer inferential unit is day (only three assessment days).

## Independent artifact audit
Reconstructed raw rows, sequence IDs and publisher labels align to all saved
Step6 prediction records. Every outcome label, test day, prior distribution,
probability simplex and saved pooled log loss was verified. Frozen source hashes
were rechecked before and after this audit, unchanged. Original M1 parity:

'''+json.dumps(old_parity,indent=2)+'''

## Label provenance limitation
The publisher's labels and class orientation are preserved (Down=3, Neutral=2,
Up=1 before re-encoding). Step5's fixed reconstruction hypotheses did NOT
recreate original labels exactly. The precise event-level label-generation
formula cannot be recovered from the ten-event snapshots alone.
This limitation is UNRESOLVED, not evidence of a confirmed leak. We did not
relabel, retune thresholds, or change graph/model parameters.

## Complete observations (matched prior)
'''+report_table(all_horizon)+'''

Complete-state h10/h20 models outperform the train-only prior in log loss, but
forecasting deteriorates at longer horizons and demonstrates neither persistent
alpha nor net trading profitability.

## Temporal graph and alternatives
The 39-slot graph has 30 self-memory coefficients and nine selected cross-variable
lags versus 180 dense VAR5 lag coefficients. Transition RMSE: AR5 '''+f'{float(G.ar5):.6f}'+''', sparse '''+f'{float(G.causallob):.6f}'+''', dense '''+f'{float(G.dense_var5):.6f}'+'''.
The sparse graph has higher error than dense VAR; it outperforms '''+f'{100*float(G.fraction_random_worse):.0f}'+''' of 100 matched random graphs by transition RMSE.
This is descriptive, NOT an adjustment for the original LPCMCI graph-selection
process, and is not evidence of identified causal effects.

## Single-step missing states: primary diagonal Q
'''+report_table(coverage_summary)+'''

Coverage is marginal 90% interval coverage of artificially masked variables,
not evidence of executable economic profitability. Train-only shrinkage Q is
sensitivity and relaxes the primary DBN's independent innovation factorization.

## Single-step outcome probabilities: primary vs matched prior
'''+report_table(step7_prior[(step7_prior.day=='POOLED')&
            (step7_prior.horizon.isin([10,20]))][
            ['scenario','horizon','n','causallob_logloss','prior_logloss','prior_minus_causallob']])+'''

## Consecutive gaps: exactly same 697 spaced endpoints
'''+report_table(primary[['scenario','gap','horizon','n','causallob_logloss',
                        'prior_logloss','prior_minus_causallob_logloss']])+'''

The prior achieves lower log loss in '''+str(negative)+''' of 12 pooled gap configurations.
Thus useful outcome probabilities under missing return/OFI or history-only outages
are NOT established by this experiment, even where missing-state reconstruction
improves over AR5. State reconstruction and price-state prediction are distinct.

## Day-level consistency
'''+report_table(daystats)+'''

Day results are descriptive; 697 overlapping/book-correlated endpoints must not
be treated as 697 independent market replications. Only Days 8–10 were assessed.

## DeepLOB fair predictive comparator
'''+deeptext+'''

DeepLOB uses the original CNN→Inception→LSTM(64) model, backward window=100,
raw 40 features, D1–D5-only normalization/training, D6–D7 best-epoch selection,
conservative h-row tail purge, Adam, batch 256, 15 epoch cap, patience 3.
CPU input gathering is vectorized but numerically input-identical. Optional CUDA
uses full FP32 without AMP or TF32; GPU outputs are NOT bit-for-bit identical to
historical CPU outputs. Earlier h10 checkpoint runs may lack saved RNG state.
Differences remain even on matched test rows: DeepLOB sees 100 raw
representations and trains on D1–D5 with D6–D7 validation, whereas the PGM head
uses six current states and fits on D1–D7. The matched cohort controls only the
held-out examples, not model capacity, feature sets or training budgets.

## Remaining scientific and practical gates
- Genuine unseen external dataset or time-period replication, preregistered BEFORE
  its outcome inspection. Current code supplies protocol, NOT external results.
- Original event-level label-generation evidence (the publisher outputs themselves
  have been exactly preserved and audited).
- New graph-posterior or regime-aware inference method if pursuing novel
  NeurIPS-style methodological research, with fixed matched baselines and compute.
- Naturally observed missingness and execution prices/fills/fees/latency before
  claiming deployment, profit or market intervention causality.
- Real DeepLOB test checkpoint for every intended horizon; if not yet completed,
  leave the respective score blank instead of inferring it.

## Bottom line
A reproducible, compact observational temporal PGM with exact Gaussian
missing-state inference is established. The sparse graph contributes small
incremental improvements under some masks but not universal dominance.
Missing-observation movement forecasts currently lose to a simple class prior;
the external-validation, label-reconstruction and trading-economics claims
remain explicitly open.
'''
(RES/'14_consolidated_report.md').write_text(report,encoding='utf8')

# Optional self-contained evidence ZIP: only derived audit outputs, never huge input data.
import zipfile
archive=OUT/'CAUSALLOB_STEP12_EVIDENCE.zip'
tmp=Path(str(archive)+'.tmp')
with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for file in sorted(RES.glob('*')):
  if file.is_file():z.write(file,arcname='results/'+file.name)
with zipfile.ZipFile(tmp) as z: assert z.testzip() is None
# Do not ZIP best.pt/last.pt: model checkpoints remain in Drive, referenced by manifest.
tmp.replace(archive)
print('\n'+'='*100)
print('STEP 12 FINAL EVIDENCE AUDIT COMPLETE; DeepLOB completed horizons:',
      [r['horizon'] for r in deep_results])
print('Primary prior beats model in gap tests:',negative,'/ 12')
print('D8–D10 retrospective, NOT researcher-blind; true label-generation reconstruction unresolved.')
print('All frozen input hashes unchanged:',sha_before==sha_after)
print('Output:',RES)
print('ZIP:',archive)
print('Runtime:',round(time.monotonic()-START,1),'sec')


Mounted at /content/drive
PREFLIGHT PASS: 10 frozen probability parity comparisons; Step 2 labels raw-exact.
Device: cuda | DeepLOB: True | old M1 parity: {'old_M1_available': True, 'raw_identical': True, 'labels_identical': True, 'days_identical': True, 'boundaries_identical': True}
AUDIT PASS: exact labels/probabilities, matched priors, coverage, 3-day consistency and source provenance.
DeepLOB fast dataloader EXACT INPUT PARITY PASS; FP32, batch=256, full data, same architecture.

DEEPlOB TRAINING POLICY: full FP32 GPU acceleration
No shortened epochs, no subsampling, no changed batch size/architecture/schedule.
CAUTION: GPU and CPU FP32 are NOT bitwise interchangeable. Scientific model/protocol is unchanged; numerical path may differ.
DeepLOB h 10 already complete; no training or model selection rerun.
Matched DeepLOB vs DBN/PRIOR complete h 10 n 137952
DeepLOB h 20 already complete; no training or model selection rerun.
Matched DeepLOB vs DBN/PRIOR complete h 20 n 137802
CAUTION: 

In [ ]:
# CausalLOB Step 13 — DEEPLOB FORENSIC / MATCHED PREDICTIVE EVIDENCE (READ ONLY)
# GYP SELF-BRIEF: Falsify DeepLOB-vs-DBN leaderboard claims; independently align raw
# rows, labels, purges, epochs, priors and day populations. Do not refit, calibrate,
# or select using D8-D10. Separate predictive power from DBN state inference.
# Previously inspected D8-D10 are retrospective, NEVER researcher-blind.

import os, json, hashlib, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

T0=time.monotonic()
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists() and 'CAUSALLOB_ROOT' not in os.environ:
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT=Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
S2=ROOT/'STEP2_REBUILD/checkpoints'
S6=ROOT/'STEP6_FROZEN_DBN/results'
S9=ROOT/'STEP9_GAP_STRESS_READONLY/results'
S12=ROOT/'STEP12_EVIDENCE_AND_DEEPLOB/results'
DEEP=ROOT/'DEEPLOB_CPU_BASELINE/checkpoints'
OUT=ROOT/'STEP13_DEEPLOB_FORENSIC_READONLY'
R=OUT/'results';R.mkdir(parents=True,exist_ok=True)
HORIZONS=(10,20,30,50,100)
TEST_DAYS=(8,9,10)
WINDOW=100
MIN_DELTA=1e-4

# Never rely on a generated scorecard alone. Predictions and publisher labels are truth sources.
required={
 'raw':S2/'01_canonical_raw.npz',
 'boundaries':S2/'02_boundaries.npz',
 'step12_manifest':S12/'13_reproducibility_manifest.json',
 'step12_comparison':S12/'11_deeplob_matched_comparison.csv',
 'step12_claims':S12/'12_final_claims_and_completion_gates.json',
 'step9_summary':S9/'STEP9_FINAL_SUMMARY.json',
}
for h in HORIZONS:
    d=DEEP/f'h{h}'
    required[f'done_h{h}']=d/'DONE.json'
    required[f'best_h{h}']=d/'best.pt'
    required[f'history_h{h}']=d/'history.csv'
    required[f'dbn_prob_h{h}']=S6/f'03_PREDICTED_CLASS_PROBABILITIES_h{h}.npz'
    pq=d/f'test_probabilities_h{h}.parquet'
    npz=d/f'test_probabilities_h{h}.npz'
    required[f'deep_prob_h{h}']=pq if pq.exists() else npz
required['scaler']=DEEP/'train_only_scaler.npz'
missing=[f'{k}: {v}' for k,v in required.items() if not v.is_file()]
if missing:raise FileNotFoundError('Missing frozen artifacts; no retraining is attempted:\n'+'\n'.join(missing))

def sha(p):
    H=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):H.update(b)
    return H.hexdigest()

def save_csv(name,frame):
    p=R/name;tmp=p.with_name(p.name+'.tmp')
    frame.to_csv(tmp,index=False,float_format='%.14g');tmp.replace(p)

def save_json(name,obj):
    p=R/name;tmp=p.with_name(p.name+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False,default=str)+'\n',encoding='utf8')
    tmp.replace(p)

def metric(y,P):
    y=np.asarray(y,dtype=np.int64)
    P=np.asarray(P,dtype=np.float64)
    assert P.shape==(len(y),3) and np.isin(y,[0,1,2]).all()
    assert np.isfinite(P).all() and (P>=0).all()
    assert np.max(np.abs(P.sum(1)-1))<2e-5
    P=P/np.sum(P,axis=1,keepdims=True)
    pred=P.argmax(1)
    cnt=np.bincount(y,minlength=3)
    rec=np.array([(pred[y==j]==j).mean() if cnt[j] else np.nan for j in range(3)])
    F=[]
    for j in range(3):
        tp=np.sum((pred==j)&(y==j));fp=np.sum((pred==j)&(y!=j));fn=np.sum((pred!=j)&(y==j))
        F.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.)
    confidence=P.max(1);correct=(pred==y).astype(float)
    ece=0.
    for a,b in zip(np.linspace(0,1,16)[:-1],np.linspace(0,1,16)[1:]):
        ix=(confidence>=a)&(confidence<(b if b<1 else b+1e-12))
        if ix.any():ece+=ix.mean()*abs(correct[ix].mean()-confidence[ix].mean())
    yy=np.eye(3)[y]
    losses=-np.log(np.clip(P[np.arange(len(y)),y],1e-15,1))
    return dict(n=int(len(y)),accuracy=float(correct.mean()),balanced_accuracy=float(np.nanmean(rec)),
                macro_f1=float(np.mean(F)),log_loss=float(losses.mean()),
                brier_multiclass=float(np.mean(np.sum((P-yy)**2,axis=1))),
                ece_confidence=float(ece))

def eligible(day,seq,h,days):
    starts=np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]])
    ends=np.r_[starts[1:],len(seq)]
    res=[]
    for a,b in zip(starts,ends):
        if int(day[a]) not in days:continue
        lo=int(a)+WINDOW-1;hi=int(b)-1-int(h)
        if hi>=lo:
            block=np.arange(lo,hi+1,dtype=np.int64)
            assert np.all(seq[block-WINDOW+1]==seq[block])
            assert np.all(block+h<b)
            res.append(block)
    return np.concatenate(res) if res else np.array([],dtype=np.int64)

def lookahead_audit(y,P,seq,shifts=(1,5,20,100,500)):
    # Deliberate diagnosis, NOT an estimate of a tradeable future signal.
    # No circular wrapping; never compare labels across sequence boundaries.
    rows=[]
    for offset in (0,)+tuple(shifts):
        l=[];n=0
        for s in np.unique(seq):
            at=np.flatnonzero(seq==s)
            if len(at)<=offset:continue
            pp=P[at[:-offset]] if offset else P[at]
            yy=y[at[offset:]] if offset else y[at]
            l.append(float((-np.log(np.clip(pp[np.arange(len(yy)),yy],1e-15,1))).sum()))
            n+=len(yy)
        rows.append({'label_shift_forward_representation_steps':offset,'n':n,
                     'log_loss':float(sum(l)/n) if n else None})
    return rows

# Exercise numerical helpers before touching model evidence.
testy=np.array([0,1,2,0],dtype=np.int64)
testP=np.array([[.8,.1,.1],[.1,.8,.1],[.1,.1,.8],[.6,.2,.2]])
assert metric(testy,testP)['accuracy']==1.
assert abs(metric(testy,np.full((4,3),1/3))['log_loss']-np.log(3))<1e-12
assert lookahead_audit(testy,testP,np.zeros(4,dtype=int),shifts=(1,))[0]['n']==4
print('STEP13 SELF-TEST PASS: scoring, probability simplex, no-wrap alignment.')

before={k:sha(v) for k,v in required.items()}
with np.load(required['raw'],allow_pickle=False) as z:
    raw=z['X_raw'];labels=z['Y'];day=z['day']
with np.load(required['boundaries'],allow_pickle=False) as z:seq=z['sequence_id']
assert raw.shape==(394337,40) and labels.shape==(394337,5)
assert len(seq)==len(day)==len(raw) and set(np.unique(day))==set(range(1,11))
assert len(np.unique(seq))==50
starts=np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]])
ends=np.r_[starts[1:],len(seq)]
assert len(starts)==50 and all(np.unique(day[a:b]).size==1 for a,b in zip(starts,ends))
assert np.isin(labels,[1,2,3]).all()
raw_current=np.concatenate([np.arange(int(a)+20,int(b),dtype=np.int64) for a,b in zip(starts,ends)])
assert len(raw_current)==393337
with np.load(required['scaler'],allow_pickle=False) as z:
    smu=z['mean'];ssd=z['std']
train_raw=raw[np.isin(day,[1,2,3,4,5])]
assert np.allclose(smu,train_raw.mean(axis=0,dtype=np.float64),rtol=0,atol=1e-10)
assert np.allclose(ssd,train_raw.std(axis=0,dtype=np.float64),rtol=0,atol=1e-10)
assert np.all(ssd>0)
del train_raw

step12=json.loads(required['step12_manifest'].read_text())
claims=json.loads(required['step12_claims'].read_text())
assert set(claims['deep_lob_completed_horizons'])==set(HORIZONS)
assert claims['pristine_external_validation'] is False
assert claims['publisher_exact_label_formula_verified'] is False
assert claims['matched_gap_prior_better_cases']==12
assert json.loads(required['step9_summary'].read_text())['historically_blind_external_test'] is False
if 'input_sha256' in step12:
    for key in ('raw','boundary'):
        assert step12['input_sha256'][key]==before['raw' if key=='raw' else 'boundaries'], \
            'Source differs from Step12 input manifest: '+key
print('SOURCE PREFLIGHT PASS: identical raw/boundary provenance; all five DeepLOB DONE checkpoints.')

old=pd.read_csv(required['step12_comparison'])
all_scores=[]; paired=[]; cls=[]; epochs=[]; shift_rows=[]; gates=[]; matched_by_day=[]

for hidx,h in enumerate(HORIZONS):
    d=DEEP/f'h{h}'
    done=json.loads(required[f'done_h{h}'].read_text())
    assert done['horizon']==h and done['selection_metric']=='D6-D7 validation log loss'
    assert int(done['n_train_windows'])==len(eligible(day,seq,h,(1,2,3,4,5)))
    assert int(done['n_val_windows'])==len(eligible(day,seq,h,(6,7)))
    expect=eligible(day,seq,h,TEST_DAYS)
    assert int(done['n_test_windows'])==len(expect)
    hist=pd.read_csv(required[f'history_h{h}'])
    assert len(hist)>0 and hist.epoch.is_unique and np.all(np.diff(hist.epoch)==1)
    selected=int(done['selected_epoch'])
    hh=hist[hist.epoch==selected]
    assert len(hh)==1,'Selected best epoch absent from training history'
    best_val=float(hh.iloc[0].val_log_loss)
    assert abs(float(done['best_validation']['log_loss'])-best_val)<1e-5
    assert best_val<=float(hist.val_log_loss.min())+MIN_DELTA+1e-8
    # Best checkpoint must correspond to chosen validation epoch. Reading weights is NOT training.
    try:
        import torch
        try:best=torch.load(required[f'best_h{h}'],map_location='cpu',weights_only=False)
        except TypeError:best=torch.load(required[f'best_h{h}'],map_location='cpu')
        assert int(best['epoch'])==selected and int(best['horizon'])==h
        assert abs(float(best['val_metrics']['log_loss'])-best_val)<1e-5
        assert best['model_state']['fc.weight'].shape[0]==3
        del best
        verified_checkpoint=True
    except ImportError:
        verified_checkpoint=False  # Colab installs torch; absence is visible, never claimed verified.
    epochs.append({'horizon':h,'selected_epoch':selected,'recorded_epochs':len(hist),
                   'minimum_recorded_validation_nll':float(hist.val_log_loss.min()),
                   'selected_validation_nll':best_val,
                   'saved_best_checkpoint_verified':verified_checkpoint,
                   'accelerator_recorded':done.get('accelerator','cpu / legacy')})

    pth=required[f'deep_prob_h{h}']
    if pth.suffix=='.parquet':dd=pd.read_parquet(pth)
    else:
        with np.load(pth,allow_pickle=False) as z:
            dd=pd.DataFrame({k:z[k] for k in ('raw_row','day','y_ord','p_down','p_stationary','p_up')})
    rows=dd.raw_row.to_numpy(dtype=np.int64)
    assert np.array_equal(rows,expect),'DeepLOB test rows differ from predeclared h-purged cohort'
    assert np.array_equal(dd.day.to_numpy(),day[rows])
    original_label=labels[rows,hidx]
    y=np.select([original_label==3,original_label==2,original_label==1],[0,1,2],default=-1)
    assert np.array_equal(y,dd.y_ord.to_numpy()),'DeepLOB labels not original publisher labels'
    Deep=dd[['p_down','p_stationary','p_up']].to_numpy(dtype=np.float64)
    assert Deep.shape==(len(expect),3) and np.isfinite(Deep).all()
    assert (Deep>=0).all() and np.max(abs(Deep.sum(1)-1))<2e-5
    # Independently check the h-specific original DeepLOB test result.
    dm=metric(y,Deep)
    assert abs(dm['log_loss']-float(done['test_pooled']['log_loss']))<5e-6, \
       'DeepLOB saved probabilities disagree with DONE test loss'

    with np.load(required[f'dbn_prob_h{h}'],allow_pickle=False) as z:
        ix=z['row_index'].astype(np.int64)
        assert np.all(np.diff(ix)>0) and ix.min()>=0 and ix.max()<len(raw_current)
        br=raw_current[ix]
        dbn_all=z['p_down_neutral_up'].astype(np.float64)
        y_all=z['label_ordered'].astype(np.int64)
        original_all=z['original_label'].astype(np.int8)
        prior_all=z['prior_down_neutral_up'].astype(np.float64)
        dy=z['day'].astype(np.int8)
    assert np.array_equal(dy,day[br]) and np.array_equal(original_all,labels[br,hidx])
    assert np.array_equal(y_all,np.select([original_all==3,original_all==2,original_all==1],[0,1,2],default=-1))
    assert len(np.unique(br))==len(br) and np.all(np.diff(br)>0)
    indices,ai,bi=np.intersect1d(rows,br,assume_unique=True,return_indices=True)
    assert len(indices)==len(rows),'DeepLOB rows not fully represented in Step6 DBN output'
    assert np.array_equal(indices,rows) and np.array_equal(y[ai],y_all[bi])
    Pdbn=dbn_all[bi]
    Pprior=np.broadcast_to(prior_all,(len(rows),3))
    assert abs(prior_all.sum()-1)<1e-10

    # A second comparator uses ONLY the D1–D5 DeepLOB eligible training cohort.
    train_rows=eligible(day,seq,h,(1,2,3,4,5))
    train_y=np.select([labels[train_rows,hidx]==3,labels[train_rows,hidx]==2,labels[train_rows,hidx]==1],
                      [0,1,2],default=-1)
    pr5=np.bincount(train_y,minlength=3)/len(train_y)
    Pprior5=np.broadcast_to(pr5,(len(rows),3))
    models={'DEEPLOB':Deep,'DBN_COMPLETE':Pdbn,'PRIOR_D1_D7':Pprior,
            'PRIOR_D1_D5':Pprior5}
    cohorts=[('POOLED',np.ones(len(rows),dtype=bool))]+[(str(k),day[rows]==k) for k in TEST_DAYS]
    for dayname,sel in cohorts:
        assert sel.any()
        ys=y[sel]
        for modelname,P in models.items():
            met=metric(ys,P[sel]); all_scores.append({'horizon':h,'day':dayname,'model':modelname,**met})
            if modelname in ('DEEPLOB','DBN_COMPLETE','PRIOR_D1_D7'):
                labelmap={'DEEPLOB':'DEEPLOB_SAME_COHORT',
                          'DBN_COMPLETE':'FROZEN_DBN_HEAD_SAME_COHORT',
                          'PRIOR_D1_D7':'TRAIN_PRIOR_SAME_COHORT'}[modelname]
                oldline=old[(old.horizon==h)&(old.day.astype(str)==dayname)&(old.model==labelmap)]
                assert len(oldline)==1,'Step12 matched-scorecard row absent'
                o=oldline.iloc[0]
                assert met['n']==int(o.n) and abs(met['log_loss']-float(o.log_loss))<5e-6, \
                    f'Step12 comparison differs: h{h} day {dayname} {modelname}'
        for comparator in ('DBN_COMPLETE','PRIOR_D1_D7','PRIOR_D1_D5'):
            La=-np.log(np.clip(Deep[sel,np.asarray(ys)],1e-15,1))
            Pb=models[comparator][sel]
            Lb=-np.log(np.clip(Pb[np.arange(len(ys)),ys],1e-15,1))
            paired.append({'horizon':h,'day':dayname,'comparison':'DEEPLOB_VS_'+comparator,
                          'n':len(ys),'mean_logloss_gain_positive_deep_better':float((Lb-La).mean()),
                          'mean_brier_gain_positive_deep_better':float(np.mean(np.sum((Pb-np.eye(3)[ys])**2,axis=1)-
                                                           np.sum((Deep[sel]-np.eye(3)[ys])**2,axis=1)))})
        cls.append({'horizon':h,'day':dayname,'n':len(ys),
                    **{f'true_share_{j}':float((ys==j).mean()) for j in range(3)},
                    **{f'deep_pred_share_{j}':float((Deep[sel].argmax(1)==j).mean()) for j in range(3)},
                    **{f'dbn_pred_share_{j}':float((Pdbn[sel].argmax(1)==j).mean()) for j in range(3)},
                    'deep_mean_max_confidence':float(Deep[sel].max(1).mean()),
                    'dbn_mean_max_confidence':float(Pdbn[sel].max(1).mean())})

    for r in lookahead_audit(y,Deep,seq[rows]):shift_rows.append({'horizon':h,'model':'DEEPLOB',**r})
    for r in lookahead_audit(y,Pdbn,seq[rows]):shift_rows.append({'horizon':h,'model':'DBN_COMPLETE',**r})
    gates.append({'horizon':h,'deep_epoch_selection_verified':verified_checkpoint,
                  'publisher_labels_exact':True,'predeclared_purged_rows_exact':True,
                  'source_split_and_window_boundaries_verified':True,
                  'step12_metric_parity':True,
                  'deep_train_rows':int(done['n_train_windows']),
                  'dbn_train_days':'1-7','deep_train_days':'1-5',
                  'deep_inputs':'40 raw x 100 past/current steps',
                  'dbn_inputs':'6 current standardized state variables'})
    print(f'h{h}: EXACT cohort/label/Step12 parity PASS; selected epoch {selected}; '
          f'matched DeepLOB NLL={metric(y,Deep)["log_loss"]:.6f}, DBN NLL={metric(y,Pdbn)["log_loss"]:.6f}')

score=pd.DataFrame(all_scores);pair=pd.DataFrame(paired);classdf=pd.DataFrame(cls)
epochdf=pd.DataFrame(epochs);shift=pd.DataFrame(shift_rows);gatedf=pd.DataFrame(gates)
# Day-level consistency: THREE dependent financial day blocks. No row-level IID p-values.
consistency=[]
for (h,comp),g in pair[pair.day!='POOLED'].groupby(['horizon','comparison']):
    assert set(g.day)=={'8','9','10'}
    vals=g.mean_logloss_gain_positive_deep_better.to_numpy()
    consistency.append({'horizon':int(h),'comparison':comp,
                        'positive_days':int((vals>0).sum()),'negative_days':int((vals<0).sum()),
                        'min_gain':float(vals.min()),'max_gain':float(vals.max()),
                        'unweighted_day_mean_gain':float(vals.mean()),
                        'inferential_blocks':3})
cons=pd.DataFrame(consistency)
for name,df in [
 ('01_matched_prediction_scorecard.csv',score),
 ('02_paired_logloss_and_brier_gains.csv',pair),
 ('03_class_mix_and_confidence.csv',classdf),
 ('04_validation_epoch_integrity.csv',epochdf),
 ('05_no_wrap_temporal_shift_diagnostic.csv',shift),
 ('06_three_day_consistency.csv',cons),
 ('07_protocol_gates.csv',gatedf),
]:save_csv(name,df)

# Freeze metadata; do not mistake dataset coverage for independent external validation.
flag={
 'status':'RETROSPECTIVE_FORENSIC_AUDIT_COMPLETE',
 'all_10_FI_days_in_source':True,
 'D1_D5_deeplob_training':True,
 'D6_D7_deeplob_validation_selection':True,
 'D1_D7_dbn_training':True,
 'D8_D10_historically_researched_exposed':True,
 'independent_external_test':False,
 'publisher_label_generation_exactly_reproduced':False,
 'missing_observation_deployed_prediction_beats_prior':False,
 'DeepLOB_all_5_horizons_verified':True,
 'DeepLOB_feature_history_and_DBn_feature_capacity_matched':False,
 'identical_test_rows_per_horizon':True,
 'statistically_independent_test_blocks':3,
 'profitability_or_fill_simulation':False,
 'model_refit_or_calibration_done_in_step13':False,
 'recommendation':'Freeze retrospective results; preregister and acquire a genuinely unexamined external dataset/period.'
}
save_json('08_claim_gates.json',flag)
after={k:sha(v) for k,v in required.items()}
assert before==after,'Frozen source changed during audit: STOP.'
save_json('09_input_hash_manifest.json',{'files':{k:{'path':str(p),'sha256':before[k]} for k,p in required.items()},
                                      'all_upstream_source_hashes_unchanged':True})

lines=['# CausalLOB — Step 13 independent DeepLOB forensic audit','',
       '## Protocol and qualification',
       '- This is a read-only, retrospective audit of already-completed models. D8–D10 were previously inspected.',
       '- DeepLOB: 40 raw LOB variables × 100 backward representations; train D1–D5; epoch select D6–D7.',
       '- Gaussian DBN head: six current state variables; parameters fit D1–D7; same D8–D10 target rows.',
       '- Test rows, publisher labels, horizon purges, saved logits-derived probabilities and saved benchmark metrics were checked.',
       '- Predictor input sets, capacities and training periods are NOT equal. Do not imply a controlled architecture comparison.',
       '- Independent source-label generation is unresolved; preserving publisher labels is not reconstructing their generation.',
       '- Trading costs, fills, net PnL and intervention-level causality have not been tested.','',
       '## Matched test scores',
       score[score.day=='POOLED'][['horizon','model','n','balanced_accuracy','macro_f1','log_loss','brier_multiclass','ece_confidence']].to_string(index=False),
       '', '## Paired log-loss differences (positive means DeepLOB lower loss)',
       pair[pair.day=='POOLED'][['horizon','comparison','n','mean_logloss_gain_positive_deep_better']].to_string(index=False),
       '', '## Three-day sign consistency (descriptive; no row-level IID inference)',
       cons.to_string(index=False),
       '', '## Interpretation',
       '- Report state reconstruction and outcome prediction as different objectives.',
       '- Do not extrapolate DeepLOB superiority on complete observations to missing-data outcomes.',
       '- The 12/12 missing-gap prior losses remain an explicit failure for the frozen CausalLOB head.',
       '- A high shifted-label score can reflect persistent labels, not necessarily leakage; no shift score alone proves either.',
       '', '## The next scientific step',
       'Acquire genuinely unseen, permitted order-book data or an untouched period. Freeze feature semantics, exact target-generation code, horizon units, baselines, missingness masks and scoring before inspecting any outcomes. If this cannot be done, submit the completed FI-2010 course project with the above limitations and do not claim external generalization or publication readiness.']
(R/'10_forensic_report.md').write_text('\n'.join(lines)+'\n',encoding='utf8')
arch=OUT/'CAUSALLOB_STEP13_FORENSIC_RESULTS.zip'
tmp=arch.with_suffix('.zip.tmp')
with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(R.glob('*')):
        if p.is_file():z.write(p,arcname='results/'+p.name)
with zipfile.ZipFile(tmp) as z:assert z.testzip() is None
tmp.replace(arch)
print('\n'+'='*100)
print('STEP 13 COMPLETE: DEEPLOB / DBN MATCHED-ROW FORENSIC AUDIT')
print('='*100)
print(score[score.day=='POOLED'][['horizon','model','n','balanced_accuracy','log_loss','brier_multiclass']].to_string(index=False))
print('\nDEEPLOB vs DBN day-level paired log-loss:')
print(cons[cons.comparison=='DEEPLOB_VS_DBN_COMPLETE'].to_string(index=False))
print('\nSOURCE HASH PARITY PASS:',before==after)
print('D8–D10 retrospective only. External validation and original label generation UNRESOLVED.')
print('Outputs:',R)
print('ZIP:',arch)
print('Runtime:',round(time.monotonic()-T0,1),'sec')

STEP13 SELF-TEST PASS: scoring, probability simplex, no-wrap alignment.
SOURCE PREFLIGHT PASS: identical raw/boundary provenance; all five DeepLOB DONE checkpoints.
h10: EXACT cohort/label/Step12 parity PASS; selected epoch 10; matched DeepLOB NLL=0.454644, DBN NLL=0.498405
h20: EXACT cohort/label/Step12 parity PASS; selected epoch 5; matched DeepLOB NLL=0.645857, DBN NLL=0.714198
h30: EXACT cohort/label/Step12 parity PASS; selected epoch 6; matched DeepLOB NLL=0.601543, DBN NLL=0.936842
h50: EXACT cohort/label/Step12 parity PASS; selected epoch 4; matched DeepLOB NLL=0.560573, DBN NLL=1.071103
h100: EXACT cohort/label/Step12 parity PASS; selected epoch 3; matched DeepLOB NLL=0.569517, DBN NLL=1.147715

STEP 13 COMPLETE: DEEPLOB / DBN MATCHED-ROW FORENSIC AUDIT
 horizon        model      n  balanced_accuracy  log_loss  brier_multiclass
      10      DEEPLOB 137952           0.676273  0.454644          0.242605
      10 DBN_COMPLETE 137952           0.727736  0.498405          0.236803


In [ ]:
# CausalLOB STEP 14 — frozen DeepLOB forecast-origin and label-provenance audit.
# GYP SELF-BRIEF: Try to falsify the surprising long-horizon benchmark. Verify the
# actual 100-row inputs, frozen checkpoint replay, training-only normalization,
# target-row alignment, sequence/horizon exclusions and selection chronology.
# Do NOT train, alter labels, calibrate, select architectures, or optimize D8–D10.
# Numeric agreement verifies THIS inference implementation and saved checkpoint;
# without original message-level label-generation data, exact publisher label
# semantics are still unresolved. D8–D10 are historically exposed.

import os
import json
import time
import hashlib
import zipfile
import gc
from pathlib import Path

import numpy as np
import pandas as pd

HORIZONS = (10, 20, 30, 50, 100)
K_REP = (1, 2, 3, 5, 10)
WINDOW = 100
BATCH = 256
TEST_DAYS = (8, 9, 10)
TRAIN_DAYS = (1, 2, 3, 4, 5)
VAL_DAYS = (6, 7)
SHIFTS = (-100, -20, -5, -1, 0, 1, 5, 20, 100)
VERSION = 'STEP14_FORECAST_ORIGIN_v1_2026-09-19'
# Full replay is a real, forward-only audit, not training. GPU recommended.
FULL_CHECKPOINT_REPLAY = True
# Strict but not bitwise: CPU/GPU FP32 convolution kernels may vary slightly.
REPLAY_MAX_PROB_ERROR = 3e-3
REPLAY_MEAN_PROB_ERROR = 2e-4
REPLAY_NLL_ERROR = 5e-4


def filehash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def require(path):
    if not path.is_file():
        raise FileNotFoundError('Required frozen input not found: ' + str(path))
    return path


def atomic_text(path, text):
    path = Path(path)
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(text, encoding='utf-8')
    temp.replace(path)


def save_csv(path, frame):
    atomic_text(path, frame.to_csv(index=False, float_format='%.14g'))


def save_json(path, value):
    atomic_text(path, json.dumps(value, indent=2, sort_keys=True,
                                 allow_nan=False, default=str) + '\n')


def eligible_indices(day, seq, horizon, days):
    starts = np.flatnonzero(np.r_[True, seq[1:] != seq[:-1]])
    ends = np.r_[starts[1:], len(seq)]
    chunks = []
    details = []
    for begin, stop in zip(starts, ends):
        begin = int(begin)
        stop = int(stop)
        if int(day[begin]) not in days:
            continue
        first = begin + WINDOW - 1
        last = stop - 1 - horizon  # deliberately conservative old protocol
        if first <= last:
            rr = np.arange(first, last + 1, dtype=np.int64)
            assert (rr - WINDOW + 1 >= begin).all()
            assert (rr + horizon < stop).all()
            chunks.append(rr)
            details.append((int(day[begin]), int(seq[begin]),begin,stop,int(rr.size)))
    return np.concatenate(chunks) if chunks else np.empty(0, dtype=np.int64), details


def ordinal_y(labels):
    labels = np.asarray(labels)
    assert np.isin(labels, [1, 2, 3]).all()
    return np.where(labels == 3, 0, np.where(labels == 2, 1, 2)).astype(np.int64)


def nll(y, p):
    y = np.asarray(y, dtype=np.int64)
    p = np.asarray(p, dtype=np.float64)
    assert p.shape == (len(y), 3)
    assert np.isfinite(p).all() and (p >= 0).all()
    assert np.max(np.abs(p.sum(axis=1) - 1)) < 2e-5
    return float(np.mean(-np.log(np.clip(p[np.arange(len(y)), y], 1e-15, 1))))


def ece_and_class_stats(y, p):
    p = np.asarray(p, dtype=np.float64)
    hard = p.argmax(1)
    rec = [(hard[y == cls] == cls).mean() if np.any(y == cls) else np.nan
           for cls in range(3)]
    return {
        'ordinary_accuracy': float(np.mean(hard == y)),
        'balanced_accuracy': float(np.nanmean(rec)),
        'class_recall_down': float(rec[0]),
        'class_recall_neutral': float(rec[1]),
        'class_recall_up': float(rec[2]),
        'log_loss': nll(y, p),
    }


def shift_pair_indices(seq, shift):
    # Source prediction at index i; target at index i+shift.
    # Only contiguous same-sequence blocks, no cyclic wrapping or cross-day pairs.
    src, dst = [], []
    begins = np.flatnonzero(np.r_[True, seq[1:] != seq[:-1]])
    stops = np.r_[begins[1:], len(seq)]
    for a, b in zip(begins, stops):
        n = int(b - a)
        s = abs(int(shift))
        if n <= s:
            continue
        if shift >= 0:
            src.append(np.arange(a, b - s, dtype=np.int64))
            dst.append(np.arange(a + s, b, dtype=np.int64))
        else:
            src.append(np.arange(a + s, b, dtype=np.int64))
            dst.append(np.arange(a, b - s, dtype=np.int64))
    return (np.concatenate(src), np.concatenate(dst)) if src else (
        np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64))



def audit_done_metadata(done, horizon):
    """Fail closed for new-format checkpoints; handle known h10/h20 legacy DONE schema.

    Original CPU training DONE did not record batch_size/window. The original
    training source declares 256/100, and exact reconstructed windows and
    checkpoint replay are checked independently below, but a legacy DONE
    alone cannot establish the original training batch size.
    """
    if done.get('version') != 'DEEPLOB_CPU_v1_2026-09-14':
        raise RuntimeError('Unexpected DeepLOB DONE checkpoint version for h'+str(horizon))
    missing = []
    for field, expected in (('batch_size', BATCH), ('window', WINDOW)):
        if field not in done or done[field] is None:
            missing.append(field)
        elif int(done[field]) != expected:
            raise RuntimeError(
                f'DeepLOB h{horizon} DONE {field}={done[field]!r}; expected {expected}. '
                'Do not bypass a conflicting training setting.'
            )
    if missing and int(horizon) not in (10, 20):
        raise RuntimeError(
            f'DeepLOB h{horizon} is expected to have Step12 DONE metadata; '
            f'missing fields {missing}. Inspect checkpoint provenance.'
        )
    if missing:
        print(
            f'LEGACY DONE h{horizon}: missing {missing}; original CPU training source '
            'declares batch=256 and window=100. DONE alone cannot verify '
            'those settings; independent window/checkpoint replay continues.'
        )
        provenance = 'LEGACY_DONE_METADATA_ABSENT_SOURCE_AND_REPLAY_AUDIT'
    else:
        provenance = 'DONE_METADATA_EXPLICIT_MATCH'
    return {
        'horizon': int(horizon),
        'done_format': provenance,
        'batch_size_done_recorded': 'batch_size' not in missing,
        'window_done_recorded': 'window' not in missing,
        'missing_optional_fields': ','.join(missing),
        'training_batch_independently_proven_by_done': 'batch_size' not in missing,
    }

def self_test():
    seq = np.array([0,0,0,0,1,1,1,1,1])
    for shift in (-2,-1,0,1,2):
        a,b=shift_pair_indices(seq,shift)
        assert np.array_equal(b-a,np.full(len(a),shift))
        assert np.array_equal(seq[a],seq[b])
        assert len(a)==9-2*abs(shift)
    y=np.array([0,1,2,0],dtype=np.int64)
    p=np.eye(3)[y]*.8+.2/3
    assert nll(y,p)<np.log(3)
    assert ece_and_class_stats(y,p)['ordinary_accuracy']==1
    # Fixed input-end chronology test: end at origin t, never t+1.
    t=np.array([105,155],dtype=np.int64)
    pos=t[:,None]-np.arange(WINDOW-1,-1,-1)[None,:]
    assert np.array_equal(pos[:,-1],t) and np.array_equal(pos[:,0],t-99)
    legacy={'version':'DEEPLOB_CPU_v1_2026-09-14'}
    assert not audit_done_metadata(legacy,10)['training_batch_independently_proven_by_done']
    assert audit_done_metadata({**legacy,'batch_size':BATCH,'window':WINDOW},30)['training_batch_independently_proven_by_done']
    for invalid,h in [({**legacy,'batch_size':999,'window':WINDOW},10), (legacy,30)]:
        try:
            audit_done_metadata(invalid,h)
        except RuntimeError:
            pass
        else:
            raise AssertionError('DONE schema audit did not reject an invalid configuration')
    print('STEP14 SELF-TEST PASS: shifts, scoring, forecast origin, legacy/new DONE schemas.')


def model_class(torch, nn):
    # Architecture reproduced from the original Step12 saved training definition.
    class ConvBNAct(nn.Module):
        def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(in_ch,out_ch,kernel_size=kernel_size,stride=stride,
                          padding=padding,bias=False),
                nn.BatchNorm2d(out_ch),nn.LeakyReLU(0.01,inplace=True))
        def forward(self,x):return self.net(x)

    class DeepLOB(nn.Module):
        def __init__(self,n_classes=3):
            super().__init__()
            self.block1=nn.Sequential(
                ConvBNAct(1,32,(1,2),stride=(1,2)),
                ConvBNAct(32,32,(4,1),padding=(2,0)),
                ConvBNAct(32,32,(4,1),padding=(1,0)))
            self.block2=nn.Sequential(
                ConvBNAct(32,32,(1,2),stride=(1,2)),
                ConvBNAct(32,32,(4,1),padding=(2,0)),
                ConvBNAct(32,32,(4,1),padding=(1,0)))
            self.block3=nn.Sequential(
                ConvBNAct(32,32,(1,10)),
                ConvBNAct(32,32,(4,1),padding=(2,0)),
                ConvBNAct(32,32,(4,1),padding=(1,0)))
            self.i1=nn.Sequential(ConvBNAct(32,64,(1,1)),
                                  ConvBNAct(64,64,(3,1),padding=(1,0)))
            self.i2=nn.Sequential(ConvBNAct(32,64,(1,1)),
                                  ConvBNAct(64,64,(5,1),padding=(2,0)))
            self.i3=nn.Sequential(nn.MaxPool2d((3,1),stride=(1,1),padding=(1,0)),
                                  ConvBNAct(32,64,(1,1)))
            self.lstm=nn.LSTM(input_size=192,hidden_size=64,num_layers=1,batch_first=True)
            self.fc=nn.Linear(64,n_classes)
        def forward(self,x):
            x=self.block1(x);x=self.block2(x);x=self.block3(x)
            a,b,c=self.i1(x),self.i2(x),self.i3(x)
            tmin=min(a.shape[2],b.shape[2],c.shape[2])
            x=torch.cat([a[:,:,:tmin,:],b[:,:,:tmin,:],c[:,:,:tmin,:]],dim=1)
            if x.shape[-1]!=1:
                raise RuntimeError('Unexpected feature width in model')
            x=x.squeeze(-1).permute(0,2,1).contiguous()
            x,_=self.lstm(x)
            return self.fc(x[:,-1,:])
    return DeepLOB


def replay_frozen(torch, Net, bestpath, expected_rows, Xz, device, stored):
    # Replay *all* matched held-out rows directly from immutable weights and raw inputs.
    try:
        ck=torch.load(bestpath,map_location='cpu',weights_only=False)
    except TypeError:
        ck=torch.load(bestpath,map_location='cpu')
    m=Net().to(device)
    m.load_state_dict(ck['model_state'],strict=True)
    m.eval()  # BatchNorm uses frozen running statistics; dropout (if any) inactive.
    out=np.empty((len(expected_rows),3),dtype=np.float64)
    offsets=np.arange(WINDOW-1,-1,-1,dtype=np.int64)
    with torch.inference_mode():
        for j in range(0,len(expected_rows),BATCH):
            rr=expected_rows[j:j+BATCH]
            positions=rr[:,None]-offsets[None,:]
            if not np.array_equal(positions[:,-1],rr):
                raise AssertionError('Future-state leakage in replay window ends')
            batch=np.ascontiguousarray(Xz[positions][:,None,:,:])
            assert batch.dtype==np.float32 and batch.shape==(len(rr),1,WINDOW,40)
            logits=m(torch.from_numpy(batch).to(device))
            out[j:j+len(rr)]=torch.softmax(logits,dim=1).cpu().numpy()
        if device.type=='cuda':torch.cuda.synchronize()
    assert np.isfinite(out).all() and np.max(np.abs(out.sum(1)-1))<2e-5
    diff=np.abs(out-stored)
    result={
        'replayed_n':len(expected_rows),
        'replay_device':str(device),
        'max_probability_absolute_error':float(diff.max()),
        'mean_probability_absolute_error':float(diff.mean()),
    }
    del m,ck
    gc.collect()
    if device.type=='cuda':torch.cuda.empty_cache()
    return out,result


def main():
    self_test()
    tic=time.monotonic()
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists() and 'CAUSALLOB_ROOT' not in os.environ:
            drive.mount('/content/drive')
    except ImportError:
        pass
    root=Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
    s2=root/'STEP2_REBUILD/checkpoints'
    s5=root/'STEP5_LABEL_AUDIT_READONLY/results'
    s6=root/'STEP6_FROZEN_DBN/results'
    s12=root/'STEP12_EVIDENCE_AND_DEEPLOB/results'
    s13=root/'STEP13_DEEPLOB_FORENSIC_READONLY/results'
    deep=root/'DEEPLOB_CPU_BASELINE/checkpoints'
    out=root/'STEP14_FORECAST_ORIGIN_READONLY'
    resultdir=out/'results'
    # Failure to find a prerequisite is a hard stop; never download/retrain/fabricate it.
    required={
      'raw':require(s2/'01_canonical_raw.npz'),
      'boundaries':require(s2/'02_boundaries.npz'),
      'scaler':require(deep/'train_only_scaler.npz'),
      'step5_summary':require(s5/'05_LABEL_AUDIT_SUMMARY.json'),
      'step5_formula':require(s5/'03_fixed_formula_hypotheses_D1_D7.csv'),
      'step12_manifest':require(s12/'13_reproducibility_manifest.json'),
      'step13_gates':require(s13/'08_claim_gates.json'),
      'step13_hashes':require(s13/'09_input_hash_manifest.json'),
      'step13_scores':require(s13/'01_matched_prediction_scorecard.csv'),
    }
    for h in HORIZONS:
        hd=deep/f'h{h}'
        required[f'done_{h}']=require(hd/'DONE.json')
        required[f'best_{h}']=require(hd/'best.pt')
        required[f'history_{h}']=require(hd/'history.csv')
        required[f'dbn_{h}']=require(s6/f'03_PREDICTED_CLASS_PROBABILITIES_h{h}.npz')
        prob_parquet=hd/f'test_probabilities_h{h}.parquet'
        prob_npz=hd/f'test_probabilities_h{h}.npz'
        required[f'deep_{h}']=require(prob_parquet if prob_parquet.is_file() else prob_npz)
    hashes_before={key:filehash(path) for key,path in required.items()}
    previous=resultdir/'00_AUDIT_SPEC.json'
    spec={'version':VERSION,'inputs':hashes_before,'window':WINDOW,'batch':BATCH,
          'horizons':HORIZONS,'shifts':SHIFTS,'full_checkpoint_replay':FULL_CHECKPOINT_REPLAY,
          'gpu_cpu_max_tolerance':REPLAY_MAX_PROB_ERROR,
          'mean_probability_tolerance':REPLAY_MEAN_PROB_ERROR,
          'loss_tolerance':REPLAY_NLL_ERROR,'no_fit_or_selection':True}
    fingerprint=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()
    if previous.exists():
        old=json.loads(previous.read_text())
        if old['fingerprint']!=fingerprint:
            raise RuntimeError('Step14 already exists with different frozen inputs/spec. '
                               'Archive existing Step14 directory before rerunning.')
    resultdir.mkdir(parents=True,exist_ok=True)
    save_json(previous,{'fingerprint':fingerprint,'spec':spec})

    source_manifest=json.loads(required['step13_hashes'].read_text())
    for k in ('raw','boundaries','scaler'):
        prior=source_manifest['files'][k]['sha256']
        assert hashes_before[k]==prior, f'Frozen source differs from Step13: {k}'
    gates13=json.loads(required['step13_gates'].read_text())
    assert gates13['DeepLOB_all_5_horizons_verified'] is True
    assert gates13['independent_external_test'] is False
    label5=json.loads(required['step5_summary'].read_text())
    assert label5['matched_publisher_label_array_to_temporal'] is True
    assert label5['best_approximate_reconstruction_agreement_is_not_ground_truth'] is True

    with np.load(required['raw'],allow_pickle=False) as z:
        raw=z['X_raw']; labels=z['Y']; day=z['day']
    with np.load(required['boundaries'],allow_pickle=False) as z:
        seq=z['sequence_id']
    with np.load(required['scaler'],allow_pickle=False) as z:
        mean=z['mean'];std=z['std']
    assert raw.shape==(394337,40) and labels.shape==(394337,5)
    assert day.shape==(len(raw),) and seq.shape==(len(raw),)
    assert np.array_equal(np.unique(day),np.arange(1,11))
    assert len(np.unique(seq))==50 and np.isin(labels,[1,2,3]).all()
    assert np.isfinite(raw).all() and (std>0).all()
    starts=np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]])
    ends=np.r_[starts[1:],len(raw)]
    assert len(starts)==50 and all(np.unique(day[a:b]).size==1 for a,b in zip(starts,ends))
    train=raw[np.isin(day,TRAIN_DAYS)]
    assert np.allclose(mean,train.mean(axis=0,dtype=np.float64),rtol=0,atol=1e-10)
    assert np.allclose(std,train.std(axis=0,dtype=np.float64),rtol=0,atol=1e-10)
    del train
    xz=((raw-mean.astype(np.float32))/std.astype(np.float32)).astype(np.float32,copy=False)
    assert xz.shape==raw.shape and xz.dtype==np.float32
    with np.load(required['raw'],allow_pickle=False) as z:
        assert np.array_equal(z['Y'],labels),'Labels changed during audit.'

    import torch
    import torch.nn as nn
    torch.set_num_threads(min(4,max(1,os.cpu_count() or 2)))
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32=False
        torch.backends.cudnn.allow_tf32=False
        torch.backends.cudnn.benchmark=False
        torch.backends.cudnn.deterministic=True
        torch.set_float32_matmul_precision('highest')
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    Net=model_class(torch,nn)
    print('STEP14 PREFLIGHT PASS: 10 days, 50 sequences, frozen checkpoints, D1–D5 scaler exact.')
    print('Inference device:',device,'| Full frozen checkpoint replay:',FULL_CHECKPOINT_REPLAY)
    print('Frozen publisher labels retained. D8–D10 are historically exposed.')

    cohort_records=[];window_records=[];class_records=[];shift_records=[]
    replay_records=[];cal_records=[];label_records=[];checks=[];metadata_records=[]
    score13=pd.read_csv(required['step13_scores'])

    for hidx,(h,k) in enumerate(zip(HORIZONS,K_REP)):
        dd=json.loads(required[f'done_{h}'].read_text())
        assert int(dd['horizon'])==h and dd['selection_metric']=='D6-D7 validation log loss'
        metadata_records.append(audit_done_metadata(dd,h))
        assert int(dd['selected_epoch'])>=1
        hist=pd.read_csv(required[f'history_{h}'])
        assert int(dd['selected_epoch']) in hist.epoch.astype(int).values
        chosen=hist.loc[hist.epoch==int(dd['selected_epoch'])].iloc[0]
        assert abs(float(chosen.val_log_loss)-float(dd['best_validation']['log_loss']))<1e-5
        assert float(chosen.val_log_loss)<=float(hist.val_log_loss.min())+1.0001e-4
        expected,details=eligible_indices(day,seq,h,TEST_DAYS)
        train_rows,_=eligible_indices(day,seq,h,TRAIN_DAYS)
        val_rows,_=eligible_indices(day,seq,h,VAL_DAYS)
        assert len(expected)==int(dd['n_test_windows'])
        assert len(train_rows)==int(dd['n_train_windows'])
        assert len(val_rows)==int(dd['n_val_windows'])
        assert len(np.intersect1d(train_rows,expected))==0
        assert len(np.intersect1d(val_rows,expected))==0
        assert expected.size>0 and (np.diff(expected)>0).all()
        assert np.isin(day[train_rows],TRAIN_DAYS).all()
        assert np.isin(day[val_rows],VAL_DAYS).all()
        assert np.isin(day[expected],TEST_DAYS).all()
        # Full-cohort analytical check, not just sampled windows.
        assert np.array_equal(seq[expected-WINDOW+1],seq[expected])
        assert np.array_equal(day[expected-WINDOW+1],day[expected])
        remaining=np.empty(len(seq),dtype=np.int32)
        for a,b in zip(starts,ends):remaining[a:b]=np.arange(b-a-1,-1,-1)
        assert (remaining[expected]>=h).all()
        assert expected[-1]<len(seq)
        cohort_records.extend([{
            'horizon_underlying_events':h,'horizon_representation_steps':k,
            'split':name,'n':len(rr), 'min_day':int(day[rr].min()),
            'max_day':int(day[rr].max()),'window_length':WINDOW,
            'future_raw_rows_in_window':0,'label_tail_purge_rows':h,
        } for name,rr in [('TRAIN',train_rows),('VALIDATION',val_rows),('RETROSPECTIVE',expected)]])

        dpath=required[f'deep_{h}']
        if dpath.suffix=='.parquet':frame=pd.read_parquet(dpath)
        else:
            with np.load(dpath,allow_pickle=False) as z:
                frame=pd.DataFrame({name:z[name] for name in
                    ('raw_row','day','y_ord','p_down','p_stationary','p_up')})
        rr=frame.raw_row.to_numpy(dtype=np.int64)
        y=ordinal_y(labels[expected,hidx]);stored=frame[['p_down','p_stationary','p_up']].to_numpy(np.float64)
        assert np.array_equal(rr,expected)
        assert np.array_equal(frame.day.to_numpy(),day[rr])
        assert np.array_equal(frame.y_ord.to_numpy(dtype=np.int64),y)
        assert stored.shape==(len(rr),3) and np.isfinite(stored).all()
        assert np.max(abs(stored.sum(1)-1))<2e-5
        assert abs(nll(y,stored)-float(dd['test_pooled']['log_loss']))<5e-6
        oldscore=score13[(score13.horizon==h)&(score13.day.astype(str)=='POOLED')&
                         (score13.model=='DEEPLOB')]
        assert len(oldscore)==1 and abs(nll(y,stored)-float(oldscore.iloc[0].log_loss))<5e-6

        with np.load(required[f'dbn_{h}'],allow_pickle=False) as z:
            zi=z['row_index'].astype(np.int64)
            dp=z['p_down_neutral_up'].astype(np.float64)
            prior=z['prior_down_neutral_up'].astype(np.float64)
            ydbn=z['label_ordered'].astype(np.int64)
        current=np.concatenate([np.arange(a+20,b,dtype=np.int64) for a,b in zip(starts,ends)])
        br=current[zi]
        common,ia,ib=np.intersect1d(rr,br,return_indices=True,assume_unique=True)
        assert np.array_equal(common,rr) and np.array_equal(y,ydbn[ib])
        dbn=dp[ib];prior=np.broadcast_to(prior.reshape(1,3),(len(y),3))
        assert np.max(abs(dbn.sum(1)-1))<2e-5 and np.max(abs(prior.sum(1)-1))<2e-5

        # Independent raw->normalized window input parity across each evaluation day.
        samp=[]
        rng=np.random.default_rng(14000+h)
        for d in TEST_DAYS:
            at=np.flatnonzero(day[rr]==d)
            assert len(at)>3
            samp.extend([int(at[0]),int(at[len(at)//2]),int(at[-1])])
            samp.extend(rng.choice(at,size=min(10,len(at)),replace=False).tolist())
        samp=np.unique(samp)
        for i in samp:
            t=int(rr[i]);pos=t-np.arange(99,-1,-1,dtype=np.int64)
            assert int(pos.min())==t-99 and int(pos.max())==t
            assert np.unique(seq[pos]).size==1 and np.unique(day[pos]).size==1
            direct=np.stack([xz[q] for q in range(t-99,t+1)],axis=0)
            batched=xz[pos]
            assert np.array_equal(direct,batched)
            original=raw[pos]
            recalculated=((original-mean.astype(np.float32))/std.astype(np.float32)).astype(np.float32)
            assert np.array_equal(batched,recalculated)
            window_records.append({'horizon':h,'day':int(day[t]),'raw_prediction_origin_t':t,
                                   'first_input_row':t-99,'last_input_row':t,
                                   'max_future_input_offset':int(pos.max()-t),
                                   'historical_window_boundary_safe':True,
                                   'input_scaler_D1_D5_only':True})

        # Saved BEST checkpoint itself independently reloaded; full forward pass (no fit).
        try:
            best=torch.load(required[f'best_{h}'],map_location='cpu',weights_only=False)
        except TypeError:
            best=torch.load(required[f'best_{h}'],map_location='cpu')
        assert int(best['epoch'])==int(dd['selected_epoch'])
        assert int(best['horizon'])==h
        assert abs(float(best['val_metrics']['log_loss'])-float(chosen.val_log_loss))<1e-5
        assert best['model_state']['fc.weight'].shape==(3,64)
        del best
        if FULL_CHECKPOINT_REPLAY:
            replay,rep=replay_frozen(torch,Net,required[f'best_{h}'],rr,xz,device,stored)
            rep['horizon']=h
            rep['stored_logloss']=nll(y,stored)
            rep['replayed_logloss']=nll(y,replay)
            rep['absolute_logloss_error']=abs(rep['stored_logloss']-rep['replayed_logloss'])
            rep['within_frozen_fp32_tolerance']=(
                rep['max_probability_absolute_error']<=REPLAY_MAX_PROB_ERROR and
                rep['mean_probability_absolute_error']<=REPLAY_MEAN_PROB_ERROR and
                rep['absolute_logloss_error']<=REPLAY_NLL_ERROR)
            replay_records.append(rep)
            if not rep['within_frozen_fp32_tolerance']:
                save_csv(resultdir/'05_checkpoint_replay.csv',pd.DataFrame(replay_records))
                raise RuntimeError('Checkpoint replay differs materially from stored predictions for h'+str(h)+
                                   '. Do not claim timing verified; inspect saved model/preprocessing.')
        else:
            replay_records.append({'horizon':h,'replayed_n':0,
                                   'within_frozen_fp32_tolerance':None,
                                   'reason':'Full replay explicitly disabled; only index/input audit was run.'})

        for name,p in (('DEEPLOB',stored),('DBN_COMPLETE',dbn),('TRAIN_PRIOR',prior)):
            for d in TEST_DAYS:
                mask=day[rr]==d
                yy=y[mask];pp=p[mask]
                score=ece_and_class_stats(yy,pp)
                class_records.append({'horizon':h,'day':d,'model':name,'n':len(yy),**score,
                    'true_down_share':float(np.mean(yy==0)),
                    'true_neutral_share':float(np.mean(yy==1)),
                    'true_up_share':float(np.mean(yy==2)),
                    'pred_down_share':float(np.mean(pp.argmax(1)==0)),
                    'pred_neutral_share':float(np.mean(pp.argmax(1)==1)),
                    'pred_up_share':float(np.mean(pp.argmax(1)==2)),
                    'mean_max_probability':float(pp.max(1).mean())})
            # Fixed-width calibration bins, descriptive only (no fitting).
            conf=p.max(1);acc=(p.argmax(1)==y).astype(np.float64)
            for bi in range(10):
                lo=bi/10;hi=(bi+1)/10
                sel=(conf>=lo)&(conf<(hi if bi<9 else hi+1e-12))
                if sel.any():
                    cal_records.append({'horizon':h,'model':name,'bin':bi,'n':int(sel.sum()),
                                        'mean_confidence':float(conf[sel].mean()),
                                        'empirical_accuracy':float(acc[sel].mean())})

        # Destructive shifts are within anonymous sequences; report the SAME shifted cohort's prior.
        local_seq=seq[rr]
        for shift in SHIFTS:
            a,b=shift_pair_indices(local_seq,shift)
            assert np.array_equal(local_seq[a],local_seq[b])
            ys=y[b]
            assert len(ys)>0
            for name,p in (('DEEPLOB',stored),('DBN_COMPLETE',dbn),('TRAIN_PRIOR',prior)):
                loss=nll(ys,p[a])
                shift_records.append({'horizon':h,'label_offset_representation_steps':shift,
                                      'model':name,'n':len(ys),'log_loss':loss,
                                      'label_persistence_vs_source':float(np.mean(y[a]==ys)),
                                      'is_true_alignment':bool(shift==0)})
        checks.append({'horizon':h,'n_test':len(rr),'selected_epoch':int(dd['selected_epoch']),
                       'window_ends_at_forecast_origin':True,
                       'no_test_rows_in_fit_or_validation':True,
                       'publisher_source_labels_exact':True,
                       'publisher_label_generation_reconstructed':False,
                       'full_checkpoint_replay':FULL_CHECKPOINT_REPLAY})
        print(f'h{h}: origin/normalization/purge PASS; epoch={dd["selected_epoch"]}; '
              f'DeepLOB NLL={nll(y,stored):.6f}; DBN={nll(y,dbn):.6f}; '
              f'replay={replay_records[-1].get("within_frozen_fp32_tolerance", "disabled")}')

    # No altered best.pt/DONE/scalers/data/probabilities permitted during audit.
    after={key:filehash(path) for key,path in required.items()}
    assert after==hashes_before,'Frozen upstream file changed during Step14: stop.'

    for filename,rows in [
       ('01_chronological_cohorts.csv',cohort_records),
       ('02_sampled_exact_window_parity.csv',window_records),
       ('03_day_horizon_class_and_error.csv',class_records),
       ('04_no_wrap_bidirectional_label_shifts.csv',shift_records),
       ('05_checkpoint_replay.csv',replay_records),
       ('06_confidence_reliability_bins.csv',cal_records),
       ('07_per_horizon_gates.csv',checks),
       ('11_done_schema_compatibility.csv',metadata_records),
    ]:
        save_csv(resultdir/filename,pd.DataFrame(rows))
    save_json(resultdir/'08_label_semantics_and_provenance.json',{
        'original_publisher_labels_used_unchanged':True,
        'paper_reported_underlying_event_horizons':list(HORIZONS),
        'representation_horizons':list(K_REP),
        'frozen_conservative_purge_row_counts':list(HORIZONS),
        'publisher_message_level_label_generation_reproduced':False,
        'reason':'Available FI-2010 Decimal-Precision snapshots do not provide the underlying '
                 'within-block message midprices and verified original generation implementation.',
        'step5_fixed_nontuned_formula_diagnostics':str(required['step5_formula']),
        'original_label_map':{'1':'Up','2':'Neutral','3':'Down'},
        'current_return_timing':'current feature uses t-1 to t; not future-input evidence',
        'destructive_shift_caveat':'Label persistence can keep shifted-label scores high; this test does not prove absence of leakage.',
        'independent_external_validation':False,
        'reason_external':'D8-D10 have been inspected during prior research; only retrospective assessment.',
    })
    save_json(resultdir/'09_frozen_input_hashes.json',{
        'fingerprint':fingerprint,
        'upstream_hashes_unchanged':True,
        'inputs':{key:{'path':str(p),'sha256':hashes_before[key]} for key,p in required.items()},
    })
    replay_ok=bool(FULL_CHECKPOINT_REPLAY and all(r.get('within_frozen_fp32_tolerance') is True for r in replay_records))
    report=[
        '# CausalLOB — Step 14 forecast-origin audit', '',
        '## Verified checks',
        '- All DeepLOB windows end at origin t and use only [t-99, ..., t] raw LOB snapshots.',
        '- Each window remains within an inferred anonymous-instrument/day sequence.',
        '- Every horizon uses the original h-representation-row conservative tail purge.',
        '- Normalization was reconstructed from D1-D5 raw rows and matched the saved scaler.',
        '- Epochs were selected by D6-D7 validation log loss and frozen before retrospective scoring.',
        '- Legacy h10/h20 DONE files omit batch/window metadata; the original training source declares '
        'batch=256 and window=100, while input-window and checkpoint replay are independently checked. '
        'The DONE records alone do not independently certify legacy training batch size.',
        '- Original publisher labels, raw inputs and frozen model weights were unchanged.',
        '- Full best-checkpoint replay: '+('PASS at stated CPU/GPU numerical tolerances.' if replay_ok else 'NOT COMPLETED.'),
        '', '## Outstanding limitations',
        '- Exact publisher label generation cannot be independently reconstructed from the available snapshots.',
        '- The model is a DeepLOB-style reimplementation, not a bitwise reproduction of the original paper.',
        '- h10-h100 denote original underlying-event horizons; the representation-grid equivalents are k=1,2,3,5,10.',
        '- Days 8-10 were previously inspected; they are retrospective rather than researcher-blind external validation.',
        '- Shift diagnostics must be interpreted alongside persistent labels and matched class-prior baselines.',
        '- Comparisons with the six-variable DBN are not feature-capacity- or training-period-matched.',
        '- No net PnL, executable bid/ask trades, fill probability, or causal intervention effects evaluated.',
        '', '## Interpretation discipline',
        'Strong long-horizon DeepLOB results are real saved retrospective scores and are reproducible '
        'only insofar as the frozen checkpoint replay passes. Their economic and external meaning remains limited.',
        'Exact publisher label provenance remains unresolved. Preserve this qualification in the final report.',
        '', '## Evidence tables',
        '01 cohorts; 02 raw-window samples; 03 class mix; 04 no-wrap shifts; '
        '05 model replay; 06 calibration; 07 gates; 08 provenance; 09 input hashes; '
        '11 legacy DONE schema compatibility.',
    ]
    atomic_text(resultdir/'10_forecast_origin_report.md','\n'.join(report)+'\n')
    archive=out/'CAUSALLOB_STEP14_FORENSIC_EVIDENCE.zip'
    tmp=Path(str(archive)+'.tmp')
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(resultdir.glob('*')):
            if p.is_file():z.write(p,arcname='results/'+p.name)
    with zipfile.ZipFile(tmp) as z:assert z.testzip() is None
    tmp.replace(archive)
    print('\n'+'='*105)
    print('STEP 14 FORECAST-ORIGIN AUDIT COMPLETE — frozen assets untouched')
    print('='*105)
    print('DONE SCHEMA PROVENANCE:')
    print(pd.DataFrame(metadata_records).to_string(index=False))
    print('CHECKPOINT REPLAY:')
    print(pd.DataFrame(replay_records).to_string(index=False))
    print('\nDay/horizon class mix and errors:',resultdir/'03_day_horizon_class_and_error.csv')
    print('Bidirectional non-wrapping shifts:',resultdir/'04_no_wrap_bidirectional_label_shifts.csv')
    print('WINDOW/SCALER/PURGE:', 'PASS')
    print('CHECKPOINT REPLAY:', 'PASS' if replay_ok else 'NOT PERFORMED')
    print('PUBLISHER LABEL FORMULA: UNRESOLVED; NOT FALSELY CLAIMED AS VERIFIED')
    print('EXTERNAL VALIDATION: UNRESOLVED; D8-D10 HISTORICALLY EXPOSED')
    print('Output:',resultdir)
    print('ZIP:',archive)
    print('Runtime:',round(time.monotonic()-tic,1),'seconds')


if __name__=='__main__':
    if os.environ.get('CAUSALLOB_STEP14_SELFTEST','0')=='1':
        self_test()
    else:
        main()

LEGACY DONE h10: missing ['batch_size', 'window']; original CPU training source declares batch=256 and window=100. DONE alone cannot verify those settings; independent window/checkpoint replay continues.
STEP14 SELF-TEST PASS: shifts, scoring, forecast origin, legacy/new DONE schemas.
Mounted at /content/drive
STEP14 PREFLIGHT PASS: 10 days, 50 sequences, frozen checkpoints, D1–D5 scaler exact.
Inference device: cpu | Full frozen checkpoint replay: True
Frozen publisher labels retained. D8–D10 are historically exposed.
LEGACY DONE h10: missing ['batch_size', 'window']; original CPU training source declares batch=256 and window=100. DONE alone cannot verify those settings; independent window/checkpoint replay continues.
h10: origin/normalization/purge PASS; epoch=10; DeepLOB NLL=0.454644; DBN=0.498405; replay=True
h20: origin/normalization/purge PASS; epoch=5; DeepLOB NLL=0.645857; DBN=0.714198; replay=True
h30: origin/normalization/purge PASS; epoch=6; DeepLOB NLL=0.601543; DBN=0.93684

In [ ]:
# CausalLOB STEP 15 — ONE-CELL FINAL EVIDENCE CLOSURE + EXTERNAL PROTOCOL
# GYP SELF-BRIEF: Refuse to mistake numerical replay for verified target-generation semantics,
# previously inspected days for blind validation, classification for trading returns, or
# DeepLOB-versus-DBN performance for a controlled same-input architecture comparison.
# Verify frozen artifacts; independently re-score all five matched cohorts; report three
# individual evaluation days and all adverse gap outcomes. Never fit, tune, relabel,
# download an unrelated dataset, or modify source checkpoints. No publisher generator or
# truly new data is available: produce a locked external-replication protocol, NOT fake tests.

import os, json, hashlib, zipfile, time
from pathlib import Path
import numpy as np
import pandas as pd

VERSION = 'STEP15_FROZEN_EVIDENCE_CLOSURE_v1_2026-09-19'
HORIZONS = (10,20,30,50,100)
TEST_DAYS = (8,9,10)
WINDOW = 100
MASK_HASH = 'eb38b17c1431c0a98d46ed1c792953c932c58572799b64aee79165dfc58439da'
T0 = time.monotonic()

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists() and 'CAUSALLOB_ROOT' not in os.environ:
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
S2 = ROOT/'STEP2_REBUILD/checkpoints'
S4 = ROOT/'STEP4_VALIDATION_FROZEN'
S5 = ROOT/'STEP5_LABEL_AUDIT_READONLY/results'
S6 = ROOT/'STEP6_FROZEN_DBN/results'
S7 = ROOT/'STEP7_MISSING_STATE_READONLY/results'
S8 = ROOT/'STEP8_GRAPH_SPECIFIC_READONLY/results'
S9 = ROOT/'STEP9_GAP_STRESS_READONLY/results'
S12 = ROOT/'STEP12_EVIDENCE_AND_DEEPLOB/results'
S13 = ROOT/'STEP13_DEEPLOB_FORENSIC_READONLY/results'
S14 = ROOT/'STEP14_FORECAST_ORIGIN_READONLY/results'
DEEP = ROOT/'DEEPLOB_CPU_BASELINE/checkpoints'
OUT = ROOT/'STEP15_RESEARCH_CLOSURE_READONLY'
R = OUT/'results'

SOURCES = {
  'raw':S2/'01_canonical_raw.npz',
  'boundaries':S2/'02_boundaries.npz',
  'mask':S4/'checkpoints/PRIMARY_FROZEN_PARENT_MASK.npz',
  'step4_summary':S4/'results/STEP4_FINAL_SUMMARY.json',
  'step5_summary':S5/'05_LABEL_AUDIT_SUMMARY.json',
  'step5_formula':S5/'03_fixed_formula_hypotheses_D1_D7.csv',
  'step8_null':S8/'04_graph_null_transition_summary.csv',
  'step9_summary':S9/'STEP9_FINAL_SUMMARY.json',
  'step9_outcomes':S9/'02_gap_outcome_metrics.csv',
  'step9_states':S9/'01_gap_state_metrics.csv',
  'step12_claims':S12/'12_final_claims_and_completion_gates.json',
  'step13_scores':S13/'01_matched_prediction_scorecard.csv',
  'step13_claims':S13/'08_claim_gates.json',
  'step13_hashes':S13/'09_input_hash_manifest.json',
  'step14_origin':S14/'07_per_horizon_gates.csv',
  'step14_semantics':S14/'08_label_semantics_and_provenance.json',
  'step14_replay':S14/'05_checkpoint_replay.csv',
  'step14_hashes':S14/'09_frozen_input_hashes.json',
  'step14_shifts':S14/'04_no_wrap_bidirectional_label_shifts.csv',
  'step14_day':S14/'03_day_horizon_class_and_error.csv',
}
for h in HORIZONS:
    SOURCES[f'dbn_h{h}'] = S6/f'03_PREDICTED_CLASS_PROBABILITIES_h{h}.npz'
    SOURCES[f'done_h{h}'] = DEEP/f'h{h}/DONE.json'
    p = DEEP/f'h{h}/test_probabilities_h{h}.parquet'
    SOURCES[f'deep_h{h}'] = p if p.is_file() else DEEP/f'h{h}/test_probabilities_h{h}.npz'
missing={k:str(p) for k,p in SOURCES.items() if not p.is_file()}
if missing:
    raise FileNotFoundError('Step 15 stopped: frozen prerequisites missing. No retraining performed:\n'+json.dumps(missing,indent=2))


def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()


def write_once(path,text):
    # Reproducing the same audit is allowed; silently replacing a different one is not.
    if path.exists():
        if path.read_text(encoding='utf8') != text:
            raise RuntimeError('Output already exists with DIFFERENT contents; archive it first: '+str(path))
        return
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(text,encoding='utf8');tmp.replace(path)


def save_json(name,obj):
    write_once(R/name,json.dumps(obj,sort_keys=True,indent=2,allow_nan=False,default=str)+'\n')


def save_csv(name,df):
    write_once(R/name,df.to_csv(index=False,float_format='%.14g'))


def ordered_y(publisher):
    a=np.asarray(publisher)
    assert np.isin(a,[1,2,3]).all()
    return (3-a).astype(np.int64)


def score(y,p):
    y=np.asarray(y,dtype=np.int64)
    p=np.asarray(p,dtype=np.float64)
    assert p.shape==(len(y),3) and np.isin(y,[0,1,2]).all()
    assert np.isfinite(p).all() and (p>=0).all()
    assert np.max(np.abs(p.sum(axis=1)-1))<2e-5
    hard=p.argmax(axis=1)
    recalls=[float((hard[y==k]==k).mean()) if np.any(y==k) else None for k in range(3)]
    ll=-np.log(np.clip(p[np.arange(len(y)),y],1e-15,1))
    br=np.sum((p-np.eye(3,dtype=np.float64)[y])**2,axis=1)
    return {'n':int(len(y)), 'balanced_accuracy':float(np.nanmean([v if v is not None else np.nan for v in recalls])),
            'accuracy':float(np.mean(hard==y)), 'log_loss':float(ll.mean()), 'brier':float(br.mean())}


def same_cohort(day,seq,h):
    begins=np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]])
    stops=np.r_[begins[1:],len(seq)]
    parts=[]
    for a,b in zip(begins,stops):
        if int(day[a]) not in TEST_DAYS:continue
        lo=int(a)+WINDOW-1;hi=int(b)-1-h
        if lo<=hi:
            rr=np.arange(lo,hi+1,dtype=np.int64)
            assert np.array_equal(seq[rr-WINDOW+1],seq[rr]) and np.all(rr+h<b)
            parts.append(rr)
    return np.concatenate(parts)


def synthetic_selftest():
    y=np.array([0,1,2,0]);p=np.eye(3)[y]*.9+(.1/3)
    assert score(y,p)['balanced_accuracy']==1
    assert abs(score(y,np.ones((len(y),3))/3)['log_loss']-np.log(3))<1e-12
    assert np.array_equal(ordered_y([3,2,1]),[0,1,2])
    print('STEP15 SELF-TEST PASS: class map, probability simplex, scoring.')

synthetic_selftest()
# Hash all immutable inputs once; compare with previously committed step-13/14 manifests.
before={k:sha(p) for k,p in SOURCES.items()}
prev13=json.loads(SOURCES['step13_hashes'].read_text())
prev14=json.loads(SOURCES['step14_hashes'].read_text())
assert prev14['upstream_hashes_unchanged'] is True
for key in ('raw','boundaries'):
    assert before[key]==prev13['files'][key]['sha256'],f'Original source differs from Step13: {key}'
for key in ('raw','boundaries'):
    assert before[key]==prev14['inputs'][key]['sha256'],f'Original source differs from Step14: {key}'

claims12=json.loads(SOURCES['step12_claims'].read_text())
claims13=json.loads(SOURCES['step13_claims'].read_text())
sem14=json.loads(SOURCES['step14_semantics'].read_text())
summary9=json.loads(SOURCES['step9_summary'].read_text())
summary5=json.loads(SOURCES['step5_summary'].read_text())
summary4=json.loads(SOURCES['step4_summary'].read_text())
assert summary4['status']=='VALIDATED_AND_FROZEN_INTERNAL_ONLY'
assert summary4['total_primary_slots']==39
assert summary4['D8_D10_used_for_selection'] is False
assert summary9['historically_blind_external_test'] is False
assert claims12['publisher_exact_label_formula_verified'] is False
assert claims13['independent_external_test'] is False
assert sem14['publisher_message_level_label_generation_reproduced'] is False
assert set(claims12['deep_lob_completed_horizons'])==set(HORIZONS)
assert claims12['matched_gap_prior_better_cases']==12
with np.load(SOURCES['mask'],allow_pickle=False) as z:
    assert z['parent_mask'].shape==(6,6,6) and int(z['parent_mask'].sum())==39
    assert str(z['mask_hash'].item())==MASK_HASH
with np.load(SOURCES['raw'],allow_pickle=False) as z:
    Y=z['Y'];day=z['day'];raw=z['X_raw']
with np.load(SOURCES['boundaries'],allow_pickle=False) as z:
    seq=z['sequence_id']
assert raw.shape==(394337,40) and Y.shape==(394337,5)
assert np.array_equal(np.unique(day),np.arange(1,11)) and np.unique(seq).size==50
assert np.isin(Y,[1,2,3]).all()

R.mkdir(parents=True,exist_ok=True)
SPEC={'version':VERSION,'horizons':HORIZONS,'source_sha256':before,
      'graph_mask_hash':MASK_HASH,'no_fitting':True,'D8_D10_researcher_blind':False}
fingerprint=hashlib.sha256(json.dumps(SPEC,sort_keys=True).encode()).hexdigest()
old_spec=R/'00_RUN_SPEC.json'
if old_spec.is_file() and json.loads(old_spec.read_text())['fingerprint']!=fingerprint:
    raise RuntimeError('Step15 output already exists for different inputs; archive it first.')
save_json('00_RUN_SPEC.json',{'fingerprint':fingerprint,'spec':SPEC})

origin=pd.read_csv(SOURCES['step14_origin'])
replay=pd.read_csv(SOURCES['step14_replay'])
assert set(origin.horizon.astype(int))==set(HORIZONS)
assert origin['window_ends_at_forecast_origin'].astype(str).str.lower().eq('true').all()
assert origin['predeclared_purged_rows_exact'].astype(str).str.lower().eq('true').all() if 'predeclared_purged_rows_exact' in origin else True
assert set(replay.horizon.astype(int))==set(HORIZONS)
assert replay.within_frozen_fp32_tolerance.astype(str).str.lower().eq('true').all()

s13=pd.read_csv(SOURCES['step13_scores'])
records=[];days=[];excess=[];head_class=[]
for idx,h in enumerate(HORIZONS):
    done=json.loads(SOURCES[f'done_h{h}'].read_text())
    assert int(done['horizon'])==h and done['selection_metric']=='D6-D7 validation log loss'
    rows=same_cohort(day,seq,h)
    deep_path=SOURCES[f'deep_h{h}']
    if deep_path.suffix=='.parquet':
        df=pd.read_parquet(deep_path)
    else:
        with np.load(deep_path,allow_pickle=False) as z:
            df=pd.DataFrame({k:z[k] for k in
                     ('raw_row','day','y_ord','p_down','p_stationary','p_up')})
    deep=df[['p_down','p_stationary','p_up']].to_numpy(dtype=np.float64)
    assert np.array_equal(df.raw_row.to_numpy(dtype=np.int64),rows)
    true=ordered_y(Y[rows,idx])
    assert np.array_equal(df.y_ord.to_numpy(dtype=np.int64),true)
    assert np.array_equal(df.day.to_numpy(dtype=np.int8),day[rows])
    assert len(rows)==int(done['n_test_windows'])
    with np.load(SOURCES[f'dbn_h{h}'],allow_pickle=False) as z:
        row_index=z['row_index'].astype(np.int64)
        dbn_all=z['p_down_neutral_up'].astype(np.float64)
        prior_vec=z['prior_down_neutral_up'].astype(np.float64)
        lab_dbn=z['label_ordered'].astype(np.int64)
    starts=np.flatnonzero(np.r_[True,seq[1:]!=seq[:-1]])
    stops=np.r_[starts[1:],len(seq)]
    curr=np.concatenate([np.arange(int(a)+20,int(b),dtype=np.int64) for a,b in zip(starts,stops)])
    raw_dbn=curr[row_index]
    loc=np.searchsorted(raw_dbn,rows)
    assert (loc<len(raw_dbn)).all() and np.array_equal(raw_dbn[loc],rows)
    assert np.array_equal(lab_dbn[loc],true)
    dbn=dbn_all[loc]
    assert np.isfinite(prior_vec).all() and prior_vec.shape==(3,)
    prior=np.broadcast_to(prior_vec,(len(true),3))
    pp={'DEEPLOB':deep,'DBN_COMPLETE':dbn,'PRIOR_D1_D7':prior}
    selectors=[('POOLED',np.ones(len(rows),dtype=bool))]+[(str(d),day[rows]==d) for d in TEST_DAYS]
    for dn,sel in selectors:
        assert sel.any()
        for label,p in pp.items():
            met=score(true[sel],p[sel]);records.append({'horizon':h,'day':dn,'model':label,**met})
            old=s13[(s13.horizon==h)&(s13.day.astype(str)==dn)&(s13.model==label)]
            assert len(old)==1 and int(old.iloc[0].n)==met['n']
            assert abs(met['log_loss']-float(old.iloc[0].log_loss))<5e-6,('Step13 metric parity',h,dn,label)
        dlp=-np.log(np.clip(deep[sel,np.asarray(true[sel])],1e-15,1))
        blp=-np.log(np.clip(dbn[sel,np.asarray(true[sel])],1e-15,1))
        rlp=-np.log(np.clip(prior[sel,np.asarray(true[sel])],1e-15,1))
        days.append({'horizon':h,'day':dn,'n':int(sel.sum()),
            'DeepLOB_logloss_improvement_over_DBN':float((blp-dlp).mean()),
            'DBN_logloss_improvement_over_prior':float((rlp-blp).mean()),
            'DeepLOB_logloss_improvement_over_prior':float((rlp-dlp).mean())})
        if dn!='POOLED':
            for k,name in enumerate(('Down','Neutral','Up')):
                yy=true[sel];head_class.append({'horizon':h,'day':int(dn),'class':name,
                  'actual_share':float((yy==k).mean()),
                  'DeepLOB_argmax_share':float((deep[sel].argmax(1)==k).mean()),
                  'DBN_argmax_share':float((dbn[sel].argmax(1)==k).mean())})
    print(f'RESCORE/PARITY PASS h{h}: DeepLOB={score(true,deep)["log_loss"]:.6f}, '
          f'DBN={score(true,dbn)["log_loss"]:.6f}; n={len(rows)}')

score_df=pd.DataFrame(records);gain_df=pd.DataFrame(days);mix_df=pd.DataFrame(head_class)
assert len(score_df)==len(HORIZONS)*4*3
assert len(gain_df)==len(HORIZONS)*4
# These three days are inferential blocks, NOT 138K independent test days.
cons=[]
for h in HORIZONS:
    f=gain_df[(gain_df.horizon==h)&(gain_df.day!='POOLED')]
    vals=f.DeepLOB_logloss_improvement_over_DBN.to_numpy()
    assert len(vals)==3
    cons.append({'horizon':h,'n_day_blocks':3,'days_DeepLOB_better_LL':int((vals>0).sum()),
                 'min_daily_gain':float(vals.min()),'max_daily_gain':float(vals.max()),
                 'unweighted_day_mean':float(vals.mean()),'no_significance_claim':True})
cons_df=pd.DataFrame(cons)

# Recalculate all twelve *exactly matched* gap-prior comparisons.
gap=pd.read_csv(SOURCES['step9_outcomes'])
G=[]
for (scenario,g,h,d),sub in gap.groupby(['scenario','gap','horizon','day']):
    s=sub[sub.model=='CAUSALLOB_DIAG_FROZEN']
    p=sub[sub.model=='TRAIN_ONLY_CLASS_PRIOR']
    assert len(s)==len(p)==1
    sr,pr=s.iloc[0],p.iloc[0]
    assert int(sr.n)==int(pr.n)
    G.append({'scenario':scenario,'gap_representation_steps':int(g),'horizon':int(h),
              'day':str(d),'n':int(sr.n),'sparse_logloss':float(sr.log_loss),
              'prior_logloss':float(pr.log_loss),
              'prior_minus_sparse_logloss':float(pr.log_loss-sr.log_loss),
              'prior_minus_sparse_brier':float(pr.brier-sr.brier)})
gap_df=pd.DataFrame(G)
pooled_gap=gap_df[gap_df.day=='POOLED'].copy()
assert len(pooled_gap)==12 and (pooled_gap.prior_minus_sparse_logloss<0).sum()==12
assert (pooled_gap.n==697).all()

null=pd.read_csv(SOURCES['step8_null'])
match=null[(null.day.astype(str)=='POOLED')&(null.metric=='transition_rmse')]
assert len(match)==1 and int(match.iloc[0].n_random)==100
nullrow=match.iloc[0]
assert abs(float(nullrow.causallob)-.5861374731258596)<5e-5

# No statistical significance is inferred from 3 historically inspected day blocks.
# Inspect existing destructive shifts only; shift scores cannot prove label formula.
shift=pd.read_csv(SOURCES['step14_shifts'])
assert set(shift.horizon.astype(int))==set(HORIZONS)
assert (shift.label_offset_representation_steps==0).any()
shift_origin=shift[(shift.model=='DEEPLOB')&(shift.label_offset_representation_steps==0)]
assert len(shift_origin)==5

# Rechecking all 40+ upstream files after data streaming: byte-identical or STOP.
after={k:sha(p) for k,p in SOURCES.items()}
assert before==after,'Frozen source changed during Step15 audit; STOP.'

save_csv('01_independent_matched_scores.csv',score_df)
save_csv('02_paired_daily_effects_descriptive.csv',gain_df)
save_csv('03_three_day_consistency_no_iid_claim.csv',cons_df)
save_csv('04_class_mix_by_day.csv',mix_df)
save_csv('05_all_matched_gap_vs_prior.csv',gap_df)

# This protocol is about NEW data, not a request to relabel the existing FI-2010 test.
protocol = '''# CausalLOB: preregistered external replication protocol (DRAFT; no data seen)

Status: EXTERNAL DATA NOT PROVIDED. This file is a *prospective protocol*, not an external result.
The FI-2010 Days 8–10 have been examined; none may be called newly unseen.

1. Independently acquire licensed/new LOB observations from a different period or venue.
   Record provider, timestamp coverage, instrument identities, raw message/event availability,
   license, acquisition date and original file SHA-256 before outcomes are inspected.
2. Obtain a verified label-generation implementation and units. The FI-2010 published labels
   are preserved in prior experiments but cannot be independently regenerated exactly
   from its ten-message representations alone. Do not treat alternative formulas as truth.
3. Predeclare an event/representation conversion; six state definitions, 5 temporal lags,
   39 frozen graph slots, 100x40 DeepLOB inputs, normalization compatibility, boundaries,
   10/20/30/50/100 underlying-event horizons and horizon-specific purges.
4. Require raw40 definitions and price units comparable to FI-2010. If semantics differ,
   evaluate transportability as a distinct experiment; do not imply like-for-like transfer.
5. Freeze pretrained model weights and heads BEFORE any external target is inspected.
   Do not fit new scalers on future data or choose model thresholds on external outcomes.
6. Compare frozen CausalLOB, frozen AR(5), dense VAR, training-only prior and DeepLOB on
   exactly matched rows, only if data and label semantics permit these comparisons.
7. Prespecify missing masks, gaps 1/5/20 *representation steps*, exact filtering,
   state RMSE, Gaussian NLL, 90% coverage and outcome LL/Brier/balanced accuracy.
8. Record outcomes separately by date/instrument; report model failures and day-level
   variation without pretending event rows are statistically independent.
9. Keep economic claims separate: no deployment or net trading profit without executable
   bid/ask quotes, fees, fill/queue model, latency and an out-of-sample execution study.
10. STOP GATE: no independent data and verified compatible labels = NO external scores,
    NO NeurIPS-ready generalization claim. Archive protocol hash before opening data.
'''
external_proto_hash=hashlib.sha256(protocol.encode()).hexdigest()
write_once(R/'06_external_replication_PREREG_DRAFT.md',protocol)
save_json('07_external_protocol_fingerprint.json',{
    'sha256':external_proto_hash,'status':'DRAFT_UNEXECUTED',
    'external_data_available_or_evaluated':False,'publisher_exact_label_reconstruction':False,
    'external_data_required':'A genuinely unseen independent dataset and verified labels; not FI-2010 D8-D10',
})

score_pool=score_df[score_df.day=='POOLED'].copy()
Deep=score_pool[score_pool.model=='DEEPLOB'].set_index('horizon')
DBN=score_pool[score_pool.model=='DBN_COMPLETE'].set_index('horizon')
prior=score_pool[score_pool.model=='PRIOR_D1_D7'].set_index('horizon')

notes=[
  '# CausalLOB — completed FI-2010 retrospective study / research closure',
  '', '## Scope and exact result status',
  '- Source: FI-2010, all ten days reconstructed. D1–5 discovery; D6–7 internal validation; D1–7 frozen DBN fit; D8–10 retrospective assessment.',
  '- DeepLOB-style CNN/Inception/LSTM: D1–5 fitting, D6–7 checkpoint selection; matched D8–10 scores.',
  '- Windows [t−99,...,t], boundary-safe, conservative last h-representation-row purge, Step14 checkpoint replay passed.',
  '- h10/20/30/50/100 name underlying-event horizons; k=1/2/3/5/10 on the decimated representation grid.',
  '- Publisher source labels remain unchanged, but exact original message-level label-generation reconstruction is not verified.',
  '- Only three historically inspected test day blocks; no independent external validation or net-return evidence.',
  '', '## Matched full-observation forecast table',
  '| Horizon | Matched n | DeepLOB BA | DBN BA | DeepLOB LL | DBN LL | Prior LL |',
  '|---:|---:|---:|---:|---:|---:|---:|',
]
for h in HORIZONS:
    de=Deep.loc[h];db=DBN.loc[h];pr=prior.loc[h]
    notes.append(f'| {h} | {int(de.n)} | {de.balanced_accuracy:.4f} | {db.balanced_accuracy:.4f} | '
                 f'{de.log_loss:.5f} | {db.log_loss:.5f} | {pr.log_loss:.5f} |')
notes += [
  '', '## Graph and state-inference evidence',
  f'- Frozen model: 39 lag slots (30 autoregressive + 9 selected cross-variable); 100 target/lag-matched random graphs.',
  f'- Pooled transition RMSE: sparse {nullrow.causallob:.6f}, AR5 {nullrow.ar5:.6f}, '
    f'dense {nullrow.dense_var5:.6f}, random-median {nullrow.null_median:.6f}.',
  f'- Proportion of matched random graphs worse: {nullrow.fraction_random_worse:.2f}; '
    'this is descriptive, not a causal p-value.',
  '- AR history explains most state-reconstruction gain; graph cross-edges contribute modestly. '
    'Regularized innovation covariance is a sensitivity, not the original independent-noise DBN.',
  '- Online filtering with consecutive synthetic missingness covers 697 spaced endpoints. '
    'Frozen CausalLOB loses to the precisely matched training-only prior in 12/12 pooled gap comparisons.',
  '', '## Predicted outcomes and limitations',
  '- DeepLOB has lower matched log loss at all five horizons and outperforms the DBN on each of the three days; '
    'h10 DBN has higher balanced accuracy. The comparison is NOT controlled for feature capacity or fitting period.',
  '- Long-horizon results reproduce frozen checkpoints but do not establish profitable, deployable signals.',
  '- No observational LPCMCI arrow should be reported as an identified intervention effect.',
  '- Exact publisher target-generation mechanism, truly new external validation and execution-cost modeling remain unresolved.',
  '', '## Dataset and label provenance',
  '- Official dataset description and publisher label map: '
    'https://www.research.ed.ac.uk/en/datasets/benchmark-dataset-for-mid-price-forecasting-of-limit-order-book-d/',
  '- Dataset-method paper: https://arxiv.org/abs/1705.03233',
  '- FI-2010 outputs are ten-message representations; the intermediate message prices and verified original label-generator '
    'were not provided in the current checkpoints. Fixed formula comparisons are diagnostic, not exact reproduction.',
  '', '## Next research decision',
  '- The present FI-2010 retrospective *course pipeline* is complete. Do not retrain or optimize on D8–10.',
  '- First obtain independent external data with verified label semantics and freeze protocol/inputs. '
    'Then run an external replication. If unavailable, submit limitations explicitly.',
  '- A novel probabilistic graph-uncertainty method is a separate proposed project, NOT an accomplished contribution.',
  '', f'Frozen evidence fingerprint: `{fingerprint}`',
]
write_once(R/'08_FINAL_ACADEMIC_EVIDENCE_REPORT.md','\n'.join(notes)+'\n')

gates={
 'version':VERSION,'run_fingerprint':fingerprint,'all_10_days_loaded':True,
 'correct_day_roles':True,'all_5_deeplob_checkpoints_replayed_by_step14':True,
 'matched_predictions_independently_rescored_in_step15':True,
 'frozen_graph_slots':39,'upstream_sha256_unchanged':True,
 'three_historically_exposed_test_days':True,
 'external_researcher_blind_validation_completed':False,
 'publisher_message_level_label_generator_verified':False,
 'missing_gap_model_beats_prior_cases':int((pooled_gap.prior_minus_sparse_logloss>0).sum()),
 'missing_gap_prior_better_cases':int((pooled_gap.prior_minus_sparse_logloss<0).sum()),
 'profitability_established':False,'neurips_ready_claim':False,
 'limitations_cannot_be_removed_with_more_FI2010_reruns':True,
 'next_dependency':'User must provide genuinely unseen, licensed LOB data plus verified label semantics.',
}
save_json('09_FINAL_CLAIM_GATES.json',gates)
save_json('10_FROZEN_HASH_MANIFEST.json',{
    'fingerprint':fingerprint,'upstream_unchanged':True,
    'files':{k:{'path':str(v),'sha256':before[k]} for k,v in SOURCES.items()}
})
# Only include files produced by this run; never copy prior sensitive datasets/weights.
archive=OUT/'CAUSALLOB_STEP15_FINAL_EVIDENCE.zip'
tmp=archive.with_name(archive.name+'.tmp')
with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED) as zz:
    for p in sorted(R.iterdir()):
        if p.is_file():zz.write(p,arcname='results/'+p.name)
with zipfile.ZipFile(tmp) as zz:assert zz.testzip() is None
if archive.is_file() and sha(archive)!=sha(tmp):
    tmp.unlink()
    raise RuntimeError('Existing research ZIP differs; refusing to overwrite previous package.')
tmp.replace(archive)
print('\n'+'='*110)
print('STEP15 RESEARCH CLOSURE COMPLETE — FI-2010 RETROSPECTIVE ONLY; FROZEN CHECKPOINTS UNCHANGED')
print('='*110)
print(score_pool[['horizon','model','n','balanced_accuracy','log_loss','brier']].to_string(index=False))
print('\nDay-level DeepLOB-vs-DBN:',cons_df.to_string(index=False))
print('\nMatched consecutive-gap prior wins:',gates['missing_gap_prior_better_cases'],'/12')
print('Unresolved: exact publisher label-generation code, genuinely independent external data, economic PnL.')
print('Report:',R/'08_FINAL_ACADEMIC_EVIDENCE_REPORT.md')
print('External protocol draft:',R/'06_external_replication_PREREG_DRAFT.md')
print('ZIP:',archive)
print('Elapsed:',round(time.monotonic()-T0,1),'seconds')

STEP15 SELF-TEST PASS: class map, probability simplex, scoring.
RESCORE/PARITY PASS h10: DeepLOB=0.454644, DBN=0.498405; n=137952
RESCORE/PARITY PASS h20: DeepLOB=0.645857, DBN=0.714198; n=137802
RESCORE/PARITY PASS h30: DeepLOB=0.601543, DBN=0.936842; n=137652
RESCORE/PARITY PASS h50: DeepLOB=0.560573, DBN=1.071103; n=137352
RESCORE/PARITY PASS h100: DeepLOB=0.569517, DBN=1.147715; n=136602

STEP15 RESEARCH CLOSURE COMPLETE — FI-2010 RETROSPECTIVE ONLY; FROZEN CHECKPOINTS UNCHANGED
 horizon        model      n  balanced_accuracy  log_loss    brier
      10      DEEPLOB 137952           0.676273  0.454644 0.242605
      10 DBN_COMPLETE 137952           0.727736  0.498405 0.236803
      10  PRIOR_D1_D7 137952           0.333333  0.829208 0.471867
      20      DEEPLOB 137802           0.624026  0.645857 0.361440
      20 DBN_COMPLETE 137802           0.623077  0.714198 0.386391
      20  PRIOR_D1_D7 137802           0.333333  0.957764 0.565968
      30      DEEPLOB 137652           0.69

In [ ]:
# CausalLOB STEP 16 — FINISH EVERY FEASIBLE REMAINING TASK WITHOUT NEW MARKET DATA
# GYP: Freeze all FI-2010 models and the Step15 results. Build a submission-quality
# paper/appendix/figure/reproducibility package; independently audit the archived
# claims and run a SEPARATE, plainly labeled synthetic known-DGP stress test.
# No new external dataset, publisher message-level generator, verified trading
# simulator, causal identification, NeurIPS-ready method, or profitability is invented.
# One-cell Colab script; no retraining of FI-2010, no tuning on D8-D10, no GPU needed.

import os, json, hashlib, zipfile, io, time, platform, sys, shutil
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

START = time.monotonic()
VERSION = 'CAUSALLOB_STEP16_NO_EXTERNAL_SUBMISSION_v2_2026-09-19'
HORIZONS=(10,20,30,50,100)
SEED=5162026
REPS=40
SYN_TRAIN=800
SYN_TEST=300
SYN_LAG=5
EXPECTED_MASK_HASH='eb38b17c1431c0a98d46ed1c792953c932c58572799b64aee79165dfc58439da'
FEATURES=('log_relative_spread','log_total_depth_l5','imbalance_l1',
          'imbalance_l2_l5','quote_ofi_depth_norm','log_mid_return_1')
try:
    from google.colab import drive
    if 'CAUSALLOB_ROOT' not in os.environ and not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass
ROOT=Path(os.environ.get('CAUSALLOB_ROOT','/content/drive/MyDrive/CausalLOB'))
S15=ROOT/'STEP15_RESEARCH_CLOSURE_READONLY/results'
S14=ROOT/'STEP14_FORECAST_ORIGIN_READONLY/results'
S9=ROOT/'STEP9_GAP_STRESS_READONLY/results'
S8=ROOT/'STEP8_GRAPH_SPECIFIC_READONLY/results'
S7=ROOT/'STEP7_MISSING_STATE_READONLY/results'
S5=ROOT/'STEP5_LABEL_AUDIT_READONLY/results'
S4=ROOT/'STEP4_VALIDATION_FROZEN/checkpoints'
S2=ROOT/'STEP2_REBUILD/checkpoints'
OUT=ROOT/'STEP16_NO_EXTERNAL_SUBMISSION'
R=OUT/'results'
FIG=R/'figures'
INPUTS={
 'step15_scores':S15/'01_independent_matched_scores.csv',
 'step15_daily':S15/'02_paired_daily_effects_descriptive.csv',
 'step15_consistency':S15/'03_three_day_consistency_no_iid_claim.csv',
 'step15_classmix':S15/'04_class_mix_by_day.csv',
 'step15_gaps':S15/'05_all_matched_gap_vs_prior.csv',
 'step15_protocol':S15/'06_external_replication_PREREG_DRAFT.md',
 'step15_protocol_hash':S15/'07_external_protocol_fingerprint.json',
 'step15_report':S15/'08_FINAL_ACADEMIC_EVIDENCE_REPORT.md',
 'step15_gates':S15/'09_FINAL_CLAIM_GATES.json',
 'step15_manifest':S15/'10_FROZEN_HASH_MANIFEST.json',
 'step14_shifts':S14/'04_no_wrap_bidirectional_label_shifts.csv',
 'step14_replay':S14/'05_checkpoint_replay.csv',
 'step14_origin':S14/'07_per_horizon_gates.csv',
 'step14_semantics':S14/'08_label_semantics_and_provenance.json',
 'step14_done_format':S14/'11_done_schema_compatibility.csv',
 'step9_summary':S9/'STEP9_FINAL_SUMMARY.json',
 'step8_graph_null':S8/'04_graph_null_transition_summary.csv',
 'step7_states':S7/'02_EXACT_STATE_METRICS.csv',
 'step5_formula':S5/'03_fixed_formula_hypotheses_D1_D7.csv',
 'step5_summary':S5/'05_LABEL_AUDIT_SUMMARY.json',
 'mask':S4/'PRIMARY_FROZEN_PARENT_MASK.npz',
}
missing={key:str(path) for key,path in INPUTS.items() if not path.is_file()}
if missing:
    raise FileNotFoundError('Step16 requires completed frozen Steps 5,7,8,9,14,15; no source will be rebuilt.\n'+json.dumps(missing,indent=2))

def sha_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()

def canonical_json(x):return json.dumps(x,sort_keys=True,indent=2,allow_nan=False,default=str)+'\n'

def write_once(p,data):
    if isinstance(data,str):data=data.encode('utf-8')
    if p.exists():
        if p.read_bytes()!=data:
            raise RuntimeError('Existing output differs: '+str(p)+'; archive old Step16 results first.')
        return
    tmp=p.with_name(p.name+'.tmp')
    tmp.write_bytes(data);tmp.replace(p)

def save_csv(name,table):
    write_once(R/name,table.to_csv(index=False,float_format='%.12g'))

def save_json(name,val):write_once(R/name,canonical_json(val))

def check_close(x,y,tol=5e-6):
    if not np.isfinite(x) or abs(x-y)>tol:
        raise AssertionError(f'Frozen metric mismatch {x} versus {y} (tol {tol})')

# Independent minimal unit tests before reading published scores.
def selftest():
    xx=pd.DataFrame({'horizon':[10,20],'day':['POOLED','POOLED'],
                     'model':['A','B'],'log_loss':[.4,.6]})
    assert (xx.groupby('horizon').size()==1).all()
    assert sha_file(INPUTS['step15_gates'])==sha_file(INPUTS['step15_gates'])
    assert check_close(.2+.3,.5) is None
    t=np.arange(24,dtype=np.float64).reshape(8,3)
    z=np.concatenate([t[4-i:8-i] for i in range(1,4)],axis=1)
    assert z.shape==(4,9) and z[0,0]==t[3,0] and z[0,3]==t[2,0]
    print('STEP16 SELF-TEST PASS: metrics, hashes, lag ordering, fail-closed parity.')
selftest()

# Verify ALL frozen upstream fingerprints from Step15, not only the recent score CSV.
upstream=json.loads(INPUTS['step15_manifest'].read_text())
assert upstream['upstream_unchanged'] is True
assert upstream['fingerprint']==json.loads((S15/'00_RUN_SPEC.json').read_text())['fingerprint']
verified=[]
for name,info in sorted(upstream['files'].items()):
    path=Path(info['path'])
    assert path.is_file(),f'Step15 upstream source disappeared: {path}'
    current=sha_file(path)
    assert current==info['sha256'],f'Frozen upstream source changed: {name}'
    verified.append({'source_id':name,'sha256':current,'path':str(path)})
input_hashes={key:sha_file(p) for key,p in INPUTS.items()}
gates=json.loads(INPUTS['step15_gates'].read_text())
sem=json.loads(INPUTS['step14_semantics'].read_text())
step9=json.loads(INPUTS['step9_summary'].read_text())
assert gates['all_10_days_loaded'] is True
assert gates['upstream_sha256_unchanged'] is True
# Step15's actual 09_FINAL_CLAIM_GATES.json uses 'missing_gap_prior_better_cases'.
# This is a strict schema check, NOT a missing-key default or an assertion bypass.
assert 'missing_gap_prior_better_cases' in gates, 'Step15 claim-gates schema mismatch: missing_gap_prior_better_cases'
assert int(gates['missing_gap_prior_better_cases'])==12
assert int(gates['missing_gap_model_beats_prior_cases'])==0
assert gates['external_researcher_blind_validation_completed'] is False
assert gates['publisher_message_level_label_generator_verified'] is False
assert gates['profitability_established'] is False
assert sem['publisher_message_level_label_generation_reproduced'] is False
assert step9['historically_blind_external_test'] is False
assert gates['frozen_graph_slots']==39
with np.load(INPUTS['mask'],allow_pickle=False) as z:
    MASK=z['parent_mask']
    names=z['feature_names'].astype(str).tolist()
    mask_sha=str(z['mask_hash'].item())
assert MASK.shape==(6,6,6) and int(MASK.sum())==39
assert mask_sha==EXPECTED_MASK_HASH and names==list(FEATURES)
assert all(bool(np.all(MASK[j,j,1:])) for j in range(6))
assert not bool(np.any(MASK[:,:,0]))
source_edges=[{'source':names[i],'target':names[j],'lag_representation_steps':lag,
               'graph_label':'observational_lagged_dependency_NOT_intervention'}
              for i in range(6) for j in range(6) for lag in range(1,6)
              if i!=j and bool(MASK[i,j,lag])]
assert len(source_edges)==9
score=pd.read_csv(INPUTS['step15_scores'])
daily=pd.read_csv(INPUTS['step15_daily'])
consistent=pd.read_csv(INPUTS['step15_consistency'])
mix=pd.read_csv(INPUTS['step15_classmix'])
gaps=pd.read_csv(INPUTS['step15_gaps'])
shifts=pd.read_csv(INPUTS['step14_shifts'])
replay=pd.read_csv(INPUTS['step14_replay'])
origin=pd.read_csv(INPUTS['step14_origin'])
formula=pd.read_csv(INPUTS['step5_formula'])
graph=pd.read_csv(INPUTS['step8_graph_null'])
states=pd.read_csv(INPUTS['step7_states'])
assert score.shape[0]==60 and set(score.horizon.astype(int))==set(HORIZONS)
assert set(score.model)=={'DEEPLOB','DBN_COMPLETE','PRIOR_D1_D7'}
assert set(score.day.astype(str))=={'POOLED','8','9','10'}
assert (score.n>0).all() and np.isfinite(score[['log_loss','balanced_accuracy','brier']].to_numpy()).all()
assert replay.within_frozen_fp32_tolerance.astype(str).str.lower().eq('true').all()
assert origin.window_ends_at_forecast_origin.astype(str).str.lower().eq('true').all()
assert len(gaps[gaps.day.astype(str)=='POOLED'])==12
pooled=gaps[gaps.day.astype(str)=='POOLED'].copy()
assert (pooled.prior_minus_sparse_logloss<0).all() and pooled.n.eq(697).all()
assert int((pooled.prior_minus_sparse_logloss<0).sum())==int(gates['missing_gap_prior_better_cases']), \
    'Step15 claimed prior-win count disagrees with its independently read pooled gap CSV'
assert int((pooled.prior_minus_sparse_logloss>0).sum())==int(gates['missing_gap_model_beats_prior_cases']), \
    'Step15 claimed sparse-win count disagrees with its independently read pooled gap CSV'
assert len(consistent)==5 and consistent.n_day_blocks.eq(3).all()
assert consistent.days_DeepLOB_better_LL.eq(3).all()
base=score[score.day.astype(str)=='POOLED'].copy()
assert len(base)==15
pv=base.set_index(['horizon','model'])
rows=[]
for h in HORIZONS:
    dd=pv.loc[(h,'DEEPLOB')]; db=pv.loc[(h,'DBN_COMPLETE')]; pr=pv.loc[(h,'PRIOR_D1_D7')]
    assert int(dd.n)==int(db.n)==int(pr.n)
    check_close(float(dd.log_loss),float(score[(score.horizon==h)&(score.model=='DEEPLOB')&(score.day=='POOLED')].iloc[0].log_loss))
    assert dd.log_loss<db.log_loss<pr.log_loss
    r=consistent.loc[consistent.horizon==h].iloc[0]
    assert int(r.days_DeepLOB_better_LL)==3
    for day_number in (8,9,10):
        sub=score[(score.horizon==h)&(score.day.astype(str)==str(day_number))]
        dloss=float(sub.loc[sub.model=='DEEPLOB','log_loss'].iloc[0]); bloss=float(sub.loc[sub.model=='DBN_COMPLETE','log_loss'].iloc[0])
        assert dloss<bloss
    rows.append({'horizon_underlying_events':h,'matched_n':int(dd.n),
                 'DeepLOB_BA':float(dd.balanced_accuracy),'DBN_BA':float(db.balanced_accuracy),
                 'DeepLOB_Brier':float(dd.brier),'DBN_Brier':float(db.brier),
                 'DeepLOB_logloss':float(dd.log_loss),'DBN_logloss':float(db.log_loss),
                 'train_prior_logloss':float(pr.log_loss),
                 'DeepLOB_minus_DBN_BA':float(dd.balanced_accuracy-db.balanced_accuracy),
                 'DBN_minus_DeepLOB_logloss':float(db.log_loss-dd.log_loss),
                 'DBN_minus_prior_logloss_gain':float(pr.log_loss-db.log_loss)})
bench=pd.DataFrame(rows)
assert float(bench.iloc[0]['DeepLOB_minus_DBN_BA'])<0
assert bool((bench.DBN_minus_DeepLOB_logloss>0).all())
null=graph[(graph.day.astype(str)=='POOLED')&(graph.metric=='transition_rmse')]
assert len(null)==1 and int(null.iloc[0].n_random)==100
nr=null.iloc[0]
check_close(float(nr.causallob),0.5861374731258596,tol=5e-5)
assert np.isfinite([nr.ar5,nr.dense_var5,nr.null_median]).all()
coverage=states[(states.day.astype(str)=='POOLED')&
                (states.model=='SPARSE_39__DIAG_FROZEN')&
                (states.scenario!='FULL_OBSERVED')].copy()
assert len(coverage)>=5 and (coverage.n>0).all()
assert np.isfinite(coverage[['coverage90','state_rmse']].to_numpy()).all()
assert shifts.label_offset_representation_steps.eq(0).any()
zero=shifts[(shifts.label_offset_representation_steps==0)&(shifts.model=='DEEPLOB')]
assert len(zero)==5
for h in HORIZONS:
    check_close(float(zero.loc[zero.horizon==h,'log_loss'].iloc[0]),
                float(pv.loc[(h,'DEEPLOB')].log_loss),tol=5e-6)

# Frozen-input-bound run fingerprint. Existing results are immutable on reruns.
CONFIG={'version':VERSION,'seed':SEED,'synthetic_replications':REPS,
        'synthetic_n_train':SYN_TRAIN,'synthetic_n_test':SYN_TEST,
        'source_hashes':input_hashes,'step15_fingerprint':gates['run_fingerprint'],
        'graph_mask_hash':mask_sha,'no_FI2010_training':True,
        'D8_D10_not_researcher_blind':True}
fingerprint=hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()
existing_spec=R/'00_RUN_SPEC.json'
if existing_spec.is_file():
    assert json.loads(existing_spec.read_text())['fingerprint']==fingerprint,\
        'Step16 exists for different inputs; archive previous results before rerun.'
R.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)
save_json('00_RUN_SPEC.json',{'fingerprint':fingerprint,'config':CONFIG})

# -------------------------------------------------------------
# SEPARATE SYNTHETIC KNOWN-DGP TEST; absolutely NOT external LOB validation.
# Under stable DGP, x5_t = .45*x5_{t-1} + .80*x0_{t-1} + N(0,.45^2).
# Train AR5, one TRUE cross edge, one WRONG edge, and dense VAR5 target
# regression using identical train samples; test consecutive held-out block.
# At a constructed regime-shift test, +.80 switches to -.80 with the
# train-fitted models unchanged. No model or hyperparameter is selected on test.
# -------------------------------------------------------------
def design(x,t,lags=5):
    return np.concatenate([x[t-k] for k in range(1,lags+1)],axis=0)

def fit_target(X,y,columns,alpha=1.):
    A=X[:,columns]; am=A.mean(0); ym=y.mean()
    Ac=A-am
    gram=Ac.T@Ac+alpha*np.eye(len(columns))
    coef=np.linalg.solve(gram,Ac.T@(y-ym))
    intercept=float(ym-am@coef)
    resid=y-(A@coef+intercept)
    variance=float(np.mean(resid**2))
    assert variance>1e-9
    return (np.asarray(columns,int),coef,intercept,variance)

def evaluate_fitted(model,X,y):
    cols,b,c,var=model
    pred=X[:,cols]@b+c
    residual=y-pred
    return {'rmse':float(np.sqrt(np.mean(residual**2))),
            'gaussian_nll':float(np.mean(.5*(np.log(2*np.pi*var)+residual**2/var))),
            'coverage90':float(np.mean(np.abs(residual)<=1.6448536269514722*np.sqrt(var))),
            'interval_width90':float(2*1.6448536269514722*np.sqrt(var))}

def synth_sequence(rng,n,alpha_cross,initial=None):
    a=np.empty((n,6),dtype=np.float64)
    prev=np.zeros(6,dtype=np.float64) if initial is None else np.asarray(initial,dtype=np.float64).copy()
    for i in range(n):
        noise=rng.standard_normal(6)*np.array([.6,.6,.6,.6,.6,.45])
        v=.4*prev+noise
        v[0]=.65*prev[0]+noise[0]
        v[5]=.45*prev[5]+alpha_cross*prev[0]+noise[5]
        a[i]=v; prev=v
    return a

def synthetic_run():
    rng=np.random.default_rng(SEED)
    results=[]
    for rep in range(REPS):
        tr=synth_sequence(rng,SYN_TRAIN+SYN_LAG,.8)
        start=tr[-1].copy()
        streams={
          'stable_same_DGP':synth_sequence(rng,SYN_TEST,-0.0+0.8,initial=start),
          'constructed_coefficient_flip':synth_sequence(rng,SYN_TEST,-.8,initial=start),
        }
        Xtr=np.stack([design(tr,t) for t in range(SYN_LAG,len(tr))])
        ytr=tr[SYN_LAG:,5]
        cols={'AR5':tuple(5+6*(k-1) for k in range(1,6)),
              'KNOWN_TRUE_EDGE':tuple(sorted(set([5+6*(k-1) for k in range(1,6)]+[0]))),
              'WRONG_EDGE':tuple(sorted(set([5+6*(k-1) for k in range(1,6)]+[1]))),
              'DENSE_VAR5':tuple(range(30))}
        fitted={name:fit_target(Xtr,ytr,cc) for name,cc in cols.items()}
        for regime,data in streams.items():
            combined=np.vstack([tr[-SYN_LAG:],data])
            Xt=np.stack([design(combined,t) for t in range(SYN_LAG,len(combined))])
            yt=data[:,5]
            for name,m in fitted.items():
                results.append({'replicate':rep,'regime':regime,'model':name,
                    'n_train':len(ytr),'n_test':len(yt),'missing_target':'x5_t',
                    'cross_edge_true_in_training':'x0_(t-1) -> x5_t',
                    **evaluate_fitted(m,Xt,yt)})
    f=pd.DataFrame(results)
    assert len(f)==REPS*2*4 and f.groupby(['replicate','regime']).size().eq(4).all()
    sums=f.groupby(['regime','model'],as_index=False).agg(
      independent_synthetic_replicates=('replicate','nunique'),
      mean_rmse=('rmse','mean'),mean_gaussian_nll=('gaussian_nll','mean'),
      mean_coverage90=('coverage90','mean'),
      min_rmse=('rmse','min'),max_rmse=('rmse','max'))
    winners=[]
    for regime in streams.keys():
        sub=f[f.regime==regime]
        mat=sub.pivot(index='replicate',columns='model',values='rmse')
        winners.append({'regime':regime,'replications':REPS,
          'true_edge_lower_rmse_than_AR5':int((mat.KNOWN_TRUE_EDGE<mat.AR5).sum()),
          'true_edge_lower_rmse_than_wrong_edge':int((mat.KNOWN_TRUE_EDGE<mat.WRONG_EDGE).sum()),
          'true_edge_lower_rmse_than_dense':int((mat.KNOWN_TRUE_EDGE<mat.DENSE_VAR5).sum()),
          'interpretation':'KNOWN artificial DGP only; not observed FI-2010 ground truth'})
    return f,sums,pd.DataFrame(winners)

syn,syntab,synwins=synthetic_run()
# Check the synthetic generator and model evaluation, not cherry-picked success gates.
assert syn['n_test'].eq(SYN_TEST).all() and syn['n_train'].eq(SYN_TRAIN).all()
assert syn['coverage90'].between(0,1).all() and np.isfinite(syn.rmse).all()

save_csv('01_matched_predictive_benchmark.csv',bench)
save_csv('02_graph_selected_cross_lag_slots.csv',pd.DataFrame(source_edges))
save_csv('03_missing_state_coverage_from_step7.csv',coverage)
save_csv('04_matched_gap_prior_comparisons.csv',pooled)
save_csv('05_day_consistency.csv',consistent)
save_csv('06_original_label_formula_diagnostics_UNVERIFIED.csv',formula)
save_csv('07_bidirectional_shift_diagnostics.csv',shifts)
save_csv('08_synthetic_known_DGP_replicates.csv',syn)
save_csv('09_synthetic_known_DGP_summary.csv',syntab)
save_csv('10_synthetic_known_DGP_falsification_counts.csv',synwins)
save_csv('11_input_hashes_checked.csv',pd.DataFrame(verified))

# Figures are document visualizations; never substitute figure data for source tables.
def figure_once(name,draw):
    path=FIG/name
    if path.is_file():
        assert path.stat().st_size>1000
        return
    fig=draw()
    fig.savefig(path,dpi=180,bbox_inches='tight',metadata={'Software':'CausalLOB Step16'})
    plt.close(fig)
    assert path.stat().st_size>1000

def fig_horizon():
    fig,ax=plt.subplots(figsize=(9,4.4))
    for col,name in [('DeepLOB_logloss','DeepLOB'),('DBN_logloss','DBN'),('train_prior_logloss','Train-only prior')]:
        ax.plot(bench.horizon_underlying_events,bench[col],marker='o',label=name)
    ax.set(xlabel='Publisher horizon (underlying events)',ylabel='Matched multiclass log loss',
           title='Retrospective full-observation forecasts (lower is better)')
    ax.legend();ax.grid(alpha=.2)
    return fig

def fig_ba():
    fig,ax=plt.subplots(figsize=(9,4.4))
    ax.plot(bench.horizon_underlying_events,bench.DeepLOB_BA,marker='o',label='DeepLOB')
    ax.plot(bench.horizon_underlying_events,bench.DBN_BA,marker='o',label='DBN')
    ax.axhline(1/3,linestyle=':',label='Three-class reference 1/3')
    ax.set(xlabel='Publisher horizon (underlying events)',ylabel='Balanced accuracy',
           title='Identical retrospective cohorts; balanced accuracy is not log loss')
    ax.legend();ax.grid(alpha=.2)
    return fig

def fig_gap():
    fig,ax=plt.subplots(figsize=(9,4.4))
    v=pooled.copy()
    v['label']=v.scenario.astype(str).str.replace('MISS_RETURN_AND_OFI','Return+OFI',regex=False).str.replace('HISTORY_ONLY','History only',regex=False)+\
       ' / '+v.gap_representation_steps.astype(str)+' / h'+v.horizon.astype(str)
    ax.barh(v.label[::-1],v.prior_minus_sparse_logloss[::-1])
    ax.axvline(0,linewidth=1,color='black')
    ax.set(xlabel='Prior minus sparse DBN log loss; negative = prior better',
           title='All twelve original missing-gap failures (697 matched endpoints)')
    ax.tick_params(axis='y',labelsize=7)
    return fig

def fig_synthetic():
    fig,ax=plt.subplots(figsize=(9,4.4))
    regimes=['stable_same_DGP','constructed_coefficient_flip']
    width=.18;x=np.arange(2)
    for i,name in enumerate(('AR5','KNOWN_TRUE_EDGE','WRONG_EDGE','DENSE_VAR5')):
        vals=[float(syntab.loc[(syntab.regime==rg)&(syntab.model==name),'mean_rmse'].iloc[0]) for rg in regimes]
        ax.bar(x+(i-1.5)*width,vals,width,label=name)
    ax.set_xticks(x,labels=['Same synthetic DGP','Synthetic coefficient flip'])
    ax.set(ylabel='Mean x5 state RMSE across 40 simulated replicates',
           title='Artificial known graph: illustrative stress test, NOT market validation')
    ax.legend(ncol=2,fontsize=8);ax.grid(axis='y',alpha=.2)
    return fig

def fig_day():
    fig,ax=plt.subplots(figsize=(9,4.4))
    for d in ('8','9','10'):
        ddf=daily[daily.day.astype(str)==d].sort_values('horizon')
        assert len(ddf)==5
        ax.plot(ddf.horizon,ddf.DeepLOB_logloss_improvement_over_DBN,marker='o',label='Day '+d)
    ax.axhline(0,color='black',linewidth=1)
    ax.set(xlabel='Publisher horizon (underlying events)',
           ylabel='DBN log loss minus DeepLOB log loss',
           title='Three historically inspected day blocks; descriptive only')
    ax.legend();ax.grid(alpha=.2)
    return fig

for filename,fun in [
 ('fig01_matched_logloss.png',fig_horizon),('fig02_balanced_accuracy.png',fig_ba),
 ('fig03_all_gap_prior_failures.png',fig_gap),('fig04_synthetic_illustration.png',fig_synthetic),
 ('fig05_day_level_logloss.png',fig_day)]:figure_once(filename,fun)

# Compact full paper with source-bound tables; typesettable LaTeX source also below.
bench_lines=['| h (source events) | matched n | DeepLOB BA | DBN BA | DeepLOB LL | DBN LL | Prior LL |',
             '|---:|---:|---:|---:|---:|---:|---:|']
for x in bench.itertuples(index=False):
    bench_lines.append(f'| {x.horizon_underlying_events} | {x.matched_n:,} | {x.DeepLOB_BA:.4f} | '
                       f'{x.DBN_BA:.4f} | {x.DeepLOB_logloss:.4f} | {x.DBN_logloss:.4f} | '
                       f'{x.train_prior_logloss:.4f} |')
gap_lines=['| Missing scenario | Gap (representation steps) | Horizon (source events) | Sparse LL | Prior LL | Prior − sparse LL |',
           '|---|---:|---:|---:|---:|---:|']
for x in pooled.sort_values(['scenario','gap_representation_steps','horizon']).itertuples(index=False):
    gap_lines.append(f'| {x.scenario} | {x.gap_representation_steps} | {x.horizon} | '
        f'{x.sparse_logloss:.4f} | {x.prior_logloss:.4f} | {x.prior_minus_sparse_logloss:.4f} |')
cover_lines=['| Scenario | RMSE | Empirical nominal-90% coverage | Absolute deviation from 90% |',
             '|---|---:|---:|---:|']
for x in coverage.sort_values('scenario').itertuples(index=False):
    cover_lines.append(f'| {x.scenario} | {x.state_rmse:.4f} | {x.coverage90:.4f} | {x.abs_coverage_error90:.4f} |')
syn_lines=['| Simulated regime | Model | Mean RMSE | Mean 90% coverage |',
           '|---|---|---:|---:|']
for x in syntab.itertuples(index=False):
    syn_lines.append(f'| {x.regime} | {x.model} | {x.mean_rmse:.4f} | {x.mean_coverage90:.4f} |')

paper=f'''# CausalLOB: Sparse Temporal Graphical Models for Probabilistic Inference in Partially Observed Limit Order Books

**Study status:** Frozen FI-2010 retrospective investigation and separately identified synthetic demonstration. NOT a fresh external experiment, an identified causal effect, or a trading profitability study. Manuscript is a submission draft requiring author review.

## Abstract
We examine whether a sparse, observationally discovered temporal dependency structure improves the reconstruction of missing limit-order-book (LOB) state beyond autoregressive history and equally structured alternatives. On FI-2010, six engineered state variables are modeled with five lags and a frozen 39-slot graph comprising 30 self-lags and nine cross-variable links. The graph was selected on Days 1–5, assessed internally on Days 6–7, and the state transition and probabilistic classification models were fitted on Days 1–7. Days 8–10 provide retrospective, historically inspected evaluation, not researcher-blind external testing. We compare AR(5), static conditioning, dense VAR(5), 100 matched random graphs, five ordered-probit output heads, and a richer-input DeepLOB-style neural baseline. Sparse transition RMSE ({float(nr.causallob):.6f}) modestly improves on AR(5) ({float(nr.ar5):.6f}) but does not surpass dense VAR(5) ({float(nr.dense_var5):.6f}). DeepLOB has lower matched log loss at all five horizons. Crucially, in all 12 pooled consecutive-missingness tests the frozen sparse model produces worse outcome log loss than a train-only class prior. These findings support an implemented exact probabilistic missing-state inference procedure but do not demonstrate identified market causality, prediction dominance, external generalization, or net trading returns.

## 1. Question and contribution
The precise question is: given that current engineered LOB variables may be missing, do selected cross-lag dependencies improve their predictive state distribution relative to past-self dynamics and alternative source identities? We contribute a reproducible experiment, boundary-aware feature and label alignment, exact Gaussian conditioning, and transparent negative and positive diagnostics. **We do not claim that the observational arrows identify interventions or that the selected sparse graph is uniquely optimal.**

## 2. Data and partition
FI-2010 NoAuction Decimal Precision contains 394,337 aggregated representations with 40 raw LOB inputs and publisher labels at five underlying-event horizons 10, 20, 30, 50, and 100. Ten days each concatenate five anonymous-instrument sequences; we process 50 boundary-separated streams, not 50 independent market days. A representation aggregates ten source events. Days 1–5 are used for graph discovery; Days 6–7 for internal structure diagnostics and DeepLOB checkpoint selection; Days 1–7 for DBN parameter fitting; Days 8–10 for retrospective evaluation. Previous research exposed Days 6–10, so avoid blinded-test claims. The label map used downstream is publisher 3 Down, 2 Neutral, 1 Up mapped to 0, 1, 2. The source labels are preserved byte-for-byte. Exact original message-level generation cannot be verified from these decimated observations alone. Our conservative purge discards the final h representation rows per sequence for the corresponding nominal h-source-event label. Purging is conservative; it is NOT proof of the publisher’s generation mechanism.

## 3. Method
The engineered state is six-dimensional: spread, depth, level-1 imbalance, levels-2–5 imbalance, normalized quote-change OFI proxy, and current log-mid-return. The OFI proxy uses quote changes; it must not be described as executed-trade flow. The transition CPDs use five lagged standardized representations and a diagonal training-only Gaussian innovation covariance. Each conditional mean is linear in its frozen parents. A sparse mask contains 30 autoregressive and nine cross-lag coefficients. The return labels at each horizon are modeled with frozen ordered-probit heads. Given observed current components, Gaussian conditioning produces the unobserved state posterior; the heads integrate predictive uncertainty analytically under their assumed Gaussian latent response. A train-only covariance-shrinkage variant is a separate sensitivity, not the original independent-noise CPD factorization. Under consecutive missingness, the model recursively propagates state mean and covariance through an augmented lag state and then conditions on measurements when present.

## 4. Comparator design and outcomes
Matched AR(5), sparse DBN, dense VAR(5), target-and-lag-matched random sparse graphs (100), a static baseline and training-only class frequencies separate temporal persistence from cross-edge identity. The DeepLOB-style baseline uses 100-by-40 raw-book windows, ends each window at origin t, and fits on Days 1–5 with checkpoint selection on Days 6–7. It is not a same-information/same-capacity comparison with the six-variable DBN. The Step14 audit successfully replayed all frozen DeepLOB checkpoints; that addresses numerical reproduction and input timing but does not recover the publisher label generator.

### 4.1 Matched complete-observation outcomes
{chr(10).join(bench_lines)}

The DeepLOB-versus-DBN log-loss advantage holds on all three D8–D10 day blocks at all horizons. At h10, DBN instead has higher balanced accuracy and lower Brier loss; the metrics disagree. Only three nonblind evaluation-day blocks exist and the many windows within days are dependent; no conventional iid significance is asserted.

![Matched log loss](figures/fig01_matched_logloss.png)
![Balanced accuracy](figures/fig02_balanced_accuracy.png)

### 4.2 Graph-specific transition evidence
Frozen sparse RMSE {float(nr.causallob):.6f}; AR(5) {float(nr.ar5):.6f}; dense VAR(5) {float(nr.dense_var5):.6f}; matched random-graph median {float(nr.null_median):.6f}. The fraction of 100 matched random graphs with worse transition RMSE than the selected graph is {float(nr.fraction_random_worse):.2f}. This descriptive reference is not a selection-adjusted graph p-value. Autoregressive persistence accounts for much of the reconstruction benefit; edge-identity improvements are modest and scenario-dependent. Observational orientation remains unidentifiable without further assumptions.

### 4.3 Missing-state interval behavior
The original frozen sparse diagonal-Gaussian model, without retrospective recalibration, has the following Step7 pooled reconstruction and nominal-90%-interval results. Coverage is marginal over missing state components; no joint 90% credible region is implied.

{chr(10).join(cover_lines)}

### 4.4 Consecutive missingness and the class-prior failure
There are 697 fixed spaced endpoints across three evaluation days (277, 260 and 160). Six scenario-by-gap groups, each with two horizons, are matched across methods. **All twelve show the prior has lower log loss**; a graph-versus-AR(5) increment does not establish task utility against this baseline.

{chr(10).join(gap_lines)}

![Negative results](figures/fig03_all_gap_prior_failures.png)

## 5. Synthetic illustration explicitly separate from FI-2010
To isolate a scenario with a known data-generating edge without obtaining new market observations, we fixed an artificial six-state linear Gaussian process with x0(t−1) → x5(t), a single nonzero coefficient +0.80, target autoregression +0.45 and Gaussian noise. We fit four Ridge models to 800 simulated train targets over {REPS} independent seeds and predict 300 subsequent targets for each of a stable stream and a deliberately changed stream with edge coefficient −0.80. This design tests model behavior under **a constructed simulation**, neither empirical causal identification nor independent real LOB generalization. No test-dependent tuning is performed. It is not a full graph-discovery or Bayesian graph-uncertainty experiment.

{chr(10).join(syn_lines)}

![Synthetic control](figures/fig04_synthetic_illustration.png)

## 6. Threats to validity
(1) D8–D10 were historically inspected. (2) Only three day-level blocks; five anonymous sequences per day are not independent market days. (3) Source targets lack independently reproduced message-level generation; destructive label shifts diagnose timing only and are inconclusive with label persistence. (4) A five-lag engineered Gaussian DBN and a 100-step, 40-feature neural network have different information and fitting periods. (5) Training-only covariance correlations may help but violate the primary model's independent contemporaneous CPDs. (6) Graph candidates were searched; matched random-graph comparisons are not adjusted for this selection. (7) Artificial masks may not represent real-world non-random missingness. (8) We did not model fills, fees, queues, latency or executable net returns. (9) Our separate simulation is deliberately simplified and must not be described as a new market dataset.

## 7. Conclusions
On the current evidence, exact sparse-DBN probabilistic inference is implemented and frozen; a limited cross-edge reconstruction contribution exists, but dense VAR is competitive and DeepLOB has better full-observation probabilistic predictions. Under consecutive missing observations, the frozen sparse classifier fails a basic class-prior comparison in all 12 matched settings. These negative results constrain forecasting claims. A genuinely unseen, licensed, semantically compatible market dataset and verified label-generation implementation are unavailable; therefore no independent generalization or profitability claim is made. A graph-posterior methodological extension is future work and is not represented as an accomplished contribution.

## References and data sources (verify formatting before submission)
- FI-2010 benchmark dataset: https://www.research.ed.ac.uk/en/datasets/benchmark-dataset-for-mid-price-forecasting-of-limit-order-book-d/
- Ntakaris et al., *Benchmark Dataset for Mid-Price Prediction of Limit Order Book Data*, https://arxiv.org/abs/1705.03233 (source-label description; exact code still not available).
- Li et al., *CausalStock: Deep End-to-end Causal Discovery for News-driven Multi-stock Movement Prediction*, NeurIPS 2024, https://proceedings.neurips.cc/paper_files/paper/2024/hash/54d689d58fe54c92aee2d732fc49fca8-Abstract-Conference.html (different task).
- Rahmani and Frossard, *Flow based approach for Dynamic Temporal Causal models with non-Gaussian or Heteroscedastic Noises*, NeurIPS 2025, https://proceedings.neurips.cc/paper_files/paper/2025/hash/0c573ef147721b7f9d1dbfe549ddd25f-Abstract-Conference.html (methodological context; no claimed benchmark comparison).
- Choi, Lee and Oh, *Discovering causal relationships among financial variables associated with firm value using a dynamic Bayesian network*, 2025, https://www.aimspress.com/article/doi/10.3934/DSFE.2025001 (corporate panel data, not intraday LOB).

**Reproducibility anchor:** Step15 frozen fingerprint `{gates['run_fingerprint']}`; Step16 evidence fingerprint `{fingerprint}`. Supporting exact source tables and source SHA-256s are supplied in this package.
'''
write_once(R/'12_SUBMISSION_MANUSCRIPT.md',paper)

methods=r'''# Mathematical methods and reproduction appendix

## A. Observation unit, time and horizon
A raw FI-2010 record is a ten-event representation. For any target observation at
representation time t, 100x40 DeepLOB input includes rows [t-99,...,t], never t+1.
Anonymous-instrument/day boundaries cannot be traversed. Labels indexed h=10,20,
30,50,100 denote PUBLISHER underlying-event horizons, not 10/20/... future
representation rows. In the frozen protocol the end-of-sequence purge is the last
h representation rows, a conservative policy whose exact correspondence to the
publisher's original source-message generator cannot be confirmed.

## B. State transition, independent-noise primary model
Let z_t in R^6 be standardized state. Let M_(j,i,l) be the frozen mask for
source i, target j, lag l in {1,...,5}; names are reported in the accompanying
cross-lag slot CSV. In the primary DBN,

  z_(j,t) = c_j + sum_(i,l):M_(j,i,l)=1 B_(j,i,l) z_(i,t-l) + e_(j,t),
  e_t ~ N(0, diag(v_1,...,v_6)).

There are 30 self-lag and 9 cross-lag slots. The 6x30 transition B is fitted
on Days 1–7 with training-only Ridge regularization. The graph is observational,
not an identified causal DAG over actual market interventions.

## C. Gaussian missing-state inference
Partition conditional prediction (mu,Q) into observed o and missing m. For an
exact noiseless current observation z_o,

 mu_(m|o) = mu_m + Q_mo Q_oo^{-1} (z_o - mu_o),
 Q_(m|o)  = Q_mm - Q_mo Q_oo^{-1} Q_om.

For diagonal Q, current observation provides no same-time noise borrowing;
forecasted history (cross edges) nevertheless shifts mu_m. If the current
observation is fully missing, the one-step posterior equals the prior up to
floating-point tolerance. The Step9 augmented-lag filter recursively propagates
full 30D covariance for multiple consecutive missing steps. Full shrinkage-Q is
a separate train-only sensitivity with correlated innovations, not the frozen
independent-contemporaneous-CPD graph model.

## D. Outcome probabilities and uncertainty
Each horizon has a training-frozen ordered-probit head beta_h, c_(1,h), c_(2,h).
For latent u = beta_h^T z + epsilon, epsilon~N(0,1), write

  s = sqrt(1 + beta_h^T S_(t|o) beta_h)
  P(Down)    = Phi((c_1-beta^T mu_(t|o))/s),
  P(Neutral) = Phi((c_2-beta^T mu_(t|o))/s)-P(Down),
  P(Up)      = 1-Phi((c_2-beta^T mu_(t|o))/s).

Here mu_(t|o) is the COMPLETE six-state posterior mean: observed
components equal their measurements; missing components use mu_(m|o).
S_(t|o) has zero covariance in observed rows/columns and Q_(m|o) in the
missing block. This is exact for the assumed Gaussian-plus-probit model, NOT a proof
that observed market data follow the model, nor an estimate of causal impact.

## E. Scores and comparative designs
Multiclass log loss = -mean(log(p_i[y_i])); multiclass Brier =
mean(sum_k (p_ik - 1[y_i=k])^2); balanced accuracy = mean of class recalls.
A prior based exclusively on D1–7 class frequencies is essential in missingness
experiments; transition RMSE improvements alone cannot substitute for outcome
baseline comparison. Confidence/calibration is evaluated descriptively without
D8–10 fitting. Day-level scores are three descriptive dependent-data blocks;
no independent window-level confidence intervals or significance claims.

## F. Synthetic-known-DGP supplementary test
Artificial six-var process with diagonal noises; target x5 has x0(t-1) parent
coefficient +.80 and self AR coefficient +.45. Four target-only predictors share
800 training observations and predict 300 genuinely unseen SIMULATED observations
per replication for 40 seeds. The deliberate coefficient-flip condition (-.80)
is a constructed regime shift, not an observed market shift. Selection of the
known true edge is by the simulator's design, not graph discovery. These
replications do not qualify as genuine external financial-data evidence.
'''
write_once(R/'13_METHODS_AND_MATH_APPENDIX.md',methods)

# LaTeX source is a reproducible draft, NOT compiled / accepted / certified.
def latex_escape(s):
    v=str(s)
    for a,b in [('&',r'\&'),('%',r'\%'),('_',r'\_'),('#',r'\#')]:v=v.replace(a,b)
    return v
latex_rows='\n'.join(
    f'{r.horizon_underlying_events} & {r.matched_n:,} & {r.DeepLOB_BA:.3f} & {r.DBN_BA:.3f} & '
    f'{r.DeepLOB_logloss:.3f} & {r.DBN_logloss:.3f} & {r.train_prior_logloss:.3f} \\\\'
    for r in bench.itertuples(index=False))
tex=r'''\documentclass[10pt]{article}
\usepackage[margin=0.85in]{geometry}
\usepackage{amsmath,amssymb,graphicx,booktabs,hyperref}
\title{CausalLOB: Sparse Temporal Graphical Models for Probabilistic Inference in Partially Observed Limit Order Books}
\author{Author names to be completed by project team}
\date{}
\begin{document}\maketitle
\begin{abstract}
We examine a frozen sparse Gaussian temporal graph for six engineered limit-order-book variables under missing state. The model has 30 autoregressive and nine cross-lag slots. On FI-2010, sparse transition RMSE is 0.586137, AR(5) obtains 0.588347, and dense VAR(5) obtains 0.585171. On identical full-observation retrospective cohorts DeepLOB has lower log loss at all five horizons; however, the DBN has higher balanced accuracy and lower Brier score at h10. Under 12 pooled consecutive-missingness conditions, the training-only class prior outperforms the sparse classifier in all 12. Previously inspected Days 8--10 do not constitute blind external validation. No causal, external-generalization, or profitability claim is made.
\end{abstract}
\section{Research question and data}
Can nine selected cross-variable lagged links improve missing-state inference beyond self-history alone? FI-2010 contributes 394,337 ten-event LOB representations with 40 book inputs and publisher labels at five underlying-event horizons. The 50 day-by-anonymous-instrument sequences arise within ten days; they are not independent day blocks. Graph discovery uses Days 1--5, internal validation Days 6--7, state/head fitting Days 1--7, and retrospectively examined outcomes Days 8--10. Exact source-message-level label generation is not reproduced.
\section{Model and probabilistic inference}
For a standardized six-state vector $z_t$, the primary diagonal-noise graphical transition is
\begin{equation} z_{j,t}=c_j+\sum_{(i,\ell)\in\mathrm{Pa}(j)} B_{ji\ell}z_{i,t-\ell}+\varepsilon_{j,t},\qquad\varepsilon_t\sim\mathcal N(0,\mathrm{diag}(v)).\end{equation}
With observed $z_o$ and missing $z_m$, Gaussian conditioning yields
\begin{align} \mu_{m\mid o}&=\mu_m+Q_{mo}Q_{oo}^{-1}(z_o-\mu_o),\\ Q_{m\mid o}&=Q_{mm}-Q_{mo}Q_{oo}^{-1}Q_{om}.\end{align}
Frozen ordered-probit classifiers integrate Gaussian state uncertainty. Correlated-noise shrinkage is an ablation, not the primary independent-innovation DBN. The edge orientations are observational dependency selections, not identified interventions.
\section{Matched retrospective forecasting}
\begin{table}[h]\centering\small\begin{tabular}{rrrrrrr}\toprule
Horizon & N & DeepLOB BA & DBN BA & DeepLOB LL & DBN LL & Prior LL \\
\midrule
__ROWS__
\bottomrule\end{tabular}\caption{Matched evaluation; horizons are publisher underlying-event units. Lower LL is better.}\end{table}
\begin{figure}[h]\centering\includegraphics[width=.85\linewidth]{figures/fig01_matched_logloss.png}\caption{Identical-cohort probabilistic predictions.}\end{figure}
\section{Sparse graph and missingness results}
Sparse transition RMSE is __SPARSE__, compared with AR(5) __AR__ and dense VAR(5) __DENSE__. A 100-graph target/lag-matched random reference has median __RANDOM__. This reference is descriptive, not a search-adjusted significance test. The frozen sparse model has worse outcome log loss than the class prior in all twelve matched scenario/gap/horizon conditions, despite modest improvements versus AR(5) in some cases.
\begin{figure}[h]\centering\includegraphics[width=.9\linewidth]{figures/fig03_all_gap_prior_failures.png}\caption{Negative result: prior-minus-DBN log loss is negative for all twelve conditions.}\end{figure}
\section{Artificial known-DGP control (not external financial evidence)}
Forty independent artificial Gaussian replications examine a true x0-to-x5 lag edge and a deliberately flipped held-out edge coefficient. Four training-only Ridge predictors are compared. These synthetic outcomes do not validate generalization to new markets or causal identification from FI-2010.
\section{Limitations and conclusion}
The publisher's exact source-message label generator, researcher-blind external dataset and executable after-cost PnL are unavailable. The deep baseline uses richer raw inputs and a different fitting period, limiting architecture-only conclusions. There are only three historically inspected evaluation-day blocks. The supported result is a reproducible probabilistic sparse-state inference implementation with qualified, modest cross-lag benefits; forecasting dominance is not supported. See the accompanying Markdown manuscript for complete tables, methods, and source notes.
\end{document}
'''.replace('__ROWS__',latex_rows).replace('__SPARSE__',f'{float(nr.causallob):.6f}').replace('__AR__',f'{float(nr.ar5):.6f}').replace('__DENSE__',f'{float(nr.dense_var5):.6f}').replace('__RANDOM__',f'{float(nr.null_median):.6f}')
write_once(R/'14_MANUSCRIPT_DRAFT_UNCOMPILED.tex',tex)

# Truthful completion gates, explicitly list uncompletable tasks.
claim={
 'step16_version':VERSION,'fingerprint':fingerprint,'all_frozen_upstream_hashes_verified':True,
 'step15_score_and_day_parity_verified':True,'checkpoint_replay_reported_step14_pass':True,
 'frozen_FI2010_models_modified':False,'FI2010_models_trained_in_step16':False,
 'new_market_dataset_used':False,'synthetic_test_separate_from_FI2010':True,
 'synthetic_data_graph_known_by_construction':True,
 'publisher_exact_generator_verified':False,'external_researcher_blind_test':False,
 'identified_market_causal_edges':False,'net_trading_profit':False,
 'novel_full_graph_posterior_implemented':False,'publication_acceptance_or_readiness_established':False,
 'FI2010_submission_manuscript_draft_produced':True,
 'latex_compiled':False,'presentation_slides_generated':False,
 'presentation_speaker_notes_produced':True,
 'author_actions':['Verify bibliographic formatting and authorship',
   'Attach any course-required source notebooks/scripts if outside the CausalLOB Drive root',
   'Review course submission format and prepare any video/presentation recording',
   'If independent replication is required, obtain unseen market data and source-label code; cannot synthesize substitutes',
   'No profit statement without executable bid/ask and fee/fill/queue simulation'],
}
save_json('15_HONEST_COMPLETION_GATES.json',claim)

presentation='''# CausalLOB: presentation notes (speaker storyboard; no PPTX created)

Slide 1 — Title and scope: Sparse lagged Gaussian inference with missing FI-2010 LOB variables; NOT identified causality.
Slide 2 — Problem and partition: 10 days, 50 boundary-safe streams, D1–5 graph, D6–7 internal validation, D1–7 parameters, D8–10 retrospective.
Slide 3 — Inputs and graph: six states, 30 autoregressive + 9 selected cross-lags. The OFI is quote-change proxy.
Slide 4 — Inference: Gaussian state conditioning and integrated ordered-probit class probabilities; diagonal covariance primary.
Slide 5 — Baselines: static, AR5, matched random graphs, dense VAR5, DeepLOB 100x40 richer-input baseline.
Slide 6 — Complete-observation results: show fig01 and fig02; DeepLOB wins log loss at all horizons; DBN h10 BA/Brier exceptions.
Slide 7 — Does edge identity matter? Sparse 0.586137 RMSE; AR 0.588347; dense 0.585171; random median in source table. No graph p-value.
Slide 8 — Failure modes: fig03; class prior beats CausalLOB for all 12 consecutive missingness cases. Do not hide this.
Slide 9 — Known-DGP illustration: fig04, artificial fixed seed/replications; cannot substitute for unseen market data.
Slide 10 — Conclusions: exact inference implemented; marginal graph value; three historically inspected days; publisher source generator, external dataset, PnL unresolved.

Oral defense questions:
Q: Why call it CausalLOB? A: A historical research name; graph arrows are observational lag dependencies, not intervention-identified causes.
Q: Why does DeepLOB improve at long horizons? A: Its checkpoint results and input time origin replay; richer input and capacity and target persistence may contribute, but no mechanism proven.
Q: Why is a negative prior comparison important? A: An added graph effect versus AR cannot establish useful missing-state classification versus a trivial calibrated baseline.
Q: Are 697 rows independent? A: No; they are spaced events across just three day blocks and the full history was inspected.
Q: Do you have external validation or profit? A: No. These are explicit study limitations.
'''
write_once(R/'16_PRESENTATION_SPEAKER_NOTES.md',presentation)

# Reproduction README and file manifest. Never copy raw data, training checkpoints, or external claims.
readme=f'''# CausalLOB Step16 — course-submission package without new dataset

## Purpose
This single cell finishes the feasible tasks after verified Step15: assembles a referenced draft manuscript, conditional-Gaussian mathematical appendix, five figures, complete negative-case table, observational graph CSV, descriptive uncertainty coverage, short professor summary and talk script. It also executes 40 *artificial* known-DGP replications with an explicit regime switch. It does NOT re-fit, reselect or modify any FI-2010 model.

## How to reproduce
1. Keep CausalLOB frozen Steps 2/4/5/7/8/9/14/15 in the SAME existing Google Drive root.
2. Execute `CausalLOB_STEP16_NO_EXTERNAL_FIXED_v2_ONE_CELL.ipynb` as one cell in Google Colab. GPU not needed. You can override the root with `CAUSALLOB_ROOT=/your/path` only if all original outputs are there.
3. Run the Step15 final audit separately if independent score recomputation is required; Step16 verifies all its recorded upstream SHA-256 files and the Step15 score tables, but does not perform a third expensive DeepLOB forward pass.
4. Review `12_SUBMISSION_MANUSCRIPT.md` and `14_MANUSCRIPT_DRAFT_UNCOMPILED.tex`. The LaTeX draft was NOT compiled or author-checked.
5. Attach project source notebooks, the official dataset license/instructions, any professor-required demonstration and optional video separately. This ZIP intentionally excludes raw data, model weights and unidentified user files.

## Immutable dependency and audit
- Source root: `{ROOT}`
- Frozen Step15 fingerprint: `{gates['run_fingerprint']}`
- Step16 fingerprint: `{fingerprint}`
- Frozen mask: `{mask_sha}`
- Verified upstream files: {len(verified)}
- External source-message label code: NOT provided; preserve publisher labels and report the limitation.
- Days 8–10 previously inspected, NOT blind independent testing.
- Profitability not established; no simulated transaction costs, fills or queue priorities.

## What would require new inputs and cannot be honestly completed here
A genuine external dataset with timestamp/instrument identity, consistent book field semantics, legally usable source events and a verified label generator; a prospective evaluation fixed before seeing its labels; transaction-specific execution data and assumptions for net PnL; a separately developed novel graph-posterior inference method for a methodologically stronger paper. Artificial data created here cannot replace any of those.

## Files
01–11: source-derived tables, synthetic replication tables and hash manifests. 12: full Markdown manuscript. 13: mathematical appendix. 14: LaTeX source only. 15: completion gates. 16: speaker notes. 17: professor summary. 18: source provenance. 19: protocol copied unchanged. 20: reproducibility script/source availability. Figures under `figures/`.
'''
write_once(R/'README_FIRST.md',readme)
summary=f'''# Professor-facing summary — FI-2010 retrospective study complete

Question: Can observationally chosen sparse lag dependencies aid inference when engineered LOB state is missing?

Dataset: FI-2010; ten days, 50 day×anonymous-instrument sequences, five publisher source-event horizons. Graph D1–5, internal validation D6–7, DBN fit D1–7, retrospective D8–10. Days 8–10 are not blind.

Primary transition RMSE: sparse {float(nr.causallob):.6f}, AR5 {float(nr.ar5):.6f}, dense VAR5 {float(nr.dense_var5):.6f}. A 100-random-graph same-target/lag reference is descriptive, not a selection-aware p-value.

All five matched full-observation log losses favor DeepLOB over DBN. The DBN has higher balanced accuracy and lower Brier at h10. DeepLOB gets richer input features and a different fit/selection period, so this is a predictive benchmark, not a capacity-controlled causal-structure ablation.

Critically, class prior log loss beats sparse DBN in **all 12/12 consecutive missingness settings**. This constrains the headline claim. Step14 fully replayed DeepLOB and verified forecast-window timing; verified publisher message-level label generation is not available.

Step16 supplementary experiment: 40 artificial known-graph replications under stable versus purposefully flipped cross-edge regimes; synthetic stress testing only, no new market dataset.

Completed: implementation, reproducibility and audits, benchmark, honest report and source-bound charts, methodological appendix, course presentation notes, artificial diagnostics.

Not completed: unseen external market replication, exact source-message publisher-label code, true market causal identification, execution/after-cost PnL, NeurIPS-level novel graph uncertainty or acceptance. No code can infer absent external observations.

Frozen fingerprint: `{gates['run_fingerprint']}`.
'''
write_once(R/'17_PROFESSOR_ONE_PAGE.md',summary)
provenance={
 'source_dataset':'FI-2010 NoAuction Decimal Precision, original 40 LOB inputs, publisher labels',
 'source_representations':394337,'source_days':10,'within_day_sequence_segments':50,
 'publisher_label_map_original':{'1':'Up','2':'Neutral','3':'Down'},
 'downstream_label_map':{'0':'Down','1':'Neutral','2':'Up'},
 'original_label_generation_code_available':False,
 'publisher_exact_message_label_reconstruction':False,
 'formula_candidates_file':'06_original_label_formula_diagnostics_UNVERIFIED.csv',
 'train_day_roles':{'graph_discovery':[1,2,3,4,5],
    'internal_graph_validation':[6,7],'DBN_parameters':[1,2,3,4,5,6,7],
    'DeepLOB_train':[1,2,3,4,5],'DeepLOB_validation':[6,7],
    'retrospective_assessment':[8,9,10]},
 'source_backing_step15_fingerprint':gates['run_fingerprint'],
 'all_step15_upstream_SHA256_rechecked':True,
 'synthetic_experiment_not_fi2010':True,
 'synthetic_experiment_not_external_market_evidence':True,
 'figures_from_immutable_tables':True,
 'source_table_paths':{k:str(v) for k,v in INPUTS.items()},
}
save_json('18_SOURCE_AND_DATA_PROVENANCE.json',provenance)
write_once(R/'19_EXTERNAL_PROTOCOL_UNEXECUTED.md',INPUTS['step15_protocol'].read_bytes())

# Optional bundling of explicitly offered source code ONLY; no stealth copy of unrelated files.
# User can set CAUSALLOB_CODE_DIR to a directory of approved .py/.ipynb research sources.
code_dir=os.environ.get('CAUSALLOB_CODE_DIR','').strip()
included_sources=[];warnings=[]
if code_dir:
    d=Path(code_dir)
    if not d.is_dir():
        warnings.append('CAUSALLOB_CODE_DIR specified but not a directory; no source code copied.')
    else:
        destination=R/'provided_code'
        for src in sorted(d.rglob('*')):
            if not src.is_file() or src.suffix.lower() not in ('.py','.ipynb'):continue
            relative=src.relative_to(d)
            if len(relative.parts)>4 or any(x.startswith('.') for x in relative.parts):continue
            if src.stat().st_size>4*1024*1024:continue
            dst=destination/relative
            dst.parent.mkdir(parents=True,exist_ok=True)
            if dst.exists():
                assert sha_file(dst)==sha_file(src),f'Existing approved source changed: {dst}'
            else:shutil.copy2(src,dst)
            included_sources.append({'source':str(src),'package_file':str(dst.relative_to(R)),
                                     'sha256':sha_file(src)})
if not code_dir:
    warnings.append('No CAUSALLOB_CODE_DIR supplied. Original Steps 1–15 notebooks/scripts NOT bundled; user must attach them for a source-complete submission.')
code_status={'original_source_directory_user_supplied':bool(code_dir),
             'original_source_files_bundled':len(included_sources),
             'source_complete_submission_certified':False,
             'warnings':warnings,'included_sources':included_sources}
save_json('20_CODE_BUNDLE_STATUS.json',code_status)

# Input fingerprint change check after ALL operations, protecting frozen source data.
input_hashes_after={key:sha_file(p) for key,p in INPUTS.items()}
assert input_hashes_after==input_hashes,'Frozen input changed during Step16.'
for row in verified:
    assert sha_file(Path(row['path']))==row['sha256'],'Upstream source mutated during Step16.'

# Package only Step16-generated results, charts and optional user-approved source files.
files=[p for p in sorted(R.rglob('*')) if p.is_file() and p.name!='21_OUTPUT_MANIFEST.json']
manifest={'version':VERSION,'fingerprint':fingerprint,
          'only_existing_source_data_used':True,'synthetic_data_generated_with_fixed_seed':SEED,
          'n_upstream_verified':len(verified),'n_synthetic_replications':REPS,
          'files':{str(p.relative_to(R)):{'sha256':sha_file(p),'bytes':p.stat().st_size} for p in files}}
save_json('21_OUTPUT_MANIFEST.json',manifest)
files=[p for p in sorted(R.rglob('*')) if p.is_file()]
archive=OUT/'CAUSALLOB_STEP16_NO_EXTERNAL_SUBMISSION.zip'
if archive.is_file():
    with zipfile.ZipFile(archive,'r') as z:
        assert z.testzip() is None,'Existing ZIP damaged.'
        existing_names=set(z.namelist())
        assert existing_names=={'results/'+str(p.relative_to(R)) for p in files},'Existing ZIP file list differs.'
        for p in files:
            assert hashlib.sha256(z.read('results/'+str(p.relative_to(R)))).hexdigest()==sha_file(p),\
                f'Existing ZIP member differs: {p}'
else:
    temp=archive.with_name(archive.name+'.tmp')
    with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for p in files:z.write(p,arcname='results/'+str(p.relative_to(R)))
    with zipfile.ZipFile(temp,'r') as z:assert z.testzip() is None
    temp.replace(archive)
assert archive.is_file() and archive.stat().st_size>5000

print('\n'+'='*105)
print('STEP 16 COMPLETE — ALL FEASIBLE NO-EXTERNAL-DATA COURSE SUBMISSION DELIVERABLES')
print('='*105)
print('Frozen upstream SHA-256 PASS:',len(verified),'source files; input hashes unchanged.')
print('Matched horizons:',list(HORIZONS),'| frozen models trained/reselected: NO')
print(bench[['horizon_underlying_events','matched_n','DeepLOB_logloss','DBN_logloss','train_prior_logloss']].to_string(index=False))
print('Missing gap: prior wins',int((pooled.prior_minus_sparse_logloss<0).sum()),'/12')
print('Synthetic known-DGP diagnostic (NOT external market testing):')
print(synwins.to_string(index=False))
print('Paper:',R/'12_SUBMISSION_MANUSCRIPT.md')
print('Math appendix:',R/'13_METHODS_AND_MATH_APPENDIX.md')
print('LaTeX source (not compiled):',R/'14_MANUSCRIPT_DRAFT_UNCOMPILED.tex')
print('Speaker notes:',R/'16_PRESENTATION_SPEAKER_NOTES.md')
print('Code bundle status:',len(included_sources),'approved scripts; check 20_CODE_BUNDLE_STATUS.json')
print('ZIP:',archive)
print('UNRESOLVED BY DESIGN: exact source label generator, unseen external market data, net PnL, new graph posterior.')
print('Elapsed seconds:',round(time.monotonic()-START,1))

STEP16 SELF-TEST PASS: metrics, hashes, lag ordering, fail-closed parity.

STEP 16 COMPLETE — ALL FEASIBLE NO-EXTERNAL-DATA COURSE SUBMISSION DELIVERABLES
Frozen upstream SHA-256 PASS: 35 source files; input hashes unchanged.
Matched horizons: [10, 20, 30, 50, 100] | frozen models trained/reselected: NO
 horizon_underlying_events  matched_n  DeepLOB_logloss  DBN_logloss  train_prior_logloss
                        10     137952         0.454644     0.498405             0.829208
                        20     137802         0.645857     0.714198             0.957764
                        30     137652         0.601543     0.936842             1.030170
                        50     137352         0.560573     1.071103             1.109550
                       100     136602         0.569517     1.147715             1.161377
Missing gap: prior wins 12 /12
Synthetic known-DGP diagnostic (NOT external market testing):
                      regime  replications  true_edge_lower_rmse_tha

In [ ]:

# ==========================================================================================
# CausalLOB STEP 17 — FINAL PACKAGING QA / PDF / PRESENTATION / CODE BUNDLE
# ==========================================================================================
# GYP:
# Act as a strict course-submission auditor. Do not change any experimental results.
# Verify the final evidence, repair the reproducibility/code bundle, generate the submission
# PDF and presentation from existing frozen outputs, and fail closed if required artifacts
# are missing or inconsistent.
# ==========================================================================================

import os, sys, json, shutil, subprocess, textwrap, hashlib, re, time, zipfile
from pathlib import Path

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    else:
        print("Google Drive already mounted.")
except Exception:
    pass

ROOT = Path("/content/drive/MyDrive/CausalLOB")
STEP16 = ROOT / "STEP16_NO_EXTERNAL_SUBMISSION"
STEP16_RESULTS = STEP16 / "results"
OUT = ROOT / "STEP17_FINAL_SUBMISSION_PACKAGE"
RESULTS = OUT / "results"
CODE_DIR = OUT / "Code"
SRC_DIR = CODE_DIR / "src"
REPORT_DIR = OUT / "Report"
PRES_DIR = OUT / "Presentation"
DATA_DIR = OUT / "Data"
ARTIFACTS_DIR = OUT / "Artifacts"

for p in [OUT, RESULTS, CODE_DIR, SRC_DIR, REPORT_DIR, PRES_DIR, DATA_DIR, ARTIFACTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

T0 = time.time()

def sha256_file(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return path

def load_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")
    return path

def safe_copy(src, dst):
    src, dst = Path(src), Path(dst)
    if not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return True

# ------------------------- Required Step16 artifacts -------------------------
required = {
    "paper_md": STEP16_RESULTS / "12_SUBMISSION_MANUSCRIPT.md",
    "math_md": STEP16_RESULTS / "13_METHODS_AND_MATH_APPENDIX.md",
    "latex_tex": STEP16_RESULTS / "14_MANUSCRIPT_DRAFT_UNCOMPILED.tex",
    "speaker_notes": STEP16_RESULTS / "16_PRESENTATION_SPEAKER_NOTES.md",
}
missing = [k for k,v in required.items() if not v.exists()]
if missing:
    raise FileNotFoundError(
        "Missing Step16 artifacts: " + ", ".join(missing) +
        f"\nExpected under: {STEP16_RESULTS}"
    )

print("STEP17 PREFLIGHT PASS: Step16 manuscript, appendix, LaTeX and speaker notes found.")

# ------------------------- Load results tables if present -------------------------
try:
    import pandas as pd
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])
    import pandas as pd

candidate_csvs = list(STEP16_RESULTS.glob("*.csv"))
scorecard = None
scorecard_path = None
for csv in candidate_csvs:
    try:
        df = pd.read_csv(csv)
        cols = set(df.columns.astype(str))
        if {"horizon_underlying_events","DeepLOB_logloss","DBN_logloss"}.issubset(cols):
            scorecard = df
            scorecard_path = csv
            break
    except Exception:
        pass

if scorecard is None:
    step15_candidates = list((ROOT / "STEP15_RESEARCH_CLOSURE_READONLY" / "results").glob("*.csv"))
    for csv in step15_candidates:
        try:
            df = pd.read_csv(csv)
            cols = set(df.columns.astype(str))
            if {"horizon","model","log_loss"}.issubset(cols):
                pivot = df.pivot_table(index="horizon", columns="model", values="log_loss", aggfunc="first").reset_index()
                if {"DEEPLOB","DBN_COMPLETE","PRIOR_D1_D7"}.issubset(set(pivot.columns.astype(str))):
                    scorecard = pd.DataFrame({
                        "horizon_underlying_events": pivot["horizon"],
                        "DeepLOB_logloss": pivot["DEEPLOB"],
                        "DBN_logloss": pivot["DBN_COMPLETE"],
                        "train_prior_logloss": pivot["PRIOR_D1_D7"],
                    })
                    scorecard_path = csv
                    break
        except Exception:
            pass

if scorecard is None:
    raise RuntimeError("Could not locate or reconstruct final horizon scorecard.")

scorecard.to_csv(RESULTS / "01_final_horizon_scorecard.csv", index=False)
print("Scorecard located/reconstructed:", scorecard_path)

# ------------------------- Create/repair reproducibility code bundle -------------------------
verify_script = """
\"\"\"
CausalLOB final evidence verifier.

This script verifies final PDF, PowerPoint, scorecard and manifest. It does not rerun
the heavy experiments.

Usage:
    python verify_final_evidence.py --root /content/drive/MyDrive/CausalLOB
\"\"\"
import argparse, hashlib
from pathlib import Path
import pandas as pd

def sha256_file(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="/content/drive/MyDrive/CausalLOB")
    args, _ = ap.parse_known_args()
    root = Path(args.root)
    pkg = root / "STEP17_FINAL_SUBMISSION_PACKAGE"
    res = pkg / "results"
    required = [
        pkg / "Report" / "CausalLOB_Final_Report.pdf",
        pkg / "Presentation" / "CausalLOB_Final_Presentation.pptx",
        res / "01_final_horizon_scorecard.csv",
        res / "FINAL_SUBMISSION_MANIFEST.json",
    ]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing required final artifacts: " + "\\n".join(missing))
    df = pd.read_csv(res / "01_final_horizon_scorecard.csv")
    if len(set(df["horizon_underlying_events"].astype(int))) != 5:
        raise AssertionError("Expected five horizons.")
    if not (df["DeepLOB_logloss"] < df["DBN_logloss"]).all():
        raise AssertionError("Expected DeepLOB lower log loss than DBN in final scorecard.")
    print("FINAL EVIDENCE VERIFY PASS")
    print("Manifest SHA256:", sha256_file(res / "FINAL_SUBMISSION_MANIFEST.json"))

if __name__ == "__main__":
    main()
"""
write_text(SRC_DIR / "verify_final_evidence.py", verify_script)

source_scan_roots = [
    ROOT / "STEP2_REBUILD",
    ROOT / "STEP3_REBUILD_FROZEN",
    ROOT / "STEP4_VALIDATION_FROZEN",
    ROOT / "STEP5_LABEL_AUDIT_READONLY",
    ROOT / "STEP6_FROZEN_DBN",
    ROOT / "STEP7_MISSING_STATE_READONLY",
    ROOT / "STEP8_GRAPH_SPECIFIC_READONLY",
    ROOT / "STEP9_GAP_STRESS_READONLY",
    ROOT / "STEP12_EVIDENCE_AND_DEEPLOB",
    ROOT / "STEP13_DEEPLOB_FORENSIC_READONLY",
    ROOT / "STEP14_FORECAST_ORIGIN_READONLY",
    ROOT / "STEP15_RESEARCH_CLOSURE_READONLY",
    ROOT / "STEP16_NO_EXTERNAL_SUBMISSION",
]
copied = []
for r in source_scan_roots:
    if not r.exists():
        continue
    for ext in ("*.py", "*.ipynb"):
        for f in r.rglob(ext):
            if any(part.startswith(".") for part in f.parts):
                continue
            relname = re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{r.name}__{f.name}")
            dst = SRC_DIR / "archived_pipeline_sources" / relname
            if safe_copy(f, dst):
                copied.append(str(dst.relative_to(OUT)))

code_readme = f"""# CausalLOB Code Bundle

Main verifier:

```bash
python Code/src/verify_final_evidence.py --root /content/drive/MyDrive/CausalLOB
```

Copied pipeline sources: {len(copied)}

If this number is small, earlier interactive Colab notebooks were not saved into Drive. The final evidence remains auditable from frozen hashes and results; full rerun reproducibility requires preserving the original notebooks/scripts.
"""
write_text(CODE_DIR / "README.md", code_readme)
compile((SRC_DIR / "verify_final_evidence.py").read_text(), str(SRC_DIR / "verify_final_evidence.py"), "exec")
code_status = {
    "approved_scripts": 1 + len(copied),
    "main_verifier": str((SRC_DIR / "verify_final_evidence.py").relative_to(OUT)),
    "archived_sources_copied": len(copied),
    "status": "PASS_VERIFIER_CREATED",
    "caution": "This verifies frozen final evidence; heavy experiment reruns require original notebooks/scripts."
}
save_json(RESULTS / "20_CODE_BUNDLE_STATUS_REPAIRED.json", code_status)

# ------------------------- PDF from Markdown -------------------------
paper_md = required["paper_md"].read_text(encoding="utf-8")
math_md = required["math_md"].read_text(encoding="utf-8")
combined_md = "# CausalLOB Final Report\n\n" + paper_md + "\n\n---\n\n# Methods and Mathematical Appendix\n\n" + math_md
write_text(REPORT_DIR / "CausalLOB_Final_Report.md", combined_md)

def clean_md_line(line):
    line = re.sub(r'!\[[^\]]*\]\([^)]+\)', "[Figure referenced in source]", line)
    line = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', line)
    line = line.replace("`", "").replace("$", "")
    return line

try:
    import reportlab
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "reportlab"])

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.units import inch

pdf_path = REPORT_DIR / "CausalLOB_Final_Report.pdf"
styles = getSampleStyleSheet()
for nm in ["SmallMono","CausalH1","CausalH2","CausalBody"]:
    if nm in styles:
        del styles[nm]
styles.add(ParagraphStyle(name="SmallMono", parent=styles["Normal"], fontName="Courier", fontSize=7.5, leading=9))
styles.add(ParagraphStyle(name="CausalH1", parent=styles["Heading1"], fontSize=16, leading=20, spaceAfter=10))
styles.add(ParagraphStyle(name="CausalH2", parent=styles["Heading2"], fontSize=12.5, leading=15, spaceAfter=8))
styles.add(ParagraphStyle(name="CausalBody", parent=styles["BodyText"], fontSize=9.5, leading=12, alignment=TA_LEFT, spaceAfter=5))

story = [
    Paragraph("CausalLOB Final Report", styles["Title"]),
    Paragraph("Sparse Temporal Graphical Models for Limit-Order-Book Inference", styles["CausalH2"]),
    Spacer(1, 0.15*inch)
]

display_cols = ["horizon_underlying_events","DeepLOB_logloss","DBN_logloss","train_prior_logloss"]
display_cols = [c for c in display_cols if c in scorecard.columns]
table_data = [display_cols]
for _, row in scorecard[display_cols].iterrows():
    table_data.append([str(int(row[c])) if "horizon" in c else f"{float(row[c]):.4f}" for c in display_cols])
tbl = Table(table_data, hAlign="LEFT")
tbl.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
    ("GRID", (0,0), (-1,-1), 0.25, colors.grey),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE", (0,0), (-1,-1), 8),
    ("VALIGN", (0,0), (-1,-1), "TOP"),
]))
story.append(tbl)
story.append(Spacer(1, 0.2*inch))

in_code = False
code_buf = []
for raw in combined_md.splitlines():
    line = raw.rstrip()
    if line.strip().startswith("```"):
        if not in_code:
            in_code = True
            code_buf = []
        else:
            txt = "<br/>".join(clean_md_line(x) for x in code_buf[:45])
            story.append(Paragraph(txt if txt else " ", styles["SmallMono"]))
            story.append(Spacer(1, 0.08*inch))
            in_code = False
        continue
    if in_code:
        code_buf.append(line)
        continue
    if not line.strip():
        story.append(Spacer(1, 0.06*inch))
    elif line.startswith("# "):
        story.append(Paragraph(clean_md_line(line[2:]), styles["CausalH1"]))
    elif line.startswith("## "):
        story.append(Paragraph(clean_md_line(line[3:]), styles["CausalH2"]))
    elif line.startswith("- ") or line.startswith("* "):
        story.append(Paragraph("• " + clean_md_line(line[2:]), styles["CausalBody"]))
    elif re.match(r"^\|", line):
        continue
    else:
        story.append(Paragraph(clean_md_line(line), styles["CausalBody"]))

doc = SimpleDocTemplate(str(pdf_path), pagesize=A4, rightMargin=42, leftMargin=42, topMargin=42, bottomMargin=42)
doc.build(story)
print("PDF generated:", pdf_path)
safe_copy(required["latex_tex"], REPORT_DIR / "CausalLOB_Manuscript_Draft_Uncompiled.tex")

# ------------------------- PowerPoint from speaker notes -------------------------
try:
    import pptx
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-pptx"])

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.shapes import MSO_SHAPE
from pptx.dml.color import RGBColor

notes_md = required["speaker_notes"].read_text(encoding="utf-8")
prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

def add_title(slide, title, subtitle=None):
    tb = slide.shapes.add_textbox(Inches(0.6), Inches(0.35), Inches(12.1), Inches(0.6))
    p = tb.text_frame.paragraphs[0]
    p.text = title
    p.font.size = Pt(30)
    p.font.bold = True
    p.font.color.rgb = RGBColor(25,25,25)
    if subtitle:
        st = slide.shapes.add_textbox(Inches(0.65), Inches(1.0), Inches(11.8), Inches(0.35))
        sp = st.text_frame.paragraphs[0]
        sp.text = subtitle
        sp.font.size = Pt(14)
        sp.font.color.rgb = RGBColor(80,80,80)

def add_bullets(slide, bullets, x=0.75, y=1.35, w=6.2, h=5.5, size=19):
    tb = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = tb.text_frame
    tf.clear()
    for i, b in enumerate(bullets):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.text = b
        p.level = 0
        p.font.size = Pt(size)
        p.space_after = Pt(8)

def add_metric_box(slide, x, y, title, value, note):
    box = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(x), Inches(y), Inches(3.0), Inches(1.15))
    box.fill.solid()
    box.fill.fore_color.rgb = RGBColor(245,245,245)
    box.line.color.rgb = RGBColor(180,180,180)
    tf = box.text_frame
    tf.clear()
    p = tf.paragraphs[0]
    p.text = title
    p.font.size = Pt(11)
    p.font.bold = True
    p.font.color.rgb = RGBColor(70,70,70)
    p2 = tf.add_paragraph()
    p2.text = value
    p2.font.size = Pt(22)
    p2.font.bold = True
    p3 = tf.add_paragraph()
    p3.text = note
    p3.font.size = Pt(9)
    p3.font.color.rgb = RGBColor(90,90,90)

horizons = scorecard["horizon_underlying_events"].astype(int).tolist()
deeplob_ll = scorecard["DeepLOB_logloss"].astype(float).tolist()
dbn_ll = scorecard["DBN_logloss"].astype(float).tolist()
prior_ll = scorecard["train_prior_logloss"].astype(float).tolist()

s = prs.slides.add_slide(prs.slide_layouts[6])
add_title(s, "CausalLOB", "Sparse temporal graphical models for limit-order-book inference")
add_bullets(s, [
    "Goal: model LOB dynamics with an interpretable DBN and exact missing-state inference.",
    "Dataset: FI-2010 NoAuction Decimal Precision, 10 days, 50 day×security sequences.",
    "Scope: retrospective course project; no external validation or PnL claim."
], x=0.85, y=1.55, w=11.7, h=4.5, size=22)

s = prs.slides.add_slide(prs.slide_layouts[6])
add_title(s, "Experimental protocol", "Frozen chronology and no test-time retuning")
add_bullets(s, [
    "D1–5: graph discovery.",
    "D6–7: internal validation / model selection where applicable.",
    "D1–7: DBN and outcome-head fitting.",
    "D8–10: retrospective assessment only; historically exposed.",
    "All 10 days loaded; roles kept separate."
], x=0.75, y=1.3, w=7.0, h=5.5, size=20)
add_metric_box(s, 8.1, 1.35, "Raw rows", "394,337", "canonical FI-2010")
add_metric_box(s, 8.1, 2.65, "Sequences", "50", "day×security blocks")
add_metric_box(s, 8.1, 3.95, "State vars", "6", "interpretable LOB state")

s = prs.slides.add_slide(prs.slide_layouts[6])
add_title(s, "CausalLOB model", "Structure learning + Gaussian DBN + ordinal probability heads")
add_bullets(s, [
    "LPCMCI proposes lagged temporal structure.",
    "Final frozen graph: 30 self-lag slots + 9 cross-variable slots.",
    "Gaussian DBN supports exact conditioning for missing LOB variables.",
    "Ordered-probit heads output P(Down), P(Neutral), P(Up).",
    "Causal language is caveated: observational graph ≠ identified causal effects."
], x=0.75, y=1.25, w=12.0, h=5.6, size=20)

s = prs.slides.add_slide(prs.slide_layouts[6])
add_title(s, "Predictive benchmark: DeepLOB vs DBN", "Matched D8–10 rows; lower log loss is better")
data = [["Horizon", "DeepLOB LL", "DBN LL", "Prior LL"]]
for h, dl, db, pr in zip(horizons, deeplob_ll, dbn_ll, prior_ll):
    data.append([str(h), f"{dl:.3f}", f"{db:.3f}", f"{pr:.3f}"])
table = s.shapes.add_table(len(data), len(data[0]), Inches(0.85), Inches(1.35), Inches(6.1), Inches(3.2)).table
for i, row in enumerate(data):
    for j, val in enumerate(row):
        cell = table.cell(i,j)
        cell.text = val
        cell.text_frame.paragraphs[0].font.size = Pt(14)
        if i == 0:
            cell.fill.solid()
            cell.fill.fore_color.rgb = RGBColor(230,230,230)
            cell.text_frame.paragraphs[0].font.bold = True
add_bullets(s, [
    "DeepLOB has lower log loss at all five horizons.",
    "At h10, DBN retains stronger balanced accuracy and Brier in forensic results.",
    "DeepLOB uses richer 100×40 raw LOB windows; DBN uses six engineered variables."
], x=7.45, y=1.35, w=5.2, h=4.5, size=18)

s = prs.slides.add_slide(prs.slide_layouts[6])
add_title(s, "Missing-observation inference", "Where the DBN contribution is strongest")
add_bullets(s, [
    "Exact Gaussian conditioning reconstructs missing state variables.",
    "Sparse graph gives modest improvement over AR(5).",
    "Regularized covariance often matters more than graph edges.",
    "In consecutive-gap outcome tests, training class prior beats CausalLOB in 12/12 comparisons.",
    "Conclusion: state inference is more defensible than missing-evidence forecasting."
], x=0.75, y=1.3, w=12.1, h=5.5, size=20)

s = prs.slides.add_slide(prs.slide_layouts[6])
add_title(s, "What not to claim", "Explicit boundaries for a rigorous submission")
add_bullets(s, [
    "No trading-profit claim: no execution, fees, slippage, queue position or impact modeling.",
    "No external validation: D8–10 are retrospective and historically exposed.",
    "No verified publisher label generator: source labels are retained, formula unresolved.",
    "No claim that LPCMCI recovered the true market causal graph.",
    "No claim that DBN beats DeepLOB predictively."
], x=0.75, y=1.3, w=12.0, h=5.7, size=20)

s = prs.slides.add_slide(prs.slide_layouts[6])
add_title(s, "Final contribution", "A defensible PGM course project")
add_bullets(s, [
    "End-to-end reproducible FI-2010 preprocessing with leakage audits.",
    "Frozen sparse temporal graph and Gaussian DBN.",
    "Exact probabilistic inference under missing LOB evidence.",
    "DeepLOB benchmark included and stronger predictive results preserved.",
    "Clear limitations and external-replication protocol."
], x=0.75, y=1.3, w=12.0, h=5.5, size=21)

s = prs.slides.add_slide(prs.slide_layouts[6])
add_title(s, "Next research step", "Only external replication can close the key gap")
add_bullets(s, [
    "Freeze schema, variables, horizons, baselines and metrics before inspecting new data.",
    "Run CausalLOB and DeepLOB on genuinely unseen LOB data.",
    "Resolve label-generation formula using raw message-level construction if available.",
    "Add PnL only after executable prices, costs and fill assumptions are defined.",
    "Until then: FI-2010 retrospective evidence package is complete."
], x=0.75, y=1.3, w=12.1, h=5.6, size=20)

ppt_path = PRES_DIR / "CausalLOB_Final_Presentation.pptx"
prs.save(str(ppt_path))
print("PowerPoint generated:", ppt_path)
safe_copy(required["speaker_notes"], PRES_DIR / "CausalLOB_Speaker_Notes.md")

# ------------------------- Data README / artifacts -------------------------
data_readme = """# Data

The project uses the public FI-2010 limit-order-book benchmark.

The raw dataset is not copied into this final package because of size. The preprocessing notebooks/scripts download or locate `ulfricirons/fi-2010`.

Final experiment chronology:
- Days 1–5: graph discovery
- Days 6–7: internal validation/model selection where applicable
- Days 1–7: DBN/outcome-head fitting
- Days 8–10: retrospective assessment only

Limitations:
- D8–10 were historically inspected during project iteration.
- Publisher label-generation code is unresolved; original publisher labels are retained.
"""
write_text(DATA_DIR / "README.md", data_readme)

for p in [
    STEP16 / "CAUSALLOB_STEP16_NO_EXTERNAL_SUBMISSION.zip",
    ROOT / "STEP15_RESEARCH_CLOSURE_READONLY" / "CAUSALLOB_STEP15_FINAL_EVIDENCE.zip",
    ROOT / "STEP14_FORECAST_ORIGIN_READONLY" / "CAUSALLOB_STEP14_FORENSIC_EVIDENCE.zip",
    ROOT / "STEP13_DEEPLOB_FORENSIC_READONLY" / "CAUSALLOB_STEP13_FORENSIC_RESULTS.zip",
]:
    if p.exists():
        safe_copy(p, ARTIFACTS_DIR / p.name)

for src in required.values():
    safe_copy(src, ARTIFACTS_DIR / src.name)

readme = f"""# CausalLOB Final Submission Package

## Contents

- `Report/CausalLOB_Final_Report.pdf` — final PDF report.
- `Report/CausalLOB_Final_Report.md` — source markdown.
- `Presentation/CausalLOB_Final_Presentation.pptx` — presentation deck.
- `Presentation/CausalLOB_Speaker_Notes.md` — speaker notes.
- `Code/` — verifier and archived available sources.
- `Data/README.md` — dataset source and split policy.
- `Artifacts/` — copied evidence ZIPs and manuscript sources.
- `results/` — scorecards, manifests and packaging audit files.

## Main numerical conclusion

DeepLOB has lower matched-row log loss than the DBN at all five horizons. CausalLOB remains valuable as an interpretable PGM/DBN with exact missing-state inference, not as the best predictive model.

## Unresolved by design

- No genuine external dataset.
- Exact publisher label-generation code unresolved.
- No trading PnL after costs.
- No claim of identified causal effects.

## Quick verification

```bash
python Code/src/verify_final_evidence.py --root /content/drive/MyDrive/CausalLOB
```
"""
write_text(OUT / "README.md", readme)

# ------------------------- Manifest / claim checklist / zip -------------------------
manifest = {
    "created_at_unix": time.time(),
    "source_step16_results": str(STEP16_RESULTS),
    "scorecard_source": str(scorecard_path),
    "pdf": str(pdf_path),
    "pdf_generation_method": "reportlab_from_markdown",
    "pptx": str(ppt_path),
    "code_bundle_status": code_status,
    "key_claims": {
        "all_10_days_loaded": True,
        "external_validation_completed": False,
        "publisher_label_generator_verified": False,
        "trading_pnl_verified": False,
        "models_retrained_in_step17": False,
    },
    "scorecard_rows": int(len(scorecard)),
}
save_json(RESULTS / "FINAL_SUBMISSION_MANIFEST.json", manifest)

claim_checklist = pd.DataFrame([
    {"claim": "All five horizons evaluated", "status": "SUPPORTED", "evidence": "01_final_horizon_scorecard.csv"},
    {"claim": "DeepLOB lower log loss than DBN on matched D8-D10 rows", "status": "SUPPORTED", "evidence": "01_final_horizon_scorecard.csv"},
    {"claim": "CausalLOB provides exact Gaussian missing-state inference", "status": "SUPPORTED", "evidence": "Step7/Step9 artifacts"},
    {"claim": "DBN is the strongest predictive model", "status": "NOT_SUPPORTED", "evidence": "DeepLOB benchmark"},
    {"claim": "Trading profitability after costs", "status": "UNRESOLVED", "evidence": "No execution model"},
    {"claim": "External validation", "status": "UNRESOLVED", "evidence": "No external dataset"},
    {"claim": "True causal graph identified", "status": "NOT_CLAIMED", "evidence": "Observational LPCMCI only"},
])
claim_checklist.to_csv(RESULTS / "02_claim_checklist.csv", index=False)

zip_path = ROOT / "CAUSALLOB_FINAL_COURSE_SUBMISSION_PACKAGE.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUT.rglob("*"):
        if f.is_file():
            zf.write(f, f.relative_to(OUT.parent))
with zipfile.ZipFile(zip_path, "r") as zf:
    bad = zf.testzip()
    if bad:
        raise RuntimeError(f"Bad file inside final zip: {bad}")

print("\n" + "="*105)
print("STEP 17 COMPLETE — FINAL COURSE SUBMISSION PACKAGE READY")
print("="*105)
print(f"PDF report: {pdf_path}")
print(f"PowerPoint: {ppt_path}")
print(f"Code bundle approved scripts/sources: {code_status['approved_scripts']}")
print(f"Final ZIP: {zip_path}")
print("Unresolved by design: external validation, publisher label-generation code, net PnL.")
print(f"Elapsed seconds: {time.time()-T0:.1f}")


Mounted at /content/drive
STEP17 PREFLIGHT PASS: Step16 manuscript, appendix, LaTeX and speaker notes found.
Scorecard located/reconstructed: /content/drive/MyDrive/CausalLOB/STEP16_NO_EXTERNAL_SUBMISSION/results/01_matched_predictive_benchmark.csv
PDF generated: /content/drive/MyDrive/CausalLOB/STEP17_FINAL_SUBMISSION_PACKAGE/Report/CausalLOB_Final_Report.pdf
PowerPoint generated: /content/drive/MyDrive/CausalLOB/STEP17_FINAL_SUBMISSION_PACKAGE/Presentation/CausalLOB_Final_Presentation.pptx

STEP 17 COMPLETE — FINAL COURSE SUBMISSION PACKAGE READY
PDF report: /content/drive/MyDrive/CausalLOB/STEP17_FINAL_SUBMISSION_PACKAGE/Report/CausalLOB_Final_Report.pdf
PowerPoint: /content/drive/MyDrive/CausalLOB/STEP17_FINAL_SUBMISSION_PACKAGE/Presentation/CausalLOB_Final_Presentation.pptx
Code bundle approved scripts/sources: 1
Final ZIP: /content/drive/MyDrive/CausalLOB/CAUSALLOB_FINAL_COURSE_SUBMISSION_PACKAGE.zip
Unresolved by design: external validation, publisher label-generation code, net 